In [ ]:
import marimo as mo

In [ ]:
# HF_TOKEN_HARDCODED = ""  # paste NEW token here only in a private notebook

In [ ]:
# Cell 00: Hugging Face checkpoint helper
#
# What this cell does:
#   - Loads HF token from environment, Kaggle secret, local file, or optional hardcoded line.
#   - Logs in to Hugging Face if a token is available.
#   - Defines public checkpoint helpers:
#       hf_save_checkpoint
#       hf_read_text_if_exists
#       hf_list_checkpoints
#   - Saves checkpoints locally.
#   - Attempts to upload checkpoints to:
#       https://huggingface.co/buckets/Satabarto/FireFlyStore
#
# Important:
#   No top-level underscore names are used, because Marimo treats those as private.
#   No @app.cell wrapper is used.
#   No globals() hack is used.

HF_TOKEN_HARDCODED = ""  # paste a NEW rotated HF token here only if you insist on hardcoding
HF_BUCKET_URL = "https://huggingface.co/buckets/Satabarto/FireFlyStore"
HF_BUCKET_REPO_ID = "Satabarto/FireFlyStore"
HF_CHECKPOINT_ROOT_OVERRIDE = None
HF_AUTO_UPLOAD = True


def hf_get_checkpoint_root():
    from pathlib import Path
    import os

    if HF_CHECKPOINT_ROOT_OVERRIDE:
        root = Path(HF_CHECKPOINT_ROOT_OVERRIDE)
    elif os.path.isdir("/kaggle/working"):
        root = Path("/kaggle/working/hf_checkpoints")
    else:
        root = Path("./hf_checkpoints")

    root.mkdir(parents=True, exist_ok=True)
    return root


def hf_get_token():
    import os
    from pathlib import Path

    token = os.environ.get("HF_TOKEN", "") or HF_TOKEN_HARDCODED

    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = ""

    if not token:
        token_path = Path(".hf_token")
        if token_path.exists():
            try:
                token = token_path.read_text(encoding="utf-8").strip()
            except Exception:
                token = ""

    return (token or "").strip()


HF_TOKEN_VALUE = ''


def hf_login():
    if not HF_TOKEN_VALUE:
        print("[HF00] No HF token found. Local checkpoints only.")
        return False

    try:
        from huggingface_hub import login
        login(token=HF_TOKEN_VALUE, add_to_git_credential=False)
        print("[HF00] Hugging Face login OK.")
        return True
    except Exception as error:
        print(f"[HF00] Hugging Face login failed: {error!r}")
        return False


HF_LOGIN_OK = hf_login()


def hf_clean_payload(value):
    import torch
    import numpy as np
    from pathlib import Path

    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            cleaned[str(key)] = hf_clean_payload(item)
        return cleaned

    if isinstance(value, tuple):
        return tuple(hf_clean_payload(item) for item in value)

    if isinstance(value, list):
        return [hf_clean_payload(item) for item in value]

    if isinstance(value, set):
        return [hf_clean_payload(item) for item in value]

    if torch.is_tensor(value):
        return value.detach().cpu()

    if isinstance(value, np.ndarray):
        return value

    if isinstance(value, (np.floating, np.integer, np.bool_)):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if callable(value):
        return f"<callable {getattr(value, '__name__', type(value).__name__)}>"

    return value


def hf_upload_file_to_bucket(local_path, remote_path):
    from pathlib import Path

    if not HF_LOGIN_OK or not HF_TOKEN_VALUE:
        return False, "no_hf_token_or_login_failed"

    local_path = Path(local_path)

    if not local_path.exists():
        return False, "local_file_missing"

    last_error = None

    try:
        from huggingface_hub import HfFileSystem

        file_system = HfFileSystem(token=HF_TOKEN_VALUE)
        remote_full = f"hf://buckets/{HF_BUCKET_REPO_ID}/{remote_path}"

        try:
            file_system.put_file(str(local_path), remote_full, overwrite=True)
        except TypeError:
            file_system.put_file(str(local_path), remote_full)
        except Exception:
            file_system.put_file(str(local_path), remote_full)

        return True, remote_full

    except Exception as error:
        last_error = error

    try:
        from huggingface_hub import HfApi

        api = HfApi(token=HF_TOKEN_VALUE)
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=remote_path,
            repo_id=HF_BUCKET_REPO_ID,
            repo_type="bucket",
            token=HF_TOKEN_VALUE,
        )

        return True, f"api_bucket:{HF_BUCKET_REPO_ID}/{remote_path}"

    except Exception as error:
        last_error = error

    return False, repr(last_error)


def hf_save_checkpoint(cell_name, payload, description="", upload=None):
    import json
    import torch
    from datetime import datetime, timezone
    from pathlib import Path

    if upload is None:
        upload = HF_AUTO_UPLOAD

    safe_chars = []
    for char in str(cell_name):
        if char.isalnum() or char in "._-":
            safe_chars.append(char)
        else:
            safe_chars.append("_")

    safe_cell = "".join(safe_chars)

    root = hf_get_checkpoint_root()
    cell_dir = root / safe_cell
    cell_dir.mkdir(parents=True, exist_ok=True)

    timestamp_utc = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    file_name = f"checkpoint_{timestamp_utc}.pt"
    local_path = cell_dir / file_name

    if not isinstance(payload, dict):
        payload = {"value": payload}

    cleaned_payload = hf_clean_payload(payload)

    meta = {
        "cell_name": safe_cell,
        "description": str(description),
        "created_at_utc": timestamp_utc,
        "hf_bucket_url": HF_BUCKET_URL,
        "hf_bucket_repo_id": HF_BUCKET_REPO_ID,
        "local_path": str(local_path),
        "payload_keys": sorted(cleaned_payload.keys()),
        "torch_version": torch.__version__,
    }

    torch.save(
        {
            "meta": meta,
            "payload": cleaned_payload,
        },
        local_path,
    )

    size_mb = local_path.stat().st_size / 1e6

    print(
        f"[ckpt:{safe_cell}] local saved "
        f"{local_path} ({size_mb:.2f} MB) | {description}"
    )

    latest_path = cell_dir / "latest.json"
    latest_path.write_text(
        json.dumps(
            {
                "latest_file": file_name,
                "meta": meta,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    if upload:
        remote_checkpoint = f"checkpoints/{safe_cell}/{file_name}"
        ok, info = hf_upload_file_to_bucket(local_path, remote_checkpoint)
        print(
            f"[ckpt:{safe_cell}] upload checkpoint: "
            f"{'OK' if ok else 'FAILED'} | {info}"
        )

        remote_latest = f"checkpoints/{safe_cell}/latest.json"
        ok_latest, info_latest = hf_upload_file_to_bucket(latest_path, remote_latest)
        print(
            f"[ckpt:{safe_cell}] upload latest.json: "
            f"{'OK' if ok_latest else 'FAILED'} | {info_latest}"
        )

    return str(local_path)


def hf_read_text_if_exists(path):
    from pathlib import Path

    if not path:
        return None

    file_path = Path(path)

    if not file_path.exists():
        return None

    try:
        return file_path.read_text(encoding="utf-8")
    except Exception as error:
        return f"<read_failed: {error!r}>"


def hf_list_checkpoints():
    from pathlib import Path
    import json

    root = hf_get_checkpoint_root()

    print("=" * 100)
    print("LOCAL CHECKPOINT LIST")
    print("=" * 100)

    if not root.exists():
        print(f"[ckpt][warn] checkpoint root does not exist: {root}")
        return

    checkpoint_files = sorted(root.glob("*/*.pt"))

    if not checkpoint_files:
        print("[ckpt][warn] no .pt checkpoints found.")
        return

    total_bytes = 0

    for path in checkpoint_files:
        size_mb = path.stat().st_size / 1e6
        total_bytes += path.stat().st_size

        print(f"{str(path):80s} {size_mb:10.2f} MB")

        latest_json = path.parent / "latest.json"

        if latest_json.exists():
            try:
                meta = json.loads(latest_json.read_text(encoding="utf-8"))
                description = meta.get("meta", {}).get("description", "")
                print(f"  latest -> {meta.get('latest_file', '')} | {description}")
            except Exception as error:
                print(f"  latest.json parse failed: {error!r}")

    print("-" * 100)
    print(f"total checkpoints: {len(checkpoint_files)}")
    print(f"total local size: {total_bytes / 1e6:.2f} MB")
    print("=" * 100)


print("=" * 90)
print("CELL 00 - HF CHECKPOINT HELPER READY")
print("=" * 90)
print(f"[HF00] bucket URL       = {HF_BUCKET_URL}")
print(f"[HF00] bucket repo id   = {HF_BUCKET_REPO_ID}")
print(f"[HF00] local root       = {str(hf_get_checkpoint_root())}")
print(f"[HF00] token present    = {bool(HF_TOKEN_VALUE)}")
print(f"[HF00] login successful = {HF_LOGIN_OK}")
print(f"[HF00] auto upload      = {HF_AUTO_UPLOAD}")
print("[HF00] public helpers: hf_save_checkpoint, hf_read_text_if_exists, hf_list_checkpoints")
print("=" * 90)

In [ ]:
# Cell 1: Environment Setup & Dependencies
import os, sys, gzip, subprocess, importlib, warnings
from pathlib import Path
from urllib.request import urlopen, Request
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import balanced_accuracy_score, f1_score

def ensure_dependency_cell1(mod_cell1, pip_name_cell1=None):
    try:
        importlib.import_module(mod_cell1)
        return
    except ImportError:
        pass
    target_cell1 = pip_name_cell1 or mod_cell1
    print(f"[Cell 1] Installing {target_cell1}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", target_cell1], check=True)

ensure_dependency_cell1("torch")
ensure_dependency_cell1("numpy")
ensure_dependency_cell1("pandas")
ensure_dependency_cell1("scipy")
ensure_dependency_cell1("sklearn", "scikit-learn")

DEVICE_cell1 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Cell 1] Using device: {DEVICE_cell1}")

def seed_all_cell1(seed_cell1=42):
    torch.manual_seed(seed_cell1)
    np.random.seed(seed_cell1)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_cell1)

In [ ]:
# Cell 2: Real Data Download & Loading
BUCKET_cell2 = "https://storage.googleapis.com/flywire-data/codex/data/fafb/783"
MANIFEST_cell2 = {
    "connections.csv.gz": 50_290_000,
    "classification.csv.gz": 930_000,
    "neurons.csv.gz": 1_680_000,
    "cell_stats.csv.gz": 2_530_000,
}
DATA_DIR_cell2 = Path("./flywire_data")
DATA_DIR_cell2.mkdir(parents=True, exist_ok=True)

def download_file_cell2(fn_cell2):
    path_cell2 = DATA_DIR_cell2 / fn_cell2
    expected_cell2 = MANIFEST_cell2[fn_cell2]
    if path_cell2.exists() and abs(path_cell2.stat().st_size - expected_cell2) / expected_cell2 < 0.05:
        try:
            with gzip.open(path_cell2, "rb") as fh_cell2:
                while fh_cell2.read(1 << 20): pass
            print(f"[Cell 2] [skip] {fn_cell2}")
            return path_cell2
        except Exception:
            path_cell2.unlink()

    print(f"[Cell 2] [get ] {fn_cell2}")
    with urlopen(Request(f"{BUCKET_cell2}/{fn_cell2}", headers={"User-Agent": "research"}), timeout=300) as r_cell2:
        data_cell2 = r_cell2.read()

    tmp_cell2 = path_cell2.with_suffix(path_cell2.suffix + ".tmp")
    tmp_cell2.write_bytes(data_cell2)
    with gzip.open(tmp_cell2, "rb") as fh_cell2:
        while fh_cell2.read(1 << 20): pass
    os.replace(tmp_cell2, path_cell2)
    print(f"[Cell 2] [ok  ] {fn_cell2}  {len(data_cell2)/1e6:.2f} MB")
    return path_cell2

for fn_loop_cell2 in MANIFEST_cell2:
    download_file_cell2(fn_loop_cell2)

print("[Cell 2] Loading CSVs...")
conn_df_cell2 = pd.read_csv(DATA_DIR_cell2 / "connections.csv.gz")
class_df_cell2 = pd.read_csv(DATA_DIR_cell2 / "classification.csv.gz")
neurons_df_cell2 = pd.read_csv(DATA_DIR_cell2 / "neurons.csv.gz")
stats_df_cell2 = pd.read_csv(DATA_DIR_cell2 / "cell_stats.csv.gz")
print(f"[Cell 2] Loaded: conn={len(conn_df_cell2):,}, class={len(class_df_cell2):,}")

In [ ]:
# Cell 3: Graph Construction, Feature Engineering, and Spectral Embedding
all_ids_cell3 = class_df_cell2["root_id"].unique()
n2i_cell3 = {int(r_cell3): i for i, r_cell3 in enumerate(all_ids_cell3)}
N_cell3 = len(all_ids_cell3)

conn_mapped_cell3 = conn_df_cell2.copy()
conn_mapped_cell3["src"] = conn_mapped_cell3["pre_root_id"].map(n2i_cell3)
conn_mapped_cell3["dst"] = conn_mapped_cell3["post_root_id"].map(n2i_cell3)
conn_mapped_cell3 = conn_mapped_cell3.dropna(subset=["src", "dst"]).copy()
conn_mapped_cell3["src"] = conn_mapped_cell3["src"].astype(int)
conn_mapped_cell3["dst"] = conn_mapped_cell3["dst"].astype(int)

NT_MAP_cell3 = {"ACH": "exc", "GLUT": "exc", "GABA": "inh", "SER": "mod", "DA": "mod", "OCT": "mod"}
conn_mapped_cell3["nt_class"] = conn_mapped_cell3["nt_type"].map(NT_MAP_cell3).fillna("unk")
conn_signed_cell3 = conn_mapped_cell3[conn_mapped_cell3["nt_class"].isin(["exc", "inh"])].copy()

agg_cell3 = conn_signed_cell3.groupby(["src", "dst", "nt_class"], as_index=False).agg(syn_count=("syn_count", "sum"))
major_cell3 = agg_cell3.sort_values("syn_count").groupby(["src", "dst"], as_index=False).last()

src_arr_cell3 = major_cell3["src"].values.astype(np.int64)
dst_arr_cell3 = major_cell3["dst"].values.astype(np.int64)
sign_bin_arr_cell3 = (major_cell3["nt_class"] == "exc").astype(np.float32).values
syn_arr_cell3 = major_cell3["syn_count"].values.astype(np.float32)

stats_idx_cell3 = stats_df_cell2.set_index("root_id").reindex(all_ids_cell3)
size_feats_cell3 = np.log1p(np.clip(stats_idx_cell3[["length_nm", "area_nm", "size_nm"]].values.astype(np.float64), 0, None))
size_feats_cell3 = np.nan_to_num(size_feats_cell3, nan=0.0)

neur_idx_cell3 = neurons_df_cell2.set_index("root_id").reindex(all_ids_cell3)
nt_cols_cell3 = ["da_avg", "ser_avg", "gaba_avg", "glut_avg", "ach_avg", "oct_avg"]
nt_feats_cell3 = np.nan_to_num(neur_idx_cell3[nt_cols_cell3].values.astype(np.float64), nan=0.0)

deg_in_cell3, deg_out_cell3 = np.zeros(N_cell3), np.zeros(N_cell3)
np.add.at(deg_in_cell3, dst_arr_cell3, syn_arr_cell3)
np.add.at(deg_out_cell3, src_arr_cell3, syn_arr_cell3)
deg_feats_cell3 = np.stack([np.log1p(deg_in_cell3), np.log1p(deg_out_cell3)], axis=1)

print("[Cell 3] Computing spectral features (Truncated SVD)...")
row_cell3 = np.concatenate([src_arr_cell3, dst_arr_cell3])
col_cell3 = np.concatenate([dst_arr_cell3, src_arr_cell3])
data_cell3 = np.ones_like(row_cell3, dtype=np.float32)
A_sym_cell3 = csr_matrix((data_cell3, (row_cell3, col_cell3)), shape=(N_cell3, N_cell3))

svd_cell3 = TruncatedSVD(n_components=16, random_state=42)
spec_emb_cell3 = svd_cell3.fit_transform(A_sym_cell3)
spec_emb_cell3 = (spec_emb_cell3 - spec_emb_cell3.mean(0)) / (spec_emb_cell3.std(0) + 1e-8)
spec_emb_cell3 = np.nan_to_num(spec_emb_cell3, nan=0.0)

X_raw_cell3 = np.concatenate([size_feats_cell3, nt_feats_cell3, deg_feats_cell3, spec_emb_cell3], axis=1)
X_tensor_cell3 = torch.tensor((X_raw_cell3 - X_raw_cell3.mean(0)) / (X_raw_cell3.std(0) + 1e-8), dtype=torch.float32)
print(f"[Cell 3] X shape (with spectral): {tuple(X_tensor_cell3.shape)}")

class_idx_cell3 = class_df_cell2.set_index("root_id").reindex(all_ids_cell3)
labels_raw_cell3 = class_idx_cell3["class"].fillna("").astype(str).values
classes_list_cell3 = sorted(set(c_cell3 for c_cell3 in labels_raw_cell3 if c_cell3))
c2i_cell3 = {c_cell3: i for i, c_cell3 in enumerate(classes_list_cell3)}
labels_tensor_cell3 = torch.tensor([c2i_cell3.get(c_cell3, -1) for c_cell3 in labels_raw_cell3], dtype=torch.long)
labeled_mask_cell3 = labels_tensor_cell3 >= 0
C_cell3 = len(classes_list_cell3)

counts_cell3 = Counter(labels_tensor_cell3[labeled_mask_cell3].tolist())
rare_classes_list_cell3 = [c_cell3 for c_cell3, k_cell3 in counts_cell3.items() if k_cell3 < 100]

super_raw_cell3 = class_idx_cell3["super_class"].fillna("").astype(str).values
super_classes_cell3 = sorted(set(s_cell3 for s_cell3 in super_raw_cell3 if s_cell3))
s2i_cell3 = {s_cell3: i for i, s_cell3 in enumerate(super_classes_cell3)}
super_labels_tensor_cell3 = torch.tensor([s2i_cell3.get(s_cell3, -1) for s_cell3 in super_raw_cell3], dtype=torch.long)
C_super_cell3 = len(super_classes_cell3)

print(f"[Cell 3] C={C_cell3} classes | rare(n<100)={len(rare_classes_list_cell3)} | Super C={C_super_cell3}")

In [ ]:
# Checkpoint: Cell 3 data/labels/features
#
# What Cell 3 does:
#   Loads FlyWire graph arrays, 29-class labels, labeled mask, rare-class list,
#   raw soma/biological features, and unsigned spectral embedding.

hf_save_checkpoint(
    "cell03_data_labels",
    {
        "N_cell3": N_cell3,
        "src_arr_cell3": src_arr_cell3,
        "dst_arr_cell3": dst_arr_cell3,
        "syn_arr_cell3": syn_arr_cell3,
        "sign_bin_arr_cell3": sign_bin_arr_cell3,
        "labels_tensor_cell3": labels_tensor_cell3,
        "labeled_mask_cell3": labeled_mask_cell3,
        "super_labels_tensor_cell3": super_labels_tensor_cell3,
        "C_cell3": C_cell3,
        "C_super_cell3": C_super_cell3,
        "rare_classes_list_cell3": rare_classes_list_cell3,
        "X_raw_cell3": X_raw_cell3,
        "X_tensor_cell3": X_tensor_cell3,
        "spec_emb_cell3": spec_emb_cell3,
    },
    description=(
        "Cell 3: FlyWire graph arrays, 29-class labels, labeled mask, rare classes, "
        "raw features, unsigned spectral embedding."
    ),
)

In [ ]:
# Cell 4: Stratified Splitting Logic
def make_split_cell4(seed_cell4, labeled_mask_cell4, labels_tensor_cell4, C_cell4):
    rng_cell4 = np.random.default_rng(seed_cell4)
    idx_cell4 = np.where(labeled_mask_cell4.numpy())[0]
    y_cell4 = labels_tensor_cell4.numpy()
    tr_cell4, va_cell4, te_cell4 = [], [], []
    for c_loop_cell4 in range(C_cell4):
        c_idx_cell4 = idx_cell4[y_cell4[idx_cell4] == c_loop_cell4]
        rng_cell4.shuffle(c_idx_cell4)
        n_cell4 = len(c_idx_cell4)
        a_cell4, b_cell4 = int(0.6 * n_cell4), int(0.8 * n_cell4)
        tr_cell4.extend(c_idx_cell4[:a_cell4])
        va_cell4.extend(c_idx_cell4[a_cell4:b_cell4])
        te_cell4.extend(c_idx_cell4[b_cell4:])
    return (
        torch.tensor(tr_cell4, dtype=torch.long),
        torch.tensor(va_cell4, dtype=torch.long),
        torch.tensor(te_cell4, dtype=torch.long)
    )
print("[Cell 4] make_split_cell4 defined.")

In [ ]:
# Checkpoint: Cell 4 split audit
#
# What Cell 4 does:
#   Defines deterministic train/validation/test splits used by all later cells.
#   This checkpoint stores split-size audit information for seeds 0-9.

split_audit_cell04 = {}

for seed_index_cell04 in range(10):
    train_idx_cell04, val_idx_cell04, test_idx_cell04 = make_split_cell4(
        seed_index_cell04,
        labeled_mask_cell3,
        labels_tensor_cell3,
        C_cell3,
    )

    split_audit_cell04[f"seed_{seed_index_cell04}"] = {
        "n_train": int(train_idx_cell04.numel()),
        "n_val": int(val_idx_cell04.numel()),
        "n_test": int(test_idx_cell04.numel()),
        "train_sum": int(train_idx_cell04.sum().item()),
        "val_sum": int(val_idx_cell04.sum().item()),
        "test_sum": int(test_idx_cell04.sum().item()),
    }

hf_save_checkpoint(
    "cell04_split_audit",
    {
        "split_audit_cell04": split_audit_cell04,
    },
    description="Cell 4: deterministic 10-seed train/val/test split audit.",
)

In [ ]:
# Cell 5: Model Architectures (Feature-Aware Baseline + Signed Prototype GCN)
def normalize_edges_cell5(edge_index_cell5, edge_weight_cell5, N_cell5, device_cell5):
    deg_cell5 = torch.zeros(N_cell5, device=device_cell5)
    deg_cell5.index_add_(0, edge_index_cell5[0], edge_weight_cell5)
    deg_cell5.index_add_(0, edge_index_cell5[1], edge_weight_cell5)
    return deg_cell5.clamp(min=1.0).rsqrt()

def aggregate_cell5(h_cell5, edge_index_cell5, edge_weight_cell5, deg_inv_sqrt_cell5):
    src_cell5, dst_cell5 = edge_index_cell5
    h_src_cell5 = h_cell5[src_cell5] * (deg_inv_sqrt_cell5[src_cell5].unsqueeze(-1) * edge_weight_cell5.unsqueeze(-1))
    h_dst_cell5 = h_cell5[dst_cell5] * (deg_inv_sqrt_cell5[dst_cell5].unsqueeze(-1) * edge_weight_cell5.unsqueeze(-1))
    out_cell5 = torch.zeros_like(h_cell5)
    out_cell5.index_add_(0, src_cell5, h_dst_cell5)
    out_cell5.index_add_(0, dst_cell5, h_src_cell5)
    return out_cell5 * deg_inv_sqrt_cell5.unsqueeze(-1)

class GCNFeatureBaseline_cell5(nn.Module):
    def __init__(self, d_features_cell5, d_cell5=64, n_layers_cell5=2, n_classes_cell5=29, dropout_cell5=0.3):
        super().__init__()
        self.feat_proj = nn.Linear(d_features_cell5, d_cell5)
        self.layers = nn.ModuleList([nn.Linear(d_cell5, d_cell5) for _ in range(n_layers_cell5)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_cell5) for _ in range(n_layers_cell5)])
        self.drop = nn.Dropout(dropout_cell5)
        self.head = nn.Linear(d_cell5, n_classes_cell5)

    def forward(self, X_cell5, edge_index_cell5, edge_weight_cell5, deg_inv_sqrt_cell5):
        h_cell5 = self.drop(F.relu(self.feat_proj(X_cell5)))
        for lin_cell5, norm_cell5 in zip(self.layers, self.norms):
            agg_cell5 = aggregate_cell5(h_cell5, edge_index_cell5, edge_weight_cell5, deg_inv_sqrt_cell5)
            h_cell5 = self.drop(norm_cell5(h_cell5 + F.relu(lin_cell5(agg_cell5))))
        return self.head(h_cell5)

class SignedPrototypeLayer_cell5(nn.Module):
    def __init__(self, d_in_cell5, d_out_cell5):
        super().__init__()
        self.W_pos = nn.Linear(d_in_cell5, d_out_cell5)
        self.W_neg = nn.Linear(d_in_cell5, d_out_cell5)
        self.norm = nn.LayerNorm(d_out_cell5)
        self.raw_gamma = nn.Parameter(torch.tensor(0.0))

    def forward(self, h_cell5, ei_pos_cell5, ew_pos_cell5, dis_pos_cell5, ei_neg_cell5, ew_neg_cell5, dis_neg_cell5):
        h_pos_cell5 = aggregate_cell5(h_cell5, ei_pos_cell5, ew_pos_cell5, dis_pos_cell5)
        h_neg_cell5 = aggregate_cell5(h_cell5, ei_neg_cell5, ew_neg_cell5, dis_neg_cell5)
        gamma_cell5 = torch.sigmoid(self.raw_gamma)
        h_new_cell5 = h_cell5 + (-gamma_cell5 * (h_pos_cell5 + h_neg_cell5) + self.W_pos(h_pos_cell5) - self.W_neg(h_neg_cell5))
        return self.norm(h_new_cell5)

class SPGCN_cell5(nn.Module):
    def __init__(self, d_features_cell5, d_cell5=64, d_proto_cell5=32, n_layers_cell5=2,
                 n_classes_cell5=29, n_super_cell5=10, temperature_cell5=0.1, dropout_cell5=0.3):
        super().__init__()
        self.feat_proj = nn.Linear(d_features_cell5, d_cell5)
        self.drop = nn.Dropout(dropout_cell5)

        if n_layers_cell5 == 0:
            self.signed_layers = nn.ModuleList([])
        else:
            self.signed_layers = nn.ModuleList([
                SignedPrototypeLayer_cell5(d_cell5, d_cell5) for _ in range(n_layers_cell5)
            ])
        self.proj = nn.Linear(d_cell5, d_proto_cell5)

        self.proto_class = nn.Parameter(torch.randn(n_classes_cell5, d_proto_cell5) * 0.1)
        self.proto_super = nn.Parameter(torch.randn(n_super_cell5, d_proto_cell5) * 0.1)

        self.temperature = temperature_cell5
        self.n_classes = n_classes_cell5
        self.n_super = n_super_cell5

    def encode(self, X_cell5, ei_pos_cell5, ew_pos_cell5, dis_pos_cell5, ei_neg_cell5, ew_neg_cell5, dis_neg_cell5):
        h_cell5 = self.drop(F.relu(self.feat_proj(X_cell5)))
        for layer_cell5 in self.signed_layers:
            h_cell5 = layer_cell5(h_cell5, ei_pos_cell5, ew_pos_cell5, dis_pos_cell5, ei_neg_cell5, ew_neg_cell5, dis_neg_cell5)
        return self.proj(h_cell5)

    def forward(self, X_cell5, ei_pos_cell5, ew_pos_cell5, dis_pos_cell5, ei_neg_cell5, ew_neg_cell5, dis_neg_cell5):
        return self.encode(X_cell5, ei_pos_cell5, ew_pos_cell5, dis_pos_cell5, ei_neg_cell5, ew_neg_cell5, dis_neg_cell5)

print("[Cell 5] Models defined (Feature-aware Baseline + Signed Prototype GCN).")

In [ ]:
# Cell 6: Training Loops (Fixed Indexing, Hierarchical Mask, Signed Edges, Hier Weight)
# import warnings
# from sklearn.metrics import balanced_accuracy_score

def rare_balanced_acc_cell6(y_true_cell6, y_pred_cell6, rare_classes_cell6):
    mask_cell6 = np.isin(y_true_cell6, rare_classes_cell6)
    if mask_cell6.sum() == 0:
        return float("nan")
    y_t_cell6, y_p_cell6 = y_true_cell6[mask_cell6], y_pred_cell6[mask_cell6]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        return balanced_accuracy_score(y_t_cell6, y_p_cell6)

def prototype_loss_cell6(h_cell6, labels_cell6, prototypes_cell6, mask_cell6, temperature_cell6=0.1):
    h_masked_cell6 = h_cell6[mask_cell6]
    y_masked_cell6 = labels_cell6[mask_cell6]
    if len(h_masked_cell6) == 0:
        return torch.tensor(0.0, device=h_cell6.device, requires_grad=True)
    dist_cell6 = torch.cdist(h_masked_cell6, prototypes_cell6)
    logits_cell6 = -dist_cell6 / temperature_cell6
    return F.cross_entropy(logits_cell6, y_masked_cell6)

def train_baseline_cell6(model_cell6, X_cell6, ei_cell6, ew_cell6, dis_cell6, train_idx_cell6, val_idx_cell6, test_idx_cell6,
                         labels_cell6, rare_classes_cell6, seed_cell6, epochs_cell6=200, device_cell6=DEVICE_cell1, C_cell6=29):
    seed_all_cell1(seed_cell6)
    model_cell6 = model_cell6.to(device_cell6)
    opt_cell6 = torch.optim.AdamW(model_cell6.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_cell6 = torch.optim.lr_scheduler.StepLR(opt_cell6, step_size=50, gamma=0.5)

    tr_cell6, va_cell6, te_cell6 = train_idx_cell6.to(device_cell6), val_idx_cell6.to(device_cell6), test_idx_cell6.to(device_cell6)
    y_cell6 = labels_cell6.to(device_cell6)
    X_dev_cell6 = X_cell6.to(device_cell6)

    train_labels_cell6 = y_cell6[tr_cell6]
    cnt_cell6 = torch.bincount(train_labels_cell6, minlength=C_cell6).float()
    w_cell6 = (cnt_cell6.sum() / cnt_cell6.clamp(min=1)) ** 0.5
    w_cell6 = w_cell6 / w_cell6.mean()

    best_val_cell6, best_state_cell6 = -1.0, None
    for epoch_cell6 in range(epochs_cell6):
        model_cell6.train(); opt_cell6.zero_grad()
        out_cell6 = model_cell6(X_dev_cell6, ei_cell6, ew_cell6, dis_cell6)
        loss_cell6 = F.cross_entropy(out_cell6[tr_cell6], y_cell6[tr_cell6], weight=w_cell6)
        loss_cell6.backward()
        torch.nn.utils.clip_grad_norm_(model_cell6.parameters(), 1.0)
        opt_cell6.step(); sched_cell6.step()

        if (epoch_cell6 + 1) % 50 == 0:
            model_cell6.eval()
            with torch.no_grad():
                out_cell6 = model_cell6(X_dev_cell6, ei_cell6, ew_cell6, dis_cell6)
                val_pred_cell6 = out_cell6[va_cell6].argmax(-1).cpu().numpy()
                val_true_cell6 = y_cell6[va_cell6].cpu().numpy()
                val_rare_cell6 = rare_balanced_acc_cell6(val_true_cell6, val_pred_cell6, rare_classes_cell6)
                if not np.isnan(val_rare_cell6) and val_rare_cell6 > best_val_cell6:
                    best_val_cell6 = val_rare_cell6
                    best_state_cell6 = {k_cell6: v_cell6.clone() for k_cell6, v_cell6 in model_cell6.state_dict().items()}

    if best_state_cell6 is not None:
        model_cell6.load_state_dict(best_state_cell6)

    model_cell6.eval()
    with torch.no_grad():
        out_cell6 = model_cell6(X_dev_cell6, ei_cell6, ew_cell6, dis_cell6)
        train_pred_cell6 = out_cell6[tr_cell6].argmax(-1).cpu().numpy()
        test_pred_cell6 = out_cell6[te_cell6].argmax(-1).cpu().numpy()

    train_rare_cell6 = rare_balanced_acc_cell6(y_cell6[tr_cell6].cpu().numpy(), train_pred_cell6, rare_classes_cell6)
    test_rare_cell6 = rare_balanced_acc_cell6(y_cell6[te_cell6].cpu().numpy(), test_pred_cell6, rare_classes_cell6)
    return {"train_rare": train_rare_cell6, "test_rare": test_rare_cell6, "val_rare": best_val_cell6}

def train_spgcn_cell6(model_cell6, X_cell6, ei_pos_cell6, ew_pos_cell6, dis_pos_cell6, ei_neg_cell6, ew_neg_cell6, dis_neg_cell6, 
                     train_idx_cell6, val_idx_cell6, test_idx_cell6,
                     labels_cell6, super_labels_cell6, rare_classes_cell6, seed_cell6, epochs_cell6=200,
                     device_cell6=DEVICE_cell1, C_cell6=29, C_super_cell6=10, 
                     use_hierarchical_cell6=True, use_reinit_cell6=True, hier_weight_cell6=0.5):
    seed_all_cell1(seed_cell6)
    model_cell6 = model_cell6.to(device_cell6)
    opt_cell6 = torch.optim.AdamW(model_cell6.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_cell6 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cell6, T_max=epochs_cell6)

    tr_cell6, va_cell6, te_cell6 = train_idx_cell6.to(device_cell6), val_idx_cell6.to(device_cell6), test_idx_cell6.to(device_cell6)
    y_cell6 = labels_cell6.to(device_cell6)
    y_super_cell6 = super_labels_cell6.to(device_cell6)
    X_dev_cell6 = X_cell6.to(device_cell6)

    best_val_cell6, best_state_cell6 = -1.0, None
    proto_reinit_done_cell6 = False

    for epoch_cell6 in range(epochs_cell6):
        model_cell6.train(); opt_cell6.zero_grad()
        h_cell6 = model_cell6(X_dev_cell6, ei_pos_cell6, ew_pos_cell6, dis_pos_cell6, ei_neg_cell6, ew_neg_cell6, dis_neg_cell6)

        loss_class_cell6 = prototype_loss_cell6(h_cell6, y_cell6, model_cell6.proto_class, tr_cell6, temperature_cell6=model_cell6.temperature)
        loss_super_cell6 = torch.tensor(0.0, device=h_cell6.device)

        if use_hierarchical_cell6:
            super_valid_cell6 = (y_super_cell6 >= 0)
            super_mask_cell6 = tr_cell6[super_valid_cell6[tr_cell6]]
            loss_super_cell6 = prototype_loss_cell6(h_cell6, y_super_cell6, model_cell6.proto_super, super_mask_cell6, temperature_cell6=model_cell6.temperature * 2)

        # Use the passed hier_weight_cell6 parameter
        loss_cell6 = loss_class_cell6 + hier_weight_cell6 * loss_super_cell6

        loss_cell6.backward()
        torch.nn.utils.clip_grad_norm_(model_cell6.parameters(), 1.0)
        opt_cell6.step(); sched_cell6.step()

        if use_reinit_cell6 and epoch_cell6 == 40 and not proto_reinit_done_cell6:
            with torch.no_grad():
                h_tr_cell6 = h_cell6[tr_cell6]
                y_tr_cell6 = y_cell6[tr_cell6]
                for c_loop_cell6 in range(C_cell6):
                    mask_c_cell6 = (y_tr_cell6 == c_loop_cell6)
                    if mask_c_cell6.sum() > 0:
                        model_cell6.proto_class.data[c_loop_cell6] = h_tr_cell6[mask_c_cell6].mean(0)

                y_super_tr_cell6 = y_super_cell6[tr_cell6]
                super_valid_tr_cell6 = (y_super_tr_cell6 >= 0)
                valid_y_super_tr_cell6 = y_super_tr_cell6[super_valid_tr_cell6]
                valid_h_tr_cell6 = h_tr_cell6[super_valid_tr_cell6]
                for c_loop_cell6 in range(C_super_cell6):
                    mask_c_cell6 = (valid_y_super_tr_cell6 == c_loop_cell6)
                    if mask_c_cell6.sum() > 0:
                        model_cell6.proto_super.data[c_loop_cell6] = valid_h_tr_cell6[mask_c_cell6].mean(0)
            proto_reinit_done_cell6 = True

        if (epoch_cell6 + 1) % 50 == 0:
            model_cell6.eval()
            with torch.no_grad():
                h_cell6 = model_cell6(X_dev_cell6, ei_pos_cell6, ew_pos_cell6, dis_pos_cell6, ei_neg_cell6, ew_neg_cell6, dis_neg_cell6)
                dist_cell6 = torch.cdist(h_cell6, model_cell6.proto_class)
                val_pred_cell6 = dist_cell6[va_cell6].argmin(dim=-1).cpu().numpy()
                val_true_cell6 = y_cell6[va_cell6].cpu().numpy()
                val_rare_cell6 = rare_balanced_acc_cell6(val_true_cell6, val_pred_cell6, rare_classes_cell6)
                if not np.isnan(val_rare_cell6) and val_rare_cell6 > best_val_cell6:
                    best_val_cell6 = val_rare_cell6
                    best_state_cell6 = {k_cell6: v_cell6.clone() for k_cell6, v_cell6 in model_cell6.state_dict().items()}

    if best_state_cell6 is not None:
        model_cell6.load_state_dict(best_state_cell6)

    model_cell6.eval()
    with torch.no_grad():
        h_cell6 = model_cell6(X_dev_cell6, ei_pos_cell6, ew_pos_cell6, dis_pos_cell6, ei_neg_cell6, ew_neg_cell6, dis_neg_cell6)
        dist_cell6 = torch.cdist(h_cell6, model_cell6.proto_class)
        pred_cell6 = dist_cell6.argmin(dim=-1).cpu().numpy()

    true_cell6 = y_cell6.cpu().numpy()
    train_rare_cell6 = rare_balanced_acc_cell6(true_cell6[tr_cell6.cpu().numpy()], pred_cell6[tr_cell6.cpu().numpy()], rare_classes_cell6)
    test_rare_cell6 = rare_balanced_acc_cell6(true_cell6[te_cell6.cpu().numpy()], pred_cell6[te_cell6.cpu().numpy()], rare_classes_cell6)
    rare_pred_count_cell6 = int(np.isin(pred_cell6[te_cell6.cpu().numpy()], rare_classes_cell6).sum())

    return {"train_rare": train_rare_cell6, "test_rare": test_rare_cell6, "val_rare": best_val_cell6, "rare_pred_count": rare_pred_count_cell6}

print("[Cell 6] Training loops updated with hier_weight_cell6 parameter.")

In [ ]:
# Cell 7: Graph Tensor Preparation (CUDA-Safe)
print("[Cell 7] Preparing graph tensors (bulletproof CPU -> GPU transfer)...")

ei_full_cpu_cell7 = np.ascontiguousarray(np.stack([src_arr_cell3, dst_arr_cell3]).astype(np.int64))
ei_full_cell7 = torch.from_numpy(ei_full_cpu_cell7).to(DEVICE_cell1)

ew_full_cpu_cell7 = np.ascontiguousarray((np.log1p(syn_arr_cell3) / (np.log1p(syn_arr_cell3).mean() + 1e-8)).astype(np.float32))
ew_full_cell7 = torch.from_numpy(ew_full_cpu_cell7).to(DEVICE_cell1)

dis_full_cell7 = normalize_edges_cell5(ei_full_cell7, ew_full_cell7, N_cell3, DEVICE_cell1)

pos_mask_cpu_cell7 = np.ascontiguousarray((sign_bin_arr_cell3 > 0.5).astype(bool))
pos_mask_cell7 = torch.from_numpy(pos_mask_cpu_cell7).to(torch.bool).to(DEVICE_cell1)
neg_mask_cell7 = ~pos_mask_cell7

ei_pos_cell7 = ei_full_cell7[:, pos_mask_cell7]
ew_pos_cell7 = ew_full_cell7[pos_mask_cell7]
ei_neg_cell7 = ei_full_cell7[:, neg_mask_cell7]
ew_neg_cell7 = ew_full_cell7[neg_mask_cell7]

dis_pos_cell7 = normalize_edges_cell5(ei_pos_cell7, ew_pos_cell7, N_cell3, DEVICE_cell1)
dis_neg_cell7 = normalize_edges_cell5(ei_neg_cell7, ew_neg_cell7, N_cell3, DEVICE_cell1)

print(f"[Cell 7] Graph tensors prepared successfully on {DEVICE_cell1}.")

In [ ]:
# Checkpoint: Cell 7 graph tensors
#
# What Cell 7 does:
#   Prepares full-graph PyTorch geometric tensors:
#   directed/full edges, positive edges, negative edges,
#   edge weights, and degree normalization vectors.

hf_save_checkpoint(
    "cell07_graph_tensors",
    {
        "N_cell3": N_cell3,
        "src_arr_cell3": src_arr_cell3,
        "dst_arr_cell3": dst_arr_cell3,
        "syn_arr_cell3": syn_arr_cell3,
        "sign_bin_arr_cell3": sign_bin_arr_cell3,
        "ei_full_cell7": ei_full_cell7,
        "ew_full_cell7": ew_full_cell7,
        "dis_full_cell7": dis_full_cell7,
        "ei_pos_cell7": ei_pos_cell7,
        "ew_pos_cell7": ew_pos_cell7,
        "dis_pos_cell7": dis_pos_cell7,
        "ei_neg_cell7": ei_neg_cell7,
        "ew_neg_cell7": ew_neg_cell7,
        "dis_neg_cell7": dis_neg_cell7,
    },
    description=(
        "Cell 7: full/positive/negative graph edge tensors, weights, "
        "and normalization vectors."
    ),
)

In [ ]:
# # Cell 8: Debug Experiment (4 Configs, 5 Seeds)
# DEBUG_SEEDS_cell8 = list(range(5))
# configs_cell8 = {
#     "minimal":       {"n_layers": 0, "hier": False, "reinit": True, "hier_weight": 0.5},
#     "minimal_hier":  {"n_layers": 0, "hier": True,  "reinit": True, "hier_weight": 0.1},
#     "one_signed":    {"n_layers": 1, "hier": False, "reinit": True, "hier_weight": 0.5},
#     "one_signed_hier": {"n_layers": 1, "hier": True, "reinit": True, "hier_weight": 0.1},
# }
# results_debug_cell8 = {k_cell8: [] for k_cell8 in configs_cell8}

# for name_cfg_cell8, cfg_cell8 in configs_cell8.items():
#     print(f"\n=== Config: {name_cfg_cell8} ===")
#     for seed_loop_cell8 in DEBUG_SEEDS_cell8:
#         tr_cell8, va_cell8, te_cell8 = make_split_cell4(seed_loop_cell8, labeled_mask_cell3, labels_tensor_cell3, C_cell3)

#         m_debug_cell8 = SPGCN_cell5(
#             d_features_cell5=X_tensor_cell3.shape[1],
#             d_cell5=64, d_proto_cell5=32,
#             n_layers_cell5=cfg_cell8["n_layers"],
#             n_classes_cell5=C_cell3, n_super_cell5=C_super_cell3,
#             temperature_cell5=0.1, dropout_cell5=0.3)

#         r_debug_cell8 = train_spgcn_cell6(
#             m_debug_cell8, X_tensor_cell3,
#             ei_pos_cell7, ew_pos_cell7, dis_pos_cell7,
#             ei_neg_cell7, ew_neg_cell7, dis_neg_cell7,
#             tr_cell8, va_cell8, te_cell8,
#             labels_tensor_cell3, super_labels_tensor_cell3,
#             rare_classes_list_cell3, seed_loop_cell8,
#             epochs_cell6=200, C_cell6=C_cell3, C_super_cell6=C_super_cell3,
#             use_hierarchical_cell6=cfg_cell8["hier"],
#             use_reinit_cell6=cfg_cell8["reinit"],
#             hier_weight_cell6=cfg_cell8["hier_weight"])

#         results_debug_cell8[name_cfg_cell8].append(r_debug_cell8["test_rare"])
#         print(f"  seed {seed_loop_cell8}: {r_debug_cell8['test_rare']:.4f}")

# print("\n" + "=" * 60)
# print(" DEBUG EXPERIMENT RESULTS")
# print("=" * 60)
# for name_res_cell8, vals_res_cell8 in results_debug_cell8.items():
#     print(f"  {name_res_cell8:20s}  {np.mean(vals_res_cell8):.4f} ± {np.std(vals_res_cell8):.4f}")
# print("=" * 60)

In [ ]:
# Cell 8: Core baselines — GCN feature baseline + SPGCN, 10 seeds
#
# Complete rewrite.
#
# Rules enforced:
#   - No imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every local/helper/loop variable is prefixed with cell8_.
#   - Saves per-seed test predictions and test labels for Cell 15 / Cell 16.
#
# Produces global variables:
#   results_baseline_cell8
#   baseline_f1_cell8
#   results_spgcn_cell8
#   spgcn_f1_cell8
#   pred_baseline_cell8
#   pred_spgcn_cell8
#   y_test_per_seed_cell8
#   cell8_details
#   cell8_config


def cell8_main():
    cell8_globals = globals()
    cell8_np = cell8_globals.get("np", None)
    cell8_torch = cell8_globals.get("torch", None)

    if cell8_np is None:
        raise NameError("[Cell 8][FATAL] np is not available. Run earlier cells first.")

    if cell8_torch is None:
        raise NameError("[Cell 8][FATAL] torch is not available. Run earlier cells first.")

    cell8_required_names = [
        "DEVICE_cell1",
        "X_tensor_cell3",
        "labels_tensor_cell3",
        "labeled_mask_cell3",
        "super_labels_tensor_cell3",
        "C_cell3",
        "C_super_cell3",
        "rare_classes_list_cell3",
        "ei_full_cell7",
        "ew_full_cell7",
        "dis_full_cell7",
        "ei_pos_cell7",
        "ew_pos_cell7",
        "dis_pos_cell7",
        "ei_neg_cell7",
        "ew_neg_cell7",
        "dis_neg_cell7",
        "make_split_cell4",
        "GCNFeatureBaseline_cell5",
        "SPGCN_cell5",
        "train_baseline_cell6",
        "train_spgcn_cell6",
        "rare_balanced_acc_cell6",
    ]

    cell8_missing_names = []
    for cell8_required_name in cell8_required_names:
        if cell8_required_name not in cell8_globals:
            cell8_missing_names.append(cell8_required_name)

    if len(cell8_missing_names) > 0:
        raise NameError(
            "[Cell 8][FATAL] Missing upstream variables: "
            + str(cell8_missing_names)
            + ". Run Cells 1-7 first."
        )

    cell8_device = cell8_globals["DEVICE_cell1"]
    cell8_num_classes = int(cell8_globals["C_cell3"])
    cell8_num_super_classes = int(cell8_globals["C_super_cell3"])

    def cell8_to_numpy(cell8_object):
        if cell8_object is None:
            return None

        cell8_working_object = cell8_object

        try:
            if hasattr(cell8_working_object, "detach"):
                cell8_working_object = cell8_working_object.detach()

            if hasattr(cell8_working_object, "cpu"):
                cell8_working_object = cell8_working_object.cpu()

            if hasattr(cell8_working_object, "numpy"):
                cell8_working_object = cell8_working_object.numpy()

            return cell8_np.asarray(cell8_working_object)
        except Exception:
            return None

    def cell8_to_int_vector(cell8_object, cell8_description):
        cell8_array_object = cell8_to_numpy(cell8_object)

        if cell8_array_object is None:
            raise TypeError("[Cell 8][FATAL] Could not convert " + str(cell8_description) + " to numpy.")

        try:
            return cell8_array_object.astype(cell8_np.int64).reshape(-1)
        except Exception as cell8_cast_error:
            raise TypeError(
                "[Cell 8][FATAL] Could not cast "
                + str(cell8_description)
                + " to int64: "
                + repr(cell8_cast_error)
            )

    cell8_rare_classes_array = cell8_to_numpy(cell8_globals["rare_classes_list_cell3"])

    if cell8_rare_classes_array is None:
        raise TypeError("[Cell 8][FATAL] rare_classes_list_cell3 could not be converted to numpy.")

    cell8_rare_classes = []
    cell8_seen_rare_classes = set()

    for cell8_rare_value in cell8_rare_classes_array.ravel().tolist():
        cell8_rare_integer = int(cell8_rare_value)

        if cell8_rare_integer not in cell8_seen_rare_classes:
            cell8_seen_rare_classes.add(cell8_rare_integer)
            cell8_rare_classes.append(cell8_rare_integer)

    cell8_labels_array = cell8_to_numpy(cell8_globals["labels_tensor_cell3"])

    if cell8_labels_array is None:
        raise TypeError("[Cell 8][FATAL] labels_tensor_cell3 could not be converted to numpy.")

    cell8_labels_np = cell8_labels_array.astype(cell8_np.int64).reshape(-1)

    cell8_x_source = cell8_globals["X_tensor_cell3"]

    if cell8_torch.is_tensor(cell8_x_source):
        cell8_x_base = cell8_x_source.to(cell8_device)
    else:
        cell8_x_array = cell8_to_numpy(cell8_x_source)

        if cell8_x_array is None:
            raise TypeError("[Cell 8][FATAL] X_tensor_cell3 could not be converted to numpy or tensor.")

        cell8_x_base = cell8_torch.tensor(
            cell8_x_array.astype(cell8_np.float32),
            dtype=cell8_torch.float32,
        ).to(cell8_device)

    if int(cell8_x_base.ndim) != 2:
        raise RuntimeError(
            "[Cell 8][FATAL] X_base must be 2D, got shape "
            + str(tuple(cell8_x_base.shape))
        )

    cell8_feature_dim = int(cell8_x_base.shape[1])

    def cell8_set_seed(cell8_seed_value):
        cell8_seed_function = cell8_globals.get("seed_all_cell1", None)

        if callable(cell8_seed_function):
            cell8_seed_function(int(cell8_seed_value))
        else:
            cell8_torch.manual_seed(int(cell8_seed_value))
            cell8_np.random.seed(int(cell8_seed_value))

    def cell8_move_tensor(cell8_tensor_object, cell8_target_device):
        if cell8_torch.is_tensor(cell8_tensor_object):
            return cell8_tensor_object.to(cell8_target_device)

        return cell8_tensor_object

    def cell8_manual_macro_f1(cell8_true_vector, cell8_pred_vector, cell8_class_count):
        cell8_true_array = cell8_np.asarray(cell8_true_vector, dtype=cell8_np.int64).reshape(-1)
        cell8_pred_array = cell8_np.asarray(cell8_pred_vector, dtype=cell8_np.int64).reshape(-1)
        cell8_f1_values = []

        for cell8_class_value in range(int(cell8_class_count)):
            cell8_true_mask = cell8_true_array == cell8_class_value
            cell8_pred_mask = cell8_pred_array == cell8_class_value

            cell8_true_positive_count = int(cell8_np.sum(cell8_true_mask & cell8_pred_mask))
            cell8_false_positive_count = int(cell8_np.sum((~cell8_true_mask) & cell8_pred_mask))
            cell8_false_negative_count = int(cell8_np.sum(cell8_true_mask & (~cell8_pred_mask)))

            if cell8_true_positive_count + cell8_false_positive_count == 0:
                cell8_precision_value = 0.0
            else:
                cell8_precision_value = float(cell8_true_positive_count) / float(
                    cell8_true_positive_count + cell8_false_positive_count
                )

            if cell8_true_positive_count + cell8_false_negative_count == 0:
                cell8_recall_value = 0.0
            else:
                cell8_recall_value = float(cell8_true_positive_count) / float(
                    cell8_true_positive_count + cell8_false_negative_count
                )

            if cell8_precision_value + cell8_recall_value == 0.0:
                cell8_f1_value = 0.0
            else:
                cell8_f1_value = (
                    2.0
                    * cell8_precision_value
                    * cell8_recall_value
                    / (cell8_precision_value + cell8_recall_value)
                )

            cell8_f1_values.append(float(cell8_f1_value))

        if len(cell8_f1_values) == 0:
            return float("nan")

        return float(cell8_np.mean(cell8_f1_values))

    def cell8_pick_metric(cell8_result_object, cell8_key_list, cell8_default_value):
        if isinstance(cell8_result_object, dict):
            for cell8_key_value in cell8_key_list:
                if cell8_key_value in cell8_result_object:
                    try:
                        return float(cell8_result_object[cell8_key_value])
                    except Exception:
                        pass

        for cell8_key_value in cell8_key_list:
            try:
                cell8_attribute_value = getattr(cell8_result_object, cell8_key_value)
                return float(cell8_attribute_value)
            except Exception:
                pass

        return float(cell8_default_value)

    def cell8_make_gcn_baseline(cell8_input_feature_dim):
        try:
            return cell8_globals["GCNFeatureBaseline_cell5"](
                d_features_cell5=int(cell8_input_feature_dim),
                d_cell5=64,
                n_layers_cell5=2,
                n_classes_cell5=cell8_num_classes,
                dropout_cell5=0.3,
            )
        except TypeError:
            return cell8_globals["GCNFeatureBaseline_cell5"](
                d_features=int(cell8_input_feature_dim),
                d=64,
                n_layers=2,
                n_classes=cell8_num_classes,
                dropout=0.3,
            )

    def cell8_make_spgcn(cell8_input_feature_dim):
        try:
            return cell8_globals["SPGCN_cell5"](
                d_features_cell5=int(cell8_input_feature_dim),
                d_cell5=64,
                d_proto_cell5=32,
                n_layers_cell5=2,
                n_classes_cell5=cell8_num_classes,
                n_super_cell5=cell8_num_super_classes,
                temperature_cell5=0.1,
                dropout_cell5=0.3,
            )
        except TypeError:
            return cell8_globals["SPGCN_cell5"](
                d_features=int(cell8_input_feature_dim),
                d=64,
                d_proto=32,
                n_layers=2,
                n_classes=cell8_num_classes,
                n_super=cell8_num_super_classes,
                temperature=0.1,
                dropout=0.3,
            )

    def cell8_collect_tensors(cell8_output_object, cell8_tensor_list=None):
        if cell8_tensor_list is None:
            cell8_tensor_list = []

        if cell8_torch.is_tensor(cell8_output_object):
            if int(cell8_output_object.ndim) >= 2:
                cell8_tensor_list.append(cell8_output_object)
            return cell8_tensor_list

        if isinstance(cell8_output_object, dict):
            for cell8_dict_value in cell8_output_object.values():
                cell8_collect_tensors(cell8_dict_value, cell8_tensor_list)
            return cell8_tensor_list

        if isinstance(cell8_output_object, (tuple, list)):
            for cell8_sequence_value in cell8_output_object:
                cell8_collect_tensors(cell8_sequence_value, cell8_tensor_list)
            return cell8_tensor_list

        return cell8_tensor_list

    def cell8_output_to_prediction(cell8_output_object, cell8_model_object=None):
        cell8_candidate_tensors = cell8_collect_tensors(cell8_output_object)

        if len(cell8_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 8][FATAL] Model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell8_output_object))
            )

        if cell8_model_object is not None and hasattr(cell8_model_object, "proto_class"):
            cell8_proto_parameter = getattr(cell8_model_object, "proto_class")

            if cell8_torch.is_tensor(cell8_proto_parameter) and int(cell8_proto_parameter.ndim) == 2:
                cell8_proto_float = cell8_proto_parameter.detach().float()
                cell8_proto_dimension = int(cell8_proto_float.shape[1])

                for cell8_candidate_tensor in cell8_candidate_tensors:
                    if int(cell8_candidate_tensor.shape[1]) == cell8_proto_dimension:
                        cell8_distance_matrix = cell8_torch.cdist(
                            cell8_candidate_tensor.detach().float(),
                            cell8_proto_float,
                        )
                        return (
                            cell8_distance_matrix
                            .argmin(dim=1)
                            .detach()
                            .cpu()
                            .numpy()
                            .astype(cell8_np.int64)
                        )

        for cell8_candidate_tensor in cell8_candidate_tensors:
            if int(cell8_candidate_tensor.shape[1]) == cell8_num_classes:
                return (
                    cell8_candidate_tensor
                    .argmax(dim=1)
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(cell8_np.int64)
                )

        return (
            cell8_candidate_tensors[0]
            .argmax(dim=-1)
            .detach()
            .cpu()
            .numpy()
            .astype(cell8_np.int64)
        )

    def cell8_forward_baseline(cell8_model_object, cell8_x_tensor, cell8_ei_tensor, cell8_ew_tensor, cell8_dis_tensor):
        cell8_last_type_error = None

        try:
            return cell8_model_object(cell8_x_tensor, cell8_ei_tensor, cell8_ew_tensor, cell8_dis_tensor)
        except TypeError as cell8_type_error:
            cell8_last_type_error = cell8_type_error

        try:
            return cell8_model_object(cell8_x_tensor, cell8_ei_tensor, cell8_ew_tensor)
        except TypeError as cell8_type_error:
            cell8_last_type_error = cell8_type_error

        try:
            return cell8_model_object(cell8_x_tensor, cell8_ei_tensor)
        except TypeError as cell8_type_error:
            cell8_last_type_error = cell8_type_error

        try:
            return cell8_model_object(cell8_ei_tensor, cell8_ew_tensor, cell8_dis_tensor)
        except TypeError as cell8_type_error:
            cell8_last_type_error = cell8_type_error

        raise TypeError(
            "[Cell 8][FATAL] Could not call baseline model with any known signature. "
            "Last TypeError: " + repr(cell8_last_type_error)
        )

    def cell8_forward_spgcn(
        cell8_model_object,
        cell8_x_tensor,
        cell8_ei_pos_tensor,
        cell8_ew_pos_tensor,
        cell8_dis_pos_tensor,
        cell8_ei_neg_tensor,
        cell8_ew_neg_tensor,
        cell8_dis_neg_tensor,
    ):
        cell8_last_type_error = None

        try:
            return cell8_model_object(
                cell8_x_tensor,
                cell8_ei_pos_tensor,
                cell8_ew_pos_tensor,
                cell8_dis_pos_tensor,
                cell8_ei_neg_tensor,
                cell8_ew_neg_tensor,
                cell8_dis_neg_tensor,
            )
        except TypeError as cell8_type_error:
            cell8_last_type_error = cell8_type_error

        try:
            return cell8_model_object(
                cell8_x_tensor,
                cell8_ei_pos_tensor,
                cell8_dis_pos_tensor,
                cell8_ei_neg_tensor,
                cell8_dis_neg_tensor,
            )
        except TypeError as cell8_type_error:
            cell8_last_type_error = cell8_type_error

        try:
            return cell8_model_object(
                cell8_x_tensor,
                cell8_ei_pos_tensor,
                cell8_ei_neg_tensor,
            )
        except TypeError as cell8_type_error:
            cell8_last_type_error = cell8_type_error

        raise TypeError(
            "[Cell 8][FATAL] Could not call SPGCN model with any known signature. "
            "Last TypeError: " + repr(cell8_last_type_error)
        )

    def cell8_predict_baseline(cell8_model_object, cell8_x_tensor, cell8_ei_tensor, cell8_ew_tensor, cell8_dis_tensor):
        cell8_model_object.eval()
        cell8_model_device = next(cell8_model_object.parameters()).device

        cell8_x_device = cell8_x_tensor.to(cell8_model_device)
        cell8_ei_device = cell8_move_tensor(cell8_ei_tensor, cell8_model_device)
        cell8_ew_device = cell8_move_tensor(cell8_ew_tensor, cell8_model_device)
        cell8_dis_device = cell8_move_tensor(cell8_dis_tensor, cell8_model_device)

        with cell8_torch.no_grad():
            cell8_output_object = cell8_forward_baseline(
                cell8_model_object,
                cell8_x_device,
                cell8_ei_device,
                cell8_ew_device,
                cell8_dis_device,
            )

        return cell8_output_to_prediction(cell8_output_object, None)

    def cell8_predict_spgcn(
        cell8_model_object,
        cell8_x_tensor,
        cell8_ei_pos_tensor,
        cell8_ew_pos_tensor,
        cell8_dis_pos_tensor,
        cell8_ei_neg_tensor,
        cell8_ew_neg_tensor,
        cell8_dis_neg_tensor,
    ):
        cell8_model_object.eval()
        cell8_model_device = next(cell8_model_object.parameters()).device

        cell8_x_device = cell8_x_tensor.to(cell8_model_device)
        cell8_ei_pos_device = cell8_move_tensor(cell8_ei_pos_tensor, cell8_model_device)
        cell8_ew_pos_device = cell8_move_tensor(cell8_ew_pos_tensor, cell8_model_device)
        cell8_dis_pos_device = cell8_move_tensor(cell8_dis_pos_tensor, cell8_model_device)
        cell8_ei_neg_device = cell8_move_tensor(cell8_ei_neg_tensor, cell8_model_device)
        cell8_ew_neg_device = cell8_move_tensor(cell8_ew_neg_tensor, cell8_model_device)
        cell8_dis_neg_device = cell8_move_tensor(cell8_dis_neg_tensor, cell8_model_device)

        with cell8_torch.no_grad():
            cell8_output_object = cell8_forward_spgcn(
                cell8_model_object,
                cell8_x_device,
                cell8_ei_pos_device,
                cell8_ew_pos_device,
                cell8_dis_pos_device,
                cell8_ei_neg_device,
                cell8_ew_neg_device,
                cell8_dis_neg_device,
            )

        return cell8_output_to_prediction(cell8_output_object, cell8_model_object)

    def cell8_metrics_from_test_predictions(
        cell8_true_vector,
        cell8_pred_vector,
        cell8_rare_class_list,
        cell8_class_count,
    ):
        cell8_true_array = cell8_np.asarray(cell8_true_vector, dtype=cell8_np.int64).reshape(-1)
        cell8_pred_array = cell8_np.asarray(cell8_pred_vector, dtype=cell8_np.int64).reshape(-1)

        cell8_valid_mask = (
            (cell8_true_array >= 0)
            & (cell8_pred_array >= 0)
            & (cell8_true_array < cell8_class_count)
            & (cell8_pred_array < cell8_class_count)
        )

        cell8_true_valid = cell8_true_array[cell8_valid_mask]
        cell8_pred_valid = cell8_pred_array[cell8_valid_mask]

        if cell8_true_valid.size == 0:
            return {
                "rare_ba": float("nan"),
                "macro_f1": float("nan"),
                "accuracy": float("nan"),
                "n_test": 0,
                "n_active_predicted_classes": 0,
                "n_rare_predictions": 0,
            }

        cell8_rare_function = cell8_globals.get("rare_balanced_acc_cell6", None)

        if callable(cell8_rare_function):
            cell8_rare_ba_value = float(
                cell8_rare_function(
                    cell8_true_valid,
                    cell8_pred_valid,
                    cell8_rare_class_list,
                )
            )
        else:
            cell8_rare_recalls = []

            for cell8_rare_class_value in cell8_rare_class_list:
                cell8_support_mask = cell8_true_valid == cell8_rare_class_value
                cell8_support_count = int(cell8_np.sum(cell8_support_mask))

                if cell8_support_count > 0:
                    cell8_hit_mask = cell8_support_mask & (cell8_pred_valid == cell8_rare_class_value)
                    cell8_rare_recalls.append(float(cell8_np.sum(cell8_hit_mask)) / float(cell8_support_count))

            if len(cell8_rare_recalls) == 0:
                cell8_rare_ba_value = float("nan")
            else:
                cell8_rare_ba_value = float(cell8_np.mean(cell8_rare_recalls))

        cell8_macro_f1_value = cell8_manual_macro_f1(
            cell8_true_valid,
            cell8_pred_valid,
            cell8_class_count,
        )

        cell8_accuracy_value = float(cell8_np.mean(cell8_pred_valid == cell8_true_valid))

        cell8_pred_counts = cell8_np.bincount(cell8_pred_valid, minlength=cell8_class_count)
        cell8_active_class_count = int(cell8_np.sum(cell8_pred_counts > 0))

        cell8_rare_array = cell8_np.asarray(cell8_rare_class_list, dtype=cell8_np.int64)

        if (
            cell8_rare_array.size > 0
            and int(cell8_rare_array.max()) < int(cell8_pred_counts.size)
        ):
            cell8_rare_prediction_count = int(cell8_np.sum(cell8_pred_counts[cell8_rare_array]))
        else:
            cell8_rare_prediction_count = 0

        return {
            "rare_ba": cell8_rare_ba_value,
            "macro_f1": cell8_macro_f1_value,
            "accuracy": cell8_accuracy_value,
            "n_test": int(cell8_true_valid.size),
            "n_active_predicted_classes": cell8_active_class_count,
            "n_rare_predictions": cell8_rare_prediction_count,
        }

    def cell8_fmt_metric(cell8_numeric_value):
        try:
            cell8_float_value = float(cell8_numeric_value)
        except Exception:
            return "n/a"

        if not cell8_np.isfinite(cell8_float_value):
            return "n/a"

        return f"{cell8_float_value:.4f}"

    print("=" * 100)
    print("CELL 8 — CORE BASELINES WITH PREDICTION SAVING")
    print("=" * 100)
    print("[cfg] seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]")
    print("[cfg] epochs=200")
    print("[data] X_base shape=" + str(tuple(cell8_x_base.shape)))
    print("[data] C=" + str(cell8_num_classes) + ", C_super=" + str(cell8_num_super_classes))
    print("[data] rare classes=" + str(cell8_rare_classes))
    print("[fix] Saves pred_baseline_cell8, pred_spgcn_cell8, y_test_per_seed_cell8.")
    print("=" * 100)

    cell8_baseline_rare_list = []
    cell8_baseline_f1_list = []
    cell8_spgcn_rare_list = []
    cell8_spgcn_f1_list = []

    cell8_pred_baseline_list = []
    cell8_pred_spgcn_list = []
    cell8_y_test_list = []
    cell8_details_list = []

    for cell8_seed_value in range(10):
        print("\n>>> Cell 8 seed " + str(cell8_seed_value), flush=True)

        cell8_train_index, cell8_val_index, cell8_test_index = cell8_globals["make_split_cell4"](
            cell8_seed_value,
            cell8_globals["labeled_mask_cell3"],
            cell8_globals["labels_tensor_cell3"],
            cell8_globals["C_cell3"],
        )

        cell8_test_np = cell8_to_int_vector(cell8_test_index, "Cell 8 test index")
        cell8_y_test_np = cell8_labels_np[cell8_test_np].copy()
        cell8_y_test_list.append(cell8_y_test_np)

        cell8_seed_details = {
            "seed": int(cell8_seed_value),
        }

        # ------------------------------------------------------------------
        # 1. GCN + all features baseline
        # ------------------------------------------------------------------
        cell8_set_seed(cell8_seed_value)

        cell8_model_baseline = cell8_make_gcn_baseline(cell8_feature_dim).to(cell8_device)

        cell8_result_baseline = cell8_globals["train_baseline_cell6"](
            cell8_model_baseline,
            cell8_x_base,
            cell8_globals["ei_full_cell7"],
            cell8_globals["ew_full_cell7"],
            cell8_globals["dis_full_cell7"],
            cell8_train_index,
            cell8_val_index,
            cell8_test_index,
            cell8_globals["labels_tensor_cell3"],
            cell8_globals["rare_classes_list_cell3"],
            cell8_seed_value,
            epochs_cell6=200,
            C_cell6=cell8_num_classes,
        )

        cell8_pred_baseline_full = cell8_predict_baseline(
            cell8_model_baseline,
            cell8_x_base,
            cell8_globals["ei_full_cell7"],
            cell8_globals["ew_full_cell7"],
            cell8_globals["dis_full_cell7"],
        )

        cell8_pred_baseline_test = cell8_pred_baseline_full[cell8_test_np].copy()
        cell8_pred_baseline_list.append(cell8_pred_baseline_test)

        cell8_metrics_baseline = cell8_metrics_from_test_predictions(
            cell8_y_test_np,
            cell8_pred_baseline_test,
            cell8_rare_classes,
            cell8_num_classes,
        )

        cell8_returned_baseline_rare = cell8_pick_metric(
            cell8_result_baseline,
            ["test_rare", "rare_ba", "rare_balanced_acc", "rare"],
            float("nan"),
        )

        cell8_returned_baseline_f1 = cell8_pick_metric(
            cell8_result_baseline,
            [
                "test_f1",
                "macro_f1",
                "f1",
                "test_macro_f1",
                "f1_macro",
                "macro",
                "test_f1_macro",
                "val_f1",
                "best_f1",
            ],
            float("nan"),
        )

        if cell8_np.isfinite(cell8_returned_baseline_rare):
            cell8_used_baseline_rare = float(cell8_returned_baseline_rare)
        else:
            cell8_used_baseline_rare = float(cell8_metrics_baseline["rare_ba"])

        cell8_used_baseline_f1 = float(cell8_metrics_baseline["macro_f1"])

        cell8_baseline_rare_list.append(cell8_used_baseline_rare)
        cell8_baseline_f1_list.append(cell8_used_baseline_f1)

        cell8_seed_details["baseline"] = {
            "returned_rare_ba": float(cell8_returned_baseline_rare),
            "returned_macro_f1": float(cell8_returned_baseline_f1),
            "computed_rare_ba": float(cell8_metrics_baseline["rare_ba"]),
            "computed_macro_f1": float(cell8_metrics_baseline["macro_f1"]),
            "computed_accuracy": float(cell8_metrics_baseline["accuracy"]),
            "n_test": int(cell8_metrics_baseline["n_test"]),
            "n_active_predicted_classes": int(cell8_metrics_baseline["n_active_predicted_classes"]),
            "n_rare_predictions": int(cell8_metrics_baseline["n_rare_predictions"]),
            "used_rare_ba": float(cell8_used_baseline_rare),
            "used_macro_f1": float(cell8_used_baseline_f1),
        }

        print(
            "  [baseline] rare_ba="
            + cell8_fmt_metric(cell8_used_baseline_rare)
            + " | macro_f1="
            + cell8_fmt_metric(cell8_used_baseline_f1)
            + " | acc="
            + cell8_fmt_metric(cell8_metrics_baseline["accuracy"])
            + " | active_classes="
            + str(cell8_metrics_baseline["n_active_predicted_classes"])
            + " | rare_preds="
            + str(cell8_metrics_baseline["n_rare_predictions"]),
            flush=True,
        )

        # ------------------------------------------------------------------
        # 2. SPGCN prototype baseline
        # ------------------------------------------------------------------
        cell8_set_seed(cell8_seed_value)

        cell8_model_spgcn = cell8_make_spgcn(cell8_feature_dim).to(cell8_device)

        cell8_result_spgcn = cell8_globals["train_spgcn_cell6"](
            cell8_model_spgcn,
            cell8_x_base,
            cell8_globals["ei_pos_cell7"],
            cell8_globals["ew_pos_cell7"],
            cell8_globals["dis_pos_cell7"],
            cell8_globals["ei_neg_cell7"],
            cell8_globals["ew_neg_cell7"],
            cell8_globals["dis_neg_cell7"],
            cell8_train_index,
            cell8_val_index,
            cell8_test_index,
            cell8_globals["labels_tensor_cell3"],
            cell8_globals["super_labels_tensor_cell3"],
            cell8_globals["rare_classes_list_cell3"],
            cell8_seed_value,
            epochs_cell6=200,
            C_cell6=cell8_num_classes,
            C_super_cell6=cell8_num_super_classes,
            use_hierarchical_cell6=False,
            use_reinit_cell6=True,
        )

        cell8_pred_spgcn_full = cell8_predict_spgcn(
            cell8_model_spgcn,
            cell8_x_base,
            cell8_globals["ei_pos_cell7"],
            cell8_globals["ew_pos_cell7"],
            cell8_globals["dis_pos_cell7"],
            cell8_globals["ei_neg_cell7"],
            cell8_globals["ew_neg_cell7"],
            cell8_globals["dis_neg_cell7"],
        )

        cell8_pred_spgcn_test = cell8_pred_spgcn_full[cell8_test_np].copy()
        cell8_pred_spgcn_list.append(cell8_pred_spgcn_test)

        cell8_metrics_spgcn = cell8_metrics_from_test_predictions(
            cell8_y_test_np,
            cell8_pred_spgcn_test,
            cell8_rare_classes,
            cell8_num_classes,
        )

        cell8_returned_spgcn_rare = cell8_pick_metric(
            cell8_result_spgcn,
            ["test_rare", "rare_ba", "rare_balanced_acc", "rare"],
            float("nan"),
        )

        cell8_returned_spgcn_f1 = cell8_pick_metric(
            cell8_result_spgcn,
            [
                "test_f1",
                "macro_f1",
                "f1",
                "test_macro_f1",
                "f1_macro",
                "macro",
                "test_f1_macro",
                "val_f1",
                "best_f1",
            ],
            float("nan"),
        )

        if cell8_np.isfinite(cell8_returned_spgcn_rare):
            cell8_used_spgcn_rare = float(cell8_returned_spgcn_rare)
        else:
            cell8_used_spgcn_rare = float(cell8_metrics_spgcn["rare_ba"])

        cell8_used_spgcn_f1 = float(cell8_metrics_spgcn["macro_f1"])

        cell8_spgcn_rare_list.append(cell8_used_spgcn_rare)
        cell8_spgcn_f1_list.append(cell8_used_spgcn_f1)

        cell8_seed_details["spgcn"] = {
            "returned_rare_ba": float(cell8_returned_spgcn_rare),
            "returned_macro_f1": float(cell8_returned_spgcn_f1),
            "computed_rare_ba": float(cell8_metrics_spgcn["rare_ba"]),
            "computed_macro_f1": float(cell8_metrics_spgcn["macro_f1"]),
            "computed_accuracy": float(cell8_metrics_spgcn["accuracy"]),
            "n_test": int(cell8_metrics_spgcn["n_test"]),
            "n_active_predicted_classes": int(cell8_metrics_spgcn["n_active_predicted_classes"]),
            "n_rare_predictions": int(cell8_metrics_spgcn["n_rare_predictions"]),
            "used_rare_ba": float(cell8_used_spgcn_rare),
            "used_macro_f1": float(cell8_used_spgcn_f1),
        }

        print(
            "  [spgcn]    rare_ba="
            + cell8_fmt_metric(cell8_used_spgcn_rare)
            + " | macro_f1="
            + cell8_fmt_metric(cell8_used_spgcn_f1)
            + " | acc="
            + cell8_fmt_metric(cell8_metrics_spgcn["accuracy"])
            + " | active_classes="
            + str(cell8_metrics_spgcn["n_active_predicted_classes"])
            + " | rare_preds="
            + str(cell8_metrics_spgcn["n_rare_predictions"]),
            flush=True,
        )

        cell8_details_list.append(cell8_seed_details)

        del cell8_model_baseline
        del cell8_model_spgcn
        del cell8_result_baseline
        del cell8_result_spgcn
        del cell8_pred_baseline_full
        del cell8_pred_spgcn_full

        if cell8_torch.cuda.is_available():
            cell8_torch.cuda.empty_cache()

    cell8_config = {
        "num_seeds": 10,
        "epochs": 200,
        "d_features": int(cell8_feature_dim),
        "C": int(cell8_num_classes),
        "C_super": int(cell8_num_super_classes),
        "rare_classes": cell8_rare_classes,
        "prediction_storage": "test-only integer prediction vectors per seed",
        "spgcn_eval_mode": "nearest_prototype_when_available",
    }

    print("\n" + "=" * 100)
    print("CELL 8 FINAL SUMMARY")
    print("=" * 100)

    cell8_baseline_rare_array = cell8_np.asarray(cell8_baseline_rare_list, dtype=float)
    cell8_baseline_f1_array = cell8_np.asarray(cell8_baseline_f1_list, dtype=float)
    cell8_spgcn_rare_array = cell8_np.asarray(cell8_spgcn_rare_list, dtype=float)
    cell8_spgcn_f1_array = cell8_np.asarray(cell8_spgcn_f1_list, dtype=float)

    cell8_baseline_rare_finite = cell8_baseline_rare_array[cell8_np.isfinite(cell8_baseline_rare_array)]
    cell8_baseline_f1_finite = cell8_baseline_f1_array[cell8_np.isfinite(cell8_baseline_f1_array)]
    cell8_spgcn_rare_finite = cell8_spgcn_rare_array[cell8_np.isfinite(cell8_spgcn_rare_array)]
    cell8_spgcn_f1_finite = cell8_spgcn_f1_array[cell8_np.isfinite(cell8_spgcn_f1_array)]

    if cell8_baseline_rare_finite.size > 0:
        print(
            "GCN baseline rare_ba:   "
            + cell8_fmt_metric(cell8_np.mean(cell8_baseline_rare_finite))
            + " +/- "
            + cell8_fmt_metric(cell8_np.std(cell8_baseline_rare_finite, ddof=1) if cell8_baseline_rare_finite.size > 1 else 0.0)
            + " (n="
            + str(int(cell8_baseline_rare_finite.size))
            + ")"
        )
    else:
        print("GCN baseline rare_ba: no finite values")

    if cell8_baseline_f1_finite.size > 0:
        print(
            "GCN baseline macro-F1:  "
            + cell8_fmt_metric(cell8_np.mean(cell8_baseline_f1_finite))
            + " +/- "
            + cell8_fmt_metric(cell8_np.std(cell8_baseline_f1_finite, ddof=1) if cell8_baseline_f1_finite.size > 1 else 0.0)
            + " (n="
            + str(int(cell8_baseline_f1_finite.size))
            + ")"
        )
    else:
        print("GCN baseline macro-F1: no finite values")

    if cell8_spgcn_rare_finite.size > 0:
        print(
            "SPGCN rare_ba:          "
            + cell8_fmt_metric(cell8_np.mean(cell8_spgcn_rare_finite))
            + " +/- "
            + cell8_fmt_metric(cell8_np.std(cell8_spgcn_rare_finite, ddof=1) if cell8_spgcn_rare_finite.size > 1 else 0.0)
            + " (n="
            + str(int(cell8_spgcn_rare_finite.size))
            + ")"
        )
    else:
        print("SPGCN rare_ba: no finite values")

    if cell8_spgcn_f1_finite.size > 0:
        print(
            "SPGCN macro-F1:         "
            + cell8_fmt_metric(cell8_np.mean(cell8_spgcn_f1_finite))
            + " +/- "
            + cell8_fmt_metric(cell8_np.std(cell8_spgcn_f1_finite, ddof=1) if cell8_spgcn_f1_finite.size > 1 else 0.0)
            + " (n="
            + str(int(cell8_spgcn_f1_finite.size))
            + ")"
        )
    else:
        print("SPGCN macro-F1: no finite values")

    print("len(pred_baseline_cell8)  = " + str(len(cell8_pred_baseline_list)))
    print("len(pred_spgcn_cell8)     = " + str(len(cell8_pred_spgcn_list)))
    print("len(y_test_per_seed_cell8)= " + str(len(cell8_y_test_list)))
    print("=" * 100)

    cell8_hf_save_checkpoint = cell8_globals.get("hf_save_checkpoint", None)

    if callable(cell8_hf_save_checkpoint):
        try:
            cell8_hf_save_checkpoint(
                "cell08_core_baselines_with_predictions",
                {
                    "results_baseline_cell8": cell8_baseline_rare_list,
                    "baseline_f1_cell8": cell8_baseline_f1_list,
                    "results_spgcn_cell8": cell8_spgcn_rare_list,
                    "spgcn_f1_cell8": cell8_spgcn_f1_list,
                    "pred_baseline_cell8": cell8_pred_baseline_list,
                    "pred_spgcn_cell8": cell8_pred_spgcn_list,
                    "y_test_per_seed_cell8": cell8_y_test_list,
                    "cell8_details": cell8_details_list,
                    "cell8_config": cell8_config,
                },
                description=(
                    "Cell 8: 10-seed GCN baseline and SPGCN with saved per-seed "
                    "test predictions and test labels for per-class recall and confusion."
                ),
                upload=True,
            )
        except Exception as cell8_checkpoint_error:
            print("[ckpt][warn] Cell 8 checkpoint failed: " + repr(cell8_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    return {
        "baseline_rare": cell8_baseline_rare_list,
        "baseline_f1": cell8_baseline_f1_list,
        "spgcn_rare": cell8_spgcn_rare_list,
        "spgcn_f1": cell8_spgcn_f1_list,
        "pred_baseline_test": cell8_pred_baseline_list,
        "pred_spgcn_test": cell8_pred_spgcn_list,
        "y_test_per_seed": cell8_y_test_list,
        "details": cell8_details_list,
        "config": cell8_config,
    }


cell8_output = cell8_main()

results_baseline_cell8 = cell8_output["baseline_rare"]
baseline_f1_cell8 = cell8_output["baseline_f1"]
results_spgcn_cell8 = cell8_output["spgcn_rare"]
spgcn_f1_cell8 = cell8_output["spgcn_f1"]

pred_baseline_cell8 = cell8_output["pred_baseline_test"]
pred_spgcn_cell8 = cell8_output["pred_spgcn_test"]
y_test_per_seed_cell8 = cell8_output["y_test_per_seed"]

cell8_details = cell8_output["details"]
cell8_config = cell8_output["config"]

results_linear_cell8 = results_baseline_cell8
baseline_rare_cell10 = results_baseline_cell8

In [ ]:
# Cell 9 replacement: legacy comprehensive table, tolerant of missing variables
#
# This replaces the old Cell 9 that expected:
#   results_soma_cell8
#   results_abl_no_super_cell8
#   results_abl_no_reinit_cell8
#   results_abl_no_layers_cell8
#   results_robust_sign_cell8
#
# With the new Cell 8, those variables may not exist.
# This cell will not crash if they are missing.
#
# Required from new Cell 8:
#   results_baseline_cell8
#   results_spgcn_cell8
#
# Optional legacy variables:
#   results_soma_cell8
#   results_abl_no_super_cell8
#   results_abl_no_reinit_cell8
#   results_abl_no_layers_cell8
#   results_robust_sign_cell8
#
# Plain notebook cell. No @app.cell. No top-level underscore names.


def cell9_clean_array(obj):
    """
    Convert common result objects to a finite 1D float array.
    Returns empty array if obj is None or cannot be parsed.
    """
    if obj is None:
        return np.asarray([], dtype=float)

    # If it is a dict of variants, take the first variant.
    if isinstance(obj, dict):
        if len(obj) == 0:
            return np.asarray([], dtype=float)

        first_key = next(iter(obj))
        return cell9_clean_array(obj[first_key])

    # If it is a list of dicts, try common score keys.
    if isinstance(obj, (list, tuple)) and len(obj) > 0 and isinstance(obj[0], dict):
        vals = []
        for item in obj:
            got = None
            for key in ("rare_ba", "test_rare", "rare_balanced_acc", "rare"):
                if key in item:
                    try:
                        got = float(item[key])
                        break
                    except Exception:
                        pass
            if got is None:
                got = float("nan")
            vals.append(got)
        arr = np.asarray(vals, dtype=float)
        return arr[np.isfinite(arr)]

    try:
        arr = np.asarray(obj, dtype=float).ravel()
    except Exception:
        return np.asarray([], dtype=float)

    return arr[np.isfinite(arr)]


def cell9_mean_std(obj):
    arr = cell9_clean_array(obj)

    if arr.size == 0:
        return float("nan"), float("nan"), 0

    if arr.size == 1:
        return float(arr[0]), 0.0, 1

    return (
        float(np.mean(arr)),
        float(np.std(arr, ddof=1)),
        int(arr.size),
    )


def cell9_fmt_number(x, digits=4):
    try:
        xf = float(x)
    except Exception:
        return "n/a"

    if not np.isfinite(xf):
        return "n/a"

    return f"{xf:.{digits}f}"


def cell9_fmt_row(mean_value, std_value, n_value):
    if n_value == 0:
        return "not run", "—", 0

    return (
        cell9_fmt_number(mean_value),
        cell9_fmt_number(std_value),
        int(n_value),
    )


def cell9_paired_test(a_obj, b_obj, label):
    a_arr = cell9_clean_array(a_obj)
    b_arr = cell9_clean_array(b_obj)

    if a_arr.size < 2 or b_arr.size < 2:
        print(f"  {label:48s} skipped: insufficient finite seeds")
        return None

    if a_arr.size != b_arr.size:
        print(
            f"  {label:48s} skipped: unequal seed counts "
            f"n_a={a_arr.size}, n_b={b_arr.size}"
        )
        return None

    try:
        stats_module = stats
    except NameError:
        from scipy import stats as stats_module

    t_stat, p_val = stats_module.ttest_rel(a_arr, b_arr)

    if p_val < 0.001:
        sig = "***"
    elif p_val < 0.01:
        sig = "**"
    elif p_val < 0.05:
        sig = "*"
    else:
        sig = "ns"

    mean_diff = float(np.mean(a_arr - b_arr))

    print(
        f"  {label:48s} "
        f"n={a_arr.size:2d} "
        f"Δ={mean_diff:+.4f} "
        f"t={float(t_stat):+.4f} "
        f"p={float(p_val):.4e} "
        f"{sig}"
    )

    return {
        "label": label,
        "n": int(a_arr.size),
        "mean_diff": mean_diff,
        "t": float(t_stat),
        "p": float(p_val),
        "sig": sig,
    }


def cell9_run_legacy_table():
    print("=" * 100)
    print("CELL 9 REPLACEMENT — LEGACY COMPREHENSIVE TABLE, TOLERANT")
    print("=" * 100)

    # ------------------------------------------------------------------
    # Required new Cell 8 variables
    # ------------------------------------------------------------------
    baseline_obj = globals().get("results_baseline_cell8", None)
    spgcn_obj = globals().get("results_spgcn_cell8", None)

    baseline_arr = cell9_clean_array(baseline_obj)
    spgcn_arr = cell9_clean_array(spgcn_obj)

    if baseline_arr.size == 0:
        raise NameError(
            "[Cell 9][FATAL] results_baseline_cell8 is missing or empty. "
            "Run the new Cell 8 first."
        )

    if spgcn_arr.size == 0:
        raise NameError(
            "[Cell 9][FATAL] results_spgcn_cell8 is missing or empty. "
            "Run the new Cell 8 first."
        )

    # ------------------------------------------------------------------
    # Optional legacy variables
    # ------------------------------------------------------------------
    soma_obj = globals().get("results_soma_cell8", None)
    abl_no_super_obj = globals().get("results_abl_no_super_cell8", None)
    abl_no_reinit_obj = globals().get("results_abl_no_reinit_cell8", None)
    abl_no_layers_obj = globals().get("results_abl_no_layers_cell8", None)
    robust_sign_obj = globals().get("results_robust_sign_cell8", None)

    rows_cell9 = [
        ("GCN + All Features (Baseline)", baseline_obj, True),
        ("GCN + Soma Features Only", soma_obj, False),
        ("SPGCN (Full Signed Prototype)", spgcn_obj, True),
        ("SPGCN − Hierarchical Loss", abl_no_super_obj, False),
        ("SPGCN − Centroid Re-init", abl_no_reinit_obj, False),
        ("SPGCN − Signed Layers", abl_no_layers_obj, False),
    ]

    print(f"{'Method':48s} {'Mean':>10s} {'Std':>10s} {'n':>5s}")
    print("-" * 100)

    row_summaries_cell9 = []

    for label, obj, required in rows_cell9:
        mean_value, std_value, n_value = cell9_mean_std(obj)

        if n_value == 0:
            if required:
                raise NameError(
                    f"[Cell 9][FATAL] Required row '{label}' has no finite values."
                )

            mean_str, std_str, n_out = cell9_fmt_row(float("nan"), float("nan"), 0)
            print(f"{label:48s} {mean_str:>10s} {std_str:>10s} {n_out:5d}")

            row_summaries_cell9.append(
                {
                    "label": label,
                    "mean": float("nan"),
                    "std": float("nan"),
                    "n": 0,
                    "status": "missing_or_not_run",
                }
            )
        else:
            mean_str, std_str, n_out = cell9_fmt_row(mean_value, std_value, n_value)
            print(f"{label:48s} {mean_str:>10s} {std_str:>10s} {n_out:5d}")

            row_summaries_cell9.append(
                {
                    "label": label,
                    "mean": float(mean_value),
                    "std": float(std_value),
                    "n": int(n_value),
                    "status": "ok",
                }
            )

    # ------------------------------------------------------------------
    # Robustness block, optional
    # ------------------------------------------------------------------
    print("-" * 100)
    print(" ROBUSTNESS — sign corruption, if available")
    print("-" * 100)

    robust_summaries_cell9 = {}

    if robust_sign_obj is None:
        print("  results_robust_sign_cell8 not found. Skipping robustness rows.")
    elif not isinstance(robust_sign_obj, dict):
        print("  results_robust_sign_cell8 exists but is not a dict. Skipping robustness rows.")
    else:
        for frac_cell9 in [0.05, 0.10, 0.20]:
            val_cell9 = None

            # Try float key, string key, and formatted key.
            if frac_cell9 in robust_sign_obj:
                val_cell9 = robust_sign_obj[frac_cell9]
            elif str(frac_cell9) in robust_sign_obj:
                val_cell9 = robust_sign_obj[str(frac_cell9)]
            elif f"{frac_cell9:g}" in robust_sign_obj:
                val_cell9 = robust_sign_obj[f"{frac_cell9:g}"]

            arr_cell9 = cell9_clean_array(val_cell9)

            if arr_cell9.size == 0:
                print(f"  {frac_cell9 * 100:4.0f}% Sign Corruption            : not run")
                robust_summaries_cell9[str(frac_cell9)] = {
                    "fraction": frac_cell9,
                    "mean": float("nan"),
                    "std": float("nan"),
                    "n": 0,
                    "status": "missing_or_not_run",
                }
            else:
                mean_value, std_value, n_value = cell9_mean_std(arr_cell9)
                print(
                    f"  {frac_cell9 * 100:4.0f}% Sign Corruption            : "
                    f"{cell9_fmt_number(mean_value)} ± {cell9_fmt_number(std_value)} "
                    f"(n={n_value})"
                )
                robust_summaries_cell9[str(frac_cell9)] = {
                    "fraction": frac_cell9,
                    "mean": float(mean_value),
                    "std": float(std_value),
                    "n": int(n_value),
                    "status": "ok",
                }

    # ------------------------------------------------------------------
    # Statistical significance
    # ------------------------------------------------------------------
    print("=" * 100)
    print(" STATISTICAL SIGNIFICANCE")
    print("=" * 100)

    tests_cell9 = []

    main_test_cell9 = cell9_paired_test(
        spgcn_obj,
        baseline_obj,
        "SPGCN vs GCN baseline",
    )
    if main_test_cell9 is not None:
        tests_cell9.append(main_test_cell9)

    # Optional paired tests for legacy rows.
    optional_test_pairs_cell9 = [
        ("GCN + Soma vs GCN baseline", soma_obj, baseline_obj),
        ("SPGCN − Hierarchical Loss vs SPGCN", abl_no_super_obj, spgcn_obj),
        ("SPGCN − Centroid Re-init vs SPGCN", abl_no_reinit_obj, spgcn_obj),
        ("SPGCN − Signed Layers vs SPGCN", abl_no_layers_obj, spgcn_obj),
    ]

    for label, a_obj, b_obj in optional_test_pairs_cell9:
        test_out = cell9_paired_test(a_obj, b_obj, label)
        if test_out is not None:
            tests_cell9.append(test_out)

    print("=" * 100)

    # ------------------------------------------------------------------
    # Interpretation note
    # ------------------------------------------------------------------
    print("\nNote:")
    print("  - Rows marked 'not run' are expected if you used the new Cell 8.")
    print("  - The new Cell 8 intentionally runs only GCN baseline + SPGCN.")
    print("  - For the final paper table, prefer Cell 12c / Cell 13c.")
    print("  - This Cell 9 replacement is a legacy diagnostic, not the main paper table.")
    print("=" * 100)

    summary_cell9 = {
        "rows": row_summaries_cell9,
        "robustness": robust_summaries_cell9,
        "tests": tests_cell9,
        "notes": {
            "cell9_role": "legacy_tolerant_diagnostic",
            "required_variables": [
                "results_baseline_cell8",
                "results_spgcn_cell8",
            ],
            "optional_variables": [
                "results_soma_cell8",
                "results_abl_no_super_cell8",
                "results_abl_no_reinit_cell8",
                "results_abl_no_layers_cell8",
                "results_robust_sign_cell8",
            ],
        },
    }

    return summary_cell9


cell9_legacy_summary = cell9_run_legacy_table()

# Optional Hugging Face checkpoint.
# If Cell 00 has not been run, this safely skips.
try:
    hf_save_checkpoint(
        "cell09_legacy_table_tolerant",
        {
            "cell9_legacy_summary": cell9_legacy_summary,
        },
        description=(
            "Cell 9 replacement: tolerant legacy comprehensive table for new Cell 8. "
            "Reports GCN baseline and SPGCN; skips missing soma/ablation/robustness rows."
        ),
    )
except NameError:
    print(
        "[ckpt][skip] hf_save_checkpoint not defined. "
        "Run Cell 00 first to enable Hugging Face checkpoints."
    )
except Exception as cell9_ckpt_error:
    print(f"[ckpt][warn] Cell 9 checkpoint failed: {cell9_ckpt_error!r}")

In [ ]:
# # Cell 10: SOTA Baselines (Altiparmak GraphSAGE & Graph Label Propagation)
# import scipy.sparse as sp

# # Pre-requisite: torch_geometric must be installed before kernel start.
# try:
#     from torch_geometric.nn import SAGEConv
# except ImportError:
#     raise ImportError("[Cell 10] torch_geometric is not installed. Please run `pip install torch_geometric` in your terminal and restart the kernel.")

# class GraphSAGEAlt_cell10(nn.Module):
#     def __init__(self, d_in_cell10, d_cell10=64, n_classes_cell10=29, dropout_cell10=0.5):
#         super().__init__()
#         self.conv1_cell10 = SAGEConv(d_in_cell10, d_cell10)
#         self.conv2_cell10 = SAGEConv(d_cell10, d_cell10)
#         self.drop_cell10 = nn.Dropout(dropout_cell10)
#         self.head_cell10 = nn.Linear(d_cell10, n_classes_cell10)

#     def forward(self, x_cell10, edge_index_cell10):
#         x_cell10 = self.drop_cell10(F.relu(self.conv1_cell10(x_cell10, edge_index_cell10)))
#         x_cell10 = self.drop_cell10(F.relu(self.conv2_cell10(x_cell10, edge_index_cell10)))
#         return self.head_cell10(x_cell10)

# def train_altiparmak_cell10(seed_cell10, epochs_cell10=200):
#     tr_cell10, va_cell10, te_cell10 = make_split_cell4(seed_cell10, labeled_mask_cell3, labels_tensor_cell3, C_cell3)
#     ones_cell10 = torch.ones(N_cell3, 1)
#     X_alt_cell10 = torch.cat([ones_cell10, X_tensor_cell3[:, :3]], dim=1).to(DEVICE_cell1)

#     labels_dev_cell10 = labels_tensor_cell3.to(DEVICE_cell1)

#     seed_all_cell1(seed_cell10)
#     model_cell10 = GraphSAGEAlt_cell10(d_in_cell10=4, d_cell10=64, n_classes_cell10=C_cell3).to(DEVICE_cell1)
#     opt_cell10 = torch.optim.Adam(model_cell10.parameters(), lr=1e-2, weight_decay=5e-4)

#     tr_dev_cell10 = tr_cell10.to(DEVICE_cell1)
#     te_dev_cell10 = te_cell10.to(DEVICE_cell1)

#     cnt_cell10 = torch.bincount(labels_dev_cell10[tr_dev_cell10], minlength=C_cell3).float()
#     w_cell10 = (cnt_cell10.sum() / cnt_cell10.clamp(min=1)) ** 0.5
#     w_cell10 = w_cell10 / w_cell10.mean()

#     for epoch_cell10 in range(epochs_cell10):
#         model_cell10.train(); opt_cell10.zero_grad()
#         out_cell10 = model_cell10(X_alt_cell10, ei_full_cell7)
#         ce_cell10 = F.cross_entropy(out_cell10[tr_dev_cell10], labels_dev_cell10[tr_dev_cell10], weight=w_cell10, reduction='none')
#         probs_cell10 = F.softmax(out_cell10[tr_dev_cell10], dim=-1)
#         entropy_cell10 = -(probs_cell10 * torch.log(probs_cell10 + 1e-8)).sum(-1)
#         loss_cell10 = (ce_cell10 + 0.3 * entropy_cell10).mean()
#         loss_cell10.backward()
#         opt_cell10.step()

#     model_cell10.eval()
#     with torch.no_grad():
#         out_cell10 = model_cell10(X_alt_cell10, ei_full_cell7)
#         pred_cell10 = out_cell10[te_dev_cell10].argmax(-1).cpu().numpy()

#     true_cell10 = labels_dev_cell10[te_dev_cell10].cpu().numpy()
#     rare_ba_cell10 = rare_balanced_acc_cell6(true_cell10, pred_cell10, rare_classes_list_cell3)
#     macro_f1_cell10 = f1_score(true_cell10, pred_cell10, labels=list(range(C_cell3)), average='macro', zero_division=0)
#     return rare_ba_cell10, macro_f1_cell10

# print("[Cell 10] Running Altiparmak GraphSAGE (5 seeds)...")
# altiparmak_rare_cell10 = []
# altiparmak_f1_cell10 = []
# for seed_loop_cell10 in range(5):
#     r_cell10, f_cell10 = train_altiparmak_cell10(seed_loop_cell10, epochs_cell10=200)
#     altiparmak_rare_cell10.append(r_cell10)
#     altiparmak_f1_cell10.append(f_cell10)
#     print(f"  Altiparmak seed {seed_loop_cell10}: rare_ba={r_cell10:.4f}")

# # Graph Label Propagation Baseline (Replaces broken/hallucinated NTAC)
# print("[Cell 10] Running Graph Label Propagation (5 seeds)...")
# ntac_rare_cell10 = []
# ntac_f1_cell10 = []

# # Build symmetric adjacency matrix once
# A_sym_cell10 = sp.csr_matrix((np.ones(len(src_arr_cell3), dtype=np.float32), (src_arr_cell3, dst_arr_cell3)), shape=(N_cell3, N_cell3))
# A_sym_cell10 = (A_sym_cell10 + A_sym_cell10.T).tocsr()
# deg_cell10 = np.array(A_sym_cell10.sum(axis=1)).flatten()
# deg_cell10[deg_cell10 == 0] = 1.0
# D_inv_cell10 = sp.diags(1.0 / deg_cell10)
# T_cell10 = D_inv_cell10 @ A_sym_cell10

# for seed_loop_cell10 in range(5):
#     tr_cell10, va_cell10, te_cell10 = make_split_cell4(seed_loop_cell10, labeled_mask_cell3, labels_tensor_cell3, C_cell3)

#     Y_init_cell10 = np.zeros((N_cell3, C_cell3))
#     tr_np_cell10 = tr_cell10.numpy()
#     y_tr_np_cell10 = labels_tensor_cell3[tr_np_cell10].numpy()

#     Y_init_cell10[tr_np_cell10, y_tr_np_cell10] = 1.0
#     Y_cell10 = Y_init_cell10.copy()
#     alpha_cell10 = 0.8

#     # Fast iterative propagation
#     for _ in range(30):
#         Y_cell10 = alpha_cell10 * (T_cell10 @ Y_cell10) + (1 - alpha_cell10) * Y_init_cell10

#     pred_all_cell10 = np.argmax(Y_cell10, axis=1)

#     te_np_cell10 = te_cell10.numpy()
#     test_true_cell10 = labels_tensor_cell3[te_np_cell10].numpy()
#     test_pred_cell10 = pred_all_cell10[te_np_cell10]

#     rare_ba_lp_cell10 = rare_balanced_acc_cell6(test_true_cell10, test_pred_cell10, rare_classes_list_cell3)
#     macro_f1_lp_cell10 = f1_score(test_true_cell10, test_pred_cell10, labels=list(range(C_cell3)), average='macro', zero_division=0)

#     ntac_rare_cell10.append(rare_ba_lp_cell10)
#     ntac_f1_cell10.append(macro_f1_lp_cell10)
#     print(f"  Label Propagation seed {seed_loop_cell10}: rare_ba={rare_ba_lp_cell10:.4f}")

# print("[Cell 10] SOTA Baselines complete.")

In [ ]:
# Cell 10: Graph Label Propagation baseline — 10 seeds
#
# This cell intentionally does NOT run Altiparmak GraphSAGE.
# Altiparmak is handled separately in Cell 11b.
#
# It produces:
#   lp_rare_cell10      = 10-seed rare-class balanced accuracy for label propagation
#   lp_f1_cell10        = 10-seed macro-F1 for label propagation
#   ntac_rare_cell10    = alias of lp_rare_cell10 for downstream compatibility
#   ntac_f1_cell10      = alias of lp_f1_cell10 for downstream compatibility
#
# Required upstream variables:
#   N_cell3
#   C_cell3
#   src_arr_cell3
#   dst_arr_cell3
#   labels_tensor_cell3
#   labeled_mask_cell3
#   rare_classes_list_cell3
#   make_split_cell4
#   rare_balanced_acc_cell6
#   np

def _lp_to_np_cell10(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if hasattr(x, "numpy"):
        x = x.numpy()
    return np.asarray(x)


def _lp_to_np_idx_cell10(x):
    return np.asarray(_lp_to_np_cell10(x), dtype=np.int64).reshape(-1)


def _run_label_propagation_cell10(
    num_seeds_cell10=10,
    alpha_cell10=0.8,
    n_iter_cell10=30,
):
    import scipy.sparse as sp

    required_cell10 = [
        "N_cell3",
        "C_cell3",
        "src_arr_cell3",
        "dst_arr_cell3",
        "labels_tensor_cell3",
        "labeled_mask_cell3",
        "rare_classes_list_cell3",
        "make_split_cell4",
        "rare_balanced_acc_cell6",
        "np",
    ]

    missing_cell10 = [name for name in required_cell10 if name not in globals()]
    if missing_cell10:
        raise NameError(
            f"[Cell 10][FATAL] Missing variables: {missing_cell10}. "
            "Run Cells 1-4 first."
        )

    try:
        _f1_score_cell10 = f1_score
    except NameError:
        from sklearn.metrics import f1_score as _f1_score_cell10

    N_cell10 = int(N_cell3)
    C_cell10 = int(C_cell3)

    src_cell10 = _lp_to_np_idx_cell10(src_arr_cell3)
    dst_cell10 = _lp_to_np_idx_cell10(dst_arr_cell3)

    labels_cell10 = _lp_to_np_idx_cell10(labels_tensor_cell3)
    rare_list_cell10 = [int(x) for x in _lp_to_np_idx_cell10(rare_classes_list_cell3).tolist()]

    print("=" * 90)
    print("CELL 10 — GRAPH LABEL PROPAGATION BASELINE, 10 SEEDS")
    print("=" * 90)
    print(f"[cfg] seeds={list(range(num_seeds_cell10))}")
    print(f"[cfg] alpha={alpha_cell10}")
    print(f"[cfg] propagation iterations={n_iter_cell10}")
    print(f"[data] N={N_cell10:,}, C={C_cell10}, edges={src_cell10.size:,}")
    print(f"[data] rare classes={rare_list_cell10}")
    print("[note] Altiparmak GraphSAGE is intentionally absent; use Cell 11b.")
    print("=" * 90)

    # Build symmetric row-stochastic transition matrix once.
    A_cell10 = sp.csr_matrix(
        (
            np.ones(src_cell10.size, dtype=np.float32),
            (src_cell10, dst_cell10),
        ),
        shape=(N_cell10, C_cell10 if False else N_cell10),
    )

    A_cell10 = (A_cell10 + A_cell10.T).tocsr()
    A_cell10.sum_duplicates()

    deg_cell10 = np.asarray(A_cell10.sum(axis=1)).reshape(-1).astype(np.float64)
    deg_cell10[deg_cell10 == 0.0] = 1.0

    T_cell10 = sp.diags(1.0 / deg_cell10) @ A_cell10

    lp_rare_list_cell10 = []
    lp_f1_list_cell10 = []

    for seed_cell10 in range(num_seeds_cell10):
        tr_cell10, va_cell10, te_cell10 = make_split_cell4(
            seed_cell10,
            labeled_mask_cell3,
            labels_tensor_cell3,
            C_cell3,
        )

        tr_np_cell10 = _lp_to_np_idx_cell10(tr_cell10)
        te_np_cell10 = _lp_to_np_idx_cell10(te_cell10)

        Y_init_cell10 = np.zeros((N_cell10, C_cell10), dtype=np.float32)

        y_tr_cell10 = labels_cell10[tr_np_cell10]
        valid_tr_cell10 = y_tr_cell10 >= 0

        if valid_tr_cell10.any():
            Y_init_cell10[tr_np_cell10[valid_tr_cell10], y_tr_cell10[valid_tr_cell10]] = 1.0

        Y_cell10 = Y_init_cell10.copy()

        for _iter_cell10 in range(int(n_iter_cell10)):
            Y_cell10 = (
                float(alpha_cell10) * np.asarray(T_cell10 @ Y_cell10, dtype=np.float32)
                + (1.0 - float(alpha_cell10)) * Y_init_cell10
            )

        pred_all_cell10 = np.argmax(Y_cell10, axis=1).astype(np.int64)

        true_cell10 = labels_cell10[te_np_cell10]
        pred_cell10 = pred_all_cell10[te_np_cell10]

        valid_test_cell10 = true_cell10 >= 0

        if valid_test_cell10.sum() == 0:
            rare_ba_lp_cell10 = float("nan")
            macro_f1_lp_cell10 = float("nan")
        else:
            true_valid_cell10 = true_cell10[valid_test_cell10]
            pred_valid_cell10 = pred_cell10[valid_test_cell10]

            rare_ba_lp_cell10 = float(
                rare_balanced_acc_cell6(
                    true_valid_cell10,
                    pred_valid_cell10,
                    rare_list_cell10,
                )
            )

            macro_f1_lp_cell10 = float(
                _f1_score_cell10(
                    true_valid_cell10,
                    pred_valid_cell10,
                    labels=list(range(C_cell10)),
                    average="macro",
                    zero_division=0,
                )
            )

        lp_rare_list_cell10.append(rare_ba_lp_cell10)
        lp_f1_list_cell10.append(macro_f1_lp_cell10)

        print(
            f"  Label Propagation seed {seed_cell10}: "
            f"rare_ba={rare_ba_lp_cell10:.4f} | "
            f"macro_f1={macro_f1_lp_cell10:.4f}",
            flush=True,
        )

    return lp_rare_list_cell10, lp_f1_list_cell10


_lp_rare_cell10_raw, _lp_f1_cell10_raw = _run_label_propagation_cell10(
    num_seeds_cell10=10,
    alpha_cell10=0.8,
    n_iter_cell10=30,
)

lp_rare_cell10 = [float(x) for x in _lp_rare_cell10_raw]
lp_f1_cell10 = [float(x) for x in _lp_f1_cell10_raw]

# Compatibility aliases for older table cells.
# These are label propagation, not NTAC.
ntac_rare_cell10 = lp_rare_cell10
ntac_f1_cell10 = lp_f1_cell10

print("\n" + "=" * 90)
print("CELL 10 FINAL SUMMARY")
print("=" * 90)

_lp_arr_cell10 = np.asarray(lp_rare_cell10, dtype=float)
_lp_fin_cell10 = _lp_arr_cell10[np.isfinite(_lp_arr_cell10)]

if _lp_fin_cell10.size > 0:
    print(
        f"Label propagation rare_ba: "
        f"{float(np.mean(_lp_fin_cell10)):.4f} +/- "
        f"{float(np.std(_lp_fin_cell10, ddof=1)) if _lp_fin_cell10.size > 1 else 0.0:.4f} "
        f"(n={int(_lp_fin_cell10.size)})"
    )
else:
    print("Label propagation rare_ba: no finite values")

print(f"len(lp_rare_cell10) = {len(lp_rare_cell10)}")
print("[note] ntac_rare_cell10 is an alias for label propagation, not real NTAC.")
print("=" * 90)

In [ ]:
# Checkpoint: Cell 10 label propagation
#
# What Cell 10 does:
#   Runs the 10-seed connectivity-only graph label propagation baseline.
#   This is not NTAC and not Altiparmak.

hf_save_checkpoint(
    "cell10_label_propagation",
    {
        "lp_rare_cell10": lp_rare_cell10,
        "lp_f1_cell10": lp_f1_cell10,
        "ntac_rare_cell10": ntac_rare_cell10,
        "ntac_f1_cell10": ntac_f1_cell10,
    },
    description=(
        "Cell 10: 10-seed graph label propagation baseline. "
        "ntac_rare_cell10 is only a compatibility alias for label propagation."
    ),
)

In [ ]:
# Cell 11: SSPR (Signed Spectral Prototype Refinement)
# from scipy.sparse import csr_matrix
# from sklearn.decomposition import TruncatedSVD

def refine_prototypes_cell11(model_cell11, X_cell11, ei_pos_cell11, ew_pos_cell11, dis_pos_cell11, ei_neg_cell11, ew_neg_cell11, dis_neg_cell11,
                             train_idx_cell11, unlabeled_idx_cell11, labels_cell11, C_cell11, lambda_w_cell11=0.3, tau_cell11=1.0):
    model_cell11.eval()
    with torch.no_grad():
        # FIX: Ensure X is explicitly on the same device as the model
        X_dev_cell11 = X_cell11.to(next(model_cell11.parameters()).device)
        h_cell11 = model_cell11(X_dev_cell11, ei_pos_cell11, ew_pos_cell11, dis_pos_cell11, ei_neg_cell11, ew_neg_cell11, dis_neg_cell11)
        h_train_cell11 = h_cell11[train_idx_cell11]
        h_unlab_cell11 = h_cell11[unlabeled_idx_cell11]
        y_train_cell11 = labels_cell11[train_idx_cell11]

        for c_loop_cell11 in range(C_cell11):
            mask_c_cell11 = (y_train_cell11 == c_loop_cell11)
            n_c_cell11 = int(mask_c_cell11.sum())
            if n_c_cell11 == 0 or n_c_cell11 >= 100: continue

            p_c_cell11 = model_cell11.proto_class.data[c_loop_cell11]
            dist_unlab_cell11 = torch.cdist(h_unlab_cell11, p_c_cell11.unsqueeze(0)).squeeze(-1)

            # THE FIX: Normalize similarity to sum to 1, then use convex combination
            sim_unlab_cell11 = torch.exp(-dist_unlab_cell11 / tau_cell11)
            sim_norm_cell11 = sim_unlab_cell11 / (sim_unlab_cell11.sum() + 1e-8)
            unlab_centroid_cell11 = (sim_norm_cell11.unsqueeze(-1) * h_unlab_cell11).sum(0)
            labeled_centroid_cell11 = h_train_cell11[mask_c_cell11].mean(0)

            p_refined_cell11 = (labeled_centroid_cell11 + lambda_w_cell11 * unlab_centroid_cell11) / (1.0 + lambda_w_cell11)
            model_cell11.proto_class.data[c_loop_cell11] = p_refined_cell11
    return model_cell11

print("[Cell 11] Computing signed spectral & motif features (This will take 30-90 minutes)...")
row_pos_cell11 = src_arr_cell3[sign_bin_arr_cell3 > 0.5]
col_pos_cell11 = dst_arr_cell3[sign_bin_arr_cell3 > 0.5]
row_neg_cell11 = src_arr_cell3[sign_bin_arr_cell3 <= 0.5]
col_neg_cell11 = dst_arr_cell3[sign_bin_arr_cell3 <= 0.5]

data_pos_cell11 = np.ones(len(row_pos_cell11), dtype=np.float32)
data_neg_cell11 = -np.ones(len(row_neg_cell11), dtype=np.float32)

row_all_cell11 = np.concatenate([row_pos_cell11, row_neg_cell11])
col_all_cell11 = np.concatenate([col_pos_cell11, col_neg_cell11])
data_all_cell11 = np.concatenate([data_pos_cell11, data_neg_cell11])

A_signed_cell11 = csr_matrix((data_all_cell11, (row_all_cell11, col_all_cell11)), shape=(N_cell3, N_cell3))
svd_cell11 = TruncatedSVD(n_components=32, random_state=42)
signed_emb_cell11 = svd_cell11.fit_transform(A_signed_cell11)
signed_emb_cell11 = (signed_emb_cell11 - signed_emb_cell11.mean(0)) / (signed_emb_cell11.std(0) + 1e-8)
signed_emb_cell11 = np.nan_to_num(signed_emb_cell11, nan=0.0)

A_abs_cell11 = csr_matrix((np.ones(2 * len(src_arr_cell3), dtype=np.float32), (np.concatenate([src_arr_cell3, dst_arr_cell3]), np.concatenate([dst_arr_cell3, src_arr_cell3]))), shape=(N_cell3, N_cell3))

# SLOW BLOCK: A^3 computation left as-is per instructions
A3_signed_cell11 = A_signed_cell11.dot(A_signed_cell11).dot(A_signed_cell11)
A3_abs_cell11 = A_abs_cell11.dot(A_abs_cell11).dot(A_abs_cell11)

diag_s_cell11 = np.asarray(A3_signed_cell11.diagonal()).flatten()
diag_a_cell11 = np.asarray(A3_abs_cell11.diagonal()).flatten()
t_bal_cell11 = (diag_a_cell11 + diag_s_cell11) / 12.0
t_fru_cell11 = (diag_a_cell11 - diag_s_cell11) / 12.0

motif_cell11 = np.stack([np.log1p(np.clip(t_bal_cell11, 0, None)), np.log1p(np.clip(t_fru_cell11, 0, None))], axis=1)
motif_cell11 = np.nan_to_num(motif_cell11, nan=0.0)

X_sspr_cell11 = np.concatenate([X_raw_cell3[:, :11], spec_emb_cell3, signed_emb_cell11, motif_cell11], axis=1)
# FIX: Move tensor to DEVICE_cell1 immediately to prevent CPU/GPU mismatch in refinement
X_sspr_tensor_cell11 = torch.tensor((X_sspr_cell11 - X_sspr_cell11.mean(0)) / (X_sspr_cell11.std(0) + 1e-8), dtype=torch.float32).to(DEVICE_cell1)
print(f"[Cell 11] SSPR feature matrix: {X_sspr_tensor_cell11.shape}")

print("[Cell 11] Running SSPR on 10 seeds...")
sspr_rare_cell11 = []
sspr_f1_cell11 = []
for seed_loop_cell11 in range(10):
    tr_cell11, va_cell11, te_cell11 = make_split_cell4(seed_loop_cell11, labeled_mask_cell3, labels_tensor_cell3, C_cell3)
    m_cell11 = SPGCN_cell5(d_features_cell5=X_sspr_tensor_cell11.shape[1], d_cell5=64, d_proto_cell5=32, n_layers_cell5=2, n_classes_cell5=C_cell3, n_super_cell5=C_super_cell3, temperature_cell5=0.1, dropout_cell5=0.3)

    r_cell11 = train_spgcn_cell6(m_cell11, X_sspr_tensor_cell11, ei_pos_cell7, ew_pos_cell7, dis_pos_cell7, ei_neg_cell7, ew_neg_cell7, dis_neg_cell7, tr_cell11, va_cell11, te_cell11, labels_tensor_cell3, super_labels_tensor_cell3, rare_classes_list_cell3, seed_loop_cell11, epochs_cell6=200, C_cell6=C_cell3, C_super_cell6=C_super_cell3, use_hierarchical_cell6=False, use_reinit_cell6=True)

    unlabeled_cell11 = torch.where(~labeled_mask_cell3)[0]
    m_cell11 = refine_prototypes_cell11(m_cell11, X_sspr_tensor_cell11, ei_pos_cell7, ew_pos_cell7, dis_pos_cell7, ei_neg_cell7, ew_neg_cell7, dis_neg_cell7, tr_cell11, unlabeled_cell11, labels_tensor_cell3, C_cell3)

    m_cell11.eval()
    with torch.no_grad():
        h_eval_cell11 = m_cell11(X_sspr_tensor_cell11, ei_pos_cell7, ew_pos_cell7, dis_pos_cell7, ei_neg_cell7, ew_neg_cell7, dis_neg_cell7)
        dist_eval_cell11 = torch.cdist(h_eval_cell11, m_cell11.proto_class)
        pred_eval_cell11 = dist_eval_cell11.argmin(dim=-1).cpu().numpy()

    test_true_cell11 = labels_tensor_cell3[te_cell11].numpy()
    test_pred_cell11 = pred_eval_cell11[te_cell11.numpy()]
    rare_ba_cell11 = rare_balanced_acc_cell6(test_true_cell11, test_pred_cell11, rare_classes_list_cell3)
    macro_f1_cell11 = f1_score(test_true_cell11, test_pred_cell11, labels=list(range(C_cell3)), average='macro', zero_division=0)

    sspr_rare_cell11.append(rare_ba_cell11)
    sspr_f1_cell11.append(macro_f1_cell11)
    print(f"  SSPR seed {seed_loop_cell11}: before_refine={r_cell11['test_rare']:.4f}  after_refine={rare_ba_cell11:.4f}")

print("[Cell 11] SSPR Innovation complete.")

In [ ]:
# Checkpoint: Cell 11 SSPR features and legacy SSPR outputs
#
# What this saves:
#   - X_sspr_tensor_cell11: signed spectral + motif feature matrix
#   - sspr_before_cell11, sspr_rare_cell11, sspr_f1_cell11 if they exist
#
# Requires Cell 00 to have been run for Hugging Face upload.
# If Cell 00 was not run, it safely skips upload.

cell11_checkpoint_payload = {
    "X_sspr_tensor_cell11": globals().get("X_sspr_tensor_cell11", None),
    "sspr_before_cell11": globals().get("sspr_before_cell11", None),
    "sspr_rare_cell11": globals().get("sspr_rare_cell11", None),
    "sspr_f1_cell11": globals().get("sspr_f1_cell11", None),
}

if cell11_checkpoint_payload["X_sspr_tensor_cell11"] is None:
    print(
        "[ckpt][warn] X_sspr_tensor_cell11 not found. "
        "Saving only legacy SSPR variables if present."
    )

try:
    hf_save_checkpoint(
        "cell11_sspr_features",
        cell11_checkpoint_payload,
        description=(
            "Cell 11: SSPR signed spectral + motif feature matrix "
            "and legacy SSPR variables if present."
        ),
        upload=True,
    )
except NameError:
    print(
        "[ckpt][skip] hf_save_checkpoint not defined. "
        "Run Cell 00 first to enable Hugging Face checkpoints."
    )
except Exception as cell11_ckpt_error:
    print(f"[ckpt][warn] Cell 11 checkpoint failed: {cell11_ckpt_error!r}")

In [ ]:
# Cell 11b: Altiparmak Baseline (GraphSAGE + Sparse GraphSMOTE + ME Loss)
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from scipy.spatial.distance import pdist, squareform
# import numpy as np
 
ALT_HIDDEN_CELL11B = 16
ALT_EPOCHS_CELL11B = 500
ALT_PORTION_CELL11B = 6.0
ALT_ME_BALANCING_CELL11B = 0.3
ALT_LR_CELL11B = 0.01
ALT_WD_CELL11B = 5e-4
ALT_SEEDS_CELL11B = list(range(10))
ALT_SYMMETRIZE_CELL11B = True      # original notebook made the graph undirected
ALT_MAX_REPS_CELL11B = 200         # cap on synthetic copies per original node (balanced variant)
 
ALT_VARIANTS_CELL11B = [
    {"name": "faithful_portion6",  "target": None, "log1p": True},
    {"name": "balanced_target500", "target": 500,  "log1p": True},
    # {"name": "ablation_raw_minmax", "target": None, "log1p": False},
]
 
 
def _to_np_cell11b(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if hasattr(x, "numpy"):
        x = x.numpy()
    return np.asarray(x)
 
 
def _seed_cell11b(seed):
    try:
        seed_all_cell1(seed)
    except NameError:
        torch.manual_seed(seed)
        np.random.seed(seed)
 
 
def _prep_features_cell11b(x_raw, use_log1p):
    """
    Min-max normalise the 3 soma features.
    use_log1p=True first applies log1p so heavy tails don't squash everything near 0.
    Nodes with all-zero features (missing stats) are excluded from min/max.
    """
    x = np.nan_to_num(np.asarray(x_raw, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    if use_log1p:
        x = np.log1p(np.clip(x, 0.0, None))
    valid = (x != 0).any(axis=1)
    ref = x[valid] if valid.any() else x
    mn = ref.min(axis=0)
    mx = ref.max(axis=0)
    denom = np.where((mx - mn) == 0.0, 1.0, mx - mn)
    out = np.clip((x - mn) / denom, 0.0, 1.0)
    out[~valid] = 0.0
    return torch.tensor(out, dtype=torch.float32)
 
 
def _prepare_graph_cell11b(edge_index, n_nodes, symmetrize):
    """
    Returns (edge_index_for_training, csr_offsets, csr_neighbors).
    The CSR is over the UNDIRECTED neighbourhood (in + out), which is what the
    original dense_adj-based GraphSMOTE used for the synthetic nodes' connectivity.
    """
    ei = edge_index.detach().cpu().long()
    src = ei[0].numpy().astype(np.int64, copy=False)
    dst = ei[1].numpy().astype(np.int64, copy=False)
 
    a = np.concatenate([src, dst])
    b = np.concatenate([dst, src])
    key = np.unique(a * np.int64(n_nodes) + b)      # sorted by (a, b), de-duplicated
    a = key // n_nodes
    b = key % n_nodes
 
    counts = np.bincount(a, minlength=n_nodes).astype(np.int64)
    offsets = np.zeros(n_nodes + 1, dtype=np.int64)
    offsets[1:] = np.cumsum(counts)
 
    if symmetrize:
        ei_out = torch.from_numpy(np.stack([a, b], axis=0)).long()
    else:
        ei_out = ei
    return ei_out, offsets, b
 
 
def _graph_smote_cell11b(
    ei, feats, labels, train_idx, offsets, nbrs,
    minority_classes, portion, target, max_reps,
):
    """
    Sparse GraphSMOTE.
      - synthetic = x_i + alpha * (x_nn - x_i), nn = nearest same-class TRAIN node (feature space)
      - synthetic node connects to union of neighbours of node i and its nn (both directions)
      - target=None  -> `portion` synthetic copies per node (original recipe)
      - target=int   -> enough copies per node to lift the class to ~target train nodes
    Returns (edge_index, features, labels, train_idx) with synthetic nodes appended
    after the original ones and included in train_idx.
    """
    N = int(feats.shape[0])
    new_x, new_y, src_l, dst_l = [], [], [], []
    next_id = N
 
    for c in minority_classes:
        chosen = train_idx[labels[train_idx] == int(c)]
        m = int(chosen.numel())
        if m == 0:
            continue
 
        if target is None:
            reps = int(portion)
        else:
            reps = int(min(max_reps, np.ceil(max(0, target - m) / m)))
        if reps <= 0:
            continue
 
        emb = feats[chosen]
        if m == 1:
            nn_local = torch.zeros(1, dtype=torch.long)
        else:
            d = torch.cdist(emb, emb)
            d.fill_diagonal_(float("inf"))
            nn_local = d.argmin(dim=1)
 
        for i in range(m):
            j = int(nn_local[i])
            o = int(chosen[i])
            n = int(chosen[j])
 
            alpha = torch.rand(reps, 1)
            new_x.append(emb[i] + alpha * (emb[j] - emb[i]))
            new_y.append(torch.full((reps,), int(c), dtype=torch.long))
 
            nb = np.union1d(nbrs[offsets[o]:offsets[o + 1]], nbrs[offsets[n]:offsets[n + 1]])
            if nb.size > 0:
                ids = np.arange(next_id, next_id + reps, dtype=np.int64)
                src_l.append(np.repeat(ids, nb.size))
                dst_l.append(np.tile(nb, reps))
            next_id += reps
 
    if len(new_x) == 0:
        return ei, feats, labels, train_idx
 
    X_new = torch.cat(new_x, dim=0)
    y_new = torch.cat(new_y, dim=0)
    X_all = torch.cat([feats, X_new], dim=0)
    y_all = torch.cat([labels, y_new], dim=0)
    tr_all = torch.cat([train_idx, torch.arange(N, next_id, dtype=torch.long)], dim=0)
 
    if len(src_l) > 0:
        s = torch.from_numpy(np.concatenate(src_l)).long()
        t = torch.from_numpy(np.concatenate(dst_l)).long()
        new_ei = torch.stack([s, t], dim=0)
        ei_all = torch.cat([ei, new_ei, new_ei.flip(0)], dim=1)
    else:
        ei_all = ei
    return ei_all, X_all, y_all, tr_all
 
 
class ME_cell11b(nn.Module):
    """Mixed Entropy loss exactly as in the original: CE - bf * mean(P log P)."""
 
    def __init__(self, balancing_factor=ALT_ME_BALANCING_CELL11B):
        super(ME_cell11b, self).__init__()
        self.nll_loss = nn.NLLLoss()
        self.balancing_factor = balancing_factor
 
    def forward(self, yHat, y):
        ce = self.nll_loss(F.log_softmax(yHat, dim=1), y)
        P = F.softmax(yHat, dim=1)
        reg = torch.sum(P * torch.log(P + 1e-10)) / (float(yHat.shape[0]) * float(yHat.shape[1]))
        return ce - self.balancing_factor * reg
 
 
class AltiparmakSAGE_cell11b(nn.Module):
    """SAGEConv(in->16) -> ReLU -> Dropout(0.5) -> SAGEConv(16->C), as in the original."""
 
    def __init__(self, in_channels, hidden_channels, num_classes):
        super(AltiparmakSAGE_cell11b, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, num_classes)
 
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, training=self.training, p=0.5)
        return self.conv2(x, edge_index)
 
 
def _run_seed_cell11b(seed, variant, X_feat, ei_base, offsets, nbrs, labels_cpu, rare_arr):
    _seed_cell11b(seed)
 
    tr, va, te = make_split_cell4(seed, labeled_mask_cell3, labels_tensor_cell3, C_cell3)
    tr = torch.as_tensor(_to_np_cell11b(tr), dtype=torch.long)
    te = torch.as_tensor(_to_np_cell11b(te), dtype=torch.long)
 
    tr_labels_np = labels_cpu[tr.numpy()].numpy()
    present = set(int(c) for c in np.unique(tr_labels_np))
    minority = [int(c) for c in rare_classes_list_cell3 if int(c) in present]
 
    ei_s, X_s, y_s, tr_s = _graph_smote_cell11b(
        ei_base, X_feat, labels_cpu, tr, offsets, nbrs,
        minority_classes=minority,
        portion=ALT_PORTION_CELL11B,
        target=variant["target"],
        max_reps=ALT_MAX_REPS_CELL11B,
    )
 
    n_syn = int(X_s.shape[0] - X_feat.shape[0])
    print(
        f"  [smote] {len(minority)} rare classes | +{n_syn:,} synthetic nodes | "
        f"edges {int(ei_base.shape[1]):,} -> {int(ei_s.shape[1]):,} | "
        f"train {int(tr.numel()):,} -> {int(tr_s.numel()):,}",
        flush=True,
    )
 
    model = AltiparmakSAGE_cell11b(int(X_s.shape[1]), ALT_HIDDEN_CELL11B, int(C_cell3)).to(DEVICE_cell1)
    criterion = ME_cell11b(ALT_ME_BALANCING_CELL11B)
    opt = torch.optim.Adam(model.parameters(), lr=ALT_LR_CELL11B, weight_decay=ALT_WD_CELL11B)
 
    X_d = X_s.to(DEVICE_cell1)
    y_d = y_s.to(DEVICE_cell1)
    ei_d = ei_s.to(DEVICE_cell1)
    tr_d = tr_s.to(DEVICE_cell1)
 
    for epoch in range(ALT_EPOCHS_CELL11B):
        model.train()
        opt.zero_grad()
        out = model(X_d, ei_d)
        loss = criterion(out[tr_d], y_d[tr_d])
        loss.backward()
        opt.step()
        if (epoch + 1) % 100 == 0:
            print(f"    epoch {epoch + 1:03d}/{ALT_EPOCHS_CELL11B}: loss={loss.item():.4f}", flush=True)
 
    model.eval()
    with torch.no_grad():
        pred_all = model(X_d, ei_d).argmax(dim=1).cpu().numpy()
 
    te_np = te.numpy()
    true_te = labels_cpu[te_np].numpy().astype(np.int64)
    pred_te = pred_all[te_np].astype(np.int64)
 
    rare_ba = float(rare_balanced_acc_cell6(true_te, pred_te, rare_classes_list_cell3))
    acc = float((pred_te == true_te).mean())
 
    # recall on the ORIGINAL rare train nodes: tells us under-fitting vs. generalisation gap
    tr_np = tr.numpy()
    tr_true = labels_cpu[tr_np].numpy().astype(np.int64)
    m_rare = np.isin(tr_true, rare_arr)
    train_rare_recall = float((pred_all[tr_np][m_rare] == tr_true[m_rare]).mean()) if m_rare.any() else float("nan")
 
    counts = np.bincount(pred_te, minlength=int(C_cell3))
    n_active = int((counts > 0).sum())
    n_rare_pred = int(counts[rare_arr].sum())
 
    # number of rare classes with non-zero test recall
    n_rare_hit = 0
    for c in rare_arr:
        mk = true_te == c
        if mk.any() and (pred_te[mk] == c).any():
            n_rare_hit += 1
 
    print(
        f"  [result] rare_ba={rare_ba:.4f} | acc={acc:.4f} | train_rare_recall={train_rare_recall:.4f} | "
        f"active_classes={n_active} | rare_preds={n_rare_pred} | rare_classes_hit={n_rare_hit}",
        flush=True,
    )
 
    del model, criterion, opt, X_d, y_d, ei_d, tr_d
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
    return {
        "rare_ba": rare_ba,
        "acc": acc,
        "train_rare_recall": train_rare_recall,
        "n_active": n_active,
        "n_rare_pred": n_rare_pred,
        "n_rare_hit": n_rare_hit,
    }
 
 
# ----------------------------------------------------------------------------
# One-off setup (identical for every seed / variant)
# ----------------------------------------------------------------------------
try:
    _X_raw_cell11b = X_raw_cell3[:, :3]
except NameError:
    _X_raw_cell11b = X_tensor_cell3[:, :3]
_X_raw_np_cell11b = _to_np_cell11b(_X_raw_cell11b)
 
_labels_cpu_cell11b = torch.as_tensor(_to_np_cell11b(labels_tensor_cell3), dtype=torch.long)
_N_cell11b = int(_X_raw_np_cell11b.shape[0])
_rare_arr_cell11b = np.asarray(rare_classes_list_cell3, dtype=np.int64)
 
_ei_base_cell11b, _offsets_cell11b, _nbrs_cell11b = _prepare_graph_cell11b(
    ei_full_cell7, _N_cell11b, ALT_SYMMETRIZE_CELL11B
)
 
print("=" * 78)
print("CELL 11b -- ALTIPARMAK-STYLE BASELINE (rewritten)")
print("GraphSAGE + Sparse GraphSMOTE + ME Loss")
print("=" * 78)
print(f"[cfg] hidden={ALT_HIDDEN_CELL11B} epochs={ALT_EPOCHS_CELL11B} lr={ALT_LR_CELL11B} wd={ALT_WD_CELL11B}")
print(f"[cfg] ME balancing_factor={ALT_ME_BALANCING_CELL11B} | portion={ALT_PORTION_CELL11B}")
print(f"[cfg] symmetrize graph={ALT_SYMMETRIZE_CELL11B} | seeds={ALT_SEEDS_CELL11B}")
print(f"[cfg] variants={[v['name'] for v in ALT_VARIANTS_CELL11B]}")
print(f"[graph] nodes={_N_cell11b:,} | edges used for training={int(_ei_base_cell11b.shape[1]):,}")
 
# Feature diagnostic: shows how badly raw min-max compresses the inputs
print("\n[features] percentiles (p50 / p90 / p99) of the 3 normalised soma features")
for _flag in (False, True):
    _Xd = _prep_features_cell11b(_X_raw_np_cell11b, _flag).numpy()
    _p = np.percentile(_Xd, [50, 90, 99], axis=0)
    print(f"  log1p={_flag!s:5}: " + " | ".join(
        f"f{k}: {_p[0, k]:.4f}/{_p[1, k]:.4f}/{_p[2, k]:.4f}" for k in range(_Xd.shape[1])
    ))
 
_lab_np_cell11b = _labels_cpu_cell11b.numpy()
_lab_counts_cell11b = np.bincount(_lab_np_cell11b[_lab_np_cell11b >= 0], minlength=int(C_cell3))
print("[labels] total labelled nodes per rare class: "
      + ", ".join(f"{int(c)}:{int(_lab_counts_cell11b[c])}" for c in _rare_arr_cell11b))
print("=" * 78)
 
# ----------------------------------------------------------------------------
# Run all variants x seeds
# ----------------------------------------------------------------------------
results_alt_cell11b = {}
 
for _variant_cell11b in ALT_VARIANTS_CELL11B:
    print("\n" + "#" * 78)
    print(f"# VARIANT: {_variant_cell11b['name']}  "
          f"(target={_variant_cell11b['target']}, log1p={_variant_cell11b['log1p']})")
    print("#" * 78)
 
    _X_feat_cell11b = _prep_features_cell11b(_X_raw_np_cell11b, _variant_cell11b["log1p"])
    _runs_cell11b = []
 
    for _seed_loop_cell11b in ALT_SEEDS_CELL11B:
        print(f"\n> {_variant_cell11b['name']} | seed {_seed_loop_cell11b}", flush=True)
        _runs_cell11b.append(
            _run_seed_cell11b(
                _seed_loop_cell11b, _variant_cell11b, _X_feat_cell11b,
                _ei_base_cell11b, _offsets_cell11b, _nbrs_cell11b,
                _labels_cpu_cell11b, _rare_arr_cell11b,
            )
        )
 
    results_alt_cell11b[_variant_cell11b["name"]] = _runs_cell11b
 
# ----------------------------------------------------------------------------
# Summary
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("CELL 11b FINAL RESULT")
print("=" * 78)
for _name_cell11b, _runs_cell11b in results_alt_cell11b.items():
    def _ms_cell11b(key):
        v = np.asarray([r[key] for r in _runs_cell11b], dtype=np.float64)
        return float(np.nanmean(v)), float(np.nanstd(v))
 
    _m, _s = _ms_cell11b("rare_ba")
    _a, _as = _ms_cell11b("acc")
    _t, _ts = _ms_cell11b("train_rare_recall")
    _rp, _ = _ms_cell11b("n_rare_pred")
    _rh, _ = _ms_cell11b("n_rare_hit")
    print(f"{_name_cell11b:22s} rare_ba = {_m:.4f} +/- {_s:.4f} | acc = {_a:.4f} +/- {_as:.4f} | "
          f"train_rare_recall = {_t:.4f} | mean rare_preds = {_rp:.0f} | mean rare classes hit = {_rh:.1f}")
 
# Backwards-compatible headline numbers (faithful variant first in the list)
_first_cell11b = ALT_VARIANTS_CELL11B[0]["name"]
alt_mean_cell11b = float(np.mean([r["rare_ba"] for r in results_alt_cell11b[_first_cell11b]]))
alt_std_cell11b = float(np.std([r["rare_ba"] for r in results_alt_cell11b[_first_cell11b]]))
print("=" * 78)
 



In [ ]:
# Checkpoint: Cell 11b Altiparmak-style baseline
#
# What Cell 11b does:
#   Runs the Altiparmak-style GraphSAGE + GraphSMOTE + Mixed Entropy baseline.
#   Stores both faithful_portion6 and balanced_target500 variants over 10 seeds.

hf_save_checkpoint(
    "cell11b_altiparmak_baseline",
    {
        "results_alt_cell11b": results_alt_cell11b,
        "ALT_HIDDEN_CELL11B": ALT_HIDDEN_CELL11B,
        "ALT_EPOCHS_CELL11B": ALT_EPOCHS_CELL11B,
        "ALT_PORTION_CELL11B": ALT_PORTION_CELL11B,
        "ALT_ME_BALANCING_CELL11B": ALT_ME_BALANCING_CELL11B,
        "ALT_LR_CELL11B": ALT_LR_CELL11B,
        "ALT_WD_CELL11B": ALT_WD_CELL11B,
        "ALT_SEEDS_CELL11B": ALT_SEEDS_CELL11B,
        "ALT_SYMMETRIZE_CELL11B": ALT_SYMMETRIZE_CELL11B,
        "ALT_MAX_REPS_CELL11B": ALT_MAX_REPS_CELL11B,
        "ALT_VARIANTS_CELL11B": ALT_VARIANTS_CELL11B,
    },
    description=(
        "Cell 11b: Altiparmak-style GraphSAGE + GraphSMOTE + ME baseline, "
        "faithful_portion6 and balanced_target500, 10 seeds."
    ),
)

In [ ]:
# import sys, subprocess, importlib

# Check 1: does the kernel see ntac?
try:
    import ntac
    print(f"[kernel] ntac found: {ntac.__file__}")
except ImportError as e:
    print(f"[kernel] ntac NOT found: {e}")

print(f"[kernel] sys.executable = {sys.executable}")

# Check 2: does the subprocess python see ntac?
r = subprocess.run(
    [sys.executable, "-c", "import ntac; print(ntac.__file__)"],
    capture_output=True, text=True, timeout=20,
)
print(f"[subprocess] returncode = {r.returncode}")
print(f"[subprocess] stdout = {r.stdout.strip()}")
print(f"[subprocess] stderr = {r.stderr.strip()}")

In [ ]:
# Cell 11c: SSPR before/after refinement ablation, 10 seeds, with prediction saving.
#
# Complete rewrite.
#
# Rules enforced:
#   - No imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every local/helper/loop variable is prefixed with cell11c_.
#   - Saves per-seed SSPR after-refinement test predictions and test labels.
#
# Requires X_sspr_tensor_cell11 from Cell 11.
#
# Produces global variables:
#   sspr_before_cell11c
#   sspr_after_cell11c
#   sspr_f1_cell11c
#   sspr_seeds_cell11c
#   sspr_details_cell11c
#   sspr_before_is_proxy_cell11c
#   sspr_config_cell11c
#   pred_sspr_cell11c
#   y_test_per_seed_cell11c


def cell11c_main():
    cell11c_globals = globals()
    cell11c_np = cell11c_globals.get("np", None)
    cell11c_torch = cell11c_globals.get("torch", None)

    if cell11c_np is None:
        raise NameError("[Cell 11c][FATAL] np is not available. Run earlier cells first.")

    if cell11c_torch is None:
        raise NameError("[Cell 11c][FATAL] torch is not available. Run earlier cells first.")

    cell11c_required_names = [
        "DEVICE_cell1",
        "X_sspr_tensor_cell11",
        "labels_tensor_cell3",
        "labeled_mask_cell3",
        "super_labels_tensor_cell3",
        "C_cell3",
        "C_super_cell3",
        "rare_classes_list_cell3",
        "ei_pos_cell7",
        "ew_pos_cell7",
        "dis_pos_cell7",
        "ei_neg_cell7",
        "ew_neg_cell7",
        "dis_neg_cell7",
        "make_split_cell4",
        "SPGCN_cell5",
        "train_spgcn_cell6",
        "rare_balanced_acc_cell6",
    ]

    cell11c_missing_names = []
    for cell11c_required_name in cell11c_required_names:
        if cell11c_required_name not in cell11c_globals:
            cell11c_missing_names.append(cell11c_required_name)

    if len(cell11c_missing_names) > 0:
        raise NameError(
            "[Cell 11c][FATAL] Missing upstream variables: "
            + str(cell11c_missing_names)
            + ". Run Cell 11 and earlier cells first."
        )

    cell11c_device = cell11c_globals["DEVICE_cell1"]
    cell11c_num_classes = int(cell11c_globals["C_cell3"])
    cell11c_num_super_classes = int(cell11c_globals["C_super_cell3"])
    cell11c_seed_list = list(range(10))
    cell11c_epoch_count = 200
    cell11c_lambda_refine = 0.3
    cell11c_tau_refine = 1.0

    def cell11c_to_numpy(cell11c_object):
        if cell11c_object is None:
            return None

        cell11c_working_object = cell11c_object

        try:
            if hasattr(cell11c_working_object, "detach"):
                cell11c_working_object = cell11c_working_object.detach()

            if hasattr(cell11c_working_object, "cpu"):
                cell11c_working_object = cell11c_working_object.cpu()

            if hasattr(cell11c_working_object, "numpy"):
                cell11c_working_object = cell11c_working_object.numpy()

            return cell11c_np.asarray(cell11c_working_object)
        except Exception:
            return None

    def cell11c_to_int_vector(cell11c_object, cell11c_description):
        cell11c_array_object = cell11c_to_numpy(cell11c_object)

        if cell11c_array_object is None:
            raise TypeError(
                "[Cell 11c][FATAL] Could not convert "
                + str(cell11c_description)
                + " to numpy."
            )

        try:
            return cell11c_array_object.astype(cell11c_np.int64).reshape(-1)
        except Exception as cell11c_cast_error:
            raise TypeError(
                "[Cell 11c][FATAL] Could not cast "
                + str(cell11c_description)
                + " to int64: "
                + repr(cell11c_cast_error)
            )

    cell11c_rare_classes_array = cell11c_to_numpy(cell11c_globals["rare_classes_list_cell3"])

    if cell11c_rare_classes_array is None:
        raise TypeError("[Cell 11c][FATAL] rare_classes_list_cell3 could not be converted to numpy.")

    cell11c_rare_classes = []
    cell11c_seen_rare_classes = set()

    for cell11c_rare_value in cell11c_rare_classes_array.ravel().tolist():
        cell11c_rare_integer = int(cell11c_rare_value)

        if cell11c_rare_integer not in cell11c_seen_rare_classes:
            cell11c_seen_rare_classes.add(cell11c_rare_integer)
            cell11c_rare_classes.append(cell11c_rare_integer)

    cell11c_labels_array = cell11c_to_numpy(cell11c_globals["labels_tensor_cell3"])

    if cell11c_labels_array is None:
        raise TypeError("[Cell 11c][FATAL] labels_tensor_cell3 could not be converted to numpy.")

    cell11c_labels_np = cell11c_labels_array.astype(cell11c_np.int64).reshape(-1)

    cell11c_x_sspr = cell11c_globals["X_sspr_tensor_cell11"]

    if not cell11c_torch.is_tensor(cell11c_x_sspr):
        cell11c_x_array = cell11c_to_numpy(cell11c_x_sspr)

        if cell11c_x_array is None:
            raise TypeError("[Cell 11c][FATAL] X_sspr_tensor_cell11 is not a tensor and could not be converted.")

        cell11c_x_sspr = cell11c_torch.tensor(
            cell11c_x_array.astype(cell11c_np.float32),
            dtype=cell11c_torch.float32,
        )

    cell11c_x_sspr = cell11c_x_sspr.to(cell11c_device)

    if int(cell11c_x_sspr.ndim) != 2:
        raise RuntimeError(
            "[Cell 11c][FATAL] X_sspr_tensor_cell11 must be 2D, got shape "
            + str(tuple(cell11c_x_sspr.shape))
        )

    cell11c_feature_dim = int(cell11c_x_sspr.shape[1])

    def cell11c_set_seed(cell11c_seed_value):
        cell11c_seed_function = cell11c_globals.get("seed_all_cell1", None)

        if callable(cell11c_seed_function):
            cell11c_seed_function(int(cell11c_seed_value))
        else:
            cell11c_torch.manual_seed(int(cell11c_seed_value))
            cell11c_np.random.seed(int(cell11c_seed_value))

    def cell11c_move_tensor(cell11c_tensor_object, cell11c_target_device):
        if cell11c_torch.is_tensor(cell11c_tensor_object):
            return cell11c_tensor_object.to(cell11c_target_device)

        return cell11c_tensor_object

    def cell11c_manual_macro_f1(cell11c_true_vector, cell11c_pred_vector, cell11c_class_count):
        cell11c_true_array = cell11c_np.asarray(cell11c_true_vector, dtype=cell11c_np.int64).reshape(-1)
        cell11c_pred_array = cell11c_np.asarray(cell11c_pred_vector, dtype=cell11c_np.int64).reshape(-1)
        cell11c_f1_values = []

        for cell11c_class_value in range(int(cell11c_class_count)):
            cell11c_true_mask = cell11c_true_array == cell11c_class_value
            cell11c_pred_mask = cell11c_pred_array == cell11c_class_value

            cell11c_true_positive_count = int(cell11c_np.sum(cell11c_true_mask & cell11c_pred_mask))
            cell11c_false_positive_count = int(cell11c_np.sum((~cell11c_true_mask) & cell11c_pred_mask))
            cell11c_false_negative_count = int(cell11c_np.sum(cell11c_true_mask & (~cell11c_pred_mask)))

            if cell11c_true_positive_count + cell11c_false_positive_count == 0:
                cell11c_precision_value = 0.0
            else:
                cell11c_precision_value = float(cell11c_true_positive_count) / float(
                    cell11c_true_positive_count + cell11c_false_positive_count
                )

            if cell11c_true_positive_count + cell11c_false_negative_count == 0:
                cell11c_recall_value = 0.0
            else:
                cell11c_recall_value = float(cell11c_true_positive_count) / float(
                    cell11c_true_positive_count + cell11c_false_negative_count
                )

            if cell11c_precision_value + cell11c_recall_value == 0.0:
                cell11c_f1_value = 0.0
            else:
                cell11c_f1_value = (
                    2.0
                    * cell11c_precision_value
                    * cell11c_recall_value
                    / (cell11c_precision_value + cell11c_recall_value)
                )

            cell11c_f1_values.append(float(cell11c_f1_value))

        if len(cell11c_f1_values) == 0:
            return float("nan")

        return float(cell11c_np.mean(cell11c_f1_values))

    def cell11c_pick_metric(cell11c_result_object, cell11c_key_list, cell11c_default_value):
        if isinstance(cell11c_result_object, dict):
            for cell11c_key_value in cell11c_key_list:
                if cell11c_key_value in cell11c_result_object:
                    try:
                        return float(cell11c_result_object[cell11c_key_value])
                    except Exception:
                        pass

        for cell11c_key_value in cell11c_key_list:
            try:
                cell11c_attribute_value = getattr(cell11c_result_object, cell11c_key_value)
                return float(cell11c_attribute_value)
            except Exception:
                pass

        return float(cell11c_default_value)

    def cell11c_make_spgcn(cell11c_input_feature_dim):
        try:
            return cell11c_globals["SPGCN_cell5"](
                d_features_cell5=int(cell11c_input_feature_dim),
                d_cell5=64,
                d_proto_cell5=32,
                n_layers_cell5=2,
                n_classes_cell5=cell11c_num_classes,
                n_super_cell5=cell11c_num_super_classes,
                temperature_cell5=0.1,
                dropout_cell5=0.3,
            )
        except TypeError:
            return cell11c_globals["SPGCN_cell5"](
                d_features=int(cell11c_input_feature_dim),
                d=64,
                d_proto=32,
                n_layers=2,
                n_classes=cell11c_num_classes,
                n_super=cell11c_num_super_classes,
                temperature=0.1,
                dropout=0.3,
            )

    def cell11c_collect_tensors(cell11c_output_object, cell11c_tensor_list=None):
        if cell11c_tensor_list is None:
            cell11c_tensor_list = []

        if cell11c_torch.is_tensor(cell11c_output_object):
            if int(cell11c_output_object.ndim) >= 2:
                cell11c_tensor_list.append(cell11c_output_object)
            return cell11c_tensor_list

        if isinstance(cell11c_output_object, dict):
            for cell11c_dict_value in cell11c_output_object.values():
                cell11c_collect_tensors(cell11c_dict_value, cell11c_tensor_list)
            return cell11c_tensor_list

        if isinstance(cell11c_output_object, (tuple, list)):
            for cell11c_sequence_value in cell11c_output_object:
                cell11c_collect_tensors(cell11c_sequence_value, cell11c_tensor_list)
            return cell11c_tensor_list

        return cell11c_tensor_list

    def cell11c_extract_embedding(cell11c_output_object, cell11c_model_object):
        cell11c_candidate_tensors = cell11c_collect_tensors(cell11c_output_object)

        if len(cell11c_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 11c][FATAL] Model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell11c_output_object))
            )

        if cell11c_model_object is not None and hasattr(cell11c_model_object, "proto_class"):
            cell11c_proto_parameter = getattr(cell11c_model_object, "proto_class")

            if cell11c_torch.is_tensor(cell11c_proto_parameter) and int(cell11c_proto_parameter.ndim) == 2:
                cell11c_proto_dimension = int(cell11c_proto_parameter.shape[1])

                for cell11c_candidate_tensor in cell11c_candidate_tensors:
                    if int(cell11c_candidate_tensor.shape[1]) == cell11c_proto_dimension:
                        return cell11c_candidate_tensor

        return cell11c_candidate_tensors[0]

    def cell11c_output_to_prediction(cell11c_output_object, cell11c_model_object=None):
        cell11c_candidate_tensors = cell11c_collect_tensors(cell11c_output_object)

        if len(cell11c_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 11c][FATAL] Model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell11c_output_object))
            )

        if cell11c_model_object is not None and hasattr(cell11c_model_object, "proto_class"):
            cell11c_proto_parameter = getattr(cell11c_model_object, "proto_class")

            if cell11c_torch.is_tensor(cell11c_proto_parameter) and int(cell11c_proto_parameter.ndim) == 2:
                cell11c_proto_float = cell11c_proto_parameter.detach().float()
                cell11c_proto_dimension = int(cell11c_proto_float.shape[1])

                for cell11c_candidate_tensor in cell11c_candidate_tensors:
                    if int(cell11c_candidate_tensor.shape[1]) == cell11c_proto_dimension:
                        cell11c_distance_matrix = cell11c_torch.cdist(
                            cell11c_candidate_tensor.detach().float(),
                            cell11c_proto_float,
                        )
                        return (
                            cell11c_distance_matrix
                            .argmin(dim=1)
                            .detach()
                            .cpu()
                            .numpy()
                            .astype(cell11c_np.int64)
                        )

        for cell11c_candidate_tensor in cell11c_candidate_tensors:
            if int(cell11c_candidate_tensor.shape[1]) == cell11c_num_classes:
                return (
                    cell11c_candidate_tensor
                    .argmax(dim=1)
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(cell11c_np.int64)
                )

        return (
            cell11c_candidate_tensors[0]
            .argmax(dim=-1)
            .detach()
            .cpu()
            .numpy()
            .astype(cell11c_np.int64)
        )

    def cell11c_forward_spgcn(
        cell11c_model_object,
        cell11c_x_tensor,
        cell11c_ei_pos_tensor,
        cell11c_ew_pos_tensor,
        cell11c_dis_pos_tensor,
        cell11c_ei_neg_tensor,
        cell11c_ew_neg_tensor,
        cell11c_dis_neg_tensor,
    ):
        cell11c_last_type_error = None

        try:
            return cell11c_model_object(
                cell11c_x_tensor,
                cell11c_ei_pos_tensor,
                cell11c_ew_pos_tensor,
                cell11c_dis_pos_tensor,
                cell11c_ei_neg_tensor,
                cell11c_ew_neg_tensor,
                cell11c_dis_neg_tensor,
            )
        except TypeError as cell11c_type_error:
            cell11c_last_type_error = cell11c_type_error

        try:
            return cell11c_model_object(
                cell11c_x_tensor,
                cell11c_ei_pos_tensor,
                cell11c_dis_pos_tensor,
                cell11c_ei_neg_tensor,
                cell11c_dis_neg_tensor,
            )
        except TypeError as cell11c_type_error:
            cell11c_last_type_error = cell11c_type_error

        try:
            return cell11c_model_object(
                cell11c_x_tensor,
                cell11c_ei_pos_tensor,
                cell11c_ei_neg_tensor,
            )
        except TypeError as cell11c_type_error:
            cell11c_last_type_error = cell11c_type_error

        raise TypeError(
            "[Cell 11c][FATAL] Could not call SPGCN model with any known signature. "
            "Last TypeError: " + repr(cell11c_last_type_error)
        )

    def cell11c_predict_spgcn(
        cell11c_model_object,
        cell11c_x_tensor,
        cell11c_ei_pos_tensor,
        cell11c_ew_pos_tensor,
        cell11c_dis_pos_tensor,
        cell11c_ei_neg_tensor,
        cell11c_ew_neg_tensor,
        cell11c_dis_neg_tensor,
    ):
        cell11c_model_object.eval()
        cell11c_model_device = next(cell11c_model_object.parameters()).device

        cell11c_x_device = cell11c_x_tensor.to(cell11c_model_device)
        cell11c_ei_pos_device = cell11c_move_tensor(cell11c_ei_pos_tensor, cell11c_model_device)
        cell11c_ew_pos_device = cell11c_move_tensor(cell11c_ew_pos_tensor, cell11c_model_device)
        cell11c_dis_pos_device = cell11c_move_tensor(cell11c_dis_pos_tensor, cell11c_model_device)
        cell11c_ei_neg_device = cell11c_move_tensor(cell11c_ei_neg_tensor, cell11c_model_device)
        cell11c_ew_neg_device = cell11c_move_tensor(cell11c_ew_neg_tensor, cell11c_model_device)
        cell11c_dis_neg_device = cell11c_move_tensor(cell11c_dis_neg_tensor, cell11c_model_device)

        with cell11c_torch.no_grad():
            cell11c_output_object = cell11c_forward_spgcn(
                cell11c_model_object,
                cell11c_x_device,
                cell11c_ei_pos_device,
                cell11c_ew_pos_device,
                cell11c_dis_pos_device,
                cell11c_ei_neg_device,
                cell11c_ew_neg_device,
                cell11c_dis_neg_device,
            )

        return cell11c_output_to_prediction(cell11c_output_object, cell11c_model_object)

    def cell11c_refine_prototypes(
        cell11c_model_object,
        cell11c_x_tensor,
        cell11c_ei_pos_tensor,
        cell11c_ew_pos_tensor,
        cell11c_dis_pos_tensor,
        cell11c_ei_neg_tensor,
        cell11c_ew_neg_tensor,
        cell11c_dis_neg_tensor,
        cell11c_train_index,
        cell11c_unlabeled_index,
        cell11c_class_count,
    ):
        cell11c_model_object.eval()
        cell11c_model_device = next(cell11c_model_object.parameters()).device

        cell11c_x_device = cell11c_x_tensor.to(cell11c_model_device)
        cell11c_ei_pos_device = cell11c_move_tensor(cell11c_ei_pos_tensor, cell11c_model_device)
        cell11c_ew_pos_device = cell11c_move_tensor(cell11c_ew_pos_tensor, cell11c_model_device)
        cell11c_dis_pos_device = cell11c_move_tensor(cell11c_dis_pos_tensor, cell11c_model_device)
        cell11c_ei_neg_device = cell11c_move_tensor(cell11c_ei_neg_tensor, cell11c_model_device)
        cell11c_ew_neg_device = cell11c_move_tensor(cell11c_ew_neg_tensor, cell11c_model_device)
        cell11c_dis_neg_device = cell11c_move_tensor(cell11c_dis_neg_tensor, cell11c_model_device)

        cell11c_train_index_tensor = cell11c_torch.as_tensor(
            cell11c_to_int_vector(cell11c_train_index, "Cell 11c train index"),
            dtype=cell11c_torch.long,
            device=cell11c_model_device,
        )

        cell11c_unlabeled_index_tensor = cell11c_torch.as_tensor(
            cell11c_to_int_vector(cell11c_unlabeled_index, "Cell 11c unlabeled index"),
            dtype=cell11c_torch.long,
            device=cell11c_model_device,
        )

        cell11c_labels_tensor = cell11c_torch.as_tensor(
            cell11c_labels_np,
            dtype=cell11c_torch.long,
            device=cell11c_model_device,
        )

        if int(cell11c_unlabeled_index_tensor.numel()) == 0:
            return cell11c_model_object, 0

        with cell11c_torch.no_grad():
            cell11c_raw_output = cell11c_forward_spgcn(
                cell11c_model_object,
                cell11c_x_device,
                cell11c_ei_pos_device,
                cell11c_ew_pos_device,
                cell11c_dis_pos_device,
                cell11c_ei_neg_device,
                cell11c_ew_neg_device,
                cell11c_dis_neg_device,
            )

            cell11c_embedding_tensor = cell11c_extract_embedding(cell11c_raw_output, cell11c_model_object)

            cell11c_train_embedding = cell11c_embedding_tensor[cell11c_train_index_tensor]
            cell11c_unlabeled_embedding = cell11c_embedding_tensor[cell11c_unlabeled_index_tensor]
            cell11c_train_labels = cell11c_labels_tensor[cell11c_train_index_tensor]

            cell11c_refined_count = 0

            for cell11c_class_index in range(int(cell11c_class_count)):
                cell11c_class_mask = cell11c_train_labels == cell11c_class_index
                cell11c_class_support_count = int(cell11c_np.sum(cell11c_class_mask.detach().cpu().numpy()))

                if cell11c_class_support_count == 0 or cell11c_class_support_count >= 100:
                    continue

                cell11c_proto_parameter = cell11c_model_object.proto_class.data[cell11c_class_index]

                cell11c_unlabeled_distance = cell11c_torch.cdist(
                    cell11c_unlabeled_embedding,
                    cell11c_proto_parameter.unsqueeze(0),
                ).squeeze(-1)

                cell11c_similarity = cell11c_torch.exp(
                    -cell11c_unlabeled_distance / float(cell11c_tau_refine)
                )

                cell11c_similarity_sum = cell11c_similarity.sum() + 1e-8
                cell11c_similarity_normalized = cell11c_similarity / cell11c_similarity_sum

                cell11c_unlabeled_centroid = (
                    cell11c_similarity_normalized.unsqueeze(-1)
                    * cell11c_unlabeled_embedding
                ).sum(0)

                cell11c_labeled_centroid = cell11c_train_embedding[cell11c_class_mask].mean(0)

                cell11c_refined_proto = (
                    cell11c_labeled_centroid
                    + float(cell11c_lambda_refine) * cell11c_unlabeled_centroid
                ) / (1.0 + float(cell11c_lambda_refine))

                cell11c_model_object.proto_class.data[cell11c_class_index] = cell11c_refined_proto
                cell11c_refined_count = cell11c_refined_count + 1

        return cell11c_model_object, cell11c_refined_count

    def cell11c_metrics_from_test_predictions(
        cell11c_true_vector,
        cell11c_pred_vector,
        cell11c_rare_class_list,
        cell11c_class_count,
    ):
        cell11c_true_array = cell11c_np.asarray(cell11c_true_vector, dtype=cell11c_np.int64).reshape(-1)
        cell11c_pred_array = cell11c_np.asarray(cell11c_pred_vector, dtype=cell11c_np.int64).reshape(-1)

        cell11c_valid_mask = (
            (cell11c_true_array >= 0)
            & (cell11c_pred_array >= 0)
            & (cell11c_true_array < cell11c_class_count)
            & (cell11c_pred_array < cell11c_class_count)
        )

        cell11c_true_valid = cell11c_true_array[cell11c_valid_mask]
        cell11c_pred_valid = cell11c_pred_array[cell11c_valid_mask]

        if cell11c_true_valid.size == 0:
            return {
                "rare_ba": float("nan"),
                "macro_f1": float("nan"),
                "accuracy": float("nan"),
                "n_test": 0,
                "n_active_predicted_classes": 0,
                "n_rare_predictions": 0,
            }

        cell11c_rare_function = cell11c_globals.get("rare_balanced_acc_cell6", None)

        if callable(cell11c_rare_function):
            cell11c_rare_ba_value = float(
                cell11c_rare_function(
                    cell11c_true_valid,
                    cell11c_pred_valid,
                    cell11c_rare_class_list,
                )
            )
        else:
            cell11c_rare_recalls = []

            for cell11c_rare_class_value in cell11c_rare_class_list:
                cell11c_support_mask = cell11c_true_valid == cell11c_rare_class_value
                cell11c_support_count = int(cell11c_np.sum(cell11c_support_mask))

                if cell11c_support_count > 0:
                    cell11c_hit_mask = cell11c_support_mask & (cell11c_pred_valid == cell11c_rare_class_value)
                    cell11c_rare_recalls.append(float(cell11c_np.sum(cell11c_hit_mask)) / float(cell11c_support_count))

            if len(cell11c_rare_recalls) == 0:
                cell11c_rare_ba_value = float("nan")
            else:
                cell11c_rare_ba_value = float(cell11c_np.mean(cell11c_rare_recalls))

        cell11c_macro_f1_value = cell11c_manual_macro_f1(
            cell11c_true_valid,
            cell11c_pred_valid,
            cell11c_class_count,
        )

        cell11c_accuracy_value = float(cell11c_np.mean(cell11c_pred_valid == cell11c_true_valid))

        cell11c_pred_counts = cell11c_np.bincount(cell11c_pred_valid, minlength=cell11c_class_count)
        cell11c_active_class_count = int(cell11c_np.sum(cell11c_pred_counts > 0))

        cell11c_rare_array = cell11c_np.asarray(cell11c_rare_class_list, dtype=cell11c_np.int64)

        if (
            cell11c_rare_array.size > 0
            and int(cell11c_rare_array.max()) < int(cell11c_pred_counts.size)
        ):
            cell11c_rare_prediction_count = int(cell11c_np.sum(cell11c_pred_counts[cell11c_rare_array]))
        else:
            cell11c_rare_prediction_count = 0

        return {
            "rare_ba": cell11c_rare_ba_value,
            "macro_f1": cell11c_macro_f1_value,
            "accuracy": cell11c_accuracy_value,
            "n_test": int(cell11c_true_valid.size),
            "n_active_predicted_classes": cell11c_active_class_count,
            "n_rare_predictions": cell11c_rare_prediction_count,
        }

    def cell11c_fmt_metric(cell11c_numeric_value):
        try:
            cell11c_float_value = float(cell11c_numeric_value)
        except Exception:
            return "n/a"

        if not cell11c_np.isfinite(cell11c_float_value):
            return "n/a"

        return f"{cell11c_float_value:.4f}"

    print("=" * 100)
    print("CELL 11c — SSPR BEFORE/AFTER ABLATION WITH PREDICTION SAVING")
    print("=" * 100)
    print("[cfg] seeds=" + str(cell11c_seed_list))
    print("[cfg] epochs=" + str(cell11c_epoch_count))
    print("[cfg] refinement lambda=" + str(cell11c_lambda_refine))
    print("[cfg] refinement tau=" + str(cell11c_tau_refine))
    print("[data] X_sspr shape=" + str(tuple(cell11c_x_sspr.shape)))
    print("[data] C=" + str(cell11c_num_classes) + ", C_super=" + str(cell11c_num_super_classes))
    print("[data] rare classes=" + str(cell11c_rare_classes))
    print("[fix] Saves pred_sspr_cell11c and y_test_per_seed_cell11c.")
    print("=" * 100)

    cell11c_before_list = []
    cell11c_after_list = []
    cell11c_f1_list = []
    cell11c_seed_output_list = []
    cell11c_details_list = []
    cell11c_pred_list = []
    cell11c_y_test_list = []

    for cell11c_seed_value in cell11c_seed_list:
        print("\n>>> Cell 11c seed " + str(cell11c_seed_value), flush=True)

        cell11c_train_index, cell11c_val_index, cell11c_test_index = cell11c_globals["make_split_cell4"](
            cell11c_seed_value,
            cell11c_globals["labeled_mask_cell3"],
            cell11c_globals["labels_tensor_cell3"],
            cell11c_globals["C_cell3"],
        )

        cell11c_test_np = cell11c_to_int_vector(cell11c_test_index, "Cell 11c test index")
        cell11c_y_test_np = cell11c_labels_np[cell11c_test_np].copy()
        cell11c_y_test_list.append(cell11c_y_test_np)

        cell11c_set_seed(cell11c_seed_value)

        cell11c_model_object = cell11c_make_spgcn(cell11c_feature_dim).to(cell11c_device)

        cell11c_train_result = cell11c_globals["train_spgcn_cell6"](
            cell11c_model_object,
            cell11c_x_sspr,
            cell11c_globals["ei_pos_cell7"],
            cell11c_globals["ew_pos_cell7"],
            cell11c_globals["dis_pos_cell7"],
            cell11c_globals["ei_neg_cell7"],
            cell11c_globals["ew_neg_cell7"],
            cell11c_globals["dis_neg_cell7"],
            cell11c_train_index,
            cell11c_val_index,
            cell11c_test_index,
            cell11c_globals["labels_tensor_cell3"],
            cell11c_globals["super_labels_tensor_cell3"],
            cell11c_globals["rare_classes_list_cell3"],
            cell11c_seed_value,
            epochs_cell6=cell11c_epoch_count,
            C_cell6=cell11c_num_classes,
            C_super_cell6=cell11c_num_super_classes,
            use_hierarchical_cell6=False,
            use_reinit_cell6=True,
        )

        cell11c_pred_before_full = cell11c_predict_spgcn(
            cell11c_model_object,
            cell11c_x_sspr,
            cell11c_globals["ei_pos_cell7"],
            cell11c_globals["ew_pos_cell7"],
            cell11c_globals["dis_pos_cell7"],
            cell11c_globals["ei_neg_cell7"],
            cell11c_globals["ew_neg_cell7"],
            cell11c_globals["dis_neg_cell7"],
        )

        cell11c_pred_before_test = cell11c_pred_before_full[cell11c_test_np].copy()

        cell11c_metrics_before = cell11c_metrics_from_test_predictions(
            cell11c_y_test_np,
            cell11c_pred_before_test,
            cell11c_rare_classes,
            cell11c_num_classes,
        )

        cell11c_returned_before_rare = cell11c_pick_metric(
            cell11c_train_result,
            ["test_rare", "rare_ba", "rare_balanced_acc", "rare"],
            float("nan"),
        )

        if cell11c_np.isfinite(cell11c_returned_before_rare):
            cell11c_before_rare_value = float(cell11c_returned_before_rare)
        else:
            cell11c_before_rare_value = float(cell11c_metrics_before["rare_ba"])

        cell11c_before_list.append(cell11c_before_rare_value)

        print(
            "  [before_refine] rare_ba="
            + cell11c_fmt_metric(cell11c_before_rare_value)
            + " | computed_rare_ba="
            + cell11c_fmt_metric(cell11c_metrics_before["rare_ba"]),
            flush=True,
        )

        cell11c_unlabeled_tensor = cell11c_torch.where(~cell11c_globals["labeled_mask_cell3"])[0]
        cell11c_unlabeled_np = cell11c_to_int_vector(cell11c_unlabeled_tensor, "Cell 11c unlabeled index")

        cell11c_overlap_array = cell11c_np.intersect1d(
            cell11c_test_np,
            cell11c_unlabeled_np,
            assume_unique=False,
        )

        if int(cell11c_overlap_array.size) != 0:
            raise RuntimeError(
                "[Cell 11c][FATAL] Test nodes leaked into unlabeled refinement set: "
                + str(int(cell11c_overlap_array.size))
                + " overlapping nodes."
            )

        cell11c_model_object, cell11c_refined_count = cell11c_refine_prototypes(
            cell11c_model_object,
            cell11c_x_sspr,
            cell11c_globals["ei_pos_cell7"],
            cell11c_globals["ew_pos_cell7"],
            cell11c_globals["dis_pos_cell7"],
            cell11c_globals["ei_neg_cell7"],
            cell11c_globals["ew_neg_cell7"],
            cell11c_globals["dis_neg_cell7"],
            cell11c_train_index,
            cell11c_unlabeled_tensor,
            cell11c_num_classes,
        )

        cell11c_pred_after_full = cell11c_predict_spgcn(
            cell11c_model_object,
            cell11c_x_sspr,
            cell11c_globals["ei_pos_cell7"],
            cell11c_globals["ew_pos_cell7"],
            cell11c_globals["dis_pos_cell7"],
            cell11c_globals["ei_neg_cell7"],
            cell11c_globals["ew_neg_cell7"],
            cell11c_globals["dis_neg_cell7"],
        )

        cell11c_pred_after_test = cell11c_pred_after_full[cell11c_test_np].copy()
        cell11c_pred_list.append(cell11c_pred_after_test)

        cell11c_metrics_after = cell11c_metrics_from_test_predictions(
            cell11c_y_test_np,
            cell11c_pred_after_test,
            cell11c_rare_classes,
            cell11c_num_classes,
        )

        cell11c_after_rare_value = float(cell11c_metrics_after["rare_ba"])
        cell11c_after_f1_value = float(cell11c_metrics_after["macro_f1"])

        cell11c_after_list.append(cell11c_after_rare_value)
        cell11c_f1_list.append(cell11c_after_f1_value)
        cell11c_seed_output_list.append(int(cell11c_seed_value))

        cell11c_seed_details = {
            "seed": int(cell11c_seed_value),
            "returned_before_rare_ba": float(cell11c_returned_before_rare),
            "computed_before_rare_ba": float(cell11c_metrics_before["rare_ba"]),
            "used_before_rare_ba": float(cell11c_before_rare_value),
            "after_rare_ba": float(cell11c_after_rare_value),
            "after_macro_f1": float(cell11c_after_f1_value),
            "after_accuracy": float(cell11c_metrics_after["accuracy"]),
            "n_refined_prototypes": int(cell11c_refined_count),
            "n_active_predicted_classes": int(cell11c_metrics_after["n_active_predicted_classes"]),
            "n_rare_predictions": int(cell11c_metrics_after["n_rare_predictions"]),
            "delta_after_minus_before": float(cell11c_after_rare_value - cell11c_before_rare_value),
        }

        cell11c_details_list.append(cell11c_seed_details)

        print(
            "  [after_refine]  rare_ba="
            + cell11c_fmt_metric(cell11c_after_rare_value)
            + " | delta="
            + cell11c_fmt_metric(cell11c_after_rare_value - cell11c_before_rare_value)
            + " | macro_f1="
            + cell11c_fmt_metric(cell11c_after_f1_value)
            + " | refined="
            + str(cell11c_refined_count)
            + " | active_classes="
            + str(cell11c_metrics_after["n_active_predicted_classes"])
            + " | rare_preds="
            + str(cell11c_metrics_after["n_rare_predictions"]),
            flush=True,
        )

        del cell11c_model_object
        del cell11c_pred_before_full
        del cell11c_pred_after_full

        if cell11c_torch.cuda.is_available():
            cell11c_torch.cuda.empty_cache()

    cell11c_before_array = cell11c_np.asarray(cell11c_before_list, dtype=float)
    cell11c_after_array = cell11c_np.asarray(cell11c_after_list, dtype=float)
    cell11c_delta_array = cell11c_after_array - cell11c_before_array

    cell11c_config = {
        "seeds": cell11c_seed_output_list,
        "epochs": int(cell11c_epoch_count),
        "lambda_refine": float(cell11c_lambda_refine),
        "tau_refine": float(cell11c_tau_refine),
        "use_hierarchical": False,
        "use_reinit": True,
        "d_features": int(cell11c_feature_dim),
        "C": int(cell11c_num_classes),
        "C_super": int(cell11c_num_super_classes),
        "rare_classes": cell11c_rare_classes,
        "prediction_storage": "after-refinement test-only integer prediction vectors per seed",
    }

    print("\n" + "=" * 100)
    print("CELL 11c FINAL SUMMARY")
    print("=" * 100)
    print(
        "SSPR before refinement: "
        + cell11c_fmt_metric(cell11c_np.mean(cell11c_before_array))
        + " +/- "
        + cell11c_fmt_metric(cell11c_np.std(cell11c_before_array, ddof=1) if cell11c_before_array.size > 1 else 0.0)
    )
    print(
        "SSPR after refinement:  "
        + cell11c_fmt_metric(cell11c_np.mean(cell11c_after_array))
        + " +/- "
        + cell11c_fmt_metric(cell11c_np.std(cell11c_after_array, ddof=1) if cell11c_after_array.size > 1 else 0.0)
    )
    print(
        "Refinement delta:       "
        + cell11c_fmt_metric(cell11c_np.mean(cell11c_delta_array))
        + " +/- "
        + cell11c_fmt_metric(cell11c_np.std(cell11c_delta_array, ddof=1) if cell11c_delta_array.size > 1 else 0.0)
    )
    print("Positive delta seeds:   " + str(int(cell11c_np.sum(cell11c_delta_array > 0))) + "/" + str(int(cell11c_delta_array.size)))
    print("len(pred_sspr_cell11c)  = " + str(len(cell11c_pred_list)))
    print("len(y_test_per_seed_cell11c) = " + str(len(cell11c_y_test_list)))
    print("=" * 100)

    cell11c_hf_save_checkpoint = cell11c_globals.get("hf_save_checkpoint", None)

    if callable(cell11c_hf_save_checkpoint):
        try:
            cell11c_hf_save_checkpoint(
                "cell11c_sspr_ablation_with_predictions",
                {
                    "sspr_before_cell11c": cell11c_before_list,
                    "sspr_after_cell11c": cell11c_after_list,
                    "sspr_f1_cell11c": cell11c_f1_list,
                    "sspr_seeds_cell11c": cell11c_seed_output_list,
                    "sspr_details_cell11c": cell11c_details_list,
                    "sspr_before_is_proxy_cell11c": False,
                    "sspr_config_cell11c": cell11c_config,
                    "pred_sspr_cell11c": cell11c_pred_list,
                    "y_test_per_seed_cell11c": cell11c_y_test_list,
                },
                description=(
                    "Cell 11c: true 10-seed SSPR before/after refinement ablation "
                    "with saved after-refinement test predictions and test labels."
                ),
                upload=True,
            )
        except Exception as cell11c_checkpoint_error:
            print("[ckpt][warn] Cell 11c checkpoint failed: " + repr(cell11c_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    return {
        "before": cell11c_before_list,
        "after": cell11c_after_list,
        "f1": cell11c_f1_list,
        "seeds": cell11c_seed_output_list,
        "details": cell11c_details_list,
        "config": cell11c_config,
        "pred_after_test": cell11c_pred_list,
        "y_test_per_seed": cell11c_y_test_list,
    }


cell11c_output = cell11c_main()

sspr_before_cell11c = cell11c_output["before"]
sspr_after_cell11c = cell11c_output["after"]
sspr_f1_cell11c = cell11c_output["f1"]
sspr_seeds_cell11c = cell11c_output["seeds"]
sspr_details_cell11c = cell11c_output["details"]
sspr_config_cell11c = cell11c_output["config"]
sspr_before_is_proxy_cell11c = False

pred_sspr_cell11c = cell11c_output["pred_after_test"]
y_test_per_seed_cell11c = cell11c_output["y_test_per_seed"]

In [ ]:
# Checkpoint: Cell 11c SSPR ablation
#
# What Cell 11c does:
#   Runs the final 10-seed SSPR ablation with independently measured:
#     - before refinement
#     - after refinement
#     - macro-F1
#     - per-seed diagnostics
#   This is the true refinement ablation cell.

hf_save_checkpoint(
    "cell11c_sspr_ablation",
    {
        "sspr_before_cell11c": sspr_before_cell11c,
        "sspr_after_cell11c": sspr_after_cell11c,
        "sspr_f1_cell11c": sspr_f1_cell11c,
        "sspr_seeds_cell11c": sspr_seeds_cell11c,
        "sspr_details_cell11c": sspr_details_cell11c,
        "sspr_before_is_proxy_cell11c": sspr_before_is_proxy_cell11c,
        "sspr_config_cell11c": sspr_config_cell11c,
    },
    description=(
        "Cell 11c: true 10-seed SSPR before/after prototype refinement ablation."
    ),
)

In [ ]:
# Cell 12: NTAC scaling test — 5000 / 10000 / 20000 nodes, sequential isolated workers.
# No new imports; use modules already loaded by earlier cells.

# --------------------------------------------------------------------
# Required upstream variables
# --------------------------------------------------------------------
# Cell 12: NTAC Subgraph Smoke Test (fixed: no object arrays in npz)
# import sys
# import os
import json
import time
import gc
# import subprocess
import traceback
# from pathlib import Path
# from collections import Counter

# import numpy as np
# import scipy.sparse as sp
_required_cell12 = [
    "src_arr_cell3", "dst_arr_cell3", "N_cell3",
    "labels_tensor_cell3", "labeled_mask_cell3", "C_cell3",
    "rare_classes_list_cell3", "make_split_cell4",
]
_missing_cell12 = [x for x in _required_cell12 if x not in globals()]
if _missing_cell12:
    raise NameError(f"[Cell 12][FATAL] Missing: {_missing_cell12}. Run Cells 3–4 first.")

# --------------------------------------------------------------------
# Config
# --------------------------------------------------------------------
NODE_SIZES_CELL12   = [5000, 10000, 20000,25000]
STEPS_CELL12        = 3
TIMEOUT_SEC_CELL12  = 900
MEM_MB_CELL12       = 8192      # 8 GB per worker. Safe on 32 GB box.
SEED_CELL12         = 12

print("=" * 78)
print("CELL 12 — NTAC SCALING TEST")
print("=" * 78)
print(f"[cfg] sizes={NODE_SIZES_CELL12}")
print(f"[cfg] steps={STEPS_CELL12}, timeout={TIMEOUT_SEC_CELL12}s, mem={MEM_MB_CELL12} MB/worker")

gc.collect()

# --------------------------------------------------------------------
# Pull upstream arrays once
# --------------------------------------------------------------------
src_full_cell12          = np.asarray(src_arr_cell3, dtype=np.int64)
dst_full_cell12          = np.asarray(dst_arr_cell3, dtype=np.int64)
N_full_cell12            = int(N_cell3)
labels_full_cell12       = labels_tensor_cell3.detach().cpu().numpy().astype(np.int64)
labeled_mask_full_cell12 = labeled_mask_cell3.detach().cpu().numpy().astype(bool)
C_full_cell12            = int(C_cell3)
rare_classes_cell12      = np.asarray(rare_classes_list_cell3, dtype=np.int64)

print(f"[data] full graph: N={N_full_cell12:,}, edges={len(src_full_cell12):,}, classes={C_full_cell12}")

# --------------------------------------------------------------------
# Fixed split (shared across all sizes)
# --------------------------------------------------------------------
tr_full_cell12, va_full_cell12, te_full_cell12 = make_split_cell4(
    0, labeled_mask_cell3, labels_tensor_cell3, C_cell3,
)
tr_set_cell12 = set(tr_full_cell12.numpy().tolist())
te_set_cell12 = set(te_full_cell12.numpy().tolist())

# --------------------------------------------------------------------
# Work dir + worker script (written once)
# --------------------------------------------------------------------
work_dir_cell12    = Path("./ntac_smoke_cell12")
work_dir_cell12.mkdir(parents=True, exist_ok=True)
worker_path_cell12 = work_dir_cell12 / "ntac_worker_cell12.py"

WORKER_CELL12 = r'''
import sys
import os
import json
import resource
import traceback

mem_mb = int(sys.argv[3])
mem_bytes = mem_mb * 1024 * 1024

if mem_mb > 0:
    try:
        resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
    except Exception:
        pass

import numpy as np
import scipy.sparse as sp
from collections import Counter

npz_path = sys.argv[1]
out_path = sys.argv[2]
steps = int(sys.argv[4])

try:
    from ntac import Ntac, GraphData
except Exception as e:
    with open(out_path, "w") as f:
        json.dump({"status": "not_installed", "error": repr(e)}, f)
    sys.exit(0)

try:
    z = np.load(npz_path, allow_pickle=False)
    src = z["src"].astype(np.int64)
    dst = z["dst"].astype(np.int64)
    labels_str = z["labels_str"].astype(str)
    labels_int = z["labels_int"].astype(np.int64)
    tr = z["tr"].astype(np.int64)
    te = z["te"].astype(np.int64)
    rare = z["rare"].astype(np.int64)
    nodes_root = z["nodes_root"].astype(np.int64)
    N = int(z["N_sub"][0])

    A = sp.csr_matrix(
        (np.ones(len(src), dtype=np.float32), (src, dst)),
        shape=(N, N),
    )
    A = (A + A.T).tocsr()
    A.sum_duplicates()

    data = None
    last_err = None
    for fn in [
        lambda: GraphData(A, labels=labels_str),
        lambda: GraphData(A, labels=labels_str.tolist()),
        lambda: GraphData(A, labels_str),
        lambda: GraphData(A, labels_str.tolist()),
        lambda: GraphData(graph=A, labels=labels_str),
        lambda: GraphData(adjacency=A, labels=labels_str),
    ]:
        try:
            data = fn(); break
        except TypeError as e:
            last_err = e; continue
        except Exception as e:
            raise RuntimeError(f"GraphData failed: {e!r}") from e
    if data is None:
        raise TypeError(f"Could not construct GraphData. Last TypeError: {last_err!r}")

    nt = None
    last_err = None
    for fn in [
        lambda: Ntac(data=data, labels=labels_str),
        lambda: Ntac(data=data, labels=labels_str.tolist()),
        lambda: Ntac(data, labels_str),
        lambda: Ntac(data, labels_str.tolist()),
        lambda: Ntac(graph_data=data, labels=labels_str),
        lambda: Ntac(data=data),
        lambda: Ntac(data),
    ]:
        try:
            nt = fn(); break
        except TypeError as e:
            last_err = e; continue
        except Exception as e:
            raise RuntimeError(f"Ntac failed: {e!r}") from e
    if nt is None:
        raise TypeError(f"Could not construct Ntac. Last TypeError: {last_err!r}")

    used_method = None
    for i in range(steps):
        if hasattr(nt, "step"):
            nt.step(); used_method = "step"
        elif hasattr(nt, "iterate"):
            nt.iterate(); used_method = "iterate"
        elif hasattr(nt, "update"):
            nt.update(); used_method = "update"
        elif hasattr(nt, "fit"):
            if i == 0:
                nt.fit(); used_method = "fit"; break
            else:
                used_method = "fit_already_called"; break
        else:
            raise AttributeError("Ntac object has no step/iterate/update/fit method.")

    part = None
    if hasattr(nt, "get_partition"):
        part = nt.get_partition()
    elif hasattr(nt, "partition"):
        p = nt.partition; part = p() if callable(p) else p
    elif hasattr(nt, "labels_"):
        part = nt.labels_
    elif hasattr(nt, "pred_"):
        part = nt.pred_
    else:
        raise AttributeError("Could not find partition output on Ntac object.")

    part_arr = np.full(N, "-1", dtype=object)

    if isinstance(part, dict):
        keys = list(part.keys()); values = list(part.values())
        if len(keys) == 0:
            raise RuntimeError("Empty partition dict.")
        root_to_new = {int(r): i for i, r in enumerate(nodes_root)} if len(nodes_root) == N else {}
        for k, v in zip(keys, values):
            try:
                ki = int(k)
            except Exception:
                continue
            if 0 <= ki < N:
                part_arr[ki] = str(v)
            elif root_to_new and ki in root_to_new:
                part_arr[root_to_new[ki]] = str(v)
    else:
        p = np.asarray(part).astype(object)
        if p.shape[0] != N:
            raise RuntimeError(f"Partition length mismatch: got {p.shape[0]}, expected {N}.")
        part_arr[:] = p[:]

    cluster_to_class = {}
    for i in tr:
        cid = str(part_arr[int(i)])
        cls = int(labels_int[int(i)])
        cluster_to_class.setdefault(cid, []).append(cls)
    cluster_to_class = {
        k: Counter(v).most_common(1)[0][0]
        for k, v in cluster_to_class.items()
    }

    pred = []; true = []
    for i in te:
        cid = str(part_arr[int(i)])
        pred.append(int(cluster_to_class.get(cid, -1)))
        true.append(int(labels_int[int(i)]))
    pred = np.asarray(pred, dtype=np.int64)
    true = np.asarray(true, dtype=np.int64)

    recalls = []
    for c in rare:
        m = (true == c)
        if m.sum() > 0:
            recalls.append(float((pred[m] == c).mean()))
    rare_ba = float(np.mean(recalls)) if len(recalls) > 0 else float("nan")

    mapped_classes = set(int(x) for x in cluster_to_class.values())
    unmapped_rare = [int(c) for c in rare if int(c) not in mapped_classes]

    out = {
        "status": "ok",
        "rare_ba": rare_ba,
        "n_clusters_seen_train": len(cluster_to_class),
        "n_classes_mapped": len(mapped_classes),
        "unmapped_rare_count": len(unmapped_rare),
        "unmapped_rare_classes": unmapped_rare,
        "used_method": used_method,
        "N_sub": N,
        "edges_sub": int(len(src)),
        "train_nodes": int(len(tr)),
        "test_nodes": int(len(te)),
    }
    with open(out_path, "w") as f:
        json.dump(out, f)

except Exception as e:
    with open(out_path, "w") as f:
        json.dump({
            "status": "failed",
            "error": repr(e),
            "traceback": traceback.format_exc(),
        }, f)
'''

worker_path_cell12.write_text(WORKER_CELL12, encoding="utf-8")

# --------------------------------------------------------------------
# Loop over sizes
# --------------------------------------------------------------------
results_cell12 = {}

for N_TARGET_cell12 in NODE_SIZES_CELL12:
    print("\n" + "=" * 78)
    print(f"NTAC SCALE TEST — {N_TARGET_cell12} nodes")
    print("=" * 78)

    # ---- sample nodes, proportional per class ----
    rng_cell12 = np.random.default_rng(SEED_CELL12)
    labeled_idx_cell12 = np.where(labeled_mask_full_cell12)[0]
    per_class_cell12 = max(3, int(np.ceil(N_TARGET_cell12 / max(1, C_full_cell12))))
    sampled_list_cell12 = []
    for c_loop_cell12 in range(C_full_cell12):
        idx_c_cell12 = labeled_idx_cell12[labels_full_cell12[labeled_idx_cell12] == c_loop_cell12]
        if len(idx_c_cell12) == 0:
            continue
        take_cell12 = min(per_class_cell12, len(idx_c_cell12))
        chosen_cell12 = rng_cell12.choice(idx_c_cell12, size=take_cell12, replace=False)
        sampled_list_cell12.extend(chosen_cell12.tolist())

    sampled_cell12 = np.unique(np.asarray(sampled_list_cell12, dtype=np.int64))
    if len(sampled_cell12) < N_TARGET_cell12:
        remaining_cell12 = N_TARGET_cell12 - len(sampled_cell12)
        pool_cell12 = np.setdiff1d(labeled_idx_cell12, sampled_cell12, assume_unique=False)
        if len(pool_cell12) > 0:
            extra_cell12 = rng_cell12.choice(
                pool_cell12,
                size=min(remaining_cell12, len(pool_cell12)),
                replace=False,
            )
            sampled_cell12 = np.unique(np.concatenate([sampled_cell12, extra_cell12]))

    sampled_cell12 = np.sort(sampled_cell12[:N_TARGET_cell12])
    N_sub_cell12 = int(len(sampled_cell12))
    print(f"[sample] subgraph nodes={N_sub_cell12:,}")

    if N_sub_cell12 < 50:
        print(f"[skip] subgraph too small at {N_TARGET_cell12}")
        results_cell12[N_TARGET_cell12] = {"status": "too_small", "elapsed_sec": None, "result": None}
        continue

    # ---- induced edges ----
    edge_mask_cell12 = (
        np.isin(src_full_cell12, sampled_cell12)
        & np.isin(dst_full_cell12, sampled_cell12)
    )
    src_old_cell12 = src_full_cell12[edge_mask_cell12]
    dst_old_cell12 = dst_full_cell12[edge_mask_cell12]
    print(f"[sample] induced edges={len(src_old_cell12):,}")

    if len(src_old_cell12) < 20:
        print(f"[skip] too few edges at {N_TARGET_cell12}")
        results_cell12[N_TARGET_cell12] = {"status": "too_few_edges", "elapsed_sec": None, "result": None}
        continue

    src_sub_cell12 = np.searchsorted(sampled_cell12, src_old_cell12).astype(np.int64)
    dst_sub_cell12 = np.searchsorted(sampled_cell12, dst_old_cell12).astype(np.int64)

    # ---- labels + split ----
    labels_sub_int_cell12 = labels_full_cell12[sampled_cell12].astype(np.int64)
    tr_sub_list_cell12, te_sub_list_cell12 = [], []
    labels_split_int_cell12 = np.full(N_sub_cell12, -1, dtype=np.int64)
    for new_i_cell12, old_i_cell12 in enumerate(sampled_cell12):
        if old_i_cell12 in tr_set_cell12:
            tr_sub_list_cell12.append(new_i_cell12)
            labels_split_int_cell12[new_i_cell12] = labels_full_cell12[old_i_cell12]
        elif old_i_cell12 in te_set_cell12:
            te_sub_list_cell12.append(new_i_cell12)

    tr_sub_cell12 = np.asarray(tr_sub_list_cell12, dtype=np.int64)
    te_sub_cell12 = np.asarray(te_sub_list_cell12, dtype=np.int64)

    if len(tr_sub_cell12) < 20 or len(te_sub_cell12) < 5:
        print(f"[warn] split too small at {N_TARGET_cell12}; falling back to random subgraph split.")
        perm_cell12 = rng_cell12.permutation(N_sub_cell12)
        n_tr_cell12 = int(0.60 * N_sub_cell12)
        n_va_cell12 = int(0.20 * N_sub_cell12)
        tr_sub_cell12 = perm_cell12[:n_tr_cell12]
        te_sub_cell12 = perm_cell12[n_tr_cell12 + n_va_cell12:]
        labels_split_int_cell12 = np.full(N_sub_cell12, -1, dtype=np.int64)
        labels_split_int_cell12[tr_sub_cell12] = labels_sub_int_cell12[tr_sub_cell12]

    labels_split_str_cell12 = np.array(
        [str(int(x)) if x >= 0 else "?" for x in labels_split_int_cell12],
        dtype="U16",
    )
    print(f"[split] train={len(tr_sub_cell12):,}, test={len(te_sub_cell12):,}")

    # ---- save npz for this size ----
    npz_path_cell12 = work_dir_cell12 / f"subgraph_{N_TARGET_cell12}.npz"
    out_json_cell12 = work_dir_cell12 / f"result_{N_TARGET_cell12}.json"
    np.savez(
        npz_path_cell12,
        src=src_sub_cell12,
        dst=dst_sub_cell12,
        labels_str=labels_split_str_cell12,
        labels_int=labels_sub_int_cell12,
        tr=tr_sub_cell12,
        te=te_sub_cell12,
        rare=rare_classes_cell12,
        nodes_root=sampled_cell12,
        N_sub=np.asarray([N_sub_cell12], dtype=np.int64),
    )
    if out_json_cell12.exists():
        out_json_cell12.unlink()

    # ---- launch worker ----
    print(f"[run] launching isolated NTAC worker (mem_limit={MEM_MB_CELL12} MB)...")
    gc.collect()
    t0_cell12 = time.perf_counter()
    status_cell12 = "unknown"
    result_cell12 = None
    dt_cell12 = 0.0

    try:
        proc_cell12 = subprocess.run(
            [
                sys.executable,
                str(worker_path_cell12),
                str(npz_path_cell12),
                str(out_json_cell12),
                str(MEM_MB_CELL12),
                str(STEPS_CELL12),
            ],
            timeout=TIMEOUT_SEC_CELL12,
            capture_output=True,
            text=True,
        )
        dt_cell12 = time.perf_counter() - t0_cell12

        if proc_cell12.returncode != 0:
            status_cell12 = "worker_nonzero_exit"
            print(f"[fail] worker exit={proc_cell12.returncode}, elapsed={dt_cell12:.2f}s")
            print("[stderr tail]")
            print((proc_cell12.stderr or "")[-2000:])
        elif not out_json_cell12.exists():
            status_cell12 = "no_output"
            print(f"[fail] no JSON produced, elapsed={dt_cell12:.2f}s")
            print("[stderr tail]")
            print((proc_cell12.stderr or "")[-2000:])
        else:
            result_cell12 = json.loads(out_json_cell12.read_text(encoding="utf-8"))
            status_cell12 = str(result_cell12.get("status", "malformed"))
            print(f"[ok] worker finished in {dt_cell12:.2f}s")

    except subprocess.TimeoutExpired:
        dt_cell12 = time.perf_counter() - t0_cell12
        status_cell12 = "timeout"
        print(f"[fail] timeout after {dt_cell12:.2f}s")

    except Exception as e_cell12:
        status_cell12 = "parent_exception"
        print(f"[fail] parent exception: {e_cell12!r}")
        traceback.print_exc()

    if status_cell12 == "ok" and result_cell12 is not None:
        rb = result_cell12.get("rare_ba", float("nan"))
        print(
            f"[result] N={result_cell12.get('N_sub')}, "
            f"edges={result_cell12.get('edges_sub')}, "
            f"rare_ba={rb:.4f}, "
            f"unmapped_rare={result_cell12.get('unmapped_rare_count')}"
        )

    results_cell12[N_TARGET_cell12] = {
        "status": status_cell12,
        "elapsed_sec": dt_cell12,
        "result": result_cell12,
    }

    # ---- cleanup between iterations ----
    for name_cell12 in (
        "sampled_cell12", "edge_mask_cell12",
        "src_old_cell12", "dst_old_cell12",
        "src_sub_cell12", "dst_sub_cell12",
        "labels_sub_int_cell12", "labels_split_int_cell12", "labels_split_str_cell12",
        "tr_sub_cell12", "te_sub_cell12",
        "tr_sub_list_cell12", "te_sub_list_cell12",
        "sampled_list_cell12",
    ):
        if name_cell12 in globals():
            try:
                del globals()[name_cell12]
            except Exception:
                pass
    gc.collect()

# --------------------------------------------------------------------
# Summary
# --------------------------------------------------------------------
print("\n" + "=" * 78)
print("NTAC SCALING SUMMARY")
print("=" * 78)
print(f"{'N':>8s}  {'status':>20s}  {'time (s)':>10s}  {'edges':>10s}  {'rare_ba':>10s}")
print("-" * 78)
for size_cell12, r_cell12 in results_cell12.items():
    res_cell12 = r_cell12.get("result") or {}
    edges_cell12 = res_cell12.get("edges_sub", "—")
    rb_cell12 = res_cell12.get("rare_ba", float("nan"))
    rb_str_cell12 = f"{rb_cell12:.4f}" if isinstance(rb_cell12, float) and not np.isnan(rb_cell12) else "—"
    t_cell12 = r_cell12.get("elapsed_sec") or 0.0
    print(f"{size_cell12:>8d}  {r_cell12['status']:>20s}  {t_cell12:>10.2f}  {str(edges_cell12):>10s}  {rb_str_cell12:>10s}")
print("=" * 78)

# Decision hint
ok_sizes_cell12 = [s for s, r in results_cell12.items() if r["status"] == "ok"]
if ok_sizes_cell12:
    largest_ok_cell12 = max(ok_sizes_cell12)
    t_largest_cell12 = results_cell12[largest_ok_cell12].get("elapsed_sec") or 0.0
    print(f"\n[decision] Largest successful size: {largest_ok_cell12} nodes in {t_largest_cell12:.2f}s")
    print(f"[decision] Full graph is 139,255 nodes ({139255/largest_ok_cell12:.1f}× larger).")
    if t_largest_cell12 < 300:
        print("[decision] Time suggests full graph may be feasible. Try a 50K-node run next.")
    elif t_largest_cell12 < 1200:
        print("[decision] Full graph is borderline. Decide based on memory: try 50K and watch RSS.")
    else:
        print("[decision] Full graph is out of reach in this environment.")
        print("[decision] Report NTAC on the largest successful subgraph as a diagnostic.")
else:
    print("\n[decision] No size completed successfully. Drop NTAC.")
print("=" * 78)

In [ ]:
# Checkpoint: Cell 12 NTAC scaling test
#
# What this saves:
#   - results_cell12: per-size NTAC scaling results
#   - summary_rows_cell12: compact table of N / status / time / edges / rare_ba
#   - decision_cell12: whether NTAC is viable / borderline / drop
#   - config_cell12: sizes, steps, timeout, memory limit, seed
#   - worker_script_cell12: the isolated NTAC worker source used by Cell 12
#   - local JSON summary under ntac_smoke_cell12/
#   - Hugging Face checkpoint if Cell 00 has been run
#
# Paste this at the end of Cell 12.


def cell12_float_or_none(value):
    try:
        value_float = float(value)
    except Exception:
        return None

    if not np.isfinite(value_float):
        return None

    return value_float


def cell12_int_list_or_none(value):
    if value is None:
        return None

    try:
        return [int(x) for x in value]
    except Exception:
        return None


cell12_results_for_ckpt = globals().get("results_cell12", {})

cell12_summary_rows = []

for cell12_size, cell12_record in cell12_results_for_ckpt.items():
    cell12_inner = {}

    if isinstance(cell12_record, dict):
        cell12_inner = cell12_record.get("result") or {}

    cell12_summary_rows.append(
        {
            "N": int(cell12_size),
            "status": str(cell12_record.get("status", "unknown"))
            if isinstance(cell12_record, dict)
            else "unknown",
            "elapsed_sec": cell12_float_or_none(
                cell12_record.get("elapsed_sec", None)
                if isinstance(cell12_record, dict)
                else None
            ),
            "edges_sub": cell12_inner.get("edges_sub", None),
            "rare_ba": cell12_float_or_none(cell12_inner.get("rare_ba", None)),
            "n_clusters_seen_train": cell12_inner.get(
                "n_clusters_seen_train", None
            ),
            "n_classes_mapped": cell12_inner.get("n_classes_mapped", None),
            "unmapped_rare_count": cell12_inner.get(
                "unmapped_rare_count", None
            ),
            "unmapped_rare_classes": cell12_int_list_or_none(
                cell12_inner.get("unmapped_rare_classes", None)
            ),
            "used_method": cell12_inner.get("used_method", None),
            "train_nodes": cell12_inner.get("train_nodes", None),
            "test_nodes": cell12_inner.get("test_nodes", None),
        }
    )

# ----------------------------------------------------------------------
# Decision text
# ----------------------------------------------------------------------
cell12_ok_rows = [
    row for row in cell12_summary_rows if row["status"] == "ok"
]

if len(cell12_ok_rows) > 0:
    cell12_largest_row = max(cell12_ok_rows, key=lambda row: row["N"])
    cell12_largest_N = int(cell12_largest_row["N"])
    cell12_largest_time = cell12_largest_row.get("elapsed_sec", None)
    cell12_scale_factor = (
        139255.0 / float(cell12_largest_N)
        if cell12_largest_N > 0
        else float("inf")
    )

    if cell12_largest_time is not None and cell12_largest_time < 300.0:
        cell12_decision = (
            f"Largest successful NTAC subgraph: {cell12_largest_N} nodes "
            f"in {cell12_largest_time:.2f}s. "
            f"Full graph is {cell12_scale_factor:.1f}x larger. "
            "Time suggests full graph may be feasible on a larger machine, "
            "but do not attempt full graph on 32 GB RAM."
        )
    elif cell12_largest_time is not None and cell12_largest_time < 1200.0:
        cell12_decision = (
            f"Largest successful NTAC subgraph: {cell12_largest_N} nodes "
            f"in {cell12_largest_time:.2f}s. "
            "Full graph is borderline. Use subgraph diagnostic only."
        )
    else:
        cell12_decision = (
            f"Largest successful NTAC subgraph: {cell12_largest_N} nodes. "
            "Full graph is out of reach in this environment. "
            "Report NTAC only as a subgraph diagnostic or drop it."
        )
else:
    cell12_decision = (
        "No NTAC subgraph size completed successfully. "
        "Drop NTAC from the main table."
    )

# ----------------------------------------------------------------------
# Payload
# ----------------------------------------------------------------------
cell12_checkpoint_payload = {
    "summary_rows": cell12_summary_rows,
    "decision": cell12_decision,
    "results_cell12": cell12_results_for_ckpt,
    "config": {
        "NODE_SIZES_CELL12": globals().get("NODE_SIZES_CELL12", []),
        "STEPS_CELL12": globals().get("STEPS_CELL12", None),
        "TIMEOUT_SEC_CELL12": globals().get("TIMEOUT_SEC_CELL12", None),
        "MEM_MB_CELL12": globals().get("MEM_MB_CELL12", None),
        "SEED_CELL12": globals().get("SEED_CELL12", None),
        "rare_classes": cell12_int_list_or_none(
            globals().get("rare_classes_cell12", None)
        ),
    },
    "graph_counts": {
        "N_full": int(globals().get("N_full_cell12", 0)),
        "edges_full": int(len(globals().get("src_full_cell12", []))),
        "C_full": int(globals().get("C_full_cell12", 0)),
    },
    "paths": {
        "work_dir": str(globals().get("work_dir_cell12", "")),
        "worker_path": str(globals().get("worker_path_cell12", "")),
    },
    "worker_script": globals().get("WORKER_CELL12", ""),
    "local_json_path": None,
}

# ----------------------------------------------------------------------
# Save human-readable local JSON summary
# ----------------------------------------------------------------------
cell12_Path_class = globals().get("Path", None)
cell12_json_module = globals().get("json", None)

if cell12_Path_class is not None and cell12_json_module is not None:
    try:
        cell12_json_path = cell12_Path_class(
            "ntac_smoke_cell12/cell12_checkpoint_summary.json"
        )
        cell12_json_path.parent.mkdir(parents=True, exist_ok=True)

        cell12_checkpoint_payload["local_json_path"] = str(cell12_json_path)

        cell12_json_path.write_text(
            cell12_json_module.dumps(
                cell12_checkpoint_payload,
                indent=2,
                default=str,
            ),
            encoding="utf-8",
        )

        print(f"[ckpt:cell12] local JSON summary saved: {cell12_json_path}")
    except Exception as cell12_json_error:
        print(
            f"[ckpt:cell12] local JSON summary failed: {cell12_json_error!r}"
        )
else:
    print(
        "[ckpt:cell12] Path/json not available; skipping local JSON summary."
    )

# ----------------------------------------------------------------------
# Hugging Face checkpoint, if Cell 00 has been run
# ----------------------------------------------------------------------
try:
    hf_save_checkpoint(
        "cell12_ntac_scaling",
        cell12_checkpoint_payload,
        description=(
            "Cell 12: NTAC subgraph scaling test results, decision text, "
            "config, worker script, and local JSON summary."
        ),
        upload=True,
    )
except NameError:
    print(
        "[ckpt][skip] hf_save_checkpoint not defined. "
        "Run Cell 00 first to enable Hugging Face checkpoints."
    )
except Exception as cell12_ckpt_error:
    print(f"[ckpt][warn] Cell 12 checkpoint failed: {cell12_ckpt_error!r}")

# ----------------------------------------------------------------------
# Optional: also upload the human-readable JSON summary to the bucket
# ----------------------------------------------------------------------
try:
    if (
        cell12_checkpoint_payload.get("local_json_path", None) is not None
        and "hf_upload_file_to_bucket" in globals()
    ):
        cell12_local_json_path = globals()["Path"](
            cell12_checkpoint_payload["local_json_path"]
        )

        cell12_remote_json = (
            "checkpoints/cell12_ntac_scaling/"
            + cell12_local_json_path.name
        )

        cell12_upload_ok, cell12_upload_info = hf_upload_file_to_bucket(
            cell12_local_json_path,
            cell12_remote_json,
        )

        print(
            "[ckpt:cell12] upload JSON summary: "
            f"{'OK' if cell12_upload_ok else 'FAILED'} | {cell12_upload_info}"
        )
except Exception as cell12_json_upload_error:
    print(
        "[ckpt:cell12] JSON summary upload failed: "
        f"{cell12_json_upload_error!r}"
    )

In [ ]:
# Cell 12c: Final comparison table — uses Cell 11c if available
#
# Paste this directly after Cell 11c.
#
# It does not train anything.
# It reads existing result variables and builds robust arrays.

def _build_final_table_cell12c():
    required_cell12c = ["np"]
    missing_cell12c = [name for name in required_cell12c if name not in globals()]
    if missing_cell12c:
        raise NameError(f"[Cell 12c][FATAL] Missing variables: {missing_cell12c}.")

    def _as_finite_array_cell12c(obj):
        if obj is None:
            return np.asarray([], dtype=float)

        if isinstance(obj, dict):
            if len(obj) == 0:
                return np.asarray([], dtype=float)

            # If dict itself is one result.
            for k in ("rare_ba", "test_rare", "rare_balanced_acc", "rare"):
                if k in obj:
                    try:
                        return np.asarray([float(obj[k])], dtype=float)
                    except Exception:
                        pass

            # Otherwise assume variant dict: take first variant.
            first_key = next(iter(obj))
            return _as_finite_array_cell12c(obj[first_key])

        if isinstance(obj, (list, tuple, np.ndarray)):
            obj = list(obj)

            if len(obj) == 0:
                return np.asarray([], dtype=float)

            if isinstance(obj[0], dict):
                for k in ("rare_ba", "test_rare", "rare_balanced_acc", "rare"):
                    if k in obj[0]:
                        vals = []
                        for r in obj:
                            try:
                                vals.append(float(r[k]))
                            except Exception:
                                vals.append(np.nan)
                        return np.asarray(vals, dtype=float)

            try:
                return np.asarray([float(x) for x in obj], dtype=float)
            except Exception:
                return np.asarray([], dtype=float)

        try:
            return np.asarray([float(obj)], dtype=float)
        except Exception:
            return np.asarray([], dtype=float)

    def _clean_finite_cell12c(arr):
        arr = np.asarray(arr, dtype=float).ravel()
        return arr[np.isfinite(arr)]

    def _mean_std_cell12c(arr):
        arr = _clean_finite_cell12c(arr)
        if arr.size == 0:
            return float("nan"), float("nan"), 0
        if arr.size == 1:
            return float(arr[0]), 0.0, 1
        return float(np.mean(arr)), float(np.std(arr, ddof=1)), int(arr.size)

    def _fmt_num_cell12c(x, digits=4):
        try:
            xf = float(x)
        except Exception:
            return "missing"
        if not np.isfinite(xf):
            return "missing"
        return f"{xf:.{digits}f}"

    def _first_nonempty_cell12c(names):
        for nm in names:
            if nm in globals():
                arr = _clean_finite_cell12c(
                    _as_finite_array_cell12c(globals()[nm])
                )
                if arr.size > 0:
                    return nm, arr
        return None, np.asarray([], dtype=float)

    def _is_variant_dict_cell12c(obj):
        if not isinstance(obj, dict):
            return False
        if len(obj) == 0:
            return False
        for v in obj.values():
            if isinstance(v, (list, tuple, np.ndarray)):
                return True
        return False

    # ------------------------------------------------------------------
    # SSPR after refinement
    # Prefer true Cell 11c output.
    # ------------------------------------------------------------------
    if "sspr_after_cell11c" in globals():
        sspr_after_cell12c = _clean_finite_cell12c(
            _as_finite_array_cell12c(globals()["sspr_after_cell11c"])
        )
        sspr_before_cell12c = _clean_finite_cell12c(
            _as_finite_array_cell12c(globals().get("sspr_before_cell11c", None))
        )
        before_is_proxy_cell12c = bool(
            globals().get("sspr_before_is_proxy_cell11c", False)
        )
        sspr_after_source_cell12c = "sspr_after_cell11c"
        sspr_before_source_cell12c = "sspr_before_cell11c"
    elif "sspr_rare_cell11" in globals():
        sspr_after_cell12c = _clean_finite_cell12c(
            _as_finite_array_cell12c(globals()["sspr_rare_cell11"])
        )
        sspr_before_cell12c = sspr_after_cell12c.copy()
        before_is_proxy_cell12c = True
        sspr_after_source_cell12c = "sspr_rare_cell11"
        sspr_before_source_cell12c = "fallback_to_after_refine"
    else:
        raise NameError(
            "[Cell 12c][FATAL] No SSPR result found. "
            "Run Cell 11c, or at least original Cell 11."
        )

    if sspr_after_cell12c.size == 0:
        raise ValueError("[Cell 12c][FATAL] SSPR after-refine array is empty.")

    if sspr_before_cell12c.size != sspr_after_cell12c.size:
        sspr_before_cell12c = sspr_after_cell12c.copy()
        before_is_proxy_cell12c = True
        sspr_before_source_cell12c = "fallback_size_mismatch"

    # ------------------------------------------------------------------
    # Baseline selection
    # Prefer explicit linear baseline if available.
    # ------------------------------------------------------------------
    baseline_priority_cell12c = [
        "results_linear_cell8",
        "baseline_rare_cell10",
        "results_baseline_cell8",
    ]

    baseline_name_cell12c, baseline_cell12c = _first_nonempty_cell12c(
        baseline_priority_cell12c
    )

    if baseline_cell12c.size == 0:
        baseline_cell12c = np.asarray([float("nan")], dtype=float)
        baseline_name_cell12c = "missing"

    # ------------------------------------------------------------------
    # SPGCN
    # ------------------------------------------------------------------
    spgcn_name_cell12c, spgcn_cell12c = _first_nonempty_cell12c(
        ["results_spgcn_cell8"]
    )

    if spgcn_cell12c.size == 0:
        spgcn_cell12c = np.asarray([float("nan")], dtype=float)
        spgcn_name_cell12c = "missing"

    # ------------------------------------------------------------------
    # Label propagation / old NTAC substitute
    # ------------------------------------------------------------------
    lp_name_cell12c, lp_cell12c = _first_nonempty_cell12c(
        [
            "lp_rare_cell10",
            "ntac_rare_cell10",
            "label_prop_rare_cell10",
        ]
    )

    if lp_cell12c.size == 0:
        lp_cell12c = np.asarray([], dtype=float)
        lp_name_cell12c = "missing"

    # ------------------------------------------------------------------
    # Altiparmak rows
    # ------------------------------------------------------------------
    alt_obj_cell12c = globals().get("results_alt_cell11b", None)
    alt_rows_cell12c = []

    if _is_variant_dict_cell12c(alt_obj_cell12c):
        for variant_name, runs in alt_obj_cell12c.items():
            arr = _clean_finite_cell12c(_as_finite_array_cell12c(runs))
            alt_rows_cell12c.append(
                (
                    f"Altiparmak [{variant_name}]",
                    arr,
                    f"alt_{variant_name}",
                )
            )
    elif alt_obj_cell12c is not None:
        arr = _clean_finite_cell12c(_as_finite_array_cell12c(alt_obj_cell12c))
        alt_rows_cell12c.append(
            (
                "Altiparmak GraphSAGE + GraphSMOTE + ME",
                arr,
                "alt_generic",
            )
        )

    if len(alt_rows_cell12c) == 0:
        alt_rows_cell12c.append(
            (
                "Altiparmak GraphSAGE + GraphSMOTE + ME",
                np.asarray([], dtype=float),
                "alt_missing",
            )
        )

    chance_cell12c = 1.0 / float(globals().get("C_cell3", 29))

    alt_primary_cell12c = None
    alt_primary_name_cell12c = None

    preferred_tags_cell12c = [
        "balanced_target500",
        "faithful_portion6",
        "alt_generic",
    ]

    for tag in preferred_tags_cell12c:
        for disp, arr, key in alt_rows_cell12c:
            if tag in key and arr.size >= 2:
                m, s, n = _mean_std_cell12c(arr)
                if np.isfinite(m) and m > chance_cell12c + 0.02:
                    alt_primary_cell12c = arr
                    alt_primary_name_cell12c = disp
                    break
        if alt_primary_cell12c is not None:
            break

    if alt_primary_cell12c is None:
        for disp, arr, key in alt_rows_cell12c:
            if arr.size >= 1:
                alt_primary_cell12c = arr
                alt_primary_name_cell12c = disp
                break

    if alt_primary_cell12c is None:
        alt_primary_cell12c = np.asarray([], dtype=float)
        alt_primary_name_cell12c = "missing"

    # ------------------------------------------------------------------
    # Warnings
    # ------------------------------------------------------------------
    warnings_cell12c = []

    if before_is_proxy_cell12c:
        warnings_cell12c.append(
            "SSPR before-refine values are proxy / not independently measured. "
            "Do not claim a refinement gain unless Cell 11c was run successfully."
        )

    if baseline_name_cell12c != "missing":
        bm, bs, bn = _mean_std_cell12c(baseline_cell12c)
        if np.isfinite(bm) and bm > 0.25:
            warnings_cell12c.append(
                f"Selected baseline '{baseline_name_cell12c}' has mean {bm:.4f}. "
                "If this was intended to be the linear-head GCN baseline, verify that it is not "
                "a prototype-head run. The historical linear baseline was around 0.117."
            )

    if alt_primary_cell12c.size > 0:
        am, asd, an = _mean_std_cell12c(alt_primary_cell12c)
        if np.isfinite(am) and am < chance_cell12c + 0.01:
            warnings_cell12c.append(
                f"Primary Altiparmak comparator '{alt_primary_name_cell12c}' has mean {am:.4f}, "
                f"near or below uniform chance 1/C = {chance_cell12c:.4f}. "
                "Do not report a large multiplicative improvement against this row without verifying "
                "the reimplementation."
            )

    if baseline_cell12c.size != sspr_after_cell12c.size:
        warnings_cell12c.append(
            f"Baseline n={baseline_cell12c.size}, SSPR n={sspr_after_cell12c.size}. "
            "Paired significance tests against baseline will be skipped unless seed counts match."
        )

    if spgcn_cell12c.size != sspr_after_cell12c.size:
        warnings_cell12c.append(
            f"SPGCN n={spgcn_cell12c.size}, SSPR n={sspr_after_cell12c.size}. "
            "Paired significance tests against SPGCN will be skipped unless seed counts match."
        )

    # ------------------------------------------------------------------
    # Build table rows
    # ------------------------------------------------------------------
    table_rows_cell12c = []

    if lp_cell12c.size > 0:
        table_rows_cell12c.append(
            ("Graph label propagation", lp_cell12c, "lp")
        )

    table_rows_cell12c.append(
        ("GCN + all features baseline", baseline_cell12c, "baseline")
    )
    table_rows_cell12c.append(
        ("SPGCN unsigned spectral", spgcn_cell12c, "spgcn")
    )
    table_rows_cell12c.append(
        ("SSPR without refinement", sspr_before_cell12c, "sspr_before")
    )
    table_rows_cell12c.append(
        ("SSPR full with refinement", sspr_after_cell12c, "sspr_after")
    )

    for disp, arr, key in alt_rows_cell12c:
        table_rows_cell12c.append((disp, arr, key))

    table_arrays_cell12c = {
        "lp": lp_cell12c,
        "baseline": baseline_cell12c,
        "spgcn": spgcn_cell12c,
        "sspr_before": sspr_before_cell12c,
        "sspr_after": sspr_after_cell12c,
        "alt_primary": alt_primary_cell12c,
    }

    # ------------------------------------------------------------------
    # Print table
    # ------------------------------------------------------------------
    print("=" * 112)
    print(
        "FINAL COMPARISON — FlyWire v783, 29-class rare-class balanced accuracy"
    )
    print("=" * 112)
    print(
        f"{'Method':58s} {'Mean':>10s} {'Std':>10s} {'n':>5s} {'Δ vs baseline':>15s}"
    )
    print("-" * 112)

    baseline_mean_for_delta, _, _ = _mean_std_cell12c(baseline_cell12c)

    for disp, arr, key in table_rows_cell12c:
        m, s, n = _mean_std_cell12c(arr)

        if key == "baseline":
            delta_str = "---"
        elif np.isfinite(m) and np.isfinite(baseline_mean_for_delta):
            delta_str = f"{m - baseline_mean_for_delta:+.4f}"
        else:
            delta_str = "missing"

        print(
            f"{disp:58s} "
            f"{_fmt_num_cell12c(m):>10s} "
            f"{_fmt_num_cell12c(s):>10s} "
            f"{n:5d} "
            f"{delta_str:>15s}"
        )

    print("=" * 112)

    print("\nMetadata:")
    print(f"  baseline_source          = {baseline_name_cell12c}")
    print(f"  spgcn_source             = {spgcn_name_cell12c}")
    print(f"  lp_source                = {lp_name_cell12c}")
    print(f"  sspr_after_source        = {sspr_after_source_cell12c}")
    print(f"  sspr_before_source       = {sspr_before_source_cell12c}")
    print(f"  before_is_proxy          = {before_is_proxy_cell12c}")
    print(f"  alt_primary_name         = {alt_primary_name_cell12c}")
    print(f"  uniform_chance_1_over_C  = {chance_cell12c:.4f}")

    if len(warnings_cell12c) > 0:
        print("\nWarnings:")
        for w in warnings_cell12c:
            print(f"  - {w}")
    else:
        print("\nWarnings: none")

    print("=" * 112)

    return {
        "rows": table_rows_cell12c,
        "arrays": table_arrays_cell12c,
        "warnings": warnings_cell12c,
        "baseline_name": baseline_name_cell12c,
        "spgcn_name": spgcn_name_cell12c,
        "lp_name": lp_name_cell12c,
        "sspr_after_source": sspr_after_source_cell12c,
        "sspr_before_source": sspr_before_source_cell12c,
        "before_is_proxy": before_is_proxy_cell12c,
        "alt_primary_name": alt_primary_name_cell12c,
        "chance": chance_cell12c,
    }


_table_cell12c = _build_final_table_cell12c()

table_rows_cell12c = _table_cell12c["rows"]
table_arrays_cell12c = _table_cell12c["arrays"]
warnings_cell12c = _table_cell12c["warnings"]

baseline_name_cell12c = _table_cell12c["baseline_name"]
spgcn_name_cell12c = _table_cell12c["spgcn_name"]
lp_name_cell12c = _table_cell12c["lp_name"]

sspr_after_source_cell12c = _table_cell12c["sspr_after_source"]
sspr_before_source_cell12c = _table_cell12c["sspr_before_source"]
before_is_proxy_cell12c = _table_cell12c["before_is_proxy"]

alt_primary_name_cell12c = _table_cell12c["alt_primary_name"]
chance_cell12c = _table_cell12c["chance"]

print("[Cell 12c] Stored: table_rows_cell12c, table_arrays_cell12c, warnings_cell12c")

In [ ]:
# Checkpoint: Cell 12c final table
#
# What Cell 12c does:
#   Assembles the final comparison table from all stored result arrays:
#   label propagation, GCN baseline, SPGCN, SSPR no-refine, SSPR full,
#   and Altiparmak variants.

hf_save_checkpoint(
    "cell12c_final_table",
    {
        "table_rows_cell12c": table_rows_cell12c,
        "table_arrays_cell12c": table_arrays_cell12c,
        "warnings_cell12c": warnings_cell12c,
        "baseline_name_cell12c": baseline_name_cell12c,
        "spgcn_name_cell12c": spgcn_name_cell12c,
        "lp_name_cell12c": lp_name_cell12c,
        "sspr_after_source_cell12c": sspr_after_source_cell12c,
        "sspr_before_source_cell12c": sspr_before_source_cell12c,
        "before_is_proxy_cell12c": before_is_proxy_cell12c,
        "alt_primary_name_cell12c": alt_primary_name_cell12c,
        "chance_cell12c": chance_cell12c,
    },
    description=(
        "Cell 12c: assembled final comparison table arrays, source metadata, "
        "and warnings."
    ),
)

In [ ]:
# Cell 13: Ablation decomposition + effect sizes + paper export
#
# Reads result arrays reconstructed from the experiment cells that actually ran
# (Cells 8, 11, 11b).  The original Cell 12 was repurposed for NTAC scaling
# tests and did not define these arrays, so we build them here.
#
# Outputs:
#   1. Marginal contribution of each component (ablation ladder)
#   2. Paired t-tests between consecutive steps
#   3. 95% CI for SSPR full mean
#   4. Cohen's d effect sizes
#   5. CSV export for the paper
#   6. LaTeX-ready table string

# ------------------------------------------------------------------
# 0. Initialize result arrays from available experiment outputs
# ------------------------------------------------------------------
# These were expected from Cell 12 but that cell was repurposed for NTAC
# testing.  Reconstruct from whichever experiment cells actually ran.

# --- SSPR after refinement (Cell 11, 10 seeds) ---
if "sspr_rare_cell11" in globals():
    sspr_after_cell12 = np.array(sspr_rare_cell11, dtype=np.float64)
else:
    sspr_after_cell12 = np.array([], dtype=np.float64)

# --- SSPR before refinement ---
# Cell 11 computed before-refine scores (r_cell11['test_rare']) but did
# not store them in a list.  Fall back to after-refine values (conservative).
sspr_before_cell12 = sspr_after_cell12.copy()

# --- Altiparmak (Cell 11b, faithful variant, 10 seeds) ---
if "results_alt_cell11b" in globals() and "faithful_portion6" in results_alt_cell11b:
    alt_primary_cell12 = np.array(
        [r["rare_ba"] for r in results_alt_cell11b["faithful_portion6"]],
        dtype=np.float64,
    )
else:
    alt_primary_cell12 = None

# --- Label propagation (Cell 10 is commented out) ---
lp_cell12 = np.array([], dtype=np.float64)

# --- Baseline GCN ---
# Try full-scale results from Cell 8/9 first; fall back to debug "minimal"
# config (0 signed layers, which is the feature-only baseline).
if "results_baseline_cell8" in globals():
    baseline_cell12 = np.array(results_baseline_cell8, dtype=np.float64)
elif "results_debug_cell8" in globals() and "minimal" in results_debug_cell8:
    baseline_cell12 = np.array(results_debug_cell8["minimal"], dtype=np.float64)
else:
    baseline_cell12 = np.array([], dtype=np.float64)

# --- SPGCN ---
if "results_spgcn_cell8" in globals():
    spgcn_cell12 = np.array(results_spgcn_cell8, dtype=np.float64)
elif "results_debug_cell8" in globals() and "one_signed" in results_debug_cell8:
    spgcn_cell12 = np.array(results_debug_cell8["one_signed"], dtype=np.float64)
else:
    spgcn_cell12 = np.array([], dtype=np.float64)

# ------------------------------------------------------------------
# 1. Ablation ladder: marginal delta at each step
# ------------------------------------------------------------------
print("=" * 90)
print("ABLATION LADDER — marginal rare-class balanced accuracy gain")
print("=" * 90)
print(f"{'Step':50s} {'Mean':>10s} {'Delta':>10s} {'%':>8s}")
print("-" * 90)

_steps_cell13 = []

if lp_cell12.size > 0:
    _steps_cell13.append(("Graph label propagation", lp_cell12))

_steps_cell13.append(("GCN + all features (baseline)", baseline_cell12))
_steps_cell13.append(("+ SPGCN (unsigned spectral)", spgcn_cell12))
_steps_cell13.append(("+ SSPR (signed spectral + motif, no refine)", sspr_before_cell12))
_steps_cell13.append(("+ SSPR full (with refinement)", sspr_after_cell12))

_prev_mean_cell13 = None

for _name_cell13, _arr_cell13 in _steps_cell13:
    if _arr_cell13.size == 0:
        print(f"{_name_cell13:50s} {'—':>10s} {'':>10s} {'':>8s}")
        continue
    _m_cell13 = float(np.nanmean(_arr_cell13))
    _delta_cell13 = "" if _prev_mean_cell13 is None else f"+{_m_cell13 - _prev_mean_cell13:.4f}"
    _pct_cell13 = "" if _prev_mean_cell13 is None or _prev_mean_cell13 == 0 else f"{(_m_cell13 - _prev_mean_cell13) / _prev_mean_cell13 * 100:.1f}%"
    print(f"{_name_cell13:50s} {_m_cell13:10.4f} {_delta_cell13:>10s} {_pct_cell13:>8s}")
    _prev_mean_cell13 = _m_cell13

print("=" * 90)

# ------------------------------------------------------------------
# 2. Paired t-tests between consecutive steps
# ------------------------------------------------------------------
print("\nPAIRED T-TESTS — consecutive ablation steps")
print("-" * 90)

_consecutive_pairs_cell13 = [
    ("SPGCN vs baseline", spgcn_cell12, baseline_cell12),
    ("SSPR-no-refine vs SPGCN", sspr_before_cell12, spgcn_cell12),
    ("SSPR-full vs SSPR-no-refine", sspr_after_cell12, sspr_before_cell12),
]

for _label_cell13, _a_cell13, _b_cell13 in _consecutive_pairs_cell13:
    if _a_cell13.size >= 2 and _b_cell13.size >= 2 and _a_cell13.size == _b_cell13.size:
        _t_cell13, _p_cell13 = stats.ttest_rel(_a_cell13, _b_cell13)
        _sig_cell13 = "***" if _p_cell13 < 0.001 else "**" if _p_cell13 < 0.01 else "*" if _p_cell13 < 0.05 else "ns"
        print(f"{_label_cell13:40s}  t={float(_t_cell13):+.4f}  p={float(_p_cell13):.4e}  {_sig_cell13}")
    else:
        print(f"{_label_cell13:40s}  skipped (unequal or insufficient seeds)")

# Also: SSPR-full vs Altiparmak
if alt_primary_cell12 is not None and alt_primary_cell12.size >= 2:
    if sspr_after_cell12.size == alt_primary_cell12.size:
        _t_cell13, _p_cell13 = stats.ttest_rel(sspr_after_cell12, alt_primary_cell12)
        _sig_cell13 = "***" if _p_cell13 < 0.001 else "**" if _p_cell13 < 0.01 else "*" if _p_cell13 < 0.05 else "ns"
        print(f"{'SSPR-full vs Altiparmak':40s}  t={float(_t_cell13):+.4f}  p={float(_p_cell13):.4e}  {_sig_cell13}")
    else:
        print(f"{'SSPR-full vs Altiparmak':40s}  skipped (unequal seeds: {sspr_after_cell12.size} vs {alt_primary_cell12.size})")

print("-" * 90)

# ------------------------------------------------------------------
# 3. 95% confidence interval for SSPR full mean
# ------------------------------------------------------------------
_n_cell13 = int(sspr_after_cell12.size)
_mean_cell13 = float(np.nanmean(sspr_after_cell12)) if _n_cell13 > 0 else float("nan")
_std_cell13 = float(np.nanstd(sspr_after_cell12, ddof=1)) if _n_cell13 > 1 else 0.0
_se_cell13 = _std_cell13 / np.sqrt(_n_cell13) if _n_cell13 > 1 else 0.0

# t-critical for 95% CI
if _n_cell13 >= 2:
    _tcrit_cell13 = stats.t.ppf(0.975, df=_n_cell13 - 1)
else:
    _tcrit_cell13 = 0.0

_ci_lo_cell13 = _mean_cell13 - _tcrit_cell13 * _se_cell13
_ci_hi_cell13 = _mean_cell13 + _tcrit_cell13 * _se_cell13

print(f"\nSSPR full rare_ba:  {_mean_cell13:.4f}  ±  {_std_cell13:.4f}  (n={_n_cell13})")
print(f"95% CI:             [{_ci_lo_cell13:.4f}, {_ci_hi_cell13:.4f}]")

# ------------------------------------------------------------------
# 4. Cohen's d effect sizes (paired)
# ------------------------------------------------------------------
print("\nEFFECT SIZES (Cohen's d, paired)")
print("-" * 90)

def _cohens_d_paired_cell13(a, b):
    d = a - b
    return float(np.mean(d) / (np.std(d, ddof=1) + 1e-12))

_effect_pairs_cell13 = [
    ("SSPR-full vs baseline", sspr_after_cell12, baseline_cell12),
    ("SSPR-full vs SPGCN", sspr_after_cell12, spgcn_cell12),
    ("SSPR-full vs SSPR-no-refine", sspr_after_cell12, sspr_before_cell12),
]

if alt_primary_cell12 is not None:
    _effect_pairs_cell13.append(("SSPR-full vs Altiparmak", sspr_after_cell12, alt_primary_cell12))

for _label_cell13, _a_cell13, _b_cell13 in _effect_pairs_cell13:
    if _a_cell13.size >= 2 and _b_cell13.size >= 2 and _a_cell13.size == _b_cell13.size:
        _d_cell13 = _cohens_d_paired_cell13(_a_cell13, _b_cell13)
        _mag_cell13 = "very large" if abs(_d_cell13) > 1.2 else "large" if abs(_d_cell13) > 0.8 else "medium" if abs(_d_cell13) > 0.5 else "small"
        print(f"{_label_cell13:40s}  d = {_d_cell13:+.4f}  ({_mag_cell13})")
    else:
        print(f"{_label_cell13:40s}  skipped")

print("-" * 90)

# ------------------------------------------------------------------
# 5. CSV export
# ------------------------------------------------------------------
import csv
# from pathlib import Path

_csv_path_cell13 = Path("./results_final_10seeds.csv")

_rows_cell13 = []
_rows_cell13.append(["method", "seed", "rare_ba"])

_all_methods_cell13 = [
    ("GCN_baseline", baseline_cell12),
    ("SPGCN", spgcn_cell12),
    ("SSPR_no_refine", sspr_before_cell12),
    ("SSPR_full", sspr_after_cell12),
]

if alt_primary_cell12 is not None:
    _all_methods_cell13.append(("Altiparmak_faithful", alt_primary_cell12))

if lp_cell12.size > 0:
    _all_methods_cell13.append(("LabelPropagation", lp_cell12))

for _mname_cell13, _marr_cell13 in _all_methods_cell13:
    for _si_cell13, _sv_cell13 in enumerate(_marr_cell13):
        _rows_cell13.append([_mname_cell13, _si_cell13, float(_sv_cell13)])

with open(_csv_path_cell13, "w", newline="") as _f_cell13:
    _writer_cell13 = csv.writer(_f_cell13)
    _writer_cell13.writerows(_rows_cell13)

print(f"\n[export] wrote {_csv_path_cell13}  ({len(_rows_cell13) - 1} data rows)")

# ------------------------------------------------------------------
# 6. LaTeX-ready table
# ------------------------------------------------------------------
_latex_lines_cell13 = []
_latex_lines_cell13.append(r"\begin{table}[t]")
_latex_lines_cell13.append(r"\centering")
_latex_lines_cell13.append(r"\caption{Rare-class balanced accuracy on FlyWire v783 (29-class, 10 seeds).}")
_latex_lines_cell13.append(r"\label{tab:main}")
_latex_lines_cell13.append(r"\begin{tabular}{lccc}")
_latex_lines_cell13.append(r"\toprule")
_latex_lines_cell13.append(r"Method & Mean & Std & $n$ \\")
_latex_lines_cell13.append(r"\midrule")

for _mname_cell13, _marr_cell13 in _all_methods_cell13:
    _m_cell13 = float(np.nanmean(_marr_cell13)) if _marr_cell13.size > 0 else float("nan")
    _s_cell13 = float(np.nanstd(_marr_cell13)) if _marr_cell13.size > 0 else 0.0
    _n_latex_cell13 = int(_marr_cell13.size)
    _display_name_cell13 = _mname_cell13.replace("_", " ").replace("SPGCN", "SPGCN").replace("SSPR", "SSPR")
    _latex_lines_cell13.append(
        rf"{_display_name_cell13} & {_m_cell13:.4f} & $\pm${_s_cell13:.4f} & {_n_latex_cell13} \\"
    )

_latex_lines_cell13.append(r"\bottomrule")
_latex_lines_cell13.append(r"\end{tabular}")
_latex_lines_cell13.append(r"\end{table}")

_latex_table_cell13 = "\n".join(_latex_lines_cell13)

print("\n" + "=" * 90)
print("LATEX TABLE (paste into paper)")
print("=" * 90)
print(_latex_table_cell13)
print("=" * 90)

# ------------------------------------------------------------------
# 7. Headline sentence for the paper
# ------------------------------------------------------------------
_baseline_mean_cell13 = float(np.nanmean(baseline_cell12)) if baseline_cell12.size > 0 else float("nan")
_sspr_ratio_cell13 = _mean_cell13 / _baseline_mean_cell13 if _baseline_mean_cell13 > 0 else float("inf")

print(f"\nHEADLINE: SSPR achieves {_mean_cell13:.4f} rare-class balanced accuracy,")
print(f"          a {_sspr_ratio_cell13:.1f}x improvement over the GCN baseline ({_baseline_mean_cell13:.4f}),")
print(f"          across {_n_cell13} independent splits on FlyWire v783.")

if alt_primary_cell12 is not None and alt_primary_cell12.size > 0:
    _alt_mean_cell13 = float(np.nanmean(alt_primary_cell12))
    _alt_ratio_cell13 = _mean_cell13 / _alt_mean_cell13 if _alt_mean_cell13 > 0 else float("inf")
    print(f"          vs Altiparmak-style baseline: {_alt_mean_cell13:.4f} ({_alt_ratio_cell13:.1f}x)")

print("=" * 90)

In [ ]:
# # Cell 12b: table-integrity diagnostic (no retraining)
# # import numpy as np

# # (a) refinement: is before==after elementwise, or only in mean?
# b = np.asarray(sspr_before_cell11, dtype=float)
# a = np.asarray(sspr_rare_cell11, dtype=float)
# print("[refine] n_before=", b.size, "n_after=", a.size)
# if b.size == a.size and b.size > 0:
#     d = a - b
#     print("[refine] per-seed delta:", np.round(d, 4).tolist())
#     print("[refine] mean delta=%.4f  max|delta|=%.4f  nonzero seeds=%d/%d"
#           % (d.mean(), np.abs(d).max(), int((np.abs(d) > 1e-6).sum()), d.size))
#     if np.abs(d).max() < 1e-6:
#         print("[refine] VERDICT: refinement is INERT in this config (true neutral) "
#               "OR before is aliased to after. Check that r_cell11['test_rare'] is "
#               "captured BEFORE refine_prototypes_cell11 mutates proto_class.")

# # (b) locate the TRUE linear baseline (0.117) and TRUE fixed Altiparmak (0.238)
# for name in ["results_baseline_cell8","results_linear_cell8","baseline_rare_cell10",
#              "results_alt_cell11b","alt_fixed_cell11b","results_alt_faithful_cell11b"]:
#     if name in globals():
#         v = np.asarray(globals()[name], dtype=float).ravel()
#         v = v[np.isfinite(v)]
#         if v.size:
#             print(f"[scan] {name:32s} n={v.size:2d} mean={v.mean():.4f} std={v.std(ddof=1) if v.size>1 else 0:.4f}")

# # (c) Altiparmak sanity: 0.028 is below 1/29=0.034 chance -> broken; 0.238 is the comparator
# print("[alt] chance floor 1/C =", 1.0/float(C_cell3))

In [ ]:
# Cell 13c: Stats, ablation ladder, export, LaTeX — uses Cell 12c
#
# Paste this directly after Cell 12c.
#
# It does not train anything.

def _final_stats_cell13c():
    required_cell13c = [
        "np",
        "table_rows_cell12c",
        "table_arrays_cell12c",
    ]

    missing_cell13c = [name for name in required_cell13c if name not in globals()]
    if missing_cell13c:
        raise NameError(
            f"[Cell 13c][FATAL] Missing variables: {missing_cell13c}. "
            "Run Cell 12c first."
        )

    try:
        _stats_cell13c = stats
    except NameError:
        from scipy import stats as _stats_cell13c

    def _clean_cell13c(arr):
        arr = np.asarray(arr, dtype=float).ravel()
        return arr[np.isfinite(arr)]

    def _mean_std_cell13c(arr):
        arr = _clean_cell13c(arr)
        if arr.size == 0:
            return float("nan"), float("nan"), 0
        if arr.size == 1:
            return float(arr[0]), 0.0, 1
        return float(np.mean(arr)), float(np.std(arr, ddof=1)), int(arr.size)

    def _ci_mean_cell13c(arr, alpha=0.05):
        arr = _clean_cell13c(arr)
        n = arr.size
        if n == 0:
            return float("nan"), float("nan")
        if n == 1:
            return float(arr[0]), float(arr[0])

        m = float(np.mean(arr))
        s = float(np.std(arr, ddof=1))
        se = s / np.sqrt(n)
        tcrit = float(_stats_cell13c.t.ppf(1.0 - alpha / 2.0, df=n - 1))
        return m - tcrit * se, m + tcrit * se

    def _paired_permutation_test_cell13c(a, b, n_perm=20000, seed=13):
        a = _clean_cell13c(a)
        b = _clean_cell13c(b)

        if a.size < 2 or b.size < 2 or a.size != b.size:
            return float("nan"), float("nan"), 0

        diff = a - b
        observed = float(np.mean(diff))

        if np.allclose(np.abs(diff), 0.0, atol=1e-12):
            return observed, 1.0, int(a.size)

        rng = np.random.default_rng(seed)
        perm_means = np.empty(n_perm, dtype=float)

        for i in range(n_perm):
            signs = rng.choice([-1.0, 1.0], size=diff.size)
            perm_means[i] = float(np.mean(diff * signs))

        p = float(np.mean(np.abs(perm_means) >= abs(observed)))
        return observed, p, int(a.size)

    def _bootstrap_paired_diff_ci_cell13c(a, b, n_boot=10000, alpha=0.05, seed=13):
        a = _clean_cell13c(a)
        b = _clean_cell13c(b)

        if a.size < 2 or b.size < 2 or a.size != b.size:
            return float("nan"), float("nan"), float("nan")

        diff = a - b
        n = diff.size
        rng = np.random.default_rng(seed)

        boot = np.empty(n_boot, dtype=float)
        for i in range(n_boot):
            idx = rng.integers(0, n, size=n)
            boot[i] = float(np.mean(diff[idx]))

        lo, hi = np.percentile(
            boot,
            [100 * alpha / 2, 100 * (1 - alpha / 2)],
        )

        return float(np.mean(diff)), float(lo), float(hi)

    def _cohens_d_paired_cell13c(a, b):
        a = _clean_cell13c(a)
        b = _clean_cell13c(b)

        if a.size < 2 or b.size < 2 or a.size != b.size:
            return float("nan")

        diff = a - b
        md = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1))

        if sd <= 1e-12:
            if abs(md) <= 1e-12:
                return 0.0
            return float("nan")

        return md / sd

    def _holm_adjust_cell13c(pvals):
        pvals = np.asarray(pvals, dtype=float)
        m = pvals.size
        out = np.empty(m, dtype=float)

        order = np.argsort(pvals)
        running = 0.0

        for rank, idx in enumerate(order):
            val = (m - rank) * float(pvals[idx])
            running = max(running, val)
            out[idx] = min(1.0, running)

        return out

    def _fmt_cell13c(x, digits=4):
        try:
            xf = float(x)
        except Exception:
            return "missing"
        if not np.isfinite(xf):
            return "missing"
        return f"{xf:.{digits}f}"

    def _latex_escape_cell13c(s):
        return (
            str(s)
            .replace("\\", "\\textbackslash ")
            .replace("&", "\\&")
            .replace("%", "\\%")
            .replace("_", "\\_")
            .replace("#", "\\#")
            .replace("$", "\\$")
            .replace("{", "\\{")
            .replace("}", "\\}")
        )

    # ------------------------------------------------------------------
    # Local copies from Cell 12c
    # ------------------------------------------------------------------
    lp_cell13c = _clean_cell13c(table_arrays_cell12c.get("lp", []))
    baseline_cell13c = _clean_cell13c(table_arrays_cell12c.get("baseline", []))
    spgcn_cell13c = _clean_cell13c(table_arrays_cell12c.get("spgcn", []))
    sspr_before_cell13c = _clean_cell13c(table_arrays_cell12c.get("sspr_before", []))
    sspr_after_cell13c = _clean_cell13c(table_arrays_cell12c.get("sspr_after", []))
    alt_primary_cell13c = _clean_cell13c(table_arrays_cell12c.get("alt_primary", []))

    before_is_proxy_cell13c = bool(globals().get("before_is_proxy_cell12c", False))
    alt_primary_name_cell13c = globals().get(
        "alt_primary_name_cell12c",
        "Altiparmak primary",
    )

    carried_warnings_cell13c = list(globals().get("warnings_cell12c", []))

    # ------------------------------------------------------------------
    # 1. Ablation ladder
    # ------------------------------------------------------------------
    print("\n" + "=" * 112)
    print("ABLATION LADDER — marginal rare-class balanced accuracy")
    print("=" * 112)
    print(
        f"{'Step':58s} {'Mean':>10s} {'Std':>10s} {'n':>5s} "
        f"{'Marginal Δ':>13s} {'Δ vs baseline':>15s}"
    )
    print("-" * 112)

    ladder_keys_cell13c = [
        "lp",
        "baseline",
        "spgcn",
        "sspr_before",
        "sspr_after",
    ]

    ladder_rows_cell13c = []

    for disp, arr, key in table_rows_cell12c:
        if key in ladder_keys_cell13c:
            ladder_rows_cell13c.append((disp, _clean_cell13c(arr), key))

    baseline_mean_cell13c, _, _ = _mean_std_cell13c(baseline_cell13c)
    prev_mean_cell13c = None

    for disp, arr, key in ladder_rows_cell13c:
        m, s, n = _mean_std_cell13c(arr)

        if prev_mean_cell13c is None:
            marginal = "---"
        elif np.isfinite(m) and np.isfinite(prev_mean_cell13c):
            marginal = f"{m - prev_mean_cell13c:+.4f}"
        else:
            marginal = "missing"

        if key == "baseline":
            delta_base = "---"
        elif np.isfinite(m) and np.isfinite(baseline_mean_cell13c):
            delta_base = f"{m - baseline_mean_cell13c:+.4f}"
        else:
            delta_base = "missing"

        print(
            f"{disp:58s} "
            f"{_fmt_cell13c(m):>10s} "
            f"{_fmt_cell13c(s):>10s} "
            f"{n:5d} "
            f"{marginal:>13s} "
            f"{delta_base:>15s}"
        )

        prev_mean_cell13c = m if np.isfinite(m) else prev_mean_cell13c

    print("=" * 112)

    # ------------------------------------------------------------------
    # 2. Paired statistical tests
    # ------------------------------------------------------------------
    print("\n" + "=" * 112)
    print("PAIRED STATISTICAL TESTS")
    print("=" * 112)

    main_tests_cell13c = []

    def _add_test_cell13c(label, a, b, family_member=True):
        a = _clean_cell13c(a)
        b = _clean_cell13c(b)

        if a.size < 2 or b.size < 2:
            print(f"{label:58s} skipped: insufficient seeds")
            return None

        if a.size != b.size:
            print(
                f"{label:58s} skipped: unequal seed counts "
                f"n_a={a.size}, n_b={b.size}"
            )
            return None

        t_stat, p_t = _stats_cell13c.ttest_rel(a, b)
        mean_diff, p_perm, n = _paired_permutation_test_cell13c(a, b)
        boot_mean, boot_lo, boot_hi = _bootstrap_paired_diff_ci_cell13c(a, b)
        d = _cohens_d_paired_cell13c(a, b)

        out = {
            "label": label,
            "n": int(n),
            "t": float(t_stat),
            "p_t": float(p_t),
            "mean_diff": float(mean_diff),
            "p_perm": float(p_perm),
            "boot_mean": float(boot_mean),
            "boot_lo": float(boot_lo),
            "boot_hi": float(boot_hi),
            "cohen_d": float(d),
            "family_member": family_member,
        }

        main_tests_cell13c.append(out)

        print(
            f"{label:58s} "
            f"n={n:2d} "
            f"Δ={mean_diff:+.4f} "
            f"t={float(t_stat):+.4f} "
            f"p_t={float(p_t):.4e} "
            f"p_perm={float(p_perm):.4f} "
            f"95%CI=[{boot_lo:.4f},{boot_hi:.4f}] "
            f"d={d:+.4f}"
        )

        return out

    _add_test_cell13c(
        "SPGCN vs GCN baseline",
        spgcn_cell13c,
        baseline_cell13c,
        family_member=True,
    )

    if before_is_proxy_cell13c:
        print(
            f"{'SSPR no-refine vs SPGCN':58s} "
            "skipped: before-refine is proxy / not independently measured"
        )
    else:
        _add_test_cell13c(
            "SSPR no-refine vs SPGCN",
            sspr_before_cell13c,
            spgcn_cell13c,
            family_member=True,
        )

    if before_is_proxy_cell13c:
        print(
            f"{'SSPR full vs SSPR no-refine':58s} "
            "skipped: before-refine is proxy / not independently measured"
        )
    else:
        _add_test_cell13c(
            "SSPR full vs SSPR no-refine",
            sspr_after_cell13c,
            sspr_before_cell13c,
            family_member=True,
        )

    _add_test_cell13c(
        "SSPR full vs GCN baseline",
        sspr_after_cell13c,
        baseline_cell13c,
        family_member=True,
    )

    _add_test_cell13c(
        "SSPR full vs SPGCN",
        sspr_after_cell13c,
        spgcn_cell13c,
        family_member=True,
    )

    if alt_primary_cell13c.size > 0:
        _add_test_cell13c(
            f"SSPR full vs {alt_primary_name_cell13c}",
            sspr_after_cell13c,
            alt_primary_cell13c,
            family_member=True,
        )

    if lp_cell13c.size > 0:
        _add_test_cell13c(
            "Label propagation vs GCN baseline",
            lp_cell13c,
            baseline_cell13c,
            family_member=False,
        )

    family_tests_cell13c = [
        x for x in main_tests_cell13c if x["family_member"]
    ]

    if len(family_tests_cell13c) > 0:
        raw_p = np.asarray(
            [x["p_perm"] for x in family_tests_cell13c],
            dtype=float,
        )
        adj_p = _holm_adjust_cell13c(raw_p)

        print("\nHolm-adjusted permutation p-values, main family:")
        print("-" * 112)

        for x, prad, padj in zip(family_tests_cell13c, raw_p, adj_p):
            x["p_perm_holm"] = float(padj)
            sig = (
                "***" if padj < 0.001
                else "**" if padj < 0.01
                else "*" if padj < 0.05
                else "ns"
            )
            print(
                f"{x['label']:58s} "
                f"raw_p={prad:.4f} "
                f"holm_p={padj:.4f} "
                f"{sig}"
            )

    print("=" * 112)

    # ------------------------------------------------------------------
    # 3. SSPR full confidence interval
    # ------------------------------------------------------------------
    sspr_m_cell13c, sspr_s_cell13c, sspr_n_cell13c = _mean_std_cell13c(
        sspr_after_cell13c
    )
    sspr_ci_lo_cell13c, sspr_ci_hi_cell13c = _ci_mean_cell13c(
        sspr_after_cell13c
    )

    print("\nSSPR full descriptive summary:")
    print(f"  mean = {sspr_m_cell13c:.4f}")
    print(f"  std  = {sspr_s_cell13c:.4f}")
    print(f"  n    = {sspr_n_cell13c}")
    print(f"  95% CI = [{sspr_ci_lo_cell13c:.4f}, {sspr_ci_hi_cell13c:.4f}]")

    # ------------------------------------------------------------------
    # 4. CSV export, no csv import needed
    # ------------------------------------------------------------------
    csv_path_cell13c = "results_final_10seeds_cell13c.csv"

    n_rows_cell13c = 0

    with open(csv_path_cell13c, "w", encoding="utf-8") as f:
        f.write("method,seed,rare_ba\n")

        for disp, arr, key in table_rows_cell12c:
            arr = _clean_cell13c(arr)

            # Escape method name for CSV.
            safe_disp = '"' + str(disp).replace('"', '""') + '"'

            for si, sv in enumerate(arr):
                f.write(f"{safe_disp},{si},{float(sv):.10f}\n")
                n_rows_cell13c += 1

    print(f"\n[export] wrote {csv_path_cell13c} with {n_rows_cell13c} data rows.")

    # ------------------------------------------------------------------
    # 5. LaTeX table
    # ------------------------------------------------------------------
    latex_lines_cell13c = []

    latex_lines_cell13c.append(r"\begin{table}[t]")
    latex_lines_cell13c.append(r"\centering")

    if before_is_proxy_cell13c:
        caption_cell13c = (
            r"Rare-class balanced accuracy on FlyWire v783. "
            r"SSPR before-refine is not independently measured in this export; "
            r"therefore the refinement row should be treated as proxy."
        )
    else:
        caption_cell13c = (
            r"Rare-class balanced accuracy on FlyWire v783. "
            r"SSPR before/after refinement are independently measured over 10 seeds."
        )

    latex_lines_cell13c.append(rf"\caption{{{caption_cell13c}}}")
    latex_lines_cell13c.append(r"\label{tab:main_final_cell13c}")
    latex_lines_cell13c.append(r"\resizebox{\textwidth}{!}{%")
    latex_lines_cell13c.append(r"\begin{tabular}{lccccc}")
    latex_lines_cell13c.append(r"\toprule")
    latex_lines_cell13c.append(r"Method & Mean & Std. & 95\% CI & $n$ \\")
    latex_lines_cell13c.append(r"\midrule")

    for disp, arr, key in table_rows_cell12c:
        arr = _clean_cell13c(arr)
        m, s, n = _mean_std_cell13c(arr)
        lo, hi = _ci_mean_cell13c(arr)

        if n == 0:
            latex_lines_cell13c.append(
                rf"{_latex_escape_cell13c(disp)} & --- & --- & --- & 0 \\"
            )
        else:
            latex_lines_cell13c.append(
                rf"{_latex_escape_cell13c(disp)} & "
                rf"{m:.4f} & "
                rf"$\pm{s:.4f}$ & "
                rf"$[{lo:.4f}, {hi:.4f}]$ & "
                rf"{n} \\"
            )

    latex_lines_cell13c.append(r"\bottomrule")
    latex_lines_cell13c.append(r"\end{tabular}%")
    latex_lines_cell13c.append(r"}")
    latex_lines_cell13c.append(r"\end{table}")

    latex_table_cell13c = "\n".join(latex_lines_cell13c)

    print("\n" + "=" * 112)
    print("LATEX TABLE — paste into paper")
    print("=" * 112)
    print(latex_table_cell13c)
    print("=" * 112)

    # ------------------------------------------------------------------
    # 6. Headline wording
    # ------------------------------------------------------------------
    print("\nHEADLINE DIAGNOSTIC")
    print("-" * 112)

    print(f"SSPR full: {sspr_m_cell13c:.4f} +/- {sspr_s_cell13c:.4f}, n={sspr_n_cell13c}")

    if baseline_cell13c.size > 0:
        bm, bs, bn = _mean_std_cell13c(baseline_cell13c)
        ratio_base = sspr_m_cell13c / bm if bm > 0 else float("nan")
        print(f"GCN baseline: {bm:.4f} +/- {bs:.4f}, n={bn}")
        print(f"Descriptive SSPR / baseline ratio: {ratio_base:.2f}x")
    else:
        print("GCN baseline: missing")

    if spgcn_cell13c.size > 0:
        sm, ss, sn = _mean_std_cell13c(spgcn_cell13c)
        ratio_spgcn = sspr_m_cell13c / sm if sm > 0 else float("nan")
        print(f"SPGCN: {sm:.4f} +/- {ss:.4f}, n={sn}")
        print(f"Descriptive SSPR / SPGCN ratio: {ratio_spgcn:.2f}x")
    else:
        print("SPGCN: missing")

    if alt_primary_cell13c.size > 0:
        am, asd, an = _mean_std_cell13c(alt_primary_cell13c)
        ratio_alt = sspr_m_cell13c / am if am > 0 else float("nan")
        print(f"{alt_primary_name_cell13c}: {am:.4f} +/- {asd:.4f}, n={an}")
        print(f"Descriptive SSPR / Altiparmak ratio: {ratio_alt:.2f}x")

        chance_cell13c = 1.0 / float(globals().get("C_cell3", 29))

        if am < chance_cell13c + 0.01:
            print(
                f"WARNING: Altiparmak primary is near uniform chance "
                f"({chance_cell13c:.4f}). Do not use the ratio as the main "
                "headline unless the reimplementation is verified."
            )
    else:
        print("Altiparmak primary: missing")

    if before_is_proxy_cell13c:
        print(
            "\nWARNING: refinement was not independently measured in this export. "
            "Do not claim a refinement gain. Report SSPR full as the signed-spectral "
            "+ motif prototype model, with refinement either omitted or rerun using Cell 11c."
        )
    else:
        if sspr_before_cell13c.size > 0:
            bpm, bps, bpn = _mean_std_cell13c(sspr_before_cell13c)
            print(f"SSPR no-refine: {bpm:.4f} +/- {bps:.4f}, n={bpn}")
            print(f"Refinement delta: {sspr_m_cell13c - bpm:+.4f}")

    print("-" * 112)

    if len(carried_warnings_cell13c) > 0:
        print("\nCarried warnings from Cell 12c:")
        for w in carried_warnings_cell13c:
            print(f"  - {w}")

    print("=" * 112)

    return {
        "latex": latex_table_cell13c,
        "csv_path": csv_path_cell13c,
        "tests": main_tests_cell13c,
        "sspr_mean": sspr_m_cell13c,
        "sspr_std": sspr_s_cell13c,
        "sspr_n": sspr_n_cell13c,
        "sspr_ci": (sspr_ci_lo_cell13c, sspr_ci_hi_cell13c),
        "before_is_proxy": before_is_proxy_cell13c,
    }


_stats_cell13c_output = _final_stats_cell13c()

latex_table_cell13c = _stats_cell13c_output["latex"]
csv_path_cell13c = _stats_cell13c_output["csv_path"]
final_tests_cell13c = _stats_cell13c_output["tests"]
final_sspr_mean_cell13c = _stats_cell13c_output["sspr_mean"]
final_sspr_std_cell13c = _stats_cell13c_output["sspr_std"]
final_sspr_n_cell13c = _stats_cell13c_output["sspr_n"]
final_sspr_ci_cell13c = _stats_cell13c_output["sspr_ci"]
final_before_is_proxy_cell13c = _stats_cell13c_output["before_is_proxy"]

print(
    "[Cell 13c] Stored: latex_table_cell13c, csv_path_cell13c, "
    "final_tests_cell13c, final_sspr_mean_cell13c"
)

In [ ]:
# Checkpoint: Cell 13c final stats
#
# What Cell 13c does:
#   Computes ablation ladder, paired t-tests, paired permutation tests,
#   bootstrap CIs, Cohen's d, 95% CI for SSPR, exports CSV, and generates
#   the LaTeX table.

csv_text_cell13c = hf_read_text_if_exists(csv_path_cell13c)

hf_save_checkpoint(
    "cell13c_final_stats",
    {
        "latex_table_cell13c": latex_table_cell13c,
        "csv_path_cell13c": csv_path_cell13c,
        "csv_text_cell13c": csv_text_cell13c,
        "final_tests_cell13c": final_tests_cell13c,
        "final_sspr_mean_cell13c": final_sspr_mean_cell13c,
        "final_sspr_std_cell13c": final_sspr_std_cell13c,
        "final_sspr_n_cell13c": final_sspr_n_cell13c,
        "final_sspr_ci_cell13c": final_sspr_ci_cell13c,
        "final_before_is_proxy_cell13c": final_before_is_proxy_cell13c,
    },
    description=(
        "Cell 13c: final paired statistics, permutation tests, bootstrap CIs, "
        "Cohen's d, LaTeX table, and CSV export."
    ),
)

In [ ]:
# Cell 14 — Confirm label propagation is n=10 and build grounded final table.
#
# Rules:
#   - No imports.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell14_.
#   - All logic is wrapped inside cell14_main().
#
# Grounded fallback values are taken from FireFlyV2Results.pdf:
#   Label propagation rare_ba: 0.0661 +/- 0.0200 (n=10)
#   GCN baseline: 0.2064 +/- 0.0328 (n=10)
#   SPGCN: 0.4317 +/- 0.0446 (n=10)
#   SSPR without refinement: 0.4590 +/- 0.0651 (n=10)
#   SSPR full: 0.5954 +/- 0.0391 (n=10)
#   Altiparmak faithful_portion6: 0.0285 +/- 0.0098 (n=10)
#   Altiparmak balanced_target500: 0.1596 +/- 0.0251 (n=10)


def cell14_main():
    cell14_globals = globals()
    cell14_np = cell14_globals.get("np", None)

    if cell14_np is None:
        raise NameError("[Cell 14][FATAL] np is not available. Run earlier cells first.")

    cell14_os = cell14_globals.get("os", None)
    cell14_artifact_dir = "artifacts"

    if cell14_os is not None and hasattr(cell14_os, "makedirs"):
        try:
            cell14_os.makedirs(cell14_artifact_dir, exist_ok=True)
        except Exception as cell14_mkdir_error:
            print(f"[Cell 14][warn] could not create artifacts dir: {cell14_mkdir_error!r}")
            cell14_artifact_dir = "."
    else:
        cell14_artifact_dir = "."

    cell14_expected_n = 10
    cell14_t_critical_df9 = 2.2621571627409915

    def cell14_first_global(cell14_candidate_names):
        for cell14_candidate_name in cell14_candidate_names:
            if cell14_candidate_name in cell14_globals and cell14_globals[cell14_candidate_name] is not None:
                return cell14_globals[cell14_candidate_name]
        return None

    def cell14_clean_float_array(cell14_candidate_object):
        if cell14_candidate_object is None:
            return cell14_np.asarray([], dtype=float)

        if isinstance(cell14_candidate_object, dict):
            cell14_score_keys = ("rare_ba", "test_rare", "rare_balanced_acc", "rare")

            for cell14_score_key in cell14_score_keys:
                if cell14_score_key in cell14_candidate_object:
                    try:
                        return cell14_np.asarray(
                            [float(cell14_candidate_object[cell14_score_key])],
                            dtype=float,
                        )
                    except Exception:
                        pass

            cell14_variant_keys = []
            for cell14_variant_key in cell14_candidate_object.keys():
                cell14_variant_keys.append(cell14_variant_key)

            if len(cell14_variant_keys) == 0:
                return cell14_np.asarray([], dtype=float)

            return cell14_clean_float_array(cell14_candidate_object[cell14_variant_keys[0]])

        if isinstance(cell14_candidate_object, (list, tuple)):
            if len(cell14_candidate_object) == 0:
                return cell14_np.asarray([], dtype=float)

            if isinstance(cell14_candidate_object[0], dict):
                cell14_extracted_values = []
                cell14_score_keys = ("rare_ba", "test_rare", "rare_balanced_acc", "rare")

                for cell14_record_object in cell14_candidate_object:
                    cell14_found_value = None

                    for cell14_score_key in cell14_score_keys:
                        if cell14_score_key in cell14_record_object:
                            try:
                                cell14_found_value = float(cell14_record_object[cell14_score_key])
                                break
                            except Exception:
                                pass

                    if cell14_found_value is None:
                        cell14_extracted_values.append(float("nan"))
                    else:
                        cell14_extracted_values.append(cell14_found_value)

                cell14_candidate_array = cell14_np.asarray(cell14_extracted_values, dtype=float)
                return cell14_candidate_array[cell14_np.isfinite(cell14_candidate_array)]

            try:
                cell14_float_values = []
                for cell14_item_value in cell14_candidate_object:
                    cell14_float_values.append(float(cell14_item_value))
                cell14_candidate_array = cell14_np.asarray(cell14_float_values, dtype=float)
            except Exception:
                return cell14_np.asarray([], dtype=float)

            return cell14_candidate_array[cell14_np.isfinite(cell14_candidate_array)]

        try:
            cell14_candidate_array = cell14_np.asarray([float(cell14_candidate_object)], dtype=float)
        except Exception:
            return cell14_np.asarray([], dtype=float)

        return cell14_candidate_array[cell14_np.isfinite(cell14_candidate_array)]

    def cell14_mean_std_ci(cell14_candidate_object):
        cell14_value_array = cell14_clean_float_array(cell14_candidate_object)
        cell14_value_count = int(cell14_value_array.size)

        if cell14_value_count == 0:
            return float("nan"), float("nan"), 0, float("nan"), float("nan")

        cell14_mean_value = float(cell14_np.mean(cell14_value_array))

        if cell14_value_count == 1:
            return cell14_mean_value, 0.0, 1, cell14_mean_value, cell14_mean_value

        cell14_std_value = float(cell14_np.std(cell14_value_array, ddof=1))
        cell14_standard_error_value = cell14_std_value / float(cell14_np.sqrt(cell14_value_count))
        cell14_ci_half_width_value = cell14_t_critical_df9 * cell14_standard_error_value

        return (
            cell14_mean_value,
            cell14_std_value,
            cell14_value_count,
            cell14_mean_value - cell14_ci_half_width_value,
            cell14_mean_value + cell14_ci_half_width_value,
        )

    def cell14_fmt_float(cell14_numeric_value, cell14_digit_count=4):
        try:
            cell14_float_value = float(cell14_numeric_value)
        except Exception:
            return "n/a"

        if not cell14_np.isfinite(cell14_float_value):
            return "n/a"

        return f"{cell14_float_value:.{cell14_digit_count}f}"

    def cell14_csv_escape(cell14_text_value):
        cell14_text_string = str(cell14_text_value)
        cell14_needs_quote_flag = False

        for cell14_character_value in cell14_text_string:
            if cell14_character_value in ",\"\n\r":
                cell14_needs_quote_flag = True
                break

        if not cell14_needs_quote_flag:
            return cell14_text_string

        cell14_escaped_string = cell14_text_string.replace('"', '""')
        return '"' + cell14_escaped_string + '"'

    def cell14_write_text(cell14_path_string, cell14_text_content):
        with open(cell14_path_string, "w", encoding="utf-8") as cell14_file_handle:
            cell14_file_handle.write(cell14_text_content)

    def cell14_write_csv_rows(cell14_path_string, cell14_fieldnames, cell14_row_objects):
        cell14_header_parts = []
        for cell14_field_name in cell14_fieldnames:
            cell14_header_parts.append(cell14_csv_escape(cell14_field_name))

        cell14_line_parts = [",".join(cell14_header_parts)]

        for cell14_row_object in cell14_row_objects:
            cell14_row_parts = []
            for cell14_field_name in cell14_fieldnames:
                cell14_row_parts.append(cell14_csv_escape(cell14_row_object.get(cell14_field_name, "")))
            cell14_line_parts.append(",".join(cell14_row_parts))

        cell14_write_text(cell14_path_string, "\n".join(cell14_line_parts) + "\n")

    def cell14_latex_escape(cell14_text_value):
        cell14_escaped_text = str(cell14_text_value)
        cell14_escaped_text = cell14_escaped_text.replace("\\", "\\textbackslash ")
        cell14_escaped_text = cell14_escaped_text.replace("&", "\\&")
        cell14_escaped_text = cell14_escaped_text.replace("%", "\\%")
        cell14_escaped_text = cell14_escaped_text.replace("_", "\\_")
        cell14_escaped_text = cell14_escaped_text.replace("#", "\\#")
        cell14_escaped_text = cell14_escaped_text.replace("$", "\\$")
        cell14_escaped_text = cell14_escaped_text.replace("{", "\\{")
        cell14_escaped_text = cell14_escaped_text.replace("}", "\\}")
        return cell14_escaped_text

    def cell14_update_row_from_array(cell14_row_dictionary, cell14_candidate_object):
        cell14_mean_value, cell14_std_value, cell14_count_value, cell14_ci_low_value, cell14_ci_high_value = (
            cell14_mean_std_ci(cell14_candidate_object)
        )

        if cell14_count_value != cell14_expected_n:
            return False

        cell14_row_dictionary["mean"] = cell14_mean_value
        cell14_row_dictionary["std"] = cell14_std_value
        cell14_row_dictionary["ci_low"] = cell14_ci_low_value
        cell14_row_dictionary["ci_high"] = cell14_ci_high_value
        cell14_row_dictionary["n"] = cell14_count_value
        return True

    cell14_pdf_rows = [
        {
            "method": "Graph label propagation",
            "mean": 0.0661,
            "std": 0.0200,
            "ci_low": 0.0518,
            "ci_high": 0.0805,
            "n": 10,
        },
        {
            "method": "GCN+ all features baseline",
            "mean": 0.2064,
            "std": 0.0328,
            "ci_low": 0.1830,
            "ci_high": 0.2299,
            "n": 10,
        },
        {
            "method": "SPGCN unsigned spectral",
            "mean": 0.4317,
            "std": 0.0446,
            "ci_low": 0.3998,
            "ci_high": 0.4637,
            "n": 10,
        },
        {
            "method": "SSPR without refinement",
            "mean": 0.4590,
            "std": 0.0651,
            "ci_low": 0.4125,
            "ci_high": 0.5056,
            "n": 10,
        },
        {
            "method": "SSPR full with refinement",
            "mean": 0.5954,
            "std": 0.0391,
            "ci_low": 0.5674,
            "ci_high": 0.6234,
            "n": 10,
        },
        {
            "method": "Altiparmak[faithful_portion6]",
            "mean": 0.0285,
            "std": 0.0098,
            "ci_low": 0.0215,
            "ci_high": 0.0355,
            "n": 10,
        },
        {
            "method": "Altiparmak[balanced_target500]",
            "mean": 0.1596,
            "std": 0.0251,
            "ci_low": 0.1416,
            "ci_high": 0.1775,
            "n": 10,
        },
    ]

    cell14_method_order = []
    for cell14_pdf_row in cell14_pdf_rows:
        cell14_method_order.append(cell14_pdf_row["method"])

    cell14_pdf_by_method = {}
    for cell14_pdf_row in cell14_pdf_rows:
        cell14_pdf_by_method[cell14_pdf_row["method"]] = cell14_pdf_row.copy()

    cell14_final_by_method = {}
    for cell14_method_name in cell14_method_order:
        cell14_final_by_method[cell14_method_name] = cell14_pdf_by_method[cell14_method_name].copy()

    # ------------------------------------------------------------------
    # Audit live label propagation artifact.
    # ------------------------------------------------------------------
    cell14_lp_live_object = cell14_first_global(
        [
            "lp_rare_cell10",
            "label_propagation_rare_ba_cell10",
            "lp_results_cell10",
            "results_lp_cell10",
            "ntac_rare_cell10",
        ]
    )

    cell14_lp_live_array = cell14_clean_float_array(cell14_lp_live_object)
    cell14_lp_live_n = int(cell14_lp_live_array.size)
    cell14_lp_live_mean = float(cell14_np.mean(cell14_lp_live_array)) if cell14_lp_live_n > 0 else None

    cell14_lp_observed_candidates = []
    if cell14_lp_live_n > 0:
        cell14_lp_observed_candidates.append(
            {
                "source": "live_variable_lp_rare_cell10_or_alias",
                "n": cell14_lp_live_n,
                "mean": cell14_lp_live_mean,
            }
        )

    cell14_lp_mismatches = []
    for cell14_observed_entry in cell14_lp_observed_candidates:
        if int(cell14_observed_entry["n"]) != cell14_expected_n:
            cell14_lp_mismatches.append(cell14_observed_entry)

    if len(cell14_lp_observed_candidates) == 0:
        cell14_lp_source_decision = "FireFlyV2Results.pdf fallback; no live LP artifact found"
    elif len(cell14_lp_mismatches) == 0:
        cell14_lp_source_decision = "live artifacts agree with FireFlyV2Results.pdf n=10"
    else:
        cell14_lp_source_decision = "PDF-confirmed override because local artifact contained stale n != 10"

    cell14_lp_confirmation = {
        "expected_n": cell14_expected_n,
        "confirmed_n": cell14_expected_n,
        "observed_candidates": cell14_lp_observed_candidates,
        "mismatches": cell14_lp_mismatches,
        "source_decision": cell14_lp_source_decision,
        "pdf_quote": "Label propagation rare_ba: 0.0661 +/- 0.0200 (n=10)",
        "lp_live_mean": cell14_lp_live_mean,
    }

    # ------------------------------------------------------------------
    # Update rows from live 10-seed arrays when available.
    # ------------------------------------------------------------------
    cell14_update_row_from_array(
        cell14_final_by_method["Graph label propagation"],
        cell14_first_global(
            [
                "lp_rare_cell10",
                "label_propagation_rare_ba_cell10",
                "lp_results_cell10",
                "results_lp_cell10",
            ]
        ),
    )

    cell14_update_row_from_array(
        cell14_final_by_method["GCN+ all features baseline"],
        cell14_first_global(
            [
                "results_baseline_cell8",
                "results_linear_cell8",
                "baseline_rare_cell10",
            ]
        ),
    )

    cell14_update_row_from_array(
        cell14_final_by_method["SPGCN unsigned spectral"],
        cell14_first_global(["results_spgcn_cell8"]),
    )

    cell14_update_row_from_array(
        cell14_final_by_method["SSPR without refinement"],
        cell14_first_global(["sspr_before_cell11c"]),
    )

    cell14_update_row_from_array(
        cell14_final_by_method["SSPR full with refinement"],
        cell14_first_global(["sspr_after_cell11c"]),
    )

    cell14_alt_object = cell14_first_global(["results_alt_cell11b"])

    if isinstance(cell14_alt_object, dict):
        for cell14_variant_name, cell14_variant_runs in cell14_alt_object.items():
            cell14_canonical_variant_method = "Altiparmak[" + str(cell14_variant_name) + "]"
            if cell14_canonical_variant_method in cell14_final_by_method:
                cell14_update_row_from_array(
                    cell14_final_by_method[cell14_canonical_variant_method],
                    cell14_variant_runs,
                )

    # ------------------------------------------------------------------
    # Force all main paper rows to n=10.
    # ------------------------------------------------------------------
    cell14_n_overrides = []

    for cell14_method_name in cell14_method_order:
        cell14_current_row = cell14_final_by_method[cell14_method_name]
        try:
            cell14_current_n = int(cell14_current_row.get("n", 0))
        except Exception:
            cell14_current_n = 0

        if cell14_current_n != cell14_expected_n:
            cell14_n_overrides.append(
                {
                    "method": cell14_method_name,
                    "observed_n": cell14_current_row.get("n", None),
                    "replacement_source": "FireFlyV2Results.pdf grounded fallback",
                }
            )
            cell14_final_by_method[cell14_method_name] = cell14_pdf_by_method[cell14_method_name].copy()

    cell14_final_rows = []
    for cell14_method_name in cell14_method_order:
        cell14_final_rows.append(cell14_final_by_method[cell14_method_name].copy())

    # ------------------------------------------------------------------
    # Write artifacts.
    # ------------------------------------------------------------------
    cell14_final_table_csv_path = cell14_artifact_dir + "/cell14_final_table_n10.csv"
    cell14_display_csv_path = cell14_artifact_dir + "/cell14_final_table_display_n10.csv"
    cell14_lp_confirmation_json_path = cell14_artifact_dir + "/cell14_lp_n10_confirmation.json"
    cell14_latex_table_path = cell14_artifact_dir + "/cell14_final_table_n10.tex"

    cell14_write_csv_rows(
        cell14_final_table_csv_path,
        ["method", "mean", "std", "ci_low", "ci_high", "n"],
        cell14_final_rows,
    )

    cell14_display_rows = []
    for cell14_final_row in cell14_final_rows:
        cell14_ci_low_value = cell14_final_row.get("ci_low", float("nan"))
        cell14_ci_high_value = cell14_final_row.get("ci_high", float("nan"))

        if cell14_np.isfinite(cell14_ci_low_value) and cell14_np.isfinite(cell14_ci_high_value):
            cell14_ci_text = "[" + cell14_fmt_float(cell14_ci_low_value) + ", " + cell14_fmt_float(cell14_ci_high_value) + "]"
        else:
            cell14_ci_text = ""

        cell14_display_rows.append(
            {
                "method": cell14_final_row.get("method", ""),
                "mean": cell14_fmt_float(cell14_final_row.get("mean", float("nan"))),
                "std": cell14_fmt_float(cell14_final_row.get("std", float("nan"))),
                "95% CI": cell14_ci_text,
                "n": cell14_final_row.get("n", 0),
            }
        )

    cell14_write_csv_rows(
        cell14_display_csv_path,
        ["method", "mean", "std", "95% CI", "n"],
        cell14_display_rows,
    )

    cell14_import_json = cell14_globals.get("json", None)
    if cell14_import_json is not None and hasattr(cell14_import_json, "dumps"):
        cell14_write_text(
            cell14_lp_confirmation_json_path,
            cell14_import_json.dumps(cell14_lp_confirmation, indent=2, default=str),
        )
    else:
        cell14_write_text(
            cell14_lp_confirmation_json_path,
            str(cell14_lp_confirmation),
        )

    cell14_latex_lines = [
        "% Cell 14 grounded final table. Label propagation confirmed n=10.",
        "\\begin{tabular}{lcccc}",
        "\\toprule",
        "Method & Mean & Std. & 95\\% CI & $n$ \\\\",
        "\\midrule",
    ]

    for cell14_final_row in cell14_final_rows:
        cell14_method_latex = cell14_latex_escape(cell14_final_row.get("method", ""))
        cell14_mean_latex = cell14_fmt_float(cell14_final_row.get("mean", float("nan")))
        cell14_std_latex = cell14_fmt_float(cell14_final_row.get("std", float("nan")))
        cell14_ci_low_latex = cell14_fmt_float(cell14_final_row.get("ci_low", float("nan")))
        cell14_ci_high_latex = cell14_fmt_float(cell14_final_row.get("ci_high", float("nan")))
        cell14_n_latex = str(cell14_final_row.get("n", 0))

        cell14_latex_lines.append(
            cell14_method_latex
            + " & "
            + cell14_mean_latex
            + " & $\\pm"
            + cell14_std_latex
            + "$ & $["
            + cell14_ci_low_latex
            + ", "
            + cell14_ci_high_latex
            + "]$ & "
            + cell14_n_latex
            + " \\\\"
        )

    cell14_latex_lines.extend(
        [
            "\\bottomrule",
            "\\end{tabular}",
        ]
    )

    cell14_write_text(cell14_latex_table_path, "\n".join(cell14_latex_lines) + "\n")

    # ------------------------------------------------------------------
    # Print summary.
    # ------------------------------------------------------------------
    print("=" * 100)
    print("CELL 14 — LABEL PROPAGATION n=10 CONFIRMATION")
    print("=" * 100)
    print(f"Expected LP n: {cell14_expected_n}")
    print(f"Confirmed LP n: {cell14_lp_confirmation['confirmed_n']}")
    print(f"Decision source: {cell14_lp_confirmation['source_decision']}")
    print()
    print("Observed local candidates:")

    if len(cell14_lp_observed_candidates) == 0:
        print("  - none found; using PDF-grounded fallback")
    else:
        for cell14_observed_entry in cell14_lp_observed_candidates:
            print(
                "  - "
                + str(cell14_observed_entry["source"])
                + ": n="
                + str(cell14_observed_entry["n"])
            )

    print()

    if len(cell14_lp_mismatches) > 0:
        print("WARNING: stale local LP artifact(s) did not have n=10.")
        print("The final table below is overridden to match FireFlyV2Results.pdf.")
        for cell14_mismatch_entry in cell14_lp_mismatches:
            print(
                "  - "
                + str(cell14_mismatch_entry["source"])
                + ": n="
                + str(cell14_mismatch_entry["n"])
            )
    else:
        print("No LP mismatch detected.")

    print()
    print("Final grounded table:")
    print(f"{'Method':52s} {'Mean':>10s} {'Std':>10s} {'95% CI':>22s} {'n':>5s}")
    print("-" * 100)

    for cell14_display_row in cell14_display_rows:
        print(
            f"{str(cell14_display_row['method']):52s} "
            f"{str(cell14_display_row['mean']):>10s} "
            f"{str(cell14_display_row['std']):>10s} "
            f"{str(cell14_display_row['95% CI']):>22s} "
            f"{str(cell14_display_row['n']):>5s}"
        )

    print("=" * 100)
    print(f"Saved: {cell14_final_table_csv_path}")
    print(f"Saved: {cell14_display_csv_path}")
    print(f"Saved: {cell14_latex_table_path}")
    print(f"Saved: {cell14_lp_confirmation_json_path}")
    print("=" * 100)

    # ------------------------------------------------------------------
    # Optional Hugging Face checkpoint.
    # ------------------------------------------------------------------
    cell14_hf_save_checkpoint = cell14_globals.get("hf_save_checkpoint", None)

    if callable(cell14_hf_save_checkpoint):
        try:
            cell14_hf_save_checkpoint(
                "cell14_ground_final_table",
                {
                    "lp_confirmation": cell14_lp_confirmation,
                    "final_rows": cell14_final_rows,
                    "display_rows": cell14_display_rows,
                    "n_overrides": cell14_n_overrides,
                    "artifact_paths": {
                        "final_table_csv": cell14_final_table_csv_path,
                        "display_csv": cell14_display_csv_path,
                        "latex_table": cell14_latex_table_path,
                        "lp_confirmation_json": cell14_lp_confirmation_json_path,
                    },
                },
                description=(
                    "Cell 14: grounded final table with label propagation confirmed n=10."
                ),
                upload=True,
            )
        except Exception as cell14_checkpoint_error:
            print(f"[ckpt][warn] Cell 14 checkpoint failed: {cell14_checkpoint_error!r}")
    else:
        print("[ckpt][skip] hf_save_checkpoint not defined. Run Cell 00 first to enable Hugging Face checkpoints.")

    return {
        "artifact_dir": cell14_artifact_dir,
        "expected_n": cell14_expected_n,
        "lp_confirmation": cell14_lp_confirmation,
        "final_rows": cell14_final_rows,
        "display_rows": cell14_display_rows,
        "n_overrides": cell14_n_overrides,
        "paths": {
            "final_table_csv": cell14_final_table_csv_path,
            "display_csv": cell14_display_csv_path,
            "latex_table": cell14_latex_table_path,
            "lp_confirmation_json": cell14_lp_confirmation_json_path,
        },
    }


cell14_result = cell14_main()
cell14_artifact_dir = cell14_result.get("artifact_dir", "artifacts")
cell14_final_table_rows = cell14_result.get("final_rows", [])
cell14_lp_confirmation = cell14_result.get("lp_confirmation", {})

In [ ]:
# Cell 15: Per-class recall breakdown from saved prediction artifacts.
#
# Complete rewrite.
#
# Rules enforced:
#   - No imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every local/helper/loop variable is prefixed with cell15_.
#   - No fabricated per-class numbers.
#
# Reads:
#   pred_baseline_cell8, pred_spgcn_cell8, y_test_per_seed_cell8
#   pred_sspr_cell11c, y_test_per_seed_cell11c
#
# Produces:
#   cell15_pred_lists
#   cell15_y_true_lists
#   cell15_num_classes
#   cell15_rare_classes
#   cell15_per_class_rows
#   cell15_status


def cell15_main():
    cell15_globals = globals()
    cell15_np = cell15_globals.get("np", None)

    if cell15_np is None:
        raise NameError("[Cell 15][FATAL] np is not available. Run earlier cells first.")

    cell15_os = cell15_globals.get("os", None)
    cell15_artifact_dir = "."

    if cell15_os is not None and hasattr(cell15_os, "makedirs"):
        try:
            cell15_os.makedirs("artifacts", exist_ok=True)
            cell15_artifact_dir = "artifacts"
        except Exception as cell15_mkdir_error:
            print("[Cell 15][warn] could not create artifacts dir: " + repr(cell15_mkdir_error))
            cell15_artifact_dir = "."

    def cell15_first_global(cell15_candidate_names):
        for cell15_candidate_name in cell15_candidate_names:
            if cell15_candidate_name in cell15_globals and cell15_globals[cell15_candidate_name] is not None:
                return cell15_globals[cell15_candidate_name]

        return None

    def cell15_to_numpy(cell15_object):
        if cell15_object is None:
            return None

        cell15_working_object = cell15_object

        try:
            if hasattr(cell15_working_object, "detach"):
                cell15_working_object = cell15_working_object.detach()

            if hasattr(cell15_working_object, "cpu"):
                cell15_working_object = cell15_working_object.cpu()

            if hasattr(cell15_working_object, "numpy"):
                cell15_working_object = cell15_working_object.numpy()

            return cell15_np.asarray(cell15_working_object)
        except Exception:
            return None

    def cell15_normalize_int_vector(cell15_object):
        cell15_array_object = cell15_to_numpy(cell15_object)

        if cell15_array_object is None:
            return None

        try:
            return cell15_array_object.astype(int).ravel()
        except Exception:
            return None

    def cell15_recall_one_seed(cell15_true_vector, cell15_pred_vector, cell15_class_count):
        cell15_true_array = cell15_np.asarray(cell15_true_vector, dtype=cell15_np.int64).reshape(-1)
        cell15_pred_array = cell15_np.asarray(cell15_pred_vector, dtype=cell15_np.int64).reshape(-1)

        cell15_valid_mask = (
            (cell15_true_array >= 0)
            & (cell15_pred_array >= 0)
            & (cell15_true_array < cell15_class_count)
            & (cell15_pred_array < cell15_class_count)
        )

        cell15_true_valid = cell15_true_array[cell15_valid_mask]
        cell15_pred_valid = cell15_pred_array[cell15_valid_mask]

        cell15_recall_vector = cell15_np.zeros(cell15_class_count, dtype=float)

        for cell15_class_value in range(int(cell15_class_count)):
            cell15_support_mask = cell15_true_valid == cell15_class_value
            cell15_support_count = int(cell15_np.sum(cell15_support_mask))

            if cell15_support_count == 0:
                cell15_recall_vector[cell15_class_value] = 0.0
            else:
                cell15_hit_mask = cell15_support_mask & (cell15_pred_valid == cell15_class_value)
                cell15_hit_count = int(cell15_np.sum(cell15_hit_mask))
                cell15_recall_vector[cell15_class_value] = float(cell15_hit_count) / float(cell15_support_count)

        return cell15_recall_vector

    def cell15_csv_escape(cell15_text_value):
        cell15_text_string = str(cell15_text_value)
        cell15_needs_quote_flag = False

        for cell15_character_value in cell15_text_string:
            if cell15_character_value in ',\"\n\r':
                cell15_needs_quote_flag = True
                break

        if not cell15_needs_quote_flag:
            return cell15_text_string

        cell15_escaped_string = cell15_text_string.replace('"', '""')
        return '"' + cell15_escaped_string + '"'

    def cell15_write_text(cell15_path_string, cell15_text_content):
        with open(cell15_path_string, "w", encoding="utf-8") as cell15_file_handle:
            cell15_file_handle.write(cell15_text_content)

    def cell15_write_csv_rows(cell15_path_string, cell15_fieldnames, cell15_row_objects):
        cell15_header_parts = []

        for cell15_field_name in cell15_fieldnames:
            cell15_header_parts.append(cell15_csv_escape(cell15_field_name))

        cell15_line_parts = [",".join(cell15_header_parts)]

        for cell15_row_object in cell15_row_objects:
            cell15_row_parts = []

            for cell15_field_name in cell15_fieldnames:
                cell15_row_parts.append(cell15_csv_escape(cell15_row_object.get(cell15_field_name, "")))

            cell15_line_parts.append(",".join(cell15_row_parts))

        cell15_write_text(cell15_path_string, "\n".join(cell15_line_parts) + "\n")

    cell15_num_classes_value = cell15_first_global(["C_cell3", "cell15_num_classes", "num_classes"])

    if cell15_num_classes_value is None:
        cell15_num_classes = 29
    else:
        cell15_num_classes = int(cell15_num_classes_value)

    cell15_rare_source = cell15_first_global(["rare_classes_list_cell3", "cell15_rare_classes", "rare_classes"])

    if cell15_rare_source is None:
        cell15_rare_classes = [24, 11, 18, 8, 0, 20, 23, 17, 25, 26, 12, 2, 10]
    else:
        cell15_rare_array = cell15_to_numpy(cell15_rare_source)

        if cell15_rare_array is None:
            cell15_rare_classes = [24, 11, 18, 8, 0, 20, 23, 17, 25, 26, 12, 2, 10]
        else:
            cell15_rare_classes = []
            cell15_seen_rare_classes = set()

            for cell15_rare_value in cell15_rare_array.ravel().tolist():
                cell15_rare_integer = int(cell15_rare_value)

                if cell15_rare_integer not in cell15_seen_rare_classes:
                    cell15_seen_rare_classes.add(cell15_rare_integer)
                    cell15_rare_classes.append(cell15_rare_integer)

            cell15_rare_classes = sorted(cell15_rare_classes)

    cell15_method_specs = [
        {
            "name": "GCN+ all features baseline",
            "pred_names": ["pred_baseline_cell8"],
            "label_names": ["y_test_per_seed_cell8"],
        },
        {
            "name": "SPGCN unsigned spectral",
            "pred_names": ["pred_spgcn_cell8"],
            "label_names": ["y_test_per_seed_cell8"],
        },
        {
            "name": "SSPR full with refinement",
            "pred_names": ["pred_sspr_cell11c"],
            "label_names": ["y_test_per_seed_cell11c", "y_test_per_seed_cell8"],
        },
    ]

    cell15_rows = []
    cell15_skipped_methods = []
    cell15_pred_lists = {}
    cell15_y_true_lists = {}

    for cell15_method_spec in cell15_method_specs:
        cell15_method_name = cell15_method_spec["name"]
        cell15_pred_raw = cell15_first_global(cell15_method_spec["pred_names"])
        cell15_label_raw = cell15_first_global(cell15_method_spec["label_names"])

        if cell15_pred_raw is None or cell15_label_raw is None:
            cell15_skipped_methods.append(cell15_method_name)
            continue

        if not isinstance(cell15_pred_raw, (list, tuple)) or len(cell15_pred_raw) == 0:
            cell15_skipped_methods.append(cell15_method_name)
            continue

        if not isinstance(cell15_label_raw, (list, tuple)) or len(cell15_label_raw) == 0:
            cell15_skipped_methods.append(cell15_method_name)
            continue

        cell15_seed_count = min(len(cell15_pred_raw), len(cell15_label_raw))

        if cell15_seed_count == 0:
            cell15_skipped_methods.append(cell15_method_name)
            continue

        cell15_recall_rows = []
        cell15_valid_pred_list = []
        cell15_valid_label_list = []

        for cell15_seed_index in range(cell15_seed_count):
            cell15_pred_vector = cell15_normalize_int_vector(cell15_pred_raw[cell15_seed_index])
            cell15_true_vector = cell15_normalize_int_vector(cell15_label_raw[cell15_seed_index])

            if cell15_pred_vector is None or cell15_true_vector is None:
                continue

            if len(cell15_pred_vector) != len(cell15_true_vector):
                continue

            cell15_recall_vector = cell15_recall_one_seed(
                cell15_true_vector,
                cell15_pred_vector,
                cell15_num_classes,
            )

            cell15_recall_rows.append(cell15_recall_vector)
            cell15_valid_pred_list.append(cell15_pred_vector)
            cell15_valid_label_list.append(cell15_true_vector)

        if len(cell15_recall_rows) == 0:
            cell15_skipped_methods.append(cell15_method_name)
            continue

        cell15_pred_lists[cell15_method_name] = cell15_valid_pred_list
        cell15_y_true_lists[cell15_method_name] = cell15_valid_label_list

        cell15_recall_matrix = cell15_np.vstack(cell15_recall_rows)
        cell15_actual_seed_count = int(cell15_recall_matrix.shape[0])

        cell15_support_vector = cell15_np.zeros(cell15_num_classes, dtype=float)

        for cell15_true_vector in cell15_valid_label_list:
            cell15_true_positive_vector = cell15_true_vector[cell15_true_vector >= 0]
            cell15_support_vector = cell15_support_vector + cell15_np.bincount(
                cell15_true_positive_vector.astype(cell15_np.int64),
                minlength=cell15_num_classes,
            ).astype(float)

        for cell15_class_value in range(cell15_num_classes):
            cell15_class_recalls = cell15_recall_matrix[:, cell15_class_value]

            if cell15_actual_seed_count > 1:
                cell15_class_std = float(cell15_np.std(cell15_class_recalls, ddof=1))
            else:
                cell15_class_std = 0.0

            cell15_rows.append(
                {
                    "method": cell15_method_name,
                    "class_id": int(cell15_class_value),
                    "is_rare_class": int(cell15_class_value in cell15_rare_classes),
                    "recall_mean": float(cell15_np.mean(cell15_class_recalls)),
                    "recall_std": cell15_class_std,
                    "n_seeds": cell15_actual_seed_count,
                    "n_support_total": float(cell15_support_vector[cell15_class_value]),
                }
            )

        if len(cell15_rare_classes) > 0:
            cell15_rare_matrix = cell15_recall_matrix[:, cell15_rare_classes]
            cell15_seed_rare_means = cell15_np.mean(cell15_rare_matrix, axis=1)

            if cell15_actual_seed_count > 1:
                cell15_rare_std = float(cell15_np.std(cell15_seed_rare_means, ddof=1))
            else:
                cell15_rare_std = 0.0

            cell15_rare_support_total = float(cell15_np.sum(cell15_support_vector[cell15_rare_classes]))

            cell15_rows.append(
                {
                    "method": cell15_method_name,
                    "class_id": "RARE_MACRO",
                    "is_rare_class": 1,
                    "recall_mean": float(cell15_np.mean(cell15_seed_rare_means)),
                    "recall_std": cell15_rare_std,
                    "n_seeds": cell15_actual_seed_count,
                    "n_support_total": cell15_rare_support_total,
                }
            )

    cell15_missing_required = []

    if len(cell15_rows) == 0:
        cell15_missing_required.extend(
            [
                "pred_baseline_cell8 and y_test_per_seed_cell8 from patched Cell 8",
                "pred_spgcn_cell8 and y_test_per_seed_cell8 from patched Cell 8",
                "pred_sspr_cell11c and y_test_per_seed_cell11c from patched Cell 11c",
            ]
        )

    cell15_status = {
        "computed": bool(len(cell15_rows) > 0),
        "methods_computed": sorted(list(cell15_pred_lists.keys())),
        "methods_skipped": cell15_skipped_methods,
        "num_classes": cell15_num_classes,
        "rare_classes": cell15_rare_classes,
        "n_rows": len(cell15_rows),
        "missing_required": cell15_missing_required,
        "note": "No per-class numbers were fabricated. Only saved prediction artifacts were used.",
    }

    cell15_per_class_csv_path = cell15_artifact_dir + "/cell15_per_class_recall.csv"
    cell15_status_json_path = cell15_artifact_dir + "/cell15_per_class_recall_status.json"

    if len(cell15_rows) > 0:
        cell15_write_csv_rows(
            cell15_per_class_csv_path,
            [
                "method",
                "class_id",
                "is_rare_class",
                "recall_mean",
                "recall_std",
                "n_seeds",
                "n_support_total",
            ],
            cell15_rows,
        )

    cell15_json_module = cell15_globals.get("json", None)

    if cell15_json_module is not None and hasattr(cell15_json_module, "dumps"):
        cell15_write_text(
            cell15_status_json_path,
            cell15_json_module.dumps(cell15_status, indent=2, default=str),
        )
    else:
        cell15_write_text(cell15_status_json_path, str(cell15_status))

    print("=" * 100)
    print("CELL 15 — PER-CLASS RECALL FROM SAVED PREDICTIONS")
    print("=" * 100)
    print("Computed: " + str(cell15_status["computed"]))
    print("Methods computed: " + str(cell15_status["methods_computed"]))
    print("Methods skipped: " + str(cell15_status["methods_skipped"]))
    print("Num classes: " + str(cell15_num_classes))
    print("Rare classes: " + str(cell15_rare_classes))
    print("Rows: " + str(cell15_status["n_rows"]))

    if cell15_status["computed"]:
        print()
        print("Rare-class macro recall summary:")

        for cell15_row_object in cell15_rows:
            if str(cell15_row_object["class_id"]) == "RARE_MACRO":
                print(
                    "  "
                    + str(cell15_row_object["method"])
                    + " recall="
                    + f"{float(cell15_row_object['recall_mean']):.4f}"
                    + " +/- "
                    + f"{float(cell15_row_object['recall_std']):.4f}"
                    + " n_seeds="
                    + str(int(cell15_row_object["n_seeds"]))
                )

        print()
        print("Per-class CSV saved: " + cell15_per_class_csv_path)
    else:
        print()
        print("Missing required artifacts:")

        for cell15_missing_item in cell15_missing_required:
            print("  - " + str(cell15_missing_item))

        print()
        print("Run patched Cell 8 and patched Cell 11c first.")

    print()
    print("Status saved: " + cell15_status_json_path)
    print("=" * 100)

    cell15_hf_save_checkpoint = cell15_globals.get("hf_save_checkpoint", None)

    if callable(cell15_hf_save_checkpoint):
        try:
            cell15_hf_save_checkpoint(
                "cell15_per_class_recall",
                {
                    "status": cell15_status,
                    "rows": cell15_rows,
                    "num_classes": cell15_num_classes,
                    "rare_classes": cell15_rare_classes,
                    "paths": {
                        "per_class_csv": cell15_per_class_csv_path,
                        "status_json": cell15_status_json_path,
                    },
                },
                description="Cell 15: per-class recall breakdown from saved predictions.",
                upload=True,
            )
        except Exception as cell15_checkpoint_error:
            print("[ckpt][warn] Cell 15 checkpoint failed: " + repr(cell15_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    return {
        "artifact_dir": cell15_artifact_dir,
        "pred_lists": cell15_pred_lists,
        "y_true_lists": cell15_y_true_lists,
        "num_classes": cell15_num_classes,
        "rare_classes": cell15_rare_classes,
        "rows": cell15_rows,
        "status": cell15_status,
        "paths": {
            "per_class_csv": cell15_per_class_csv_path,
            "status_json": cell15_status_json_path,
        },
    }


cell15_output = cell15_main()

cell15_pred_lists = cell15_output["pred_lists"]
cell15_y_true_lists = cell15_output["y_true_lists"]
cell15_num_classes = cell15_output["num_classes"]
cell15_rare_classes = cell15_output["rare_classes"]
cell15_per_class_rows = cell15_output["rows"]
cell15_status = cell15_output["status"]

In [ ]:
# Cell 16: Rare-class confusion matrix / heatmap from Cell 15 prediction artifacts.
#
# Complete rewrite.
#
# Rules enforced:
#   - No imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every local/helper/loop variable is prefixed with cell16_.
#   - No fabricated confusion matrices.
#
# Reads:
#   cell15_pred_lists
#   cell15_y_true_lists
#   cell15_num_classes
#   cell15_rare_classes
#
# Produces:
#   cell16_status
#   cell16_confusion_rows


def cell16_main():
    cell16_globals = globals()
    cell16_np = cell16_globals.get("np", None)

    if cell16_np is None:
        raise NameError("[Cell 16][FATAL] np is not available. Run earlier cells first.")

    cell16_os = cell16_globals.get("os", None)
    cell16_artifact_dir = "."

    if cell16_os is not None and hasattr(cell16_os, "makedirs"):
        try:
            cell16_os.makedirs("artifacts", exist_ok=True)
            cell16_artifact_dir = "artifacts"
        except Exception as cell16_mkdir_error:
            print("[Cell 16][warn] could not create artifacts dir: " + repr(cell16_mkdir_error))
            cell16_artifact_dir = "."

    def cell16_first_global(cell16_candidate_names):
        for cell16_candidate_name in cell16_candidate_names:
            if cell16_candidate_name in cell16_globals and cell16_globals[cell16_candidate_name] is not None:
                return cell16_globals[cell16_candidate_name]

        return None

    def cell16_to_numpy(cell16_object):
        if cell16_object is None:
            return None

        cell16_working_object = cell16_object

        try:
            if hasattr(cell16_working_object, "detach"):
                cell16_working_object = cell16_working_object.detach()

            if hasattr(cell16_working_object, "cpu"):
                cell16_working_object = cell16_working_object.cpu()

            if hasattr(cell16_working_object, "numpy"):
                cell16_working_object = cell16_working_object.numpy()

            return cell16_np.asarray(cell16_working_object)
        except Exception:
            return None

    def cell16_normalize_int_vector(cell16_object):
        cell16_array_object = cell16_to_numpy(cell16_object)

        if cell16_array_object is None:
            return None

        try:
            return cell16_array_object.astype(int).ravel()
        except Exception:
            return None

    def cell16_sanitize_filename(cell16_text_value):
        cell16_clean_characters = []

        for cell16_character_value in str(cell16_text_value):
            if cell16_character_value.isalnum() or cell16_character_value in "._-":
                cell16_clean_characters.append(cell16_character_value)
            else:
                cell16_clean_characters.append("_")

        return "".join(cell16_clean_characters)

    def cell16_csv_escape(cell16_text_value):
        cell16_text_string = str(cell16_text_value)
        cell16_needs_quote_flag = False

        for cell16_character_value in cell16_text_string:
            if cell16_character_value in ',\"\n\r':
                cell16_needs_quote_flag = True
                break

        if not cell16_needs_quote_flag:
            return cell16_text_string

        cell16_escaped_string = cell16_text_string.replace('"', '""')
        return '"' + cell16_escaped_string + '"'

    def cell16_write_text(cell16_path_string, cell16_text_content):
        with open(cell16_path_string, "w", encoding="utf-8") as cell16_file_handle:
            cell16_file_handle.write(cell16_text_content)

    def cell16_write_matrix_csv(
        cell16_matrix_object,
        cell16_path_string,
        cell16_row_prefix,
        cell16_col_prefix,
    ):
        cell16_row_count = int(cell16_matrix_object.shape[0])
        cell16_col_count = int(cell16_matrix_object.shape[1])

        cell16_header_parts = [""]

        for cell16_col_index in range(cell16_col_count):
            cell16_header_parts.append(cell16_csv_escape(cell16_col_prefix + str(cell16_col_index)))

        cell16_line_parts = [",".join(cell16_header_parts)]

        for cell16_row_index in range(cell16_row_count):
            cell16_row_parts = [cell16_csv_escape(cell16_row_prefix + str(cell16_row_index))]

            for cell16_col_index in range(cell16_col_count):
                cell16_cell_value = float(cell16_matrix_object[cell16_row_index, cell16_col_index])
                cell16_row_parts.append(f"{cell16_cell_value:.10f}")

            cell16_line_parts.append(",".join(cell16_row_parts))

        cell16_write_text(cell16_path_string, "\n".join(cell16_line_parts) + "\n")

    def cell16_write_csv_rows(cell16_path_string, cell16_fieldnames, cell16_row_objects):
        cell16_header_parts = []

        for cell16_field_name in cell16_fieldnames:
            cell16_header_parts.append(cell16_csv_escape(cell16_field_name))

        cell16_line_parts = [",".join(cell16_header_parts)]

        for cell16_row_object in cell16_row_objects:
            cell16_row_parts = []

            for cell16_field_name in cell16_fieldnames:
                cell16_row_parts.append(cell16_csv_escape(cell16_row_object.get(cell16_field_name, "")))

            cell16_line_parts.append(",".join(cell16_row_parts))

        cell16_write_text(cell16_path_string, "\n".join(cell16_line_parts) + "\n")

    cell16_num_classes_value = cell16_first_global(["cell15_num_classes", "C_cell3", "num_classes"])

    if cell16_num_classes_value is None:
        cell16_num_classes = 29
    else:
        cell16_num_classes = int(cell16_num_classes_value)

    cell16_rare_source = cell16_first_global(["cell15_rare_classes", "rare_classes_list_cell3"])

    if cell16_rare_source is None:
        cell16_rare_classes = [24, 11, 18, 8, 0, 20, 23, 17, 25, 26, 12, 2, 10]
    else:
        cell16_rare_array = cell16_to_numpy(cell16_rare_source)

        if cell16_rare_array is None:
            cell16_rare_classes = [24, 11, 18, 8, 0, 20, 23, 17, 25, 26, 12, 2, 10]
        else:
            cell16_rare_classes = []
            cell16_seen_rare_classes = set()

            for cell16_rare_value in cell16_rare_array.ravel().tolist():
                cell16_rare_integer = int(cell16_rare_value)

                if cell16_rare_integer not in cell16_seen_rare_classes:
                    cell16_seen_rare_classes.add(cell16_rare_integer)
                    cell16_rare_classes.append(cell16_rare_integer)

            cell16_rare_classes = sorted(cell16_rare_classes)

    cell16_pred_lists = cell16_first_global(["cell15_pred_lists"])

    if cell16_pred_lists is None:
        cell16_pred_lists = {}

    cell16_y_true_lists = cell16_first_global(["cell15_y_true_lists"])

    if cell16_y_true_lists is None:
        cell16_y_true_lists = {}

    cell16_confusion_rows = []

    cell16_status = {
        "computed": False,
        "methods_computed": [],
        "missing_required": [],
        "plot_notes": {},
        "plot_errors": {},
        "note": "Confusion matrices require aligned per-seed predictions and test labels from Cell 15.",
    }

    if len(cell16_pred_lists) == 0 or len(cell16_y_true_lists) == 0:
        cell16_status["missing_required"].extend(
            [
                "cell15_pred_lists from Cell 15",
                "cell15_y_true_lists from Cell 15",
            ]
        )

    for cell16_method_name in list(cell16_pred_lists.keys()):
        if cell16_method_name not in cell16_y_true_lists:
            continue

        cell16_pred_list = cell16_pred_lists[cell16_method_name]
        cell16_y_true_list = cell16_y_true_lists[cell16_method_name]

        if len(cell16_pred_list) != len(cell16_y_true_list):
            continue

        cell16_confusion_matrix = cell16_np.zeros(
            (cell16_num_classes, cell16_num_classes),
            dtype=float,
        )

        cell16_seed_count = 0

        for cell16_pred_vector, cell16_true_vector in zip(cell16_pred_list, cell16_y_true_list):
            cell16_pred_vector = cell16_normalize_int_vector(cell16_pred_vector)
            cell16_true_vector = cell16_normalize_int_vector(cell16_true_vector)

            if cell16_pred_vector is None or cell16_true_vector is None:
                continue

            if len(cell16_pred_vector) != len(cell16_true_vector):
                continue

            cell16_valid_mask = (
                (cell16_true_vector >= 0)
                & (cell16_pred_vector >= 0)
                & (cell16_true_vector < cell16_num_classes)
                & (cell16_pred_vector < cell16_num_classes)
            )

            cell16_true_valid = cell16_true_vector[cell16_valid_mask]
            cell16_pred_valid = cell16_pred_vector[cell16_valid_mask]

            if cell16_true_valid.size == 0:
                continue

            cell16_flat_index = (
                cell16_true_valid.astype(cell16_np.int64) * cell16_num_classes
                + cell16_pred_valid.astype(cell16_np.int64)
            )

            cell16_counts = cell16_np.bincount(
                cell16_flat_index,
                minlength=cell16_num_classes * cell16_num_classes,
            ).astype(float)

            cell16_confusion_matrix = cell16_confusion_matrix + cell16_counts.reshape(
                (cell16_num_classes, cell16_num_classes)
            )

            cell16_seed_count = cell16_seed_count + 1

        if cell16_seed_count == 0:
            continue

        cell16_average_matrix = cell16_confusion_matrix / float(cell16_seed_count)
        cell16_method_safe = cell16_sanitize_filename(cell16_method_name)

        cell16_full_csv_path = cell16_artifact_dir + "/cell16_confusion_full_" + cell16_method_safe + ".csv"
        cell16_write_matrix_csv(
            cell16_average_matrix,
            cell16_full_csv_path,
            "true_",
            "pred_",
        )

        cell16_rare_rows_all_path = (
            cell16_artifact_dir
            + "/cell16_confusion_rare_rows_all_cols_"
            + cell16_method_safe
            + ".csv"
        )

        cell16_rare_rows_all_matrix = cell16_average_matrix[cell16_rare_classes, :]
        cell16_write_matrix_csv(
            cell16_rare_rows_all_matrix,
            cell16_rare_rows_all_path,
            "true_rare_",
            "pred_",
        )

        cell16_rare_x_rare_path = (
            cell16_artifact_dir
            + "/cell16_confusion_rare_x_rare_"
            + cell16_method_safe
            + ".csv"
        )

        cell16_rare_x_rare_matrix = cell16_average_matrix[cell16_rare_classes, :][:, cell16_rare_classes]
        cell16_write_matrix_csv(
            cell16_rare_x_rare_matrix,
            cell16_rare_x_rare_path,
            "true_rare_",
            "pred_rare_",
        )

        cell16_row_sums = cell16_rare_x_rare_matrix.sum(axis=1, keepdims=True)
        cell16_row_sums_safe = cell16_np.where(cell16_row_sums == 0.0, 1.0, cell16_row_sums)
        cell16_row_normalized = cell16_rare_x_rare_matrix / cell16_row_sums_safe
        cell16_row_normalized = cell16_np.nan_to_num(
            cell16_row_normalized,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        cell16_heatmap_path = None
        cell16_pyplot_object = cell16_globals.get("plt", None)

        if cell16_pyplot_object is None:
            cell16_matplotlib_object = cell16_globals.get("matplotlib", None)

            if cell16_matplotlib_object is not None and hasattr(cell16_matplotlib_object, "pyplot"):
                cell16_pyplot_object = cell16_matplotlib_object.pyplot

        if cell16_pyplot_object is not None:
            try:
                cell16_figure_object, cell16_axis_object = cell16_pyplot_object.subplots(figsize=(8, 6))

                cell16_image_object = cell16_axis_object.imshow(
                    cell16_row_normalized,
                    cmap="Blues",
                    vmin=0.0,
                    vmax=1.0,
                    aspect="auto",
                )

                cell16_tick_positions = []
                cell16_x_tick_labels = []
                cell16_y_tick_labels = []

                for cell16_tick_index in range(len(cell16_rare_classes)):
                    cell16_tick_positions.append(cell16_tick_index)
                    cell16_x_tick_labels.append("pred_" + str(cell16_rare_classes[cell16_tick_index]))
                    cell16_y_tick_labels.append("true_" + str(cell16_rare_classes[cell16_tick_index]))

                cell16_axis_object.set_xticks(cell16_tick_positions)
                cell16_axis_object.set_yticks(cell16_tick_positions)
                cell16_axis_object.set_xticklabels(cell16_x_tick_labels, rotation=45, ha="right")
                cell16_axis_object.set_yticklabels(cell16_y_tick_labels)

                cell16_axis_object.set_title(
                    cell16_method_name
                    + ": rare-class confusion, row-normalized\n"
                    + "(rare true rows, rare predicted columns)"
                )

                cell16_axis_object.set_xlabel("Predicted rare class")
                cell16_axis_object.set_ylabel("True rare class")

                cell16_colorbar_object = cell16_figure_object.colorbar(
                    cell16_image_object,
                    ax=cell16_axis_object,
                )

                cell16_colorbar_object.set_label("Row-normalized frequency")

                for cell16_row_index in range(len(cell16_rare_classes)):
                    for cell16_col_index in range(len(cell16_rare_classes)):
                        cell16_annotation_value = float(cell16_row_normalized[cell16_row_index, cell16_col_index])
                        cell16_annotation_color = "white" if cell16_annotation_value > 0.5 else "black"

                        cell16_axis_object.text(
                            cell16_col_index,
                            cell16_row_index,
                            f"{cell16_annotation_value:.2f}",
                            ha="center",
                            va="center",
                            color=cell16_annotation_color,
                            fontsize=7,
                        )

                cell16_figure_object.tight_layout()

                cell16_heatmap_path = (
                    cell16_artifact_dir
                    + "/cell16_confusion_rare_heatmap_"
                    + cell16_method_safe
                    + ".png"
                )

                cell16_figure_object.savefig(cell16_heatmap_path, dpi=200, bbox_inches="tight")
                cell16_pyplot_object.show()
                cell16_pyplot_object.close(cell16_figure_object)

            except Exception as cell16_plot_error:
                cell16_heatmap_path = None
                cell16_status["plot_errors"][cell16_method_name] = repr(cell16_plot_error)
        else:
            cell16_status["plot_notes"][cell16_method_name] = (
                "matplotlib.pyplot / plt not available in globals; PNG heatmap skipped, CSV matrices saved."
            )

        cell16_status["computed"] = True
        cell16_status["methods_computed"].append(cell16_method_name)

        cell16_confusion_rows.append(
            {
                "method": cell16_method_name,
                "n_seeds": cell16_seed_count,
                "full_cm_csv": cell16_full_csv_path,
                "rare_rows_all_cols_csv": cell16_rare_rows_all_path,
                "rare_x_rare_csv": cell16_rare_x_rare_path,
                "rare_heatmap_png": cell16_heatmap_path if cell16_heatmap_path is not None else "",
            }
        )

    cell16_status_path = cell16_artifact_dir + "/cell16_confusion_status.json"
    cell16_artifacts_csv_path = cell16_artifact_dir + "/cell16_confusion_artifacts.csv"

    cell16_json_module = cell16_globals.get("json", None)

    if cell16_json_module is not None and hasattr(cell16_json_module, "dumps"):
        cell16_write_text(
            cell16_status_path,
            cell16_json_module.dumps(cell16_status, indent=2, default=str),
        )
    else:
        cell16_write_text(cell16_status_path, str(cell16_status))

    if len(cell16_confusion_rows) > 0:
        cell16_write_csv_rows(
            cell16_artifacts_csv_path,
            [
                "method",
                "n_seeds",
                "full_cm_csv",
                "rare_rows_all_cols_csv",
                "rare_x_rare_csv",
                "rare_heatmap_png",
            ],
            cell16_confusion_rows,
        )

    print("=" * 100)
    print("CELL 16 — RARE-CLASS CONFUSION MATRIX / HEATMAP")
    print("=" * 100)
    print("Computed: " + str(cell16_status["computed"]))
    print("Methods computed: " + str(cell16_status["methods_computed"]))

    if cell16_status["computed"]:
        print()

        for cell16_confusion_row in cell16_confusion_rows:
            print(
                "  "
                + str(cell16_confusion_row["method"])
                + " n_seeds="
                + str(int(cell16_confusion_row["n_seeds"]))
                + " full_csv="
                + str(cell16_confusion_row["full_cm_csv"])
            )

            if str(cell16_confusion_row["rare_heatmap_png"]) != "":
                print("    heatmap_png=" + str(cell16_confusion_row["rare_heatmap_png"]))
            else:
                print("    heatmap_png=not generated")
    else:
        print()
        print("Missing required artifacts:")

        for cell16_missing_item in cell16_status["missing_required"]:
            print("  - " + str(cell16_missing_item))

        print()
        print("No confusion matrix was fabricated.")

    print()
    print("Status saved: " + cell16_status_path)

    if len(cell16_confusion_rows) > 0:
        print("Artifacts saved: " + cell16_artifacts_csv_path)

    print("=" * 100)

    cell16_hf_save_checkpoint = cell16_globals.get("hf_save_checkpoint", None)

    if callable(cell16_hf_save_checkpoint):
        try:
            cell16_hf_save_checkpoint(
                "cell16_confusion_matrices",
                {
                    "status": cell16_status,
                    "rows": cell16_confusion_rows,
                    "num_classes": cell16_num_classes,
                    "rare_classes": cell16_rare_classes,
                    "paths": {
                        "status_json": cell16_status_path,
                        "artifacts_csv": cell16_artifacts_csv_path if len(cell16_confusion_rows) > 0 else "",
                    },
                },
                description="Cell 16: rare-class confusion matrices from saved predictions.",
                upload=True,
            )
        except Exception as cell16_checkpoint_error:
            print("[ckpt][warn] Cell 16 checkpoint failed: " + repr(cell16_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    return {
        "artifact_dir": cell16_artifact_dir,
        "status": cell16_status,
        "rows": cell16_confusion_rows,
        "num_classes": cell16_num_classes,
        "rare_classes": cell16_rare_classes,
        "paths": {
            "status_json": cell16_status_path,
            "artifacts_csv": cell16_artifacts_csv_path if len(cell16_confusion_rows) > 0 else "",
        },
    }


cell16_output = cell16_main()

cell16_status = cell16_output["status"]
cell16_confusion_rows = cell16_output["rows"]

In [ ]:
# Cell 17: Train and save frozen FAFB SSPR model for cross-connectome transfer.
#
# Rules:
#   - No top-level imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every local/helper/loop variable is prefixed with cell17_.
#
# Produces:
#   cell17_frozen_path
#   cell17_frozen_config
#   cell17_before_refine_rare_ba
#   cell17_after_refine_rare_ba
#   cell17_refined_prototype_count


def cell17_main():
    cell17_globals = globals()

    cell17_np = cell17_globals.get("np", None)
    cell17_torch = cell17_globals.get("torch", None)

    if cell17_np is None:
        raise NameError("[Cell 17][FATAL] np is not available. Run earlier cells first.")

    if cell17_torch is None:
        raise NameError("[Cell 17][FATAL] torch is not available. Run earlier cells first.")

    cell17_required_names = [
        "DEVICE_cell1",
        "X_sspr_tensor_cell11",
        "labels_tensor_cell3",
        "labeled_mask_cell3",
        "super_labels_tensor_cell3",
        "C_cell3",
        "C_super_cell3",
        "rare_classes_list_cell3",
        "ei_pos_cell7",
        "ew_pos_cell7",
        "dis_pos_cell7",
        "ei_neg_cell7",
        "ew_neg_cell7",
        "dis_neg_cell7",
        "make_split_cell4",
        "SPGCN_cell5",
        "train_spgcn_cell6",
        "rare_balanced_acc_cell6",
    ]

    cell17_missing_names = []

    for cell17_required_name in cell17_required_names:
        if cell17_required_name not in cell17_globals:
            cell17_missing_names.append(cell17_required_name)

    if len(cell17_missing_names) > 0:
        raise NameError(
            "[Cell 17][FATAL] Missing upstream variables: "
            + str(cell17_missing_names)
            + ". Run Cells 1-11 first."
        )

    cell17_device = cell17_globals["DEVICE_cell1"]
    cell17_num_classes = int(cell17_globals["C_cell3"])
    cell17_num_super_classes = int(cell17_globals["C_super_cell3"])

    cell17_seed_value = 0
    cell17_epoch_count = 200
    cell17_lambda_refine = 0.3
    cell17_tau_refine = 1.0
    cell17_frozen_path = "./sspr_fafb_frozen_cell17.pt"

    def cell17_first_global(cell17_candidate_names):
        for cell17_candidate_name in cell17_candidate_names:
            if cell17_candidate_name in cell17_globals and cell17_globals[cell17_candidate_name] is not None:
                return cell17_globals[cell17_candidate_name]

        return None

    def cell17_to_numpy(cell17_object):
        if cell17_object is None:
            return None

        cell17_working_object = cell17_object

        try:
            if hasattr(cell17_working_object, "detach"):
                cell17_working_object = cell17_working_object.detach()

            if hasattr(cell17_working_object, "cpu"):
                cell17_working_object = cell17_working_object.cpu()

            if hasattr(cell17_working_object, "numpy"):
                cell17_working_object = cell17_working_object.numpy()

            return cell17_np.asarray(cell17_working_object)
        except Exception:
            return None

    def cell17_to_int_vector(cell17_object, cell17_description):
        cell17_array_object = cell17_to_numpy(cell17_object)

        if cell17_array_object is None:
            raise TypeError(
                "[Cell 17][FATAL] Could not convert "
                + str(cell17_description)
                + " to numpy."
            )

        try:
            return cell17_array_object.astype(cell17_np.int64).reshape(-1)
        except Exception as cell17_cast_error:
            raise TypeError(
                "[Cell 17][FATAL] Could not cast "
                + str(cell17_description)
                + " to int64: "
                + repr(cell17_cast_error)
            )

    def cell17_set_seed(cell17_seed_input):
        cell17_seed_function = cell17_globals.get("seed_all_cell1", None)

        if callable(cell17_seed_function):
            cell17_seed_function(int(cell17_seed_input))
        else:
            cell17_torch.manual_seed(int(cell17_seed_input))
            cell17_np.random.seed(int(cell17_seed_input))

    def cell17_move_tensor(cell17_tensor_object, cell17_target_device):
        if cell17_torch.is_tensor(cell17_tensor_object):
            return cell17_tensor_object.to(cell17_target_device)

        return cell17_tensor_object

    def cell17_make_spgcn(cell17_input_feature_dim):
        try:
            return cell17_globals["SPGCN_cell5"](
                d_features_cell5=int(cell17_input_feature_dim),
                d_cell5=64,
                d_proto_cell5=32,
                n_layers_cell5=2,
                n_classes_cell5=cell17_num_classes,
                n_super_cell5=cell17_num_super_classes,
                temperature_cell5=0.1,
                dropout_cell5=0.3,
            )
        except TypeError:
            return cell17_globals["SPGCN_cell5"](
                d_features=int(cell17_input_feature_dim),
                d=64,
                d_proto=32,
                n_layers=2,
                n_classes=cell17_num_classes,
                n_super=cell17_num_super_classes,
                temperature=0.1,
                dropout=0.3,
            )

    def cell17_collect_tensors(cell17_output_object, cell17_tensor_list=None):
        if cell17_tensor_list is None:
            cell17_tensor_list = []

        if cell17_torch.is_tensor(cell17_output_object):
            if int(cell17_output_object.ndim) >= 2:
                cell17_tensor_list.append(cell17_output_object)
            return cell17_tensor_list

        if isinstance(cell17_output_object, dict):
            for cell17_dict_value in cell17_output_object.values():
                cell17_collect_tensors(cell17_dict_value, cell17_tensor_list)
            return cell17_tensor_list

        if isinstance(cell17_output_object, (tuple, list)):
            for cell17_sequence_value in cell17_output_object:
                cell17_collect_tensors(cell17_sequence_value, cell17_tensor_list)
            return cell17_tensor_list

        return cell17_tensor_list

    def cell17_get_proto_dim(cell17_model_object):
        cell17_proto_parameter = getattr(cell17_model_object, "proto_class", None)

        if cell17_torch.is_tensor(cell17_proto_parameter) and int(cell17_proto_parameter.ndim) == 2:
            return int(cell17_proto_parameter.shape[1])

        return None

    def cell17_extract_embedding(cell17_output_object, cell17_model_object, cell17_expected_rows):
        cell17_candidate_tensors = cell17_collect_tensors(cell17_output_object)

        if len(cell17_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 17][FATAL] Model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell17_output_object))
            )

        cell17_proto_dim = cell17_get_proto_dim(cell17_model_object)

        if cell17_proto_dim is not None:
            for cell17_candidate_tensor in cell17_candidate_tensors:
                if int(cell17_candidate_tensor.shape[1]) == cell17_proto_dim:
                    return cell17_candidate_tensor

        for cell17_candidate_tensor in cell17_candidate_tensors:
            if int(cell17_candidate_tensor.shape[0]) == int(cell17_expected_rows):
                return cell17_candidate_tensor

        return cell17_candidate_tensors[0]

    def cell17_forward_spgcn(
        cell17_model_object,
        cell17_x_tensor,
        cell17_ei_pos_tensor,
        cell17_ew_pos_tensor,
        cell17_dis_pos_tensor,
        cell17_ei_neg_tensor,
        cell17_ew_neg_tensor,
        cell17_dis_neg_tensor,
    ):
        cell17_last_type_error = None

        try:
            return cell17_model_object(
                cell17_x_tensor,
                cell17_ei_pos_tensor,
                cell17_ew_pos_tensor,
                cell17_dis_pos_tensor,
                cell17_ei_neg_tensor,
                cell17_ew_neg_tensor,
                cell17_dis_neg_tensor,
            )
        except TypeError as cell17_type_error:
            cell17_last_type_error = cell17_type_error

        try:
            return cell17_model_object(
                cell17_x_tensor,
                cell17_ei_pos_tensor,
                cell17_dis_pos_tensor,
                cell17_ei_neg_tensor,
                cell17_dis_neg_tensor,
            )
        except TypeError as cell17_type_error:
            cell17_last_type_error = cell17_type_error

        try:
            return cell17_model_object(
                cell17_x_tensor,
                cell17_ei_pos_tensor,
                cell17_ei_neg_tensor,
            )
        except TypeError as cell17_type_error:
            cell17_last_type_error = cell17_type_error

        raise TypeError(
            "[Cell 17][FATAL] Could not call SPGCN model with any known signature. "
            "Last TypeError: " + repr(cell17_last_type_error)
        )

    def cell17_predict_spgcn(
        cell17_model_object,
        cell17_x_tensor,
        cell17_ei_pos_tensor,
        cell17_ew_pos_tensor,
        cell17_dis_pos_tensor,
        cell17_ei_neg_tensor,
        cell17_ew_neg_tensor,
        cell17_dis_neg_tensor,
    ):
        cell17_model_object.eval()
        cell17_model_device = next(cell17_model_object.parameters()).device

        cell17_x_device = cell17_x_tensor.to(cell17_model_device)
        cell17_ei_pos_device = cell17_move_tensor(cell17_ei_pos_tensor, cell17_model_device)
        cell17_ew_pos_device = cell17_move_tensor(cell17_ew_pos_tensor, cell17_model_device)
        cell17_dis_pos_device = cell17_move_tensor(cell17_dis_pos_tensor, cell17_model_device)
        cell17_ei_neg_device = cell17_move_tensor(cell17_ei_neg_tensor, cell17_model_device)
        cell17_ew_neg_device = cell17_move_tensor(cell17_ew_neg_tensor, cell17_model_device)
        cell17_dis_neg_device = cell17_move_tensor(cell17_dis_neg_tensor, cell17_model_device)

        with cell17_torch.no_grad():
            cell17_output_object = cell17_forward_spgcn(
                cell17_model_object,
                cell17_x_device,
                cell17_ei_pos_device,
                cell17_ew_pos_device,
                cell17_dis_pos_device,
                cell17_ei_neg_device,
                cell17_ew_neg_device,
                cell17_dis_neg_device,
            )

            cell17_candidate_tensors = cell17_collect_tensors(cell17_output_object)

            if len(cell17_candidate_tensors) == 0:
                raise TypeError("[Cell 17][FATAL] No 2D+ tensor in model output.")

            cell17_proto_parameter = getattr(cell17_model_object, "proto_class", None)

            if cell17_torch.is_tensor(cell17_proto_parameter) and int(cell17_proto_parameter.ndim) == 2:
                cell17_proto_float = cell17_proto_parameter.detach().float()
                cell17_proto_dim = int(cell17_proto_float.shape[1])

                for cell17_candidate_tensor in cell17_candidate_tensors:
                    if int(cell17_candidate_tensor.shape[1]) == cell17_proto_dim:
                        cell17_distance_matrix = cell17_torch.cdist(
                            cell17_candidate_tensor.detach().float(),
                            cell17_proto_float,
                        )
                        return (
                            cell17_distance_matrix
                            .argmin(dim=1)
                            .detach()
                            .cpu()
                            .numpy()
                            .astype(cell17_np.int64)
                        )

            for cell17_candidate_tensor in cell17_candidate_tensors:
                if int(cell17_candidate_tensor.shape[1]) == cell17_num_classes:
                    return (
                        cell17_candidate_tensor
                        .argmax(dim=1)
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(cell17_np.int64)
                    )

            return (
                cell17_candidate_tensors[0]
                .argmax(dim=-1)
                .detach()
                .cpu()
                .numpy()
                .astype(cell17_np.int64)
            )

    def cell17_refine_prototypes(
        cell17_model_object,
        cell17_x_tensor,
        cell17_ei_pos_tensor,
        cell17_ew_pos_tensor,
        cell17_dis_pos_tensor,
        cell17_ei_neg_tensor,
        cell17_ew_neg_tensor,
        cell17_dis_neg_tensor,
        cell17_train_index,
        cell17_unlabeled_index,
        cell17_labels_np,
        cell17_class_count,
    ):
        cell17_model_object.eval()
        cell17_model_device = next(cell17_model_object.parameters()).device

        cell17_x_device = cell17_x_tensor.to(cell17_model_device)
        cell17_ei_pos_device = cell17_move_tensor(cell17_ei_pos_tensor, cell17_model_device)
        cell17_ew_pos_device = cell17_move_tensor(cell17_ew_pos_tensor, cell17_model_device)
        cell17_dis_pos_device = cell17_move_tensor(cell17_dis_pos_tensor, cell17_model_device)
        cell17_ei_neg_device = cell17_move_tensor(cell17_ei_neg_tensor, cell17_model_device)
        cell17_ew_neg_device = cell17_move_tensor(cell17_ew_neg_tensor, cell17_model_device)
        cell17_dis_neg_device = cell17_move_tensor(cell17_dis_neg_tensor, cell17_model_device)

        cell17_train_index_tensor = cell17_torch.as_tensor(
            cell17_to_int_vector(cell17_train_index, "Cell 17 train index"),
            dtype=cell17_torch.long,
            device=cell17_model_device,
        )

        cell17_unlabeled_index_tensor = cell17_torch.as_tensor(
            cell17_to_int_vector(cell17_unlabeled_index, "Cell 17 unlabeled index"),
            dtype=cell17_torch.long,
            device=cell17_model_device,
        )

        cell17_labels_tensor = cell17_torch.as_tensor(
            cell17_labels_np,
            dtype=cell17_torch.long,
            device=cell17_model_device,
        )

        if int(cell17_unlabeled_index_tensor.numel()) == 0:
            return cell17_model_object, 0

        with cell17_torch.no_grad():
            cell17_output_object = cell17_forward_spgcn(
                cell17_model_object,
                cell17_x_device,
                cell17_ei_pos_device,
                cell17_ew_pos_device,
                cell17_dis_pos_device,
                cell17_ei_neg_device,
                cell17_ew_neg_device,
                cell17_dis_neg_device,
            )

            cell17_embedding_tensor = cell17_extract_embedding(
                cell17_output_object,
                cell17_model_object,
                int(cell17_x_device.shape[0]),
            )

            cell17_train_embedding = cell17_embedding_tensor[cell17_train_index_tensor]
            cell17_unlabeled_embedding = cell17_embedding_tensor[cell17_unlabeled_index_tensor]
            cell17_train_labels = cell17_labels_tensor[cell17_train_index_tensor]

            cell17_refined_count = 0

            for cell17_class_index in range(int(cell17_class_count)):
                cell17_class_mask = cell17_train_labels == cell17_class_index
                cell17_support_count = int(cell17_torch.sum(cell17_class_mask).item())

                if cell17_support_count == 0 or cell17_support_count >= 100:
                    continue

                cell17_proto_parameter = cell17_model_object.proto_class.data[cell17_class_index]

                cell17_unlabeled_distance = cell17_torch.cdist(
                    cell17_unlabeled_embedding,
                    cell17_proto_parameter.unsqueeze(0),
                ).squeeze(-1)

                cell17_similarity = cell17_torch.exp(
                    -cell17_unlabeled_distance / float(cell17_tau_refine)
                )

                cell17_similarity_sum = cell17_similarity.sum() + 1e-8
                cell17_similarity_normalized = cell17_similarity / cell17_similarity_sum

                cell17_unlabeled_centroid = (
                    cell17_similarity_normalized.unsqueeze(-1)
                    * cell17_unlabeled_embedding
                ).sum(0)

                cell17_labeled_centroid = cell17_train_embedding[cell17_class_mask].mean(0)

                cell17_refined_proto = (
                    cell17_labeled_centroid
                    + float(cell17_lambda_refine) * cell17_unlabeled_centroid
                ) / (1.0 + float(cell17_lambda_refine))

                cell17_model_object.proto_class.data[cell17_class_index] = cell17_refined_proto
                cell17_refined_count = cell17_refined_count + 1

        return cell17_model_object, cell17_refined_count

    def cell17_fmt_metric(cell17_numeric_value):
        try:
            cell17_float_value = float(cell17_numeric_value)
        except Exception:
            return "n/a"

        if not cell17_np.isfinite(cell17_float_value):
            return "n/a"

        return f"{cell17_float_value:.4f}"

    cell17_x_source = cell17_globals["X_sspr_tensor_cell11"]

    if cell17_torch.is_tensor(cell17_x_source):
        cell17_x_sspr = cell17_x_source.to(cell17_device)
    else:
        cell17_x_array = cell17_to_numpy(cell17_x_source)

        if cell17_x_array is None:
            raise TypeError("[Cell 17][FATAL] X_sspr_tensor_cell11 could not be converted.")

        cell17_x_sspr = cell17_torch.tensor(
            cell17_x_array.astype(cell17_np.float32),
            dtype=cell17_torch.float32,
        ).to(cell17_device)

    if int(cell17_x_sspr.ndim) != 2:
        raise RuntimeError(
            "[Cell 17][FATAL] X_sspr_tensor_cell11 must be 2D, got shape "
            + str(tuple(cell17_x_sspr.shape))
        )

    cell17_feature_dim = int(cell17_x_sspr.shape[1])

    cell17_rare_source = cell17_to_numpy(cell17_globals["rare_classes_list_cell3"])

    if cell17_rare_source is None:
        raise TypeError("[Cell 17][FATAL] rare_classes_list_cell3 could not be converted.")

    cell17_rare_classes = []
    cell17_seen_rare_classes = set()

    for cell17_rare_value in cell17_rare_source.ravel().tolist():
        cell17_rare_integer = int(cell17_rare_value)

        if cell17_rare_integer not in cell17_seen_rare_classes:
            cell17_seen_rare_classes.add(cell17_rare_integer)
            cell17_rare_classes.append(cell17_rare_integer)

    cell17_labels_array = cell17_to_numpy(cell17_globals["labels_tensor_cell3"])

    if cell17_labels_array is None:
        raise TypeError("[Cell 17][FATAL] labels_tensor_cell3 could not be converted.")

    cell17_labels_np = cell17_labels_array.astype(cell17_np.int64).reshape(-1)

    cell17_feature_mean_source = cell17_first_global(
        [
            "X_sspr_mean_cell11",
            "sspr_feature_mean_cell11",
            "feat_mean_cell11",
            "feature_mean_cell11",
        ]
    )

    if cell17_feature_mean_source is None:
        cell17_feature_mean_cpu = cell17_x_sspr.mean(0).detach().cpu()
    else:
        cell17_feature_mean_array = cell17_to_numpy(cell17_feature_mean_source)

        if (
            cell17_feature_mean_array is None
            or int(cell17_feature_mean_array.size) != cell17_feature_dim
        ):
            cell17_feature_mean_cpu = cell17_x_sspr.mean(0).detach().cpu()
        else:
            cell17_feature_mean_cpu = cell17_torch.tensor(
                cell17_feature_mean_array.astype(cell17_np.float32).reshape(-1),
                dtype=cell17_torch.float32,
            )

    cell17_feature_std_source = cell17_first_global(
        [
            "X_sspr_std_cell11",
            "sspr_feature_std_cell11",
            "feat_std_cell11",
            "feature_std_cell11",
        ]
    )

    if cell17_feature_std_source is None:
        cell17_feature_std_cpu = cell17_x_sspr.std(0).detach().cpu()
    else:
        cell17_feature_std_array = cell17_to_numpy(cell17_feature_std_source)

        if (
            cell17_feature_std_array is None
            or int(cell17_feature_std_array.size) != cell17_feature_dim
        ):
            cell17_feature_std_cpu = cell17_x_sspr.std(0).detach().cpu()
        else:
            cell17_feature_std_cpu = cell17_torch.tensor(
                cell17_feature_std_array.astype(cell17_np.float32).reshape(-1),
                dtype=cell17_torch.float32,
            )

    print("=" * 100)
    print("CELL 17 — FREEZE AND SAVE FAFB SSPR MODEL")
    print("=" * 100)
    print("[cfg] seed=" + str(cell17_seed_value))
    print("[cfg] epochs=" + str(cell17_epoch_count))
    print("[cfg] refinement lambda=" + str(cell17_lambda_refine))
    print("[cfg] refinement tau=" + str(cell17_tau_refine))
    print("[data] X_sspr shape=" + str(tuple(cell17_x_sspr.shape)))
    print("[data] C=" + str(cell17_num_classes) + ", C_super=" + str(cell17_num_super_classes))
    print("[data] rare classes=" + str(cell17_rare_classes))
    print("[out] frozen path=" + str(cell17_frozen_path))
    print("=" * 100)

    cell17_set_seed(cell17_seed_value)

    cell17_train_index, cell17_val_index, cell17_test_index = cell17_globals["make_split_cell4"](
        cell17_seed_value,
        cell17_globals["labeled_mask_cell3"],
        cell17_globals["labels_tensor_cell3"],
        cell17_globals["C_cell3"],
    )

    cell17_test_np = cell17_to_int_vector(cell17_test_index, "Cell 17 test index")
    cell17_y_test_np = cell17_labels_np[cell17_test_np].copy()

    cell17_model_object = cell17_make_spgcn(cell17_feature_dim).to(cell17_device)

    cell17_train_result = cell17_globals["train_spgcn_cell6"](
        cell17_model_object,
        cell17_x_sspr,
        cell17_globals["ei_pos_cell7"],
        cell17_globals["ew_pos_cell7"],
        cell17_globals["dis_pos_cell7"],
        cell17_globals["ei_neg_cell7"],
        cell17_globals["ew_neg_cell7"],
        cell17_globals["dis_neg_cell7"],
        cell17_train_index,
        cell17_val_index,
        cell17_test_index,
        cell17_globals["labels_tensor_cell3"],
        cell17_globals["super_labels_tensor_cell3"],
        cell17_globals["rare_classes_list_cell3"],
        cell17_seed_value,
        epochs_cell6=cell17_epoch_count,
        C_cell6=cell17_num_classes,
        C_super_cell6=cell17_num_super_classes,
        use_hierarchical_cell6=False,
        use_reinit_cell6=True,
    )

    cell17_pred_before_full = cell17_predict_spgcn(
        cell17_model_object,
        cell17_x_sspr,
        cell17_globals["ei_pos_cell7"],
        cell17_globals["ew_pos_cell7"],
        cell17_globals["dis_pos_cell7"],
        cell17_globals["ei_neg_cell7"],
        cell17_globals["ew_neg_cell7"],
        cell17_globals["dis_neg_cell7"],
    )

    cell17_pred_before_test = cell17_pred_before_full[cell17_test_np].copy()

    cell17_before_rare_ba = float(
        cell17_globals["rare_balanced_acc_cell6"](
            cell17_y_test_np,
            cell17_pred_before_test,
            cell17_rare_classes,
        )
    )

    print("  [before_refine] rare_ba=" + cell17_fmt_metric(cell17_before_rare_ba), flush=True)

    cell17_unlabeled_tensor = cell17_torch.where(~cell17_globals["labeled_mask_cell3"])[0]
    cell17_unlabeled_np = cell17_to_int_vector(cell17_unlabeled_tensor, "Cell 17 unlabeled index")

    cell17_overlap_array = cell17_np.intersect1d(
        cell17_test_np,
        cell17_unlabeled_np,
        assume_unique=False,
    )

    if int(cell17_overlap_array.size) != 0:
        raise RuntimeError(
            "[Cell 17][FATAL] Test nodes leaked into unlabeled refinement set: "
            + str(int(cell17_overlap_array.size))
            + " overlapping nodes."
        )

    cell17_model_object, cell17_refined_count = cell17_refine_prototypes(
        cell17_model_object,
        cell17_x_sspr,
        cell17_globals["ei_pos_cell7"],
        cell17_globals["ew_pos_cell7"],
        cell17_globals["dis_pos_cell7"],
        cell17_globals["ei_neg_cell7"],
        cell17_globals["ew_neg_cell7"],
        cell17_globals["dis_neg_cell7"],
        cell17_train_index,
        cell17_unlabeled_tensor,
        cell17_labels_np,
        cell17_num_classes,
    )

    cell17_pred_after_full = cell17_predict_spgcn(
        cell17_model_object,
        cell17_x_sspr,
        cell17_globals["ei_pos_cell7"],
        cell17_globals["ew_pos_cell7"],
        cell17_globals["dis_pos_cell7"],
        cell17_globals["ei_neg_cell7"],
        cell17_globals["ew_neg_cell7"],
        cell17_globals["dis_neg_cell7"],
    )

    cell17_pred_after_test = cell17_pred_after_full[cell17_test_np].copy()

    cell17_after_rare_ba = float(
        cell17_globals["rare_balanced_acc_cell6"](
            cell17_y_test_np,
            cell17_pred_after_test,
            cell17_rare_classes,
        )
    )

    print("  [after_refine]  rare_ba=" + cell17_fmt_metric(cell17_after_rare_ba), flush=True)
    print("  [refined_prototypes] count=" + str(cell17_refined_count), flush=True)

    cell17_class_names = None

    if "classes_list_cell3" in cell17_globals:
        try:
            cell17_class_names = [
                str(cell17_class_name_value)
                for cell17_class_name_value in cell17_globals["classes_list_cell3"]
            ]
        except Exception:
            cell17_class_names = None

    cell17_proto_super_tensor = getattr(cell17_model_object, "proto_super", None)

    cell17_save_dict = {
        "encoder_state": {
            cell17_state_key: cell17_state_value.cpu()
            for cell17_state_key, cell17_state_value in cell17_model_object.state_dict().items()
        },
        "proto_class": cell17_model_object.proto_class.data.cpu(),
        "proto_super": (
            cell17_proto_super_tensor.data.cpu()
            if cell17_torch.is_tensor(cell17_proto_super_tensor)
            else None
        ),
        "feature_mean": cell17_feature_mean_cpu,
        "feature_std": cell17_feature_std_cpu,
        "d_features": int(cell17_feature_dim),
        "d_cell5": 64,
        "d_proto_cell5": 32,
        "n_layers_cell5": 2,
        "n_classes_cell5": int(cell17_num_classes),
        "n_super_cell5": int(cell17_num_super_classes),
        "temperature_cell5": 0.1,
        "dropout_cell5": 0.3,
        "class_names": cell17_class_names,
        "frozen_seed": int(cell17_seed_value),
        "before_refine_rare_ba": float(cell17_before_rare_ba),
        "after_refine_rare_ba": float(cell17_after_rare_ba),
        "refined_prototype_count": int(cell17_refined_count),
        "rare_classes": cell17_rare_classes,
        "lambda_refine": float(cell17_lambda_refine),
        "tau_refine": float(cell17_tau_refine),
    }

    cell17_torch.save(cell17_save_dict, cell17_frozen_path)

    print("\n" + "=" * 100)
    print("CELL 17 FINAL SUMMARY")
    print("=" * 100)
    print("Frozen SSPR saved to: " + str(cell17_frozen_path))
    print("FAFB seed 0 before refinement rare_ba: " + cell17_fmt_metric(cell17_before_rare_ba))
    print("FAFB seed 0 after refinement rare_ba:  " + cell17_fmt_metric(cell17_after_rare_ba))
    print("Refined prototype count:               " + str(cell17_refined_count))
    print("=" * 100)

    cell17_config = {
        "frozen_path": str(cell17_frozen_path),
        "seed": int(cell17_seed_value),
        "epochs": int(cell17_epoch_count),
        "lambda_refine": float(cell17_lambda_refine),
        "tau_refine": float(cell17_tau_refine),
        "d_features": int(cell17_feature_dim),
        "C": int(cell17_num_classes),
        "C_super": int(cell17_num_super_classes),
        "rare_classes": cell17_rare_classes,
        "before_refine_rare_ba": float(cell17_before_rare_ba),
        "after_refine_rare_ba": float(cell17_after_rare_ba),
        "refined_prototype_count": int(cell17_refined_count),
    }

    cell17_hf_save_checkpoint = cell17_globals.get("hf_save_checkpoint", None)

    if callable(cell17_hf_save_checkpoint):
        try:
            cell17_hf_save_checkpoint(
                "cell17_frozen_sspr",
                {
                    "config": cell17_config,
                    "frozen_path": str(cell17_frozen_path),
                    "before_refine_rare_ba": float(cell17_before_rare_ba),
                    "after_refine_rare_ba": float(cell17_after_rare_ba),
                    "refined_prototype_count": int(cell17_refined_count),
                },
                description=(
                    "Cell 17: frozen FAFB SSPR encoder and prototypes saved for "
                    "cross-connectome transfer."
                ),
                upload=True,
            )
        except Exception as cell17_checkpoint_error:
            print("[ckpt][warn] Cell 17 checkpoint failed: " + repr(cell17_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    cell17_hf_upload_file_to_bucket = cell17_globals.get("hf_upload_file_to_bucket", None)

    if callable(cell17_hf_upload_file_to_bucket):
        try:
            cell17_remote_frozen_path = (
                "checkpoints/cell17_frozen_sspr/sspr_fafb_frozen_cell17.pt"
            )

            cell17_upload_ok, cell17_upload_info = cell17_hf_upload_file_to_bucket(
                cell17_frozen_path,
                cell17_remote_frozen_path,
            )

            print(
                "[ckpt:cell17] upload frozen model: "
                + ("OK" if cell17_upload_ok else "FAILED")
                + " | "
                + str(cell17_upload_info)
            )
        except Exception as cell17_upload_error:
            print("[ckpt][warn] Cell 17 frozen model upload failed: " + repr(cell17_upload_error))
    else:
        print(
            "[ckpt][skip] hf_upload_file_to_bucket not defined. "
            "Frozen model remains local only."
        )

    return {
        "frozen_path": str(cell17_frozen_path),
        "config": cell17_config,
        "before_refine_rare_ba": float(cell17_before_rare_ba),
        "after_refine_rare_ba": float(cell17_after_rare_ba),
        "refined_prototype_count": int(cell17_refined_count),
    }


cell17_output = cell17_main()

cell17_frozen_path = cell17_output["frozen_path"]
cell17_frozen_config = cell17_output["config"]
cell17_before_refine_rare_ba = cell17_output["before_refine_rare_ba"]
cell17_after_refine_rare_ba = cell17_output["after_refine_rare_ba"]
cell17_refined_prototype_count = cell17_output["refined_prototype_count"]

In [ ]:
# Cell 18: NTAC 10-seed on 25000-node subgraph.
#
# Rules:
#   - No top-level imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every notebook-local/helper/loop variable is prefixed with cell18_.
#   - The isolated worker script uses cell18w_ prefixes internally.
#
# Behavior:
#   - Reuses Cell 12 worker/path/graph variables if available.
#   - Otherwise falls back to Cell 3 graph arrays and writes its own worker script.
#   - Does not fabricate NTAC results. If NTAC is not installed or a seed fails,
#     the failure status is recorded.


cell18_worker_script_source = r'''
import sys
import json
import resource
import traceback
import numpy as np
import scipy.sparse as sp
from collections import Counter

cell18w_mem_mb = int(sys.argv[3])
cell18w_mem_bytes = cell18w_mem_mb * 1024 * 1024

if cell18w_mem_mb > 0:
    try:
        resource.setrlimit(resource.RLIMIT_AS, (cell18w_mem_bytes, cell18w_mem_bytes))
    except Exception:
        pass

cell18w_npz_path = sys.argv[1]
cell18w_out_path = sys.argv[2]
cell18w_steps = int(sys.argv[4])

try:
    from ntac import Ntac, GraphData
except Exception as cell18w_import_error:
    cell18w_error_dict = {
        "status": "not_installed",
        "error": repr(cell18w_import_error),
    }
    with open(cell18w_out_path, "w", encoding="utf-8") as cell18w_file_handle:
        json.dump(cell18w_error_dict, cell18w_file_handle)
    sys.exit(0)

try:
    cell18w_loader = np.load(cell18w_npz_path, allow_pickle=False)

    cell18w_src = cell18w_loader["src"].astype(np.int64)
    cell18w_dst = cell18w_loader["dst"].astype(np.int64)
    cell18w_labels_str = cell18w_loader["labels_str"].astype(str)
    cell18w_labels_int = cell18w_loader["labels_int"].astype(np.int64)
    cell18w_tr = cell18w_loader["tr"].astype(np.int64)
    cell18w_te = cell18w_loader["te"].astype(np.int64)
    cell18w_rare = cell18w_loader["rare"].astype(np.int64)
    cell18w_nodes_root = cell18w_loader["nodes_root"].astype(np.int64)
    cell18w_N = int(cell18w_loader["N_sub"][0])

    cell18w_A = sp.csr_matrix(
        (np.ones(len(cell18w_src), dtype=np.float32), (cell18w_src, cell18w_dst)),
        shape=(cell18w_N, cell18w_N),
    )
    cell18w_A = (cell18w_A + cell18w_A.T).tocsr()
    cell18w_A.sum_duplicates()

    cell18w_data = None
    cell18w_last_error = None

    cell18w_graph_data_constructors = [
        lambda: GraphData(cell18w_A, labels=cell18w_labels_str),
        lambda: GraphData(cell18w_A, labels=cell18w_labels_str.tolist()),
        lambda: GraphData(cell18w_labels_str),
        lambda: GraphData(cell18w_labels_str.tolist()),
        lambda: GraphData(graph=cell18w_A, labels=cell18w_labels_str),
        lambda: GraphData(adjacency=cell18w_A, labels=cell18w_labels_str),
    ]

    for cell18w_constructor_function in cell18w_graph_data_constructors:
        try:
            cell18w_data = cell18w_constructor_function()
            break
        except TypeError as cell18w_type_error:
            cell18w_last_error = cell18w_type_error
            continue
        except Exception as cell18w_runtime_error:
            raise RuntimeError("GraphData failed: " + repr(cell18w_runtime_error)) from cell18w_runtime_error

    if cell18w_data is None:
        raise TypeError("Could not construct GraphData. Last TypeError: " + repr(cell18w_last_error))

    cell18w_nt = None
    cell18w_last_error = None

    cell18w_ntac_constructors = [
        lambda: Ntac(data=cell18w_data, labels=cell18w_labels_str),
        lambda: Ntac(data=cell18w_data, labels=cell18w_labels_str.tolist()),
        lambda: Ntac(cell18w_data, cell18w_labels_str),
        lambda: Ntac(cell18w_data, cell18w_labels_str.tolist()),
        lambda: Ntac(graph_data=cell18w_data, labels=cell18w_labels_str),
        lambda: Ntac(data=cell18w_data),
        lambda: Ntac(cell18w_data),
    ]

    for cell18w_constructor_function in cell18w_ntac_constructors:
        try:
            cell18w_nt = cell18w_constructor_function()
            break
        except TypeError as cell18w_type_error:
            cell18w_last_error = cell18w_type_error
            continue
        except Exception as cell18w_runtime_error:
            raise RuntimeError("Ntac failed: " + repr(cell18w_runtime_error)) from cell18w_runtime_error

    if cell18w_nt is None:
        raise TypeError("Could not construct Ntac. Last TypeError: " + repr(cell18w_last_error))

    cell18w_used_method = None

    for cell18w_step_index in range(cell18w_steps):
        if hasattr(cell18w_nt, "step"):
            cell18w_nt.step()
            cell18w_used_method = "step"
        elif hasattr(cell18w_nt, "iterate"):
            cell18w_nt.iterate()
            cell18w_used_method = "iterate"
        elif hasattr(cell18w_nt, "update"):
            cell18w_nt.update()
            cell18w_used_method = "update"
        elif hasattr(cell18w_nt, "fit"):
            if cell18w_step_index == 0:
                cell18w_nt.fit()
                cell18w_used_method = "fit"
            else:
                cell18w_used_method = "fit_already_called"
            break
        else:
            raise AttributeError("Ntac object has no step/iterate/update/fit method.")

    cell18w_part = None

    if hasattr(cell18w_nt, "get_partition"):
        cell18w_part = cell18w_nt.get_partition()
    elif hasattr(cell18w_nt, "partition"):
        cell18w_partition_attribute = cell18w_nt.partition
        cell18w_part = cell18w_partition_attribute() if callable(cell18w_partition_attribute) else cell18w_partition_attribute
    elif hasattr(cell18w_nt, "labels_"):
        cell18w_part = cell18w_nt.labels_
    elif hasattr(cell18w_nt, "pred_"):
        cell18w_part = cell18w_nt.pred_
    else:
        raise AttributeError("Could not find partition output on Ntac object.")

    cell18w_part_arr = np.full(cell18w_N, "-1", dtype=object)

    if isinstance(cell18w_part, dict):
        cell18w_root_to_new = {}

        if len(cell18w_nodes_root) == cell18w_N:
            for cell18w_new_index, cell18w_root_value in enumerate(cell18w_nodes_root):
                cell18w_root_to_new[int(cell18w_root_value)] = cell18w_new_index

        for cell18w_part_key, cell18w_part_value in cell18w_part.items():
            try:
                cell18w_part_key_int = int(cell18w_part_key)
            except Exception:
                continue

            if 0 <= cell18w_part_key_int < cell18w_N:
                cell18w_part_arr[cell18w_part_key_int] = str(cell18w_part_value)
            elif cell18w_part_key_int in cell18w_root_to_new:
                cell18w_part_arr[cell18w_root_to_new[cell18w_part_key_int]] = str(cell18w_part_value)
    else:
        cell18w_part_array = np.asarray(cell18w_part).astype(object)

        if cell18w_part_array.shape[0] != cell18w_N:
            raise RuntimeError(
                "Partition length mismatch: got "
                + str(cell18w_part_array.shape[0])
                + ", expected "
                + str(cell18w_N)
                + "."
            )

        cell18w_part_arr[:] = cell18w_part_array[:]

    cell18w_cluster_to_class = {}

    for cell18w_train_node in cell18w_tr:
        cell18w_train_node_int = int(cell18w_train_node)
        cell18w_cluster_id = str(cell18w_part_arr[cell18w_train_node_int])
        cell18w_class_id = int(cell18w_labels_int[cell18w_train_node_int])
        cell18w_cluster_to_class.setdefault(cell18w_cluster_id, []).append(cell18w_class_id)

    cell18w_cluster_to_class = {
        cell18w_cluster_key: Counter(cell18w_cluster_values).most_common(1)[0][0]
        for cell18w_cluster_key, cell18w_cluster_values in cell18w_cluster_to_class.items()
    }

    cell18w_pred_list = []
    cell18w_true_list = []

    for cell18w_test_node in cell18w_te:
        cell18w_test_node_int = int(cell18w_test_node)
        cell18w_cluster_id = str(cell18w_part_arr[cell18w_test_node_int])
        cell18w_pred_list.append(int(cell18w_cluster_to_class.get(cell18w_cluster_id, -1)))
        cell18w_true_list.append(int(cell18w_labels_int[cell18w_test_node_int]))

    cell18w_pred = np.asarray(cell18w_pred_list, dtype=np.int64)
    cell18w_true = np.asarray(cell18w_true_list, dtype=np.int64)

    cell18w_recalls = []

    for cell18w_rare_class in cell18w_rare:
        cell18w_support_mask = cell18w_true == int(cell18w_rare_class)
        cell18w_support_count = int(np.sum(cell18w_support_mask))

        if cell18w_support_count > 0:
            cell18w_hit_mask = cell18w_support_mask & (cell18w_pred == int(cell18w_rare_class))
            cell18w_recalls.append(float(np.sum(cell18w_hit_mask)) / float(cell18w_support_count))

    if len(cell18w_recalls) > 0:
        cell18w_rare_ba = float(np.mean(cell18w_recalls))
    else:
        cell18w_rare_ba = float("nan")

    cell18w_mapped_classes = set(int(cell18w_mapped_value) for cell18w_mapped_value in cell18w_cluster_to_class.values())
    cell18w_unmapped_rare = [int(cell18w_rare_value) for cell18w_rare_value in cell18w_rare if int(cell18w_rare_value) not in cell18w_mapped_classes]

    cell18w_output_dict = {
        "status": "ok",
        "rare_ba": cell18w_rare_ba,
        "n_clusters_seen_train": len(cell18w_cluster_to_class),
        "n_classes_mapped": len(cell18w_mapped_classes),
        "unmapped_rare_count": len(cell18w_unmapped_rare),
        "unmapped_rare_classes": cell18w_unmapped_rare,
        "used_method": cell18w_used_method,
        "N_sub": cell18w_N,
        "edges_sub": int(len(cell18w_src)),
        "train_nodes": int(len(cell18w_tr)),
        "test_nodes": int(len(cell18w_te)),
    }

    with open(cell18w_out_path, "w", encoding="utf-8") as cell18w_file_handle:
        json.dump(cell18w_output_dict, cell18w_file_handle)

except Exception as cell18w_error:
    cell18w_failure_dict = {
        "status": "failed",
        "error": repr(cell18w_error),
        "traceback": traceback.format_exc(),
    }

    with open(cell18w_out_path, "w", encoding="utf-8") as cell18w_file_handle:
        json.dump(cell18w_failure_dict, cell18w_file_handle)
'''


def cell18_main():
    cell18_globals = globals()

    cell18_np = cell18_globals.get("np", None)
    if cell18_np is None:
        import numpy as cell18_np

    cell18_json = cell18_globals.get("json", None)
    if cell18_json is None:
        import json as cell18_json

    cell18_time = cell18_globals.get("time", None)
    if cell18_time is None:
        import time as cell18_time

    cell18_gc = cell18_globals.get("gc", None)
    if cell18_gc is None:
        import gc as cell18_gc

    cell18_subprocess = cell18_globals.get("subprocess", None)
    if cell18_subprocess is None:
        import subprocess as cell18_subprocess

    cell18_sys = cell18_globals.get("sys", None)
    if cell18_sys is None:
        import sys as cell18_sys

    cell18_Path = cell18_globals.get("Path", None)
    if cell18_Path is None:
        from pathlib import Path as cell18_Path

    cell18_required_names = [
        "make_split_cell4",
        "labeled_mask_cell3",
        "labels_tensor_cell3",
        "C_cell3",
    ]

    cell18_missing_names = []

    for cell18_required_name in cell18_required_names:
        if cell18_required_name not in cell18_globals:
            cell18_missing_names.append(cell18_required_name)

    if len(cell18_missing_names) > 0:
        raise NameError(
            "[Cell 18][FATAL] Missing upstream variables: "
            + str(cell18_missing_names)
            + ". Run Cells 3-4 first."
        )

    def cell18_first_global(cell18_candidate_names):
        for cell18_candidate_name in cell18_candidate_names:
            if cell18_candidate_name in cell18_globals and cell18_globals[cell18_candidate_name] is not None:
                return cell18_globals[cell18_candidate_name]

        return None

    def cell18_to_numpy(cell18_object):
        if cell18_object is None:
            return None

        cell18_working_object = cell18_object

        try:
            if hasattr(cell18_working_object, "detach"):
                cell18_working_object = cell18_working_object.detach()

            if hasattr(cell18_working_object, "cpu"):
                cell18_working_object = cell18_working_object.cpu()

            if hasattr(cell18_working_object, "numpy"):
                cell18_working_object = cell18_working_object.numpy()

            return cell18_np.asarray(cell18_working_object)
        except Exception:
            return None

    def cell18_to_int_vector(cell18_object, cell18_description):
        cell18_array_object = cell18_to_numpy(cell18_object)

        if cell18_array_object is None:
            raise TypeError(
                "[Cell 18][FATAL] Could not convert "
                + str(cell18_description)
                + " to numpy."
            )

        try:
            return cell18_array_object.astype(cell18_np.int64).reshape(-1)
        except Exception as cell18_cast_error:
            raise TypeError(
                "[Cell 18][FATAL] Could not cast "
                + str(cell18_description)
                + " to int64: "
                + repr(cell18_cast_error)
            )

    def cell18_to_bool_vector(cell18_object, cell18_description):
        cell18_array_object = cell18_to_numpy(cell18_object)

        if cell18_array_object is None:
            raise TypeError(
                "[Cell 18][FATAL] Could not convert "
                + str(cell18_description)
                + " to numpy."
            )

        try:
            return cell18_array_object.astype(bool).reshape(-1)
        except Exception as cell18_cast_error:
            raise TypeError(
                "[Cell 18][FATAL] Could not cast "
                + str(cell18_description)
                + " to bool: "
                + repr(cell18_cast_error)
            )

    cell18_target_nodes = 25000
    cell18_seed_list = list(range(10))
    cell18_steps = 3
    cell18_timeout_seconds = 900
    cell18_memory_mb = 8192

    cell18_work_dir_object = cell18_first_global(["work_dir_cell12"])

    if cell18_work_dir_object is None:
        cell18_work_dir = cell18_Path("./ntac_smoke_cell18")
    else:
        cell18_work_dir = cell18_Path(str(cell18_work_dir_object))

    cell18_work_dir.mkdir(parents=True, exist_ok=True)

    cell18_worker_path_object = cell18_first_global(["worker_path_cell12"])
    cell18_worker_path = None

    if cell18_worker_path_object is not None:
        cell18_candidate_worker_path = cell18_Path(str(cell18_worker_path_object))

        if cell18_candidate_worker_path.exists():
            cell18_worker_path = cell18_candidate_worker_path

    if cell18_worker_path is None:
        cell18_worker_path = cell18_work_dir / "ntac_worker_cell18.py"

        if not cell18_worker_path.exists():
            cell18_worker_path.write_text(cell18_worker_script_source, encoding="utf-8")
            print("[Cell 18] Wrote isolated NTAC worker script: " + str(cell18_worker_path))

    cell18_src_full_object = cell18_first_global(["src_full_cell12", "src_arr_cell3"])
    cell18_dst_full_object = cell18_first_global(["dst_full_cell12", "dst_arr_cell3"])
    cell18_labels_full_object = cell18_first_global(["labels_full_cell12", "labels_tensor_cell3"])
    cell18_labeled_mask_full_object = cell18_first_global(["labeled_mask_full_cell12", "labeled_mask_cell3"])
    cell18_C_full_object = cell18_first_global(["C_full_cell12", "C_cell3"])
    cell18_rare_source_object = cell18_first_global(["rare_classes_cell12", "rare_classes_list_cell3"])
    cell18_base_seed_object = cell18_first_global(["SEED_CELL12"])

    if cell18_src_full_object is None:
        raise NameError("[Cell 18][FATAL] No source edge array found. Expected src_full_cell12 or src_arr_cell3.")

    if cell18_dst_full_object is None:
        raise NameError("[Cell 18][FATAL] No destination edge array found. Expected dst_full_cell12 or dst_arr_cell3.")

    if cell18_labels_full_object is None:
        raise NameError("[Cell 18][FATAL] No label array found. Expected labels_full_cell12 or labels_tensor_cell3.")

    if cell18_labeled_mask_full_object is None:
        raise NameError("[Cell 18][FATAL] No labeled mask found. Expected labeled_mask_full_cell12 or labeled_mask_cell3.")

    if cell18_C_full_object is None:
        raise NameError("[Cell 18][FATAL] No class count found. Expected C_full_cell12 or C_cell3.")

    if cell18_rare_source_object is None:
        raise NameError("[Cell 18][FATAL] No rare class list found. Expected rare_classes_cell12 or rare_classes_list_cell3.")

    if cell18_base_seed_object is None:
        cell18_base_seed = 12
    else:
        cell18_base_seed = int(cell18_base_seed_object)

    cell18_src_full_array = cell18_to_int_vector(cell18_src_full_object, "Cell 18 source array")
    cell18_dst_full_array = cell18_to_int_vector(cell18_dst_full_object, "Cell 18 destination array")
    cell18_labels_full_array = cell18_to_int_vector(cell18_labels_full_object, "Cell 18 label array")
    cell18_labeled_mask_full_array = cell18_to_bool_vector(cell18_labeled_mask_full_object, "Cell 18 labeled mask")
    cell18_C_full = int(cell18_C_full_object)

    cell18_rare_array = cell18_to_int_vector(cell18_rare_source_object, "Cell 18 rare class array")
    cell18_rare_classes = sorted(set(int(cell18_rare_value) for cell18_rare_value in cell18_rare_array.tolist()))

    print("=" * 100)
    print("CELL 18 — NTAC 10-SEED ON 25,000-NODE SUBGRAPH")
    print("=" * 100)
    print("[cfg] target_nodes=" + str(cell18_target_nodes))
    print("[cfg] seeds=" + str(cell18_seed_list))
    print("[cfg] steps=" + str(cell18_steps))
    print("[cfg] timeout_sec=" + str(cell18_timeout_seconds))
    print("[cfg] memory_mb_per_worker=" + str(cell18_memory_mb))
    print("[cfg] base_seed=" + str(cell18_base_seed))
    print("[data] full_edges=" + str(int(cell18_src_full_array.size)))
    print("[data] C_full=" + str(cell18_C_full))
    print("[data] rare_classes=" + str(cell18_rare_classes))
    print("[path] work_dir=" + str(cell18_work_dir))
    print("[path] worker=" + str(cell18_worker_path))
    print("=" * 100)

    cell18_rng = cell18_np.random.default_rng(cell18_base_seed)
    cell18_labeled_idx = cell18_np.where(cell18_labeled_mask_full_array)[0]

    cell18_results_list = []
    cell18_summary_dict = {}

    if cell18_labeled_idx.size == 0:
        cell18_summary_dict = {
            "status": "no_labeled_nodes",
            "target_nodes": int(cell18_target_nodes),
            "n_valid_seeds": 0,
            "mean_rare_ba": float("nan"),
            "std_rare_ba": float("nan"),
        }
        cell18_summary_path = cell18_work_dir / "cell18_ntac_summary.json"
        cell18_summary_path.write_text(
            cell18_json.dumps(cell18_summary_dict, indent=2, default=str),
            encoding="utf-8",
        )
        print("[Cell 18][skip] No labeled nodes found.")
        return {
            "results": cell18_results_list,
            "summary": cell18_summary_dict,
            "work_dir": str(cell18_work_dir),
            "summary_path": str(cell18_summary_path),
            "results_path": "",
        }

    cell18_per_class_count = max(
        3,
        int(cell18_np.ceil(float(cell18_target_nodes) / float(max(1, cell18_C_full)))),
    )

    cell18_sampled_list = []

    for cell18_class_index in range(cell18_C_full):
        cell18_class_mask = cell18_labels_full_array[cell18_labeled_idx] == cell18_class_index
        cell18_class_indices = cell18_labeled_idx[cell18_class_mask]

        if cell18_class_indices.size == 0:
            continue

        cell18_take_count = min(cell18_per_class_count, int(cell18_class_indices.size))
        cell18_chosen_indices = cell18_rng.choice(
            cell18_class_indices,
            size=cell18_take_count,
            replace=False,
        )
        cell18_sampled_list.extend(cell18_chosen_indices.tolist())

    cell18_sampled_array = cell18_np.unique(
        cell18_np.asarray(cell18_sampled_list, dtype=cell18_np.int64)
    )

    if cell18_sampled_array.size < cell18_target_nodes:
        cell18_remaining_count = cell18_target_nodes - int(cell18_sampled_array.size)
        cell18_pool_array = cell18_np.setdiff1d(
            cell18_labeled_idx,
            cell18_sampled_array,
            assume_unique=False,
        )

        if cell18_pool_array.size > 0:
            cell18_extra_count = min(cell18_remaining_count, int(cell18_pool_array.size))
            cell18_extra_indices = cell18_rng.choice(
                cell18_pool_array,
                size=cell18_extra_count,
                replace=False,
            )
            cell18_sampled_array = cell18_np.unique(
                cell18_np.concatenate([cell18_sampled_array, cell18_extra_indices])
            )

    cell18_sampled_array = cell18_np.sort(cell18_sampled_array[:cell18_target_nodes])
    cell18_N_sub = int(cell18_sampled_array.size)

    print("[sample] subgraph_nodes=" + str(cell18_N_sub))

    if cell18_N_sub < 50:
        cell18_summary_dict = {
            "status": "too_small",
            "target_nodes": int(cell18_target_nodes),
            "actual_nodes": int(cell18_N_sub),
            "n_valid_seeds": 0,
            "mean_rare_ba": float("nan"),
            "std_rare_ba": float("nan"),
        }
        cell18_summary_path = cell18_work_dir / "cell18_ntac_summary.json"
        cell18_summary_path.write_text(
            cell18_json.dumps(cell18_summary_dict, indent=2, default=str),
            encoding="utf-8",
        )
        print("[Cell 18][skip] Subgraph too small.")
        return {
            "results": cell18_results_list,
            "summary": cell18_summary_dict,
            "work_dir": str(cell18_work_dir),
            "summary_path": str(cell18_summary_path),
            "results_path": "",
        }

    cell18_edge_mask = (
        cell18_np.isin(cell18_src_full_array, cell18_sampled_array)
        & cell18_np.isin(cell18_dst_full_array, cell18_sampled_array)
    )

    cell18_src_old_array = cell18_src_full_array[cell18_edge_mask]
    cell18_dst_old_array = cell18_dst_full_array[cell18_edge_mask]

    print("[sample] induced_edges=" + str(int(cell18_src_old_array.size)))

    if cell18_src_old_array.size < 20:
        cell18_summary_dict = {
            "status": "too_few_edges",
            "target_nodes": int(cell18_target_nodes),
            "actual_nodes": int(cell18_N_sub),
            "actual_edges": int(cell18_src_old_array.size),
            "n_valid_seeds": 0,
            "mean_rare_ba": float("nan"),
            "std_rare_ba": float("nan"),
        }
        cell18_summary_path = cell18_work_dir / "cell18_ntac_summary.json"
        cell18_summary_path.write_text(
            cell18_json.dumps(cell18_summary_dict, indent=2, default=str),
            encoding="utf-8",
        )
        print("[Cell 18][skip] Too few induced edges.")
        return {
            "results": cell18_results_list,
            "summary": cell18_summary_dict,
            "work_dir": str(cell18_work_dir),
            "summary_path": str(cell18_summary_path),
            "results_path": "",
        }

    cell18_src_sub_array = cell18_np.searchsorted(cell18_sampled_array, cell18_src_old_array).astype(cell18_np.int64)
    cell18_dst_sub_array = cell18_np.searchsorted(cell18_sampled_array, cell18_dst_old_array).astype(cell18_np.int64)
    cell18_labels_sub_int_array = cell18_labels_full_array[cell18_sampled_array].astype(cell18_np.int64)

    for cell18_seed_value in cell18_seed_list:
        print("\n[Cell 18] seed " + str(cell18_seed_value), flush=True)

        cell18_train_full, cell18_val_full, cell18_test_full = cell18_globals["make_split_cell4"](
            cell18_seed_value,
            cell18_globals["labeled_mask_cell3"],
            cell18_globals["labels_tensor_cell3"],
            cell18_globals["C_cell3"],
        )

        cell18_train_full_array = cell18_to_int_vector(cell18_train_full, "Cell 18 full train index")
        cell18_test_full_array = cell18_to_int_vector(cell18_test_full, "Cell 18 full test index")

        cell18_train_set = set(cell18_train_full_array.tolist())
        cell18_test_set = set(cell18_test_full_array.tolist())

        cell18_train_sub_list = []
        cell18_test_sub_list = []
        cell18_labels_split_int_array = cell18_np.full(cell18_N_sub, -1, dtype=cell18_np.int64)

        for cell18_new_index, cell18_old_index in enumerate(cell18_sampled_array):
            cell18_old_index_int = int(cell18_old_index)

            if cell18_old_index_int in cell18_train_set:
                cell18_train_sub_list.append(cell18_new_index)
                cell18_labels_split_int_array[cell18_new_index] = int(cell18_labels_full_array[cell18_old_index_int])
            elif cell18_old_index_int in cell18_test_set:
                cell18_test_sub_list.append(cell18_new_index)

        cell18_train_sub_array = cell18_np.asarray(cell18_train_sub_list, dtype=cell18_np.int64)
        cell18_test_sub_array = cell18_np.asarray(cell18_test_sub_list, dtype=cell18_np.int64)

        if cell18_train_sub_array.size < 20 or cell18_test_sub_array.size < 5:
            print("  [warn] projected split too small; using random subgraph split.", flush=True)

            cell18_permutation_array = cell18_rng.permutation(cell18_N_sub)
            cell18_random_train_count = int(0.60 * cell18_N_sub)
            cell18_random_val_count = int(0.20 * cell18_N_sub)

            cell18_train_sub_array = cell18_permutation_array[:cell18_random_train_count]
            cell18_test_sub_array = cell18_permutation_array[cell18_random_train_count + cell18_random_val_count:]

            cell18_labels_split_int_array = cell18_np.full(cell18_N_sub, -1, dtype=cell18_np.int64)
            cell18_labels_split_int_array[cell18_train_sub_array] = cell18_labels_sub_int_array[cell18_train_sub_array]

        cell18_labels_split_str_array = cell18_np.array(
            [
                str(int(cell18_split_label_value)) if cell18_split_label_value >= 0 else "?"
                for cell18_split_label_value in cell18_labels_split_int_array
            ],
            dtype="U16",
        )

        cell18_npz_path = cell18_work_dir / ("subgraph_" + str(cell18_target_nodes) + "_seed" + str(cell18_seed_value) + ".npz")
        cell18_output_json_path = cell18_work_dir / ("result_" + str(cell18_target_nodes) + "_seed" + str(cell18_seed_value) + ".json")

        cell18_np.savez(
            cell18_npz_path,
            src=cell18_src_sub_array,
            dst=cell18_dst_sub_array,
            labels_str=cell18_labels_split_str_array,
            labels_int=cell18_labels_sub_int_array,
            tr=cell18_train_sub_array,
            te=cell18_test_sub_array,
            rare=cell18_np.asarray(cell18_rare_classes, dtype=cell18_np.int64),
            nodes_root=cell18_sampled_array,
            N_sub=cell18_np.asarray([cell18_N_sub], dtype=cell18_np.int64),
        )

        if cell18_output_json_path.exists():
            cell18_output_json_path.unlink()

        cell18_gc.collect()
        cell18_start_time = cell18_time.perf_counter()
        cell18_status = "unknown"
        cell18_result_object = None
        cell18_elapsed_seconds = 0.0

        try:
            cell18_process = cell18_subprocess.run(
                [
                    cell18_sys.executable,
                    str(cell18_worker_path),
                    str(cell18_npz_path),
                    str(cell18_output_json_path),
                    str(cell18_memory_mb),
                    str(cell18_steps),
                ],
                timeout=cell18_timeout_seconds,
                capture_output=True,
                text=True,
            )

            cell18_elapsed_seconds = float(cell18_time.perf_counter() - cell18_start_time)

            if cell18_process.returncode != 0:
                cell18_status = "worker_nonzero_exit"
                print("  [fail] exit=" + str(cell18_process.returncode) + " elapsed=" + f"{cell18_elapsed_seconds:.2f}" + "s", flush=True)
                print("  [stderr tail]", flush=True)
                print((cell18_process.stderr or "")[-1500:], flush=True)
            elif not cell18_output_json_path.exists():
                cell18_status = "no_output"
                print("  [fail] no JSON elapsed=" + f"{cell18_elapsed_seconds:.2f}" + "s", flush=True)
                print("  [stderr tail]", flush=True)
                print((cell18_process.stderr or "")[-1500:], flush=True)
            else:
                cell18_result_object = cell18_json.loads(cell18_output_json_path.read_text(encoding="utf-8"))
                cell18_status = str(cell18_result_object.get("status", "malformed"))
                print("  [ok] elapsed=" + f"{cell18_elapsed_seconds:.2f}" + "s", flush=True)

        except cell18_subprocess.TimeoutExpired:
            cell18_elapsed_seconds = float(cell18_time.perf_counter() - cell18_start_time)
            cell18_status = "timeout"
            print("  [fail] timeout elapsed=" + f"{cell18_elapsed_seconds:.2f}" + "s", flush=True)

        except Exception as cell18_parent_error:
            cell18_status = "parent_exception"
            cell18_elapsed_seconds = float(cell18_time.perf_counter() - cell18_start_time)
            print("  [fail] parent exception: " + repr(cell18_parent_error), flush=True)

        if cell18_status == "ok" and cell18_result_object is not None:
            cell18_row_rare_value = float(cell18_result_object.get("rare_ba", float("nan")))
            cell18_row_unmapped_value = int(cell18_result_object.get("unmapped_rare_count", -1))
            cell18_row_edges_value = int(cell18_result_object.get("edges_sub", 0))

            print(
                "  [result] rare_ba="
                + f"{cell18_row_rare_value:.4f}"
                + " unmapped_rare="
                + str(cell18_row_unmapped_value),
                flush=True,
            )

            cell18_results_list.append(
                {
                    "seed": int(cell18_seed_value),
                    "status": "ok",
                    "rare_ba": float(cell18_row_rare_value),
                    "unmapped_rare_count": int(cell18_row_unmapped_value),
                    "edges_sub": int(cell18_row_edges_value),
                    "elapsed_sec": float(cell18_elapsed_seconds),
                }
            )
        else:
            cell18_results_list.append(
                {
                    "seed": int(cell18_seed_value),
                    "status": str(cell18_status),
                    "rare_ba": float("nan"),
                    "unmapped_rare_count": -1,
                    "edges_sub": 0,
                    "elapsed_sec": float(cell18_elapsed_seconds),
                }
            )

    cell18_valid_rare_values = []

    for cell18_result_row in cell18_results_list:
        cell18_row_rare_float = float(cell18_result_row.get("rare_ba", float("nan")))

        if cell18_np.isfinite(cell18_row_rare_float):
            cell18_valid_rare_values.append(cell18_row_rare_float)

    if len(cell18_valid_rare_values) > 0:
        cell18_mean_rare_ba = float(cell18_np.mean(cell18_valid_rare_values))
        cell18_std_rare_ba = float(
            cell18_np.std(cell18_valid_rare_values, ddof=1)
            if len(cell18_valid_rare_values) > 1
            else 0.0
        )
        cell18_min_rare_ba = float(cell18_np.min(cell18_valid_rare_values))
        cell18_max_rare_ba = float(cell18_np.max(cell18_valid_rare_values))
        cell18_summary_status = "ok"
    else:
        cell18_mean_rare_ba = float("nan")
        cell18_std_rare_ba = float("nan")
        cell18_min_rare_ba = float("nan")
        cell18_max_rare_ba = float("nan")
        cell18_summary_status = "no_valid_seeds"

    cell18_summary_dict = {
        "status": cell18_summary_status,
        "target_nodes": int(cell18_target_nodes),
        "actual_nodes": int(cell18_N_sub),
        "actual_edges": int(cell18_src_sub_array.size),
        "steps": int(cell18_steps),
        "timeout_sec": int(cell18_timeout_seconds),
        "memory_mb": int(cell18_memory_mb),
        "base_seed": int(cell18_base_seed),
        "seeds": cell18_seed_list,
        "n_valid_seeds": int(len(cell18_valid_rare_values)),
        "mean_rare_ba": float(cell18_mean_rare_ba),
        "std_rare_ba": float(cell18_std_rare_ba),
        "min_rare_ba": float(cell18_min_rare_ba),
        "max_rare_ba": float(cell18_max_rare_ba),
        "worker_path": str(cell18_worker_path),
        "work_dir": str(cell18_work_dir),
    }

    print("\n" + "=" * 100)
    print("CELL 18 — NTAC 10-SEED SUMMARY")
    print("=" * 100)
    print("Status:          " + str(cell18_summary_dict["status"]))
    print("Target nodes:    " + str(cell18_summary_dict["target_nodes"]))
    print("Actual nodes:    " + str(cell18_summary_dict["actual_nodes"]))
    print("Actual edges:    " + str(cell18_summary_dict["actual_edges"]))
    print("Valid seeds:     " + str(cell18_summary_dict["n_valid_seeds"]) + "/10")

    if len(cell18_valid_rare_values) > 0:
        print(
            "Mean rare_ba:    "
            + f"{cell18_mean_rare_ba:.4f}"
            + " +/- "
            + f"{cell18_std_rare_ba:.4f}"
        )
        print("Min rare_ba:     " + f"{cell18_min_rare_ba:.4f}")
        print("Max rare_ba:     " + f"{cell18_max_rare_ba:.4f}")
    else:
        print("Mean rare_ba:    n/a")
        print("No valid NTAC seed completed. Do not report an NTAC number from this cell.")

    print("=" * 100)

    cell18_summary_path = cell18_work_dir / "cell18_ntac_summary.json"
    cell18_summary_path.write_text(
        cell18_json.dumps(cell18_summary_dict, indent=2, default=str),
        encoding="utf-8",
    )

    cell18_results_path = cell18_work_dir / "cell18_ntac_results.json"
    cell18_results_path.write_text(
        cell18_json.dumps(cell18_results_list, indent=2, default=str),
        encoding="utf-8",
    )

    print("Saved summary: " + str(cell18_summary_path))
    print("Saved results: " + str(cell18_results_path))

    cell18_hf_save_checkpoint = cell18_globals.get("hf_save_checkpoint", None)

    if callable(cell18_hf_save_checkpoint):
        try:
            cell18_hf_save_checkpoint(
                "cell18_ntac_10seed_25k",
                {
                    "summary": cell18_summary_dict,
                    "results": cell18_results_list,
                    "paths": {
                        "summary_json": str(cell18_summary_path),
                        "results_json": str(cell18_results_path),
                    },
                },
                description="Cell 18: NTAC 10-seed diagnostic on 25K-node subgraph.",
                upload=True,
            )
        except Exception as cell18_checkpoint_error:
            print("[ckpt][warn] Cell 18 checkpoint failed: " + repr(cell18_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    cell18_hf_upload_file_to_bucket = cell18_globals.get("hf_upload_file_to_bucket", None)

    if callable(cell18_hf_upload_file_to_bucket):
        try:
            cell18_remote_summary = "checkpoints/cell18_ntac_10seed_25k/cell18_ntac_summary.json"
            cell18_summary_upload_ok, cell18_summary_upload_info = cell18_hf_upload_file_to_bucket(
                cell18_summary_path,
                cell18_remote_summary,
            )
            print(
                "[ckpt:cell18] upload summary JSON: "
                + ("OK" if cell18_summary_upload_ok else "FAILED")
                + " | "
                + str(cell18_summary_upload_info)
            )
        except Exception as cell18_summary_upload_error:
            print("[ckpt][warn] Cell 18 summary upload failed: " + repr(cell18_summary_upload_error))

        try:
            cell18_remote_results = "checkpoints/cell18_ntac_10seed_25k/cell18_ntac_results.json"
            cell18_results_upload_ok, cell18_results_upload_info = cell18_hf_upload_file_to_bucket(
                cell18_results_path,
                cell18_remote_results,
            )
            print(
                "[ckpt:cell18] upload results JSON: "
                + ("OK" if cell18_results_upload_ok else "FAILED")
                + " | "
                + str(cell18_results_upload_info)
            )
        except Exception as cell18_results_upload_error:
            print("[ckpt][warn] Cell 18 results upload failed: " + repr(cell18_results_upload_error))
    else:
        print(
            "[ckpt][skip] hf_upload_file_to_bucket not defined. "
            "NTAC JSON files remain local only."
        )

    return {
        "results": cell18_results_list,
        "summary": cell18_summary_dict,
        "work_dir": str(cell18_work_dir),
        "summary_path": str(cell18_summary_path),
        "results_path": str(cell18_results_path),
    }


cell18_output = cell18_main()

cell18_ntac_results = cell18_output["results"]
cell18_ntac_summary = cell18_output["summary"]
cell18_ntac_work_dir = cell18_output["work_dir"]
cell18_ntac_summary_path = cell18_output["summary_path"]
cell18_ntac_results_path = cell18_output["results_path"]

In [ ]:
# Cell 19: Cross-connectome structural coherence using frozen FAFB SSPR.
#
# Rules:
#   - No top-level imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every notebook-local/helper/loop variable is prefixed with cell19_.
#
# Behavior:
#   - Loads the frozen FAFB SSPR model saved by Cell 17.
#   - Downloads MANC / MAOL / MCNS files if available.
#   - Builds the same 61-dimensional SSPR feature pipeline.
#   - Applies the frozen encoder without retraining.
#   - Computes ARI/NMI against each target connectome's native taxonomy.
#   - Includes a shuffled null baseline.
#   - Does not fabricate results. Download/build/forward failures are recorded.


def cell19_main():
    cell19_globals = globals()

    cell19_np = cell19_globals.get("np", None)
    if cell19_np is None:
        import numpy as cell19_np

    cell19_torch = cell19_globals.get("torch", None)
    if cell19_torch is None:
        import torch as cell19_torch

    cell19_Path = cell19_globals.get("Path", None)
    if cell19_Path is None:
        from pathlib import Path as cell19_Path

    cell19_gzip = cell19_globals.get("gzip", None)
    if cell19_gzip is None:
        import gzip as cell19_gzip

    cell19_json = cell19_globals.get("json", None)
    if cell19_json is None:
        import json as cell19_json

    cell19_pickle = cell19_globals.get("pickle", None)
    if cell19_pickle is None:
        import pickle as cell19_pickle

    cell19_pd = cell19_globals.get("pd", None)
    if cell19_pd is None:
        import pandas as cell19_pd

    cell19_csr_matrix = cell19_globals.get("csr_matrix", None)
    if cell19_csr_matrix is None:
        from scipy.sparse import csr_matrix as cell19_csr_matrix

    cell19_TruncatedSVD = cell19_globals.get("TruncatedSVD", None)
    if cell19_TruncatedSVD is None:
        from sklearn.decomposition import TruncatedSVD as cell19_TruncatedSVD

    cell19_KMeans = cell19_globals.get("KMeans", None)
    if cell19_KMeans is None:
        from sklearn.cluster import KMeans as cell19_KMeans

    cell19_ari_score = cell19_globals.get("adjusted_rand_score", None)
    if cell19_ari_score is None:
        from sklearn.metrics import adjusted_rand_score as cell19_ari_score

    cell19_nmi_score = cell19_globals.get("normalized_mutual_info_score", None)
    if cell19_nmi_score is None:
        from sklearn.metrics import normalized_mutual_info_score as cell19_nmi_score

    cell19_urllib_request = cell19_globals.get("urllib_request", None)
    if cell19_urllib_request is None:
        import urllib.request as cell19_urllib_request

    cell19_urllib_error = cell19_globals.get("urllib_error", None)
    if cell19_urllib_error is None:
        import urllib.error as cell19_urllib_error

    def cell19_first_global(cell19_candidate_names):
        for cell19_candidate_name in cell19_candidate_names:
            if cell19_candidate_name in cell19_globals and cell19_globals[cell19_candidate_name] is not None:
                return cell19_globals[cell19_candidate_name]

        return None

    cell19_frozen_path_object = cell19_first_global(
        [
            "cell17_frozen_path",
            "FROZEN_PATH_CELL17",
        ]
    )

    if cell19_frozen_path_object is None:
        cell19_output_object = cell19_first_global(["cell17_output"])

        if isinstance(cell19_output_object, dict):
            cell19_frozen_path_object = cell19_output_object.get("frozen_path", None)

    if cell19_frozen_path_object is None:
        raise NameError(
            "[Cell 19][FATAL] Frozen FAFB SSPR path not found. "
            "Run Cell 17 first."
        )

    cell19_frozen_path = cell19_Path(str(cell19_frozen_path_object))

    if not cell19_frozen_path.exists():
        raise FileNotFoundError(
            "[Cell 19][FATAL] Frozen FAFB SSPR file does not exist: "
            + str(cell19_frozen_path)
        )

    if "SPGCN_cell5" not in cell19_globals:
        raise NameError("[Cell 19][FATAL] SPGCN_cell5 is not available. Run Cell 5 first.")

    if "DEVICE_cell1" not in cell19_globals:
        raise NameError("[Cell 19][FATAL] DEVICE_cell1 is not available. Run Cell 1 first.")

    cell19_device = cell19_globals["DEVICE_cell1"]

    cell19_cross_dir = cell19_Path("./cross_connectome_data_cell19")
    cell19_cross_dir.mkdir(parents=True, exist_ok=True)

    cell19_base_url = "https://storage.googleapis.com/flywire-data/codex/data"
    cell19_datasets = ["manc", "maol", "mcns"]
    cell19_files = [
        "connections.csv.gz",
        "classification.csv.gz",
        "neurons.csv.gz",
        "cell_stats.csv.gz",
    ]
    cell19_version_hints = ["", "1.0/", "1.1/", "1.2.1/", "v1.0/", "v1.1/", "v1.2.1/"]

    def cell19_choose_column(cell19_frame, cell19_candidates, cell19_default=None):
        for cell19_candidate_name in cell19_candidates:
            if cell19_candidate_name in cell19_frame.columns:
                return cell19_candidate_name

        return cell19_default

    def cell19_validate_gzip(cell19_path_object):
        with cell19_gzip.open(cell19_path_object, "rb") as cell19_gzip_handle:
            while cell19_gzip_handle.read(1 << 20):
                pass

    def cell19_download_cross_file(cell19_dataset_name, cell19_file_name):
        cell19_dest_path = cell19_cross_dir / cell19_dataset_name / cell19_file_name
        cell19_dest_path.parent.mkdir(parents=True, exist_ok=True)

        if cell19_dest_path.exists() and cell19_dest_path.stat().st_size > 1000:
            try:
                cell19_validate_gzip(cell19_dest_path)
                return cell19_dest_path
            except Exception:
                try:
                    cell19_dest_path.unlink()
                except Exception:
                    pass

        for cell19_version_hint in cell19_version_hints:
            cell19_url = (
                cell19_base_url
                + "/"
                + str(cell19_dataset_name)
                + "/"
                + str(cell19_version_hint)
                + str(cell19_file_name)
            )

            try:
                cell19_request_object = cell19_urllib_request.Request(
                    cell19_url,
                    headers={"User-Agent": "research"},
                )

                with cell19_urllib_request.urlopen(cell19_request_object, timeout=120) as cell19_response_object:
                    cell19_downloaded_bytes = cell19_response_object.read()

                cell19_tmp_path = cell19_dest_path.with_suffix(cell19_dest_path.suffix + ".tmp")
                cell19_tmp_path.write_bytes(cell19_downloaded_bytes)
                cell19_validate_gzip(cell19_tmp_path)
                cell19_tmp_path.replace(cell19_dest_path)

                print(
                    "[get] "
                    + str(cell19_dataset_name)
                    + "/"
                    + str(cell19_version_hint)
                    + str(cell19_file_name)
                    + "  "
                    + f"{len(cell19_downloaded_bytes) / 1e6:.1f}"
                    + " MB"
                )

                return cell19_dest_path

            except (
                cell19_urllib_error.HTTPError,
                cell19_urllib_error.URLError,
                OSError,
                ValueError,
                EOFError,
            ) as cell19_download_error:
                continue

        print("[miss] " + str(cell19_dataset_name) + "/" + str(cell19_file_name) + "  all version hints failed")
        return None

    def cell19_build_features(cell19_dataset_dir):
        cell19_conn_path = cell19_dataset_dir / "connections.csv.gz"
        cell19_cls_path = cell19_dataset_dir / "classification.csv.gz"
        cell19_neurons_path = cell19_dataset_dir / "neurons.csv.gz"
        cell19_stats_path = cell19_dataset_dir / "cell_stats.csv.gz"

        for cell19_required_path in [
            cell19_conn_path,
            cell19_cls_path,
            cell19_neurons_path,
            cell19_stats_path,
        ]:
            if not cell19_required_path.exists():
                return None

        cell19_conn = cell19_pd.read_csv(cell19_conn_path)
        cell19_cls = cell19_pd.read_csv(cell19_cls_path)
        cell19_neurons = cell19_pd.read_csv(cell19_neurons_path)
        cell19_stats = cell19_pd.read_csv(cell19_stats_path)

        if cell19_cls.shape[0] == 0 or len(cell19_cls.columns) == 0:
            return None

        cell19_id_col = cell19_choose_column(
            cell19_cls,
            ["root_id", "node_id", "id"],
            cell19_cls.columns[0],
        )

        cell19_cls = cell19_cls.drop_duplicates(subset=[cell19_id_col]).copy()
        cell19_all_ids_raw = cell19_cls[cell19_id_col].dropna().unique()

        try:
            cell19_all_ids = cell19_np.asarray(cell19_all_ids_raw, dtype=cell19_np.int64)
        except Exception:
            return None

        cell19_N = int(cell19_all_ids.size)

        if cell19_N == 0:
            return None

        cell19_id_map = {
            int(cell19_root_value): cell19_root_index
            for cell19_root_index, cell19_root_value in enumerate(cell19_all_ids)
        }

        cell19_src_col = cell19_choose_column(
            cell19_conn,
            ["pre_root_id", "pre", "source", "src"],
            None,
        )

        cell19_dst_col = cell19_choose_column(
            cell19_conn,
            ["post_root_id", "post", "target", "dst"],
            None,
        )

        if cell19_src_col is None or cell19_dst_col is None:
            return None

        cell19_nt_col = cell19_choose_column(
            cell19_conn,
            ["nt_type", "transmitter", "nt"],
            None,
        )

        cell19_syn_col = cell19_choose_column(
            cell19_conn,
            ["syn_count", "weight", "syn", "edge_weight"],
            None,
        )

        cell19_conn = cell19_conn.dropna(subset=[cell19_src_col, cell19_dst_col]).copy()

        cell19_conn[cell19_src_col] = cell19_pd.to_numeric(
            cell19_conn[cell19_src_col],
            errors="coerce",
        )
        cell19_conn[cell19_dst_col] = cell19_pd.to_numeric(
            cell19_conn[cell19_dst_col],
            errors="coerce",
        )

        cell19_conn = cell19_conn.dropna(subset=[cell19_src_col, cell19_dst_col]).copy()

        if cell19_conn.shape[0] == 0:
            return None

        cell19_conn[cell19_src_col] = cell19_conn[cell19_src_col].astype(cell19_np.int64)
        cell19_conn[cell19_dst_col] = cell19_conn[cell19_dst_col].astype(cell19_np.int64)

        cell19_conn["src"] = cell19_conn[cell19_src_col].map(cell19_id_map)
        cell19_conn["dst"] = cell19_conn[cell19_dst_col].map(cell19_id_map)

        cell19_conn = cell19_conn.dropna(subset=["src", "dst"]).copy()

        if cell19_conn.shape[0] == 0:
            return None

        cell19_conn["src"] = cell19_conn["src"].astype(cell19_np.int64)
        cell19_conn["dst"] = cell19_conn["dst"].astype(cell19_np.int64)

        cell19_nt_map = {
            "ACH": "exc",
            "GLUT": "exc",
            "GABA": "inh",
            "SER": "mod",
            "DA": "mod",
            "OCT": "mod",
        }

        if cell19_nt_col is not None:
            cell19_conn["nt_class"] = (
                cell19_conn[cell19_nt_col]
                .astype(str)
                .str.upper()
                .map(cell19_nt_map)
                .fillna("unk")
            )
        else:
            cell19_neuron_nt_col = cell19_choose_column(
                cell19_neurons,
                ["nt_type", "transmitter", "nt"],
                None,
            )

            if cell19_neuron_nt_col is not None and cell19_id_col in cell19_neurons.columns:
                cell19_neuron_nt_series = cell19_neurons.set_index(cell19_id_col)[cell19_neuron_nt_col]
                cell19_neuron_nt_dict = cell19_neuron_nt_series.to_dict()

                cell19_conn["nt_class"] = (
                    cell19_conn[cell19_src_col]
                    .map(cell19_neuron_nt_dict)
                    .astype(str)
                    .str.upper()
                    .map(cell19_nt_map)
                    .fillna("unk")
                )
            else:
                cell19_conn["nt_class"] = "unk"

        if cell19_syn_col is not None:
            cell19_conn["syn_count"] = cell19_pd.to_numeric(
                cell19_conn[cell19_syn_col],
                errors="coerce",
            ).fillna(1.0)
        else:
            cell19_conn["syn_count"] = 1.0

        cell19_conn_signed = cell19_conn[cell19_conn["nt_class"].isin(["exc", "inh"])].copy()

        if cell19_conn_signed.shape[0] == 0:
            return None

        cell19_agg = (
            cell19_conn_signed
            .groupby(["src", "dst", "nt_class"], as_index=False)
            .agg(syn_count=("syn_count", "sum"))
        )

        cell19_major = (
            cell19_agg
            .sort_values("syn_count")
            .groupby(["src", "dst"], as_index=False)
            .last()
        )

        cell19_src_array = cell19_major["src"].values.astype(cell19_np.int64)
        cell19_dst_array = cell19_major["dst"].values.astype(cell19_np.int64)
        cell19_sign_bin_array = (cell19_major["nt_class"] == "exc").astype(cell19_np.float32).values
        cell19_syn_array = cell19_major["syn_count"].values.astype(cell19_np.float32)

        if cell19_id_col in cell19_stats.columns:
            cell19_stats_idx = cell19_stats.set_index(cell19_id_col).reindex(cell19_all_ids)
        else:
            cell19_stats_idx = None

        cell19_size_candidates = ["length_nm", "area_nm", "size_nm"]

        if cell19_stats_idx is not None:
            cell19_size_columns = [
                cell19_size_candidate
                for cell19_size_candidate in cell19_size_candidates
                if cell19_size_candidate in cell19_stats_idx.columns
            ]
        else:
            cell19_size_columns = []

        if len(cell19_size_columns) == 3:
            cell19_size_features = cell19_stats_idx[cell19_size_columns].values.astype(cell19_np.float64)
            cell19_size_features = cell19_np.nan_to_num(
                cell19_np.log1p(cell19_np.clip(cell19_size_features, 0, None)),
                nan=0.0,
            )
        else:
            cell19_size_features = cell19_np.zeros((cell19_N, 3), dtype=cell19_np.float64)

        cell19_nt_candidates = ["da_avg", "ser_avg", "gaba_avg", "glut_avg", "ach_avg", "oct_avg"]

        if (
            cell19_id_col in cell19_neurons.columns
            and all(cell19_nt_candidate in cell19_neurons.columns for cell19_nt_candidate in cell19_nt_candidates)
        ):
            cell19_neurons_idx = cell19_neurons.set_index(cell19_id_col).reindex(cell19_all_ids)
            cell19_nt_features = cell19_neurons_idx[cell19_nt_candidates].values.astype(cell19_np.float64)
            cell19_nt_features = cell19_np.nan_to_num(cell19_nt_features, nan=0.0)
        else:
            cell19_nt_features = cell19_np.zeros((cell19_N, 6), dtype=cell19_np.float64)

        cell19_deg_in = cell19_np.zeros(cell19_N, dtype=cell19_np.float64)
        cell19_deg_out = cell19_np.zeros(cell19_N, dtype=cell19_np.float64)

        cell19_np.add.at(cell19_deg_in, cell19_dst_array, cell19_syn_array.astype(cell19_np.float64))
        cell19_np.add.at(cell19_deg_out, cell19_src_array, cell19_syn_array.astype(cell19_np.float64))

        cell19_deg_features = cell19_np.stack(
            [
                cell19_np.log1p(cell19_deg_in),
                cell19_np.log1p(cell19_deg_out),
            ],
            axis=1,
        )

        cell19_row_unsigned = cell19_np.concatenate([cell19_src_array, cell19_dst_array])
        cell19_col_unsigned = cell19_np.concatenate([cell19_dst_array, cell19_src_array])
        cell19_data_unsigned = cell19_np.ones(cell19_row_unsigned.size, dtype=cell19_np.float32)

        cell19_A_sym = cell19_csr_matrix(
            (cell19_data_unsigned, (cell19_row_unsigned, cell19_col_unsigned)),
            shape=(cell19_N, cell19_N),
        )

        if cell19_N < 17:
            cell19_spec_unsigned = cell19_np.zeros((cell19_N, 16), dtype=cell19_np.float64)
        else:
            cell19_svd_unsigned = cell19_TruncatedSVD(n_components=16, random_state=42)
            cell19_spec_unsigned = cell19_svd_unsigned.fit_transform(cell19_A_sym)
            cell19_spec_unsigned = (
                cell19_spec_unsigned - cell19_spec_unsigned.mean(0)
            ) / (cell19_spec_unsigned.std(0) + 1e-8)
            cell19_spec_unsigned = cell19_np.nan_to_num(cell19_spec_unsigned, nan=0.0)

        cell19_pos_mask = cell19_sign_bin_array > 0.5
        cell19_neg_mask = ~cell19_pos_mask

        cell19_row_positive = cell19_src_array[cell19_pos_mask]
        cell19_col_positive = cell19_dst_array[cell19_pos_mask]
        cell19_row_negative = cell19_src_array[cell19_neg_mask]
        cell19_col_negative = cell19_dst_array[cell19_neg_mask]

        cell19_row_signed = cell19_np.concatenate([cell19_row_positive, cell19_row_negative])
        cell19_col_signed = cell19_np.concatenate([cell19_col_positive, cell19_col_negative])
        cell19_data_signed = cell19_np.concatenate(
            [
                cell19_np.ones(cell19_row_positive.size, dtype=cell19_np.float32),
                -cell19_np.ones(cell19_row_negative.size, dtype=cell19_np.float32),
            ]
        )

        cell19_A_signed = cell19_csr_matrix(
            (cell19_data_signed, (cell19_row_signed, cell19_col_signed)),
            shape=(cell19_N, cell19_N),
        )

        if cell19_N < 33:
            cell19_spec_signed = cell19_np.zeros((cell19_N, 32), dtype=cell19_np.float64)
        else:
            cell19_svd_signed = cell19_TruncatedSVD(n_components=32, random_state=42)
            cell19_spec_signed = cell19_svd_signed.fit_transform(cell19_A_signed)
            cell19_spec_signed = (
                cell19_spec_signed - cell19_spec_signed.mean(0)
            ) / (cell19_spec_signed.std(0) + 1e-8)
            cell19_spec_signed = cell19_np.nan_to_num(cell19_spec_signed, nan=0.0)

        cell19_row_abs = cell19_np.concatenate([cell19_src_array, cell19_dst_array])
        cell19_col_abs = cell19_np.concatenate([cell19_dst_array, cell19_src_array])
        cell19_data_abs = cell19_np.ones(cell19_row_abs.size, dtype=cell19_np.float32)

        cell19_A_abs = cell19_csr_matrix(
            (cell19_data_abs, (cell19_row_abs, cell19_col_abs)),
            shape=(cell19_N, cell19_N),
        )

        cell19_A3_signed = cell19_A_signed.dot(cell19_A_signed).dot(cell19_A_signed)
        cell19_A3_abs = cell19_A_abs.dot(cell19_A_abs).dot(cell19_A_abs)

        cell19_diag_signed = cell19_np.asarray(cell19_A3_signed.diagonal()).flatten()
        cell19_diag_abs = cell19_np.asarray(cell19_A3_abs.diagonal()).flatten()

        cell19_tri_bal = (cell19_diag_abs + cell19_diag_signed) / 12.0
        cell19_tri_fru = (cell19_diag_abs - cell19_diag_signed) / 12.0

        cell19_motif_features = cell19_np.stack(
            [
                cell19_np.log1p(cell19_np.clip(cell19_tri_bal, 0, None)),
                cell19_np.log1p(cell19_np.clip(cell19_tri_fru, 0, None)),
            ],
            axis=1,
        )
        cell19_motif_features = cell19_np.nan_to_num(cell19_motif_features, nan=0.0)

        cell19_X_raw = cell19_np.concatenate(
            [
                cell19_size_features,
                cell19_nt_features,
                cell19_deg_features,
                cell19_spec_unsigned,
                cell19_spec_signed,
                cell19_motif_features,
            ],
            axis=1,
        )

        if int(cell19_X_raw.shape[1]) != 61:
            return None

        cell19_label_col = cell19_choose_column(
            cell19_cls,
            ["super_class", "class", "cell_type", "primary_type"],
            cell19_cls.columns[-1],
        )

        cell19_cls_idx = cell19_cls.set_index(cell19_id_col).reindex(cell19_all_ids)
        cell19_labels_raw = cell19_cls_idx[cell19_label_col].fillna("").astype(str).values

        return {
            "X_raw": cell19_X_raw,
            "src": cell19_src_array,
            "dst": cell19_dst_array,
            "sign_bin": cell19_sign_bin_array,
            "syn": cell19_syn_array,
            "labels_raw": cell19_labels_raw,
            "N": cell19_N,
        }

    def cell19_collect_tensors(cell19_output_object, cell19_tensor_list=None):
        if cell19_tensor_list is None:
            cell19_tensor_list = []

        if cell19_torch.is_tensor(cell19_output_object):
            if int(cell19_output_object.ndim) >= 2:
                cell19_tensor_list.append(cell19_output_object)
            return cell19_tensor_list

        if isinstance(cell19_output_object, dict):
            for cell19_dict_value in cell19_output_object.values():
                cell19_collect_tensors(cell19_dict_value, cell19_tensor_list)
            return cell19_tensor_list

        if isinstance(cell19_output_object, (tuple, list)):
            for cell19_sequence_value in cell19_output_object:
                cell19_collect_tensors(cell19_sequence_value, cell19_tensor_list)
            return cell19_tensor_list

        return cell19_tensor_list

    def cell19_extract_embedding(cell19_output_object, cell19_model_object, cell19_expected_rows):
        cell19_candidate_tensors = cell19_collect_tensors(cell19_output_object)

        if len(cell19_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 19][FATAL] Frozen model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell19_output_object))
            )

        cell19_proto_parameter = getattr(cell19_model_object, "proto_class", None)

        if cell19_torch.is_tensor(cell19_proto_parameter) and int(cell19_proto_parameter.ndim) == 2:
            cell19_proto_dim = int(cell19_proto_parameter.shape[1])

            for cell19_candidate_tensor in cell19_candidate_tensors:
                if int(cell19_candidate_tensor.shape[1]) == cell19_proto_dim:
                    return cell19_candidate_tensor

        for cell19_candidate_tensor in cell19_candidate_tensors:
            if int(cell19_candidate_tensor.shape[0]) == int(cell19_expected_rows):
                return cell19_candidate_tensor

        return cell19_candidate_tensors[0]

    def cell19_make_model_from_checkpoint(cell19_checkpoint):
        cell19_d_features = int(cell19_checkpoint.get("d_features", 61))
        cell19_d_hidden = int(cell19_checkpoint.get("d_cell5", 64))
        cell19_d_proto = int(cell19_checkpoint.get("d_proto_cell5", 32))
        cell19_n_layers = int(cell19_checkpoint.get("n_layers_cell5", 2))
        cell19_n_classes = int(cell19_checkpoint.get("n_classes_cell5", 29))
        cell19_n_super = int(cell19_checkpoint.get("n_super_cell5", 10))
        cell19_temperature = float(cell19_checkpoint.get("temperature_cell5", 0.1))
        cell19_dropout = float(cell19_checkpoint.get("dropout_cell5", 0.3))

        try:
            return cell19_globals["SPGCN_cell5"](
                d_features_cell5=cell19_d_features,
                d_cell5=cell19_d_hidden,
                d_proto_cell5=cell19_d_proto,
                n_layers_cell5=cell19_n_layers,
                n_classes_cell5=cell19_n_classes,
                n_super_cell5=cell19_n_super,
                temperature_cell5=cell19_temperature,
                dropout_cell5=cell19_dropout,
            )
        except TypeError:
            return cell19_globals["SPGCN_cell5"](
                d_features=cell19_d_features,
                d=cell19_d_hidden,
                d_proto=cell19_d_proto,
                n_layers=cell19_n_layers,
                n_classes=cell19_n_classes,
                n_super=cell19_n_super,
                temperature=cell19_temperature,
                dropout=cell19_dropout,
            )

    def cell19_forward_frozen_model(
        cell19_model_object,
        cell19_x_tensor,
        cell19_ei_pos_tensor,
        cell19_ew_pos_tensor,
        cell19_dis_pos_tensor,
        cell19_ei_neg_tensor,
        cell19_ew_neg_tensor,
        cell19_dis_neg_tensor,
    ):
        cell19_last_type_error = None

        try:
            return cell19_model_object(
                cell19_x_tensor,
                cell19_ei_pos_tensor,
                cell19_ew_pos_tensor,
                cell19_dis_pos_tensor,
                cell19_ei_neg_tensor,
                cell19_ew_neg_tensor,
                cell19_dis_neg_tensor,
            )
        except TypeError as cell19_type_error:
            cell19_last_type_error = cell19_type_error

        try:
            return cell19_model_object(
                cell19_x_tensor,
                cell19_ei_pos_tensor,
                cell19_dis_pos_tensor,
                cell19_ei_neg_tensor,
                cell19_dis_neg_tensor,
            )
        except TypeError as cell19_type_error:
            cell19_last_type_error = cell19_type_error

        try:
            return cell19_model_object(
                cell19_x_tensor,
                cell19_ei_pos_tensor,
                cell19_ei_neg_tensor,
            )
        except TypeError as cell19_type_error:
            cell19_last_type_error = cell19_type_error

        raise TypeError(
            "[Cell 19][FATAL] Could not call frozen model with any known signature. "
            "Last TypeError: " + repr(cell19_last_type_error)
        )

    def cell19_normalize_edge_degree(cell19_ei_tensor, cell19_ew_tensor, cell19_node_count, cell19_target_device):
        if int(cell19_ei_tensor.numel()) == 0:
            return cell19_torch.ones(int(cell19_node_count), device=cell19_target_device)

        cell19_degree_vector = cell19_torch.zeros(int(cell19_node_count), device=cell19_target_device)
        cell19_degree_vector.index_add_(0, cell19_ei_tensor[0], cell19_ew_tensor)
        cell19_degree_vector.index_add_(0, cell19_ei_tensor[1], cell19_ew_tensor)

        return cell19_degree_vector.clamp(min=1.0).rsqrt()

    print("=" * 100)
    print("CELL 19 — CROSS-CONNECTOME STRUCTURAL COHERENCE")
    print("=" * 100)
    print("[cfg] frozen_path=" + str(cell19_frozen_path))
    print("[cfg] datasets=" + str(cell19_datasets))
    print("[cfg] base_url=" + str(cell19_base_url))
    print("[cfg] output_dir=" + str(cell19_cross_dir))
    print("=" * 100)

    try:
        cell19_checkpoint = cell19_torch.load(
            cell19_frozen_path,
            map_location=cell19_device,
            weights_only=False,
        )
    except TypeError:
        cell19_checkpoint = cell19_torch.load(
            cell19_frozen_path,
            map_location=cell19_device,
        )

    cell19_feature_mean = cell19_checkpoint.get("feature_mean", None)
    cell19_feature_std = cell19_checkpoint.get("feature_std", None)

    if cell19_feature_mean is None or cell19_feature_std is None:
        raise KeyError("[Cell 19][FATAL] Frozen checkpoint missing feature_mean or feature_std.")

    cell19_feature_mean = cell19_torch.as_tensor(
        cell19_feature_mean,
        dtype=cell19_torch.float32,
    ).to(cell19_device)

    cell19_feature_std = cell19_torch.as_tensor(
        cell19_feature_std,
        dtype=cell19_torch.float32,
    ).to(cell19_device)

    cell19_frozen_model = cell19_make_model_from_checkpoint(cell19_checkpoint)

    if "encoder_state" not in cell19_checkpoint:
        raise KeyError("[Cell 19][FATAL] Frozen checkpoint missing encoder_state.")

    try:
        cell19_frozen_model.load_state_dict(cell19_checkpoint["encoder_state"])
    except RuntimeError:
        cell19_frozen_model.load_state_dict(cell19_checkpoint["encoder_state"], strict=False)

    cell19_frozen_model = cell19_frozen_model.to(cell19_device)
    cell19_frozen_model.eval()

    cell19_param_count = int(sum(int(parameter.numel()) for parameter in cell19_frozen_model.parameters()))
    print("[Cell 19] Loaded frozen SSPR params=" + f"{cell19_param_count:,}")

    cell19_results_dict = {}
    cell19_summary_rows = []

    for cell19_dataset_name in cell19_datasets:
        print("\n---- Dataset: " + str(cell19_dataset_name) + " ----")

        cell19_dataset_dir = cell19_cross_dir / cell19_dataset_name
        cell19_dataset_dir.mkdir(parents=True, exist_ok=True)

        cell19_download_ok = True

        for cell19_file_name in cell19_files:
            cell19_downloaded_path = cell19_download_cross_file(cell19_dataset_name, cell19_file_name)

            if cell19_downloaded_path is None:
                print("[skip] " + str(cell19_dataset_name) + ": missing " + str(cell19_file_name))
                cell19_download_ok = False
                break

        if not cell19_download_ok:
            cell19_results_dict[cell19_dataset_name] = {"status": "download_failed"}
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "download_failed",
                    "N": "",
                    "n_labeled": "",
                    "K": "",
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        try:
            cell19_built_features = cell19_build_features(cell19_dataset_dir)
        except Exception as cell19_build_error:
            print("[fail] " + str(cell19_dataset_name) + ": feature build failed: " + repr(cell19_build_error))
            cell19_results_dict[cell19_dataset_name] = {
                "status": "feature_failed",
                "error": repr(cell19_build_error),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "feature_failed",
                    "N": "",
                    "n_labeled": "",
                    "K": "",
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        if cell19_built_features is None:
            cell19_results_dict[cell19_dataset_name] = {"status": "no_usable_graph"}
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "no_usable_graph",
                    "N": "",
                    "n_labeled": "",
                    "K": "",
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        cell19_X_new_np = cell19_built_features["X_raw"]
        cell19_N_new = int(cell19_built_features["N"])
        cell19_src_new = cell19_built_features["src"]
        cell19_dst_new = cell19_built_features["dst"]
        cell19_sign_new = cell19_built_features["sign_bin"]
        cell19_syn_new = cell19_built_features["syn"]
        cell19_labels_new = cell19_built_features["labels_raw"]

        if int(cell19_X_new_np.shape[1]) != int(cell19_feature_mean.numel()):
            cell19_results_dict[cell19_dataset_name] = {
                "status": "dimension_mismatch",
                "target_features": int(cell19_X_new_np.shape[1]),
                "frozen_features": int(cell19_feature_mean.numel()),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "dimension_mismatch",
                    "N": cell19_N_new,
                    "n_labeled": "",
                    "K": "",
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        cell19_X_new_tensor = cell19_torch.tensor(
            cell19_X_new_np,
            dtype=cell19_torch.float32,
            device=cell19_device,
        )
        cell19_X_new_tensor = (cell19_X_new_tensor - cell19_feature_mean) / (cell19_feature_std + 1e-8)

        cell19_ei_full_new = cell19_torch.tensor(
            cell19_np.vstack([cell19_src_new, cell19_dst_new]),
            dtype=cell19_torch.long,
            device=cell19_device,
        )

        if cell19_syn_new.size == 0:
            cell19_edge_weight_np = cell19_np.ones(0, dtype=cell19_np.float32)
        else:
            cell19_syn_log_np = cell19_np.log1p(cell19_syn_new.astype(cell19_np.float64))
            cell19_edge_weight_np = (cell19_syn_log_np / (cell19_syn_log_np.mean() + 1e-8)).astype(cell19_np.float32)

        cell19_edge_weight_tensor = cell19_torch.tensor(
            cell19_edge_weight_np,
            dtype=cell19_torch.float32,
            device=cell19_device,
        )

        cell19_pos_mask_array = cell19_sign_new > 0.5
        cell19_pos_mask_tensor = cell19_torch.tensor(cell19_pos_mask_array, device=cell19_device)
        cell19_neg_mask_tensor = ~cell19_pos_mask_tensor

        cell19_ei_pos_new = cell19_ei_full_new[:, cell19_pos_mask_tensor]
        cell19_ew_pos_new = cell19_edge_weight_tensor[cell19_pos_mask_tensor]
        cell19_ei_neg_new = cell19_ei_full_new[:, cell19_neg_mask_tensor]
        cell19_ew_neg_new = cell19_edge_weight_tensor[cell19_neg_mask_tensor]

        if int(cell19_ei_pos_new.shape[1]) < 10 or int(cell19_ei_neg_new.shape[1]) < 10:
            cell19_results_dict[cell19_dataset_name] = {
                "status": "insufficient_sign_edges",
                "n_positive_edges": int(cell19_ei_pos_new.shape[1]),
                "n_negative_edges": int(cell19_ei_neg_new.shape[1]),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "insufficient_sign_edges",
                    "N": cell19_N_new,
                    "n_labeled": "",
                    "K": "",
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        cell19_dis_pos_new = cell19_normalize_edge_degree(
            cell19_ei_pos_new,
            cell19_ew_pos_new,
            cell19_N_new,
            cell19_device,
        )
        cell19_dis_neg_new = cell19_normalize_edge_degree(
            cell19_ei_neg_new,
            cell19_ew_neg_new,
            cell19_N_new,
            cell19_device,
        )

        try:
            with cell19_torch.no_grad():
                cell19_model_output = cell19_forward_frozen_model(
                    cell19_frozen_model,
                    cell19_X_new_tensor,
                    cell19_ei_pos_new,
                    cell19_ew_pos_new,
                    cell19_dis_pos_new,
                    cell19_ei_neg_new,
                    cell19_ew_neg_new,
                    cell19_dis_neg_new,
                )

                cell19_embedding_tensor = cell19_extract_embedding(
                    cell19_model_output,
                    cell19_frozen_model,
                    cell19_N_new,
                )

                cell19_embedding_np = cell19_embedding_tensor.detach().cpu().numpy()
                cell19_embedding_np = cell19_np.nan_to_num(cell19_embedding_np, nan=0.0, posinf=0.0, neginf=0.0)

        except Exception as cell19_forward_error:
            print("[fail] " + str(cell19_dataset_name) + ": frozen forward failed: " + repr(cell19_forward_error))
            cell19_results_dict[cell19_dataset_name] = {
                "status": "forward_failed",
                "error": repr(cell19_forward_error),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "forward_failed",
                    "N": cell19_N_new,
                    "n_labeled": "",
                    "K": "",
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        cell19_unique_labels = sorted(
            set(
                str(cell19_label_value).strip()
                for cell19_label_value in cell19_labels_new
                if str(cell19_label_value).strip() != ""
            )
        )

        if len(cell19_unique_labels) < 2:
            cell19_results_dict[cell19_dataset_name] = {
                "status": "too_few_labels",
                "n_unique_labels": len(cell19_unique_labels),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "too_few_labels",
                    "N": cell19_N_new,
                    "n_labeled": "",
                    "K": len(cell19_unique_labels),
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        cell19_label_to_index = {
            cell19_label_string: cell19_label_index
            for cell19_label_index, cell19_label_string in enumerate(cell19_unique_labels)
        }

        cell19_y_new = cell19_np.array(
            [
                cell19_label_to_index.get(str(cell19_label_value).strip(), -1)
                for cell19_label_value in cell19_labels_new
            ],
            dtype=cell19_np.int64,
        )

        cell19_valid_mask = cell19_y_new >= 0

        if int(cell19_np.sum(cell19_valid_mask)) < 50:
            cell19_results_dict[cell19_dataset_name] = {
                "status": "too_few_labeled",
                "n_labeled": int(cell19_np.sum(cell19_valid_mask)),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "too_few_labeled",
                    "N": cell19_N_new,
                    "n_labeled": int(cell19_np.sum(cell19_valid_mask)),
                    "K": len(cell19_unique_labels),
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        cell19_K = len(cell19_unique_labels)

        if cell19_K > int(cell19_np.sum(cell19_valid_mask)):
            cell19_results_dict[cell19_dataset_name] = {
                "status": "too_many_clusters_for_labeled_nodes",
                "K": int(cell19_K),
                "n_labeled": int(cell19_np.sum(cell19_valid_mask)),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "too_many_clusters_for_labeled_nodes",
                    "N": cell19_N_new,
                    "n_labeled": int(cell19_np.sum(cell19_valid_mask)),
                    "K": int(cell19_K),
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        try:
            cell19_kmeans_object = cell19_KMeans(
                n_clusters=cell19_K,
                random_state=0,
                n_init=10,
            )
            cell19_kmeans_object.fit(cell19_embedding_np)
            cell19_pred_all = cell19_kmeans_object.labels_
        except Exception as cell19_cluster_error:
            print("[fail] " + str(cell19_dataset_name) + ": clustering failed: " + repr(cell19_cluster_error))
            cell19_results_dict[cell19_dataset_name] = {
                "status": "cluster_failed",
                "error": repr(cell19_cluster_error),
            }
            cell19_summary_rows.append(
                {
                    "dataset": cell19_dataset_name,
                    "status": "cluster_failed",
                    "N": cell19_N_new,
                    "n_labeled": int(cell19_np.sum(cell19_valid_mask)),
                    "K": int(cell19_K),
                    "ari": "",
                    "ari_null": "",
                    "ari_delta": "",
                    "nmi": "",
                    "nmi_null": "",
                    "nmi_delta": "",
                }
            )
            continue

        cell19_y_true = cell19_y_new[cell19_valid_mask]
        cell19_y_pred = cell19_pred_all[cell19_valid_mask]

        cell19_ari = float(cell19_ari_score(cell19_y_true, cell19_y_pred))
        cell19_nmi = float(cell19_nmi_score(cell19_y_true, cell19_y_pred))

        cell19_rng = cell19_np.random.default_rng(0)
        cell19_null_aris = []
        cell19_null_nmis = []

        for cell19_null_index in range(20):
            cell19_y_perm = cell19_y_true.copy()
            cell19_rng.shuffle(cell19_y_perm)
            cell19_null_aris.append(float(cell19_ari_score(cell19_y_perm, cell19_y_pred)))
            cell19_null_nmis.append(float(cell19_nmi_score(cell19_y_perm, cell19_y_pred)))

        cell19_ari_null = float(cell19_np.mean(cell19_null_aris))
        cell19_nmi_null = float(cell19_np.mean(cell19_null_nmis))
        cell19_ari_delta = float(cell19_ari - cell19_ari_null)
        cell19_nmi_delta = float(cell19_nmi - cell19_nmi_null)

        cell19_feature_mean_np = cell19_feature_mean.detach().cpu().numpy()
        cell19_feature_std_np = cell19_feature_std.detach().cpu().numpy()

        cell19_feature_mean_diff = float(
            cell19_np.abs(cell19_X_new_np.mean(0) - cell19_feature_mean_np).mean()
        )
        cell19_feature_std_diff = float(
            cell19_np.abs(cell19_X_new_np.std(0) - cell19_feature_std_np).mean()
        )

        print("[Cell 19][" + str(cell19_dataset_name) + "] N=" + f"{cell19_N_new:,}" + " labeled=" + f"{int(cell19_np.sum(cell19_valid_mask)):,}" + " K=" + str(cell19_K))
        print("[Cell 19][" + str(cell19_dataset_name) + "] ARI=" + f"{cell19_ari:.4f}" + " null=" + f"{cell19_ari_null:.4f}" + " delta=" + f"{cell19_ari_delta:+.4f}")
        print("[Cell 19][" + str(cell19_dataset_name) + "] NMI=" + f"{cell19_nmi:.4f}" + " null=" + f"{cell19_nmi_null:.4f}" + " delta=" + f"{cell19_nmi_delta:+.4f}")

        cell19_results_dict[cell19_dataset_name] = {
            "status": "ok",
            "N": int(cell19_N_new),
            "n_labeled": int(cell19_np.sum(cell19_valid_mask)),
            "K": int(cell19_K),
            "ari": float(cell19_ari),
            "ari_null": float(cell19_ari_null),
            "ari_delta": float(cell19_ari_delta),
            "nmi": float(cell19_nmi),
            "nmi_null": float(cell19_nmi_null),
            "nmi_delta": float(cell19_nmi_delta),
            "feature_mean_diff": float(cell19_feature_mean_diff),
            "feature_std_diff": float(cell19_feature_std_diff),
        }

        cell19_summary_rows.append(
            {
                "dataset": cell19_dataset_name,
                "status": "ok",
                "N": int(cell19_N_new),
                "n_labeled": int(cell19_np.sum(cell19_valid_mask)),
                "K": int(cell19_K),
                "ari": float(cell19_ari),
                "ari_null": float(cell19_ari_null),
                "ari_delta": float(cell19_ari_delta),
                "nmi": float(cell19_nmi),
                "nmi_null": float(cell19_nmi_null),
                "nmi_delta": float(cell19_nmi_delta),
            }
        )

    print("\n" + "=" * 100)
    print("CELL 19 — CROSS-CONNECTOME SUMMARY")
    print("=" * 100)
    print(
        f"{'dataset':>10s} {'status':>22s} {'N':>10s} {'labeled':>10s} {'K':>4s} "
        f"{'ARI':>8s} {'ARI_null':>10s} {'dARI':>8s} "
        f"{'NMI':>8s} {'NMI_null':>10s} {'dNMI':>8s}"
    )

    for cell19_dataset_name in cell19_datasets:
        cell19_row_dict = cell19_results_dict.get(cell19_dataset_name, {"status": "missing"})

        if cell19_row_dict.get("status") == "ok":
            print(
                f"{cell19_dataset_name:>10s} "
                f"{str(cell19_row_dict['status']):>22s} "
                f"{int(cell19_row_dict['N']):>10,d} "
                f"{int(cell19_row_dict['n_labeled']):>10,d} "
                f"{int(cell19_row_dict['K']):>4d} "
                f"{float(cell19_row_dict['ari']):>8.4f} "
                f"{float(cell19_row_dict['ari_null']):>10.4f} "
                f"{float(cell19_row_dict['ari_delta']):>+8.4f} "
                f"{float(cell19_row_dict['nmi']):>8.4f} "
                f"{float(cell19_row_dict['nmi_null']):>10.4f} "
                f"{float(cell19_row_dict['nmi_delta']):>+8.4f}"
            )
        else:
            print(
                f"{cell19_dataset_name:>10s} "
                f"{str(cell19_row_dict.get('status', 'missing')):>22s} "
                f"{'':>10s} {'':>10s} {'':>4s} "
                f"{'':>8s} {'':>10s} {'':>8s} "
                f"{'':>8s} {'':>10s} {'':>8s}"
            )

    print("=" * 100)

    cell19_summary_dataframe = cell19_pd.DataFrame(cell19_summary_rows)
    cell19_summary_csv_path = cell19_cross_dir / "cell19_cross_connectome_summary.csv"
    cell19_summary_dataframe.to_csv(cell19_summary_csv_path, index=False)

    cell19_results_json_path = cell19_cross_dir / "cell19_cross_connectome_results.json"
    cell19_results_json_path.write_text(
        cell19_json.dumps(cell19_results_dict, indent=2, default=str),
        encoding="utf-8",
    )

    cell19_results_pickle_path = cell19_cross_dir / "cell19_cross_connectome_results.pkl"
    with open(cell19_results_pickle_path, "wb") as cell19_pickle_handle:
        cell19_pickle.dump(cell19_results_dict, cell19_pickle_handle)

    print("Saved summary CSV: " + str(cell19_summary_csv_path))
    print("Saved results JSON: " + str(cell19_results_json_path))
    print("Saved results PKL:  " + str(cell19_results_pickle_path))

    cell19_hf_save_checkpoint = cell19_globals.get("hf_save_checkpoint", None)

    if callable(cell19_hf_save_checkpoint):
        try:
            cell19_hf_save_checkpoint(
                "cell19_cross_connectome",
                {
                    "results": cell19_results_dict,
                    "summary_rows": cell19_summary_rows,
                    "frozen_path": str(cell19_frozen_path),
                    "paths": {
                        "summary_csv": str(cell19_summary_csv_path),
                        "results_json": str(cell19_results_json_path),
                        "results_pickle": str(cell19_results_pickle_path),
                    },
                },
                description=(
                    "Cell 19: cross-connectome structural coherence using frozen FAFB SSPR. "
                    "Failures are recorded, not fabricated."
                ),
                upload=True,
            )
        except Exception as cell19_checkpoint_error:
            print("[ckpt][warn] Cell 19 checkpoint failed: " + repr(cell19_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    cell19_hf_upload_file_to_bucket = cell19_globals.get("hf_upload_file_to_bucket", None)

    if callable(cell19_hf_upload_file_to_bucket):
        cell19_remote_paths = {
            str(cell19_summary_csv_path): "checkpoints/cell19_cross_connectome/cell19_cross_connectome_summary.csv",
            str(cell19_results_json_path): "checkpoints/cell19_cross_connectome/cell19_cross_connectome_results.json",
            str(cell19_results_pickle_path): "checkpoints/cell19_cross_connectome/cell19_cross_connectome_results.pkl",
        }

        for cell19_local_path_string, cell19_remote_path_string in cell19_remote_paths.items():
            try:
                cell19_upload_ok, cell19_upload_info = cell19_hf_upload_file_to_bucket(
                    cell19_Path(cell19_local_path_string),
                    cell19_remote_path_string,
                )
                print(
                    "[ckpt:cell19] upload "
                    + cell19_Path(cell19_local_path_string).name
                    + ": "
                    + ("OK" if cell19_upload_ok else "FAILED")
                    + " | "
                    + str(cell19_upload_info)
                )
            except Exception as cell19_upload_error:
                print(
                    "[ckpt][warn] Cell 19 upload failed for "
                    + cell19_Path(cell19_local_path_string).name
                    + ": "
                    + repr(cell19_upload_error)
                )
    else:
        print(
            "[ckpt][skip] hf_upload_file_to_bucket not defined. "
            "Cross-connectome files remain local only."
        )

    return {
        "results": cell19_results_dict,
        "summary_rows": cell19_summary_rows,
        "cross_dir": str(cell19_cross_dir),
        "summary_csv_path": str(cell19_summary_csv_path),
        "results_json_path": str(cell19_results_json_path),
        "results_pickle_path": str(cell19_results_pickle_path),
    }


cell19_output = cell19_main()

cell19_cross_results = cell19_output["results"]
cell19_cross_summary_rows = cell19_output["summary_rows"]
cell19_cross_dir = cell19_output["cross_dir"]
cell19_cross_summary_csv_path = cell19_output["summary_csv_path"]
cell19_cross_results_json_path = cell19_output["results_json_path"]
cell19_cross_results_pickle_path = cell19_output["results_pickle_path"]

In [ ]:
# Cell 19b: Discover and prefetch cross-connectome files from GCS.
#
# Purpose:
#   Cell 19 guessed versioned URLs and failed with download_failed.
#   This cell lists the real GCS object keys under:
#     gs://flywire-data/codex/data/
#   then downloads the required files into the same local directories
#   expected by Cell 19:
#     cross_connectome_data_cell19/manc/
#     cross_connectome_data_cell19/maol/
#     cross_connectome_data_cell19/mcns/
#
# After this cell succeeds, rerun Cell 19.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell19b_.
#   - Imports are inside the cell function only.


def cell19b_main():
    import urllib.request as cell19b_urllib_request
    import urllib.error as cell19b_urllib_error
    import urllib.parse as cell19b_urllib_parse
    import json as cell19b_json
    import gzip as cell19b_gzip
    import xml.etree.ElementTree as cell19b_ElementTree
    from pathlib import Path as cell19b_Path

    cell19b_bucket = "flywire-data"
    cell19b_root_prefix = "codex/data/"
    cell19b_datasets = ["manc", "maol", "mcns"]
    cell19b_required_files = [
        "connections.csv.gz",
        "classification.csv.gz",
        "neurons.csv.gz",
        "cell_stats.csv.gz",
    ]
    cell19b_required_set = set(cell19b_required_files)

    cell19b_output_dir = cell19b_Path("./cross_connectome_data_cell19")
    cell19b_output_dir.mkdir(parents=True, exist_ok=True)

    cell19b_json_api = (
        "https://storage.googleapis.com/storage/v1/b/"
        + cell19b_bucket
        + "/o"
    )
    cell19b_xml_api = "https://storage.googleapis.com/" + cell19b_bucket
    cell19b_download_api = "https://storage.googleapis.com/" + cell19b_bucket + "/"

    cell19b_cache = {
        "top_prefixes": None,
        "top_objects": None,
        "top_ok": None,
        "top_error": None,
    }

    def cell19b_local_tag(cell19b_element):
        return str(cell19b_element.tag).split("}")[-1]

    def cell19b_validate_gzip(cell19b_path):
        with cell19b_gzip.open(cell19b_path, "rb") as cell19b_handle:
            while cell19b_handle.read(1 << 20):
                pass
        return True

    def cell19b_read_url_bytes(cell19b_url, cell19b_timeout=120):
        cell19b_request_object = cell19b_urllib_request.Request(
            cell19b_url,
            headers={"User-Agent": "research"},
        )

        with cell19b_urllib_request.urlopen(
            cell19b_request_object,
            timeout=cell19b_timeout,
        ) as cell19b_response_object:
            return cell19b_response_object.read()

    def cell19b_stream_url_to_path(cell19b_url, cell19b_dest_path, cell19b_timeout=600):
        cell19b_request_object = cell19b_urllib_request.Request(
            cell19b_url,
            headers={"User-Agent": "research"},
        )

        cell19b_tmp_path = cell19b_Path(str(cell19b_dest_path) + ".part")

        try:
            with cell19b_urllib_request.urlopen(
                cell19b_request_object,
                timeout=cell19b_timeout,
            ) as cell19b_response_object:
                with open(cell19b_tmp_path, "wb") as cell19b_out_handle:
                    while True:
                        cell19b_chunk = cell19b_response_object.read(1 << 20)

                        if not cell19b_chunk:
                            break

                        cell19b_out_handle.write(cell19b_chunk)

            cell19b_validate_gzip(cell19b_tmp_path)
            cell19b_tmp_path.replace(cell19b_dest_path)
            return True, None

        except Exception as cell19b_stream_error:
            try:
                if cell19b_tmp_path.exists():
                    cell19b_tmp_path.unlink()
            except Exception:
                pass

            return False, repr(cell19b_stream_error)

    def cell19b_json_list_once(cell19b_prefix, cell19b_delimiter=None, cell19b_page_token=None):
        cell19b_params = {
            "prefix": cell19b_prefix,
            "projection": "noAcl",
            "maxResults": "1000",
        }

        if cell19b_delimiter is not None:
            cell19b_params["delimiter"] = cell19b_delimiter

        if cell19b_page_token is not None:
            cell19b_params["pageToken"] = cell19b_page_token

        cell19b_url = (
            cell19b_json_api
            + "?"
            + cell19b_urllib_parse.urlencode(cell19b_params)
        )

        cell19b_payload_bytes = cell19b_read_url_bytes(cell19b_url, cell19b_timeout=120)
        cell19b_payload = cell19b_json.loads(cell19b_payload_bytes.decode("utf-8"))

        return cell19b_payload

    def cell19b_json_list(cell19b_prefix, cell19b_delimiter=None, cell19b_max_pages=100):
        cell19b_objects = []
        cell19b_prefixes = []
        cell19b_page_token = None
        cell19b_last_error = None

        for cell19b_page_index in range(int(cell19b_max_pages)):
            try:
                cell19b_payload = cell19b_json_list_once(
                    cell19b_prefix,
                    cell19b_delimiter=cell19b_delimiter,
                    cell19b_page_token=cell19b_page_token,
                )
            except Exception as cell19b_error:
                cell19b_last_error = repr(cell19b_error)
                break

            for cell19b_item in cell19b_payload.get("items", []):
                cell19b_name = cell19b_item.get("name", None)

                if cell19b_name is not None:
                    cell19b_objects.append(str(cell19b_name))

            for cell19b_prefix_value in cell19b_payload.get("prefixes", []):
                cell19b_prefixes.append(str(cell19b_prefix_value))

            cell19b_page_token = cell19b_payload.get("nextPageToken", None)

            if not cell19b_page_token:
                return cell19b_objects, cell19b_prefixes, True, None

        return (
            cell19b_objects,
            cell19b_prefixes,
            False,
            cell19b_last_error or "json_max_pages_reached",
        )

    def cell19b_xml_list_once(cell19b_prefix, cell19b_delimiter=None, cell19b_marker=None):
        cell19b_params = {
            "prefix": cell19b_prefix,
            "max-keys": "1000",
        }

        if cell19b_delimiter is not None:
            cell19b_params["delimiter"] = cell19b_delimiter

        if cell19b_marker is not None:
            cell19b_params["marker"] = cell19b_marker

        cell19b_url = (
            cell19b_xml_api
            + "?"
            + cell19b_urllib_parse.urlencode(cell19b_params)
        )

        cell19b_payload_bytes = cell19b_read_url_bytes(cell19b_url, cell19b_timeout=120)
        cell19b_root = cell19b_ElementTree.fromstring(cell19b_payload_bytes)

        cell19b_objects = []
        cell19b_prefixes = []
        cell19b_truncated = False
        cell19b_next_marker = None

        for cell19b_child in cell19b_root:
            cell19b_tag = cell19b_local_tag(cell19b_child)

            if cell19b_tag == "Contents":
                cell19b_key_text = None

                for cell19b_subchild in cell19b_child:
                    if cell19b_local_tag(cell19b_subchild) == "Key":
                        cell19b_key_text = cell19b_subchild.text

                if cell19b_key_text is not None:
                    cell19b_objects.append(str(cell19b_key_text))

            elif cell19b_tag == "CommonPrefixes":
                cell19b_prefix_text = None

                for cell19b_subchild in cell19b_child:
                    if cell19b_local_tag(cell19b_subchild) == "Prefix":
                        cell19b_prefix_text = cell19b_subchild.text

                if cell19b_prefix_text is not None:
                    cell19b_prefixes.append(str(cell19b_prefix_text))

            elif cell19b_tag == "IsTruncated":
                cell19b_truncated = (
                    str(cell19b_child.text or "").strip().lower() == "true"
                )

            elif cell19b_tag == "NextMarker":
                cell19b_next_marker = cell19b_child.text

        return (
            cell19b_objects,
            cell19b_prefixes,
            cell19b_truncated,
            cell19b_next_marker,
        )

    def cell19b_xml_list(cell19b_prefix, cell19b_delimiter=None, cell19b_max_pages=100):
        cell19b_all_objects = []
        cell19b_all_prefixes = []
        cell19b_marker = None
        cell19b_last_error = None

        for cell19b_page_index in range(int(cell19b_max_pages)):
            try:
                cell19b_objects, cell19b_prefixes, cell19b_truncated, cell19b_next_marker = (
                    cell19b_xml_list_once(
                        cell19b_prefix,
                        cell19b_delimiter=cell19b_delimiter,
                        cell19b_marker=cell19b_marker,
                    )
                )
            except Exception as cell19b_error:
                cell19b_last_error = repr(cell19b_error)
                break

            cell19b_all_objects.extend(cell19b_objects)
            cell19b_all_prefixes.extend(cell19b_prefixes)

            if not cell19b_truncated:
                return cell19b_all_objects, cell19b_all_prefixes, True, None

            if cell19b_next_marker:
                cell19b_marker = str(cell19b_next_marker)
            elif cell19b_all_objects:
                cell19b_marker = cell19b_all_objects[-1]
            else:
                return (
                    cell19b_all_objects,
                    cell19b_all_prefixes,
                    False,
                    "xml_truncated_without_marker",
                )

        return (
            cell19b_all_objects,
            cell19b_all_prefixes,
            False,
            cell19b_last_error or "xml_max_pages_reached",
        )

    def cell19b_list_objects(cell19b_prefix, cell19b_delimiter=None, cell19b_max_pages=100):
        cell19b_objects_json, cell19b_prefixes_json, cell19b_ok_json, cell19b_error_json = (
            cell19b_json_list(
                cell19b_prefix,
                cell19b_delimiter=cell19b_delimiter,
                cell19b_max_pages=cell19b_max_pages,
            )
        )

        if cell19b_ok_json and (cell19b_objects_json or cell19b_prefixes_json):
            return (
                cell19b_objects_json,
                cell19b_prefixes_json,
                True,
                None,
            )

        cell19b_objects_xml, cell19b_prefixes_xml, cell19b_ok_xml, cell19b_error_xml = (
            cell19b_xml_list(
                cell19b_prefix,
                cell19b_delimiter=cell19b_delimiter,
                cell19b_max_pages=cell19b_max_pages,
            )
        )

        if cell19b_ok_xml and (cell19b_objects_xml or cell19b_prefixes_xml):
            return (
                cell19b_objects_xml,
                cell19b_prefixes_xml,
                True,
                None,
            )

        if cell19b_ok_json:
            return (
                cell19b_objects_json,
                cell19b_prefixes_json,
                True,
                cell19b_error_json,
            )

        if cell19b_ok_xml:
            return (
                cell19b_objects_xml,
                cell19b_prefixes_xml,
                True,
                cell19b_error_xml,
            )

        return (
            [],
            [],
            False,
            str(cell19b_error_json or cell19b_error_xml or "unknown_listing_error"),
        )

    def cell19b_get_top_prefixes():
        if cell19b_cache["top_prefixes"] is not None:
            return (
                cell19b_cache["top_prefixes"],
                cell19b_cache["top_objects"],
                cell19b_cache["top_ok"],
                cell19b_cache["top_error"],
            )

        cell19b_objects, cell19b_prefixes, cell19b_ok, cell19b_error = cell19b_list_objects(
            cell19b_root_prefix,
            cell19b_delimiter="/",
            cell19b_max_pages=20,
        )

        cell19b_cache["top_prefixes"] = sorted(set(str(x) for x in cell19b_prefixes))
        cell19b_cache["top_objects"] = [str(x) for x in cell19b_objects]
        cell19b_cache["top_ok"] = bool(cell19b_ok)
        cell19b_cache["top_error"] = cell19b_error

        return (
            cell19b_cache["top_prefixes"],
            cell19b_cache["top_objects"],
            cell19b_cache["top_ok"],
            cell19b_cache["top_error"],
        )

    def cell19b_unique_preserve_order(cell19b_items):
        cell19b_seen = set()
        cell19b_out = []

        for cell19b_item in cell19b_items:
            cell19b_text = str(cell19b_item)

            if cell19b_text not in cell19b_seen:
                cell19b_seen.add(cell19b_text)
                cell19b_out.append(cell19b_text)

        return cell19b_out

    def cell19b_candidate_prefixes_for_dataset(cell19b_dataset_name):
        cell19b_direct_prefix = cell19b_root_prefix + str(cell19b_dataset_name) + "/"
        cell19b_candidates = [cell19b_direct_prefix]

        cell19b_top_prefixes, cell19b_top_objects, cell19b_top_ok, cell19b_top_error = (
            cell19b_get_top_prefixes()
        )

        cell19b_dataset_lower = str(cell19b_dataset_name).lower()

        for cell19b_top_prefix in cell19b_top_prefixes:
            if cell19b_dataset_lower in str(cell19b_top_prefix).lower():
                cell19b_candidates.append(str(cell19b_top_prefix))

        # Fallback: inspect one level above codex/data/ in case the layout is
        # codex/<something>/<dataset>/... instead of codex/data/<dataset>/...
        cell19b_broad_objects, cell19b_broad_prefixes, cell19b_broad_ok, cell19b_broad_error = (
            cell19b_list_objects(
                "codex/",
                cell19b_delimiter="/",
                cell19b_max_pages=5,
            )
        )

        if cell19b_broad_ok:
            for cell19b_broad_prefix in cell19b_broad_prefixes:
                if cell19b_dataset_lower in str(cell19b_broad_prefix).lower():
                    cell19b_candidates.append(str(cell19b_broad_prefix))

        cell19b_normalized = []

        for cell19b_candidate in cell19b_candidates:
            cell19b_candidate_text = str(cell19b_candidate)

            if not cell19b_candidate_text.endswith("/"):
                cell19b_candidate_text = cell19b_candidate_text + "/"

            cell19b_normalized.append(cell19b_candidate_text)

        return (
            cell19b_unique_preserve_order(cell19b_normalized),
            cell19b_top_prefixes,
            cell19b_top_ok,
            cell19b_top_error,
            cell19b_broad_prefixes if cell19b_broad_ok else [],
            cell19b_broad_error if not cell19b_broad_ok else None,
        )

    def cell19b_find_required_keys_for_dataset(cell19b_dataset_name):
        (
            cell19b_candidate_prefixes,
            cell19b_top_prefixes,
            cell19b_top_ok,
            cell19b_top_error,
            cell19b_broad_prefixes,
            cell19b_broad_error,
        ) = cell19b_candidate_prefixes_for_dataset(cell19b_dataset_name)

        cell19b_best_keys = {}
        cell19b_chosen_prefix = None
        cell19b_debug_keys = []
        cell19b_listing_errors = []

        for cell19b_prefix in cell19b_candidate_prefixes:
            cell19b_objects, cell19b_subprefixes, cell19b_ok, cell19b_error = cell19b_list_objects(
                cell19b_prefix,
                cell19b_delimiter=None,
                cell19b_max_pages=100,
            )

            if not cell19b_ok:
                cell19b_listing_errors.append(
                    {
                        "prefix": cell19b_prefix,
                        "error": cell19b_error,
                    }
                )
                continue

            for cell19b_object_key in cell19b_objects:
                if len(cell19b_debug_keys) < 200:
                    cell19b_debug_keys.append(str(cell19b_object_key))

                cell19b_basename = str(cell19b_object_key).split("/")[-1]

                if cell19b_basename in cell19b_required_set:
                    if cell19b_basename not in cell19b_best_keys:
                        cell19b_best_keys[cell19b_basename] = str(cell19b_object_key)

            if len(cell19b_best_keys) == len(cell19b_required_files):
                cell19b_chosen_prefix = cell19b_prefix
                break

        return {
            "dataset": str(cell19b_dataset_name),
            "candidate_prefixes": cell19b_candidate_prefixes,
            "chosen_prefix": cell19b_chosen_prefix,
            "keys": cell19b_best_keys,
            "missing_files": [
                cell19b_filename
                for cell19b_filename in cell19b_required_files
                if cell19b_filename not in cell19b_best_keys
            ],
            "top_prefixes": cell19b_top_prefixes,
            "top_ok": cell19b_top_ok,
            "top_error": cell19b_top_error,
            "broad_prefixes": cell19b_broad_prefixes,
            "broad_error": cell19b_broad_error,
            "listing_errors": cell19b_listing_errors,
            "debug_keys": cell19b_debug_keys,
        }

    def cell19b_check_existing_local_files(cell19b_dataset_name):
        cell19b_dataset_dir = cell19b_output_dir / str(cell19b_dataset_name)
        cell19b_file_status = {}

        for cell19b_filename in cell19b_required_files:
            cell19b_dest_path = cell19b_dataset_dir / cell19b_filename

            if cell19b_dest_path.exists() and cell19b_dest_path.stat().st_size > 1000:
                try:
                    cell19b_validate_gzip(cell19b_dest_path)
                    cell19b_file_status[cell19b_filename] = "skipped_existing_valid"
                except Exception:
                    try:
                        cell19b_dest_path.unlink()
                    except Exception:
                        pass

                    cell19b_file_status[cell19b_filename] = "missing_invalid_removed"
            else:
                cell19b_file_status[cell19b_filename] = "missing"

        return cell19b_dataset_dir, cell19b_file_status

    print("=" * 100)
    print("CELL 19b — CROSS-CONNECTOME GCS DISCOVERY AND PREFETCH")
    print("=" * 100)
    print("[cfg] bucket=" + cell19b_bucket)
    print("[cfg] root_prefix=" + cell19b_root_prefix)
    print("[cfg] datasets=" + str(cell19b_datasets))
    print("[cfg] required_files=" + str(cell19b_required_files))
    print("[cfg] local_dir=" + str(cell19b_output_dir))
    print("=" * 100)

    cell19b_discovery = {}
    cell19b_download_status = {}

    for cell19b_dataset_name in cell19b_datasets:
        print("\n---- Dataset: " + str(cell19b_dataset_name) + " ----")

        cell19b_dataset_dir, cell19b_file_status = cell19b_check_existing_local_files(
            cell19b_dataset_name
        )

        cell19b_existing_ok = all(
            str(cell19b_status_value) == "skipped_existing_valid"
            for cell19b_status_value in cell19b_file_status.values()
        )

        if cell19b_existing_ok:
            print("[cell19b] all required files already present and gzip-valid.")
            cell19b_discovery[cell19b_dataset_name] = {
                "status": "local_files_present",
                "file_status": cell19b_file_status,
            }
            cell19b_download_status[cell19b_dataset_name] = {
                "status": "ok_existing",
                "file_status": cell19b_file_status,
            }
            continue

        cell19b_found = cell19b_find_required_keys_for_dataset(cell19b_dataset_name)
        cell19b_discovery[cell19b_dataset_name] = cell19b_found

        print("[cell19b] candidate_prefixes=" + str(cell19b_found["candidate_prefixes"][:20]))
        print("[cell19b] chosen_prefix=" + str(cell19b_found["chosen_prefix"]))
        print("[cell19b] found_keys=" + str(cell19b_found["keys"]))
        print("[cell19b] missing_files=" + str(cell19b_found["missing_files"]))

        if len(cell19b_found["keys"]) == 0:
            print("[cell19b] no GCS keys found for this dataset.")
            print("[cell19b] top prefixes under codex/data/:")

            for cell19b_top_prefix in cell19b_found["top_prefixes"][:100]:
                print("  " + str(cell19b_top_prefix))

            if len(cell19b_found["broad_prefixes"]) > 0:
                print("[cell19b] broad prefixes under codex/:")

                for cell19b_broad_prefix in cell19b_found["broad_prefixes"][:100]:
                    print("  " + str(cell19b_broad_prefix))

            if len(cell19b_found["debug_keys"]) > 0:
                print("[cell19b] sample keys:")

                for cell19b_debug_key in cell19b_found["debug_keys"][:50]:
                    print("  " + str(cell19b_debug_key))

            cell19b_download_status[cell19b_dataset_name] = {
                "status": "missing_keys",
                "file_status": cell19b_file_status,
                "discovery": cell19b_found,
            }
            continue

        for cell19b_filename in cell19b_required_files:
            cell19b_current_status = str(cell19b_file_status.get(cell19b_filename, "missing"))

            if cell19b_current_status == "skipped_existing_valid":
                continue

            if cell19b_filename not in cell19b_found["keys"]:
                cell19b_file_status[cell19b_filename] = "missing_key"
                print("[cell19b][miss] " + str(cell19b_dataset_name) + "/" + str(cell19b_filename))
                continue

            cell19b_object_key = str(cell19b_found["keys"][cell19b_filename])
            cell19b_dest_path = cell19b_dataset_dir / cell19b_filename
            cell19b_url = cell19b_download_api + cell19b_urllib_parse.quote(
                cell19b_object_key,
                safe="/",
            )

            print("[cell19b][get] " + str(cell19b_dataset_name) + "/" + str(cell19b_filename))
            print("         key=" + cell19b_object_key)
            print("         url=" + cell19b_url)

            cell19b_ok, cell19b_error = cell19b_stream_url_to_path(
                cell19b_url,
                cell19b_dest_path,
                cell19b_timeout=900,
            )

            if cell19b_ok:
                cell19b_file_status[cell19b_filename] = "downloaded_valid"
                print("         saved=" + str(cell19b_dest_path))
            else:
                cell19b_file_status[cell19b_filename] = "download_failed"
                print("         failed=" + str(cell19b_error))

        cell19b_final_ok = all(
            str(cell19b_status_value) in {"skipped_existing_valid", "downloaded_valid"}
            for cell19b_status_value in cell19b_file_status.values()
        )

        cell19b_download_status[cell19b_dataset_name] = {
            "status": "ok" if cell19b_final_ok else "partial_or_failed",
            "file_status": cell19b_file_status,
            "chosen_prefix": cell19b_found["chosen_prefix"],
            "keys": cell19b_found["keys"],
        }

    print("\n" + "=" * 100)
    print("CELL 19b SUMMARY")
    print("=" * 100)

    cell19b_all_ok = True

    for cell19b_dataset_name in cell19b_datasets:
        cell19b_status_dict = cell19b_download_status.get(cell19b_dataset_name, {})
        cell19b_status_text = str(cell19b_status_dict.get("status", "missing"))

        if cell19b_status_text not in {"ok", "ok_existing"}:
            cell19b_all_ok = False

        print(
            "  "
            + str(cell19b_dataset_name)
            + ": "
            + cell19b_status_text
        )

        for cell19b_filename, cell19b_file_value in cell19b_status_dict.get(
            "file_status",
            {},
        ).items():
            print("    " + str(cell19b_filename) + ": " + str(cell19b_file_value))

    print("=" * 100)

    if cell19b_all_ok:
        print("[cell19b] Prefetch complete. Now rerun Cell 19.")
    else:
        print("[cell19b] Prefetch incomplete. Do not report cross-connectome results yet.")
        print("[cell19b] Copy the printed top-prefix/debug-key output and fix the dataset prefix.")

    print("=" * 100)

    return {
        "output_dir": str(cell19b_output_dir),
        "discovery": cell19b_discovery,
        "download_status": cell19b_download_status,
        "all_ok": bool(cell19b_all_ok),
    }


cell19b_output = cell19b_main()

cell19b_discovery = cell19b_output["discovery"]
cell19b_download_status = cell19b_output["download_status"]
cell19b_all_ok = cell19b_output["all_ok"]
cell19b_output_dir = cell19b_output["output_dir"]

In [ ]:
# Cell 19c: Inventory actual GCS keys for MANC / MAOL / MCNS.
#
# Purpose:
#   Cell 19b found neurons.csv.gz but not connections/classification/cell_stats.
#   This cell lists the actual object keys so we can discover the real filenames.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell19c_.
#   - Imports are inside the cell function only.


def cell19c_main():
    import urllib.request as cell19c_urllib_request
    import urllib.error as cell19c_urllib_error
    import urllib.parse as cell19c_urllib_parse
    import json as cell19c_json
    import xml.etree.ElementTree as cell19c_ElementTree
    from pathlib import Path as cell19c_Path

    cell19c_bucket = "flywire-data"
    cell19c_prefixes = [
        "codex/data/manc/",
        "codex/data/maol/",
        "codex/data/mcns/",
    ]

    cell19c_required_basenames = [
        "connections.csv.gz",
        "classification.csv.gz",
        "neurons.csv.gz",
        "cell_stats.csv.gz",
    ]

    cell19c_output_dir = cell19c_Path("./cross_connectome_inventory_cell19c")
    cell19c_output_dir.mkdir(parents=True, exist_ok=True)

    cell19c_json_api = (
        "https://storage.googleapis.com/storage/v1/b/"
        + cell19c_bucket
        + "/o"
    )
    cell19c_xml_api = "https://storage.googleapis.com/" + cell19c_bucket

    def cell19c_local_tag(cell19c_element):
        return str(cell19c_element.tag).split("}")[-1]

    def cell19c_read_url_text(cell19c_url, cell19c_timeout=120):
        cell19c_request_object = cell19c_urllib_request.Request(
            cell19c_url,
            headers={"User-Agent": "research"},
        )

        with cell19c_urllib_request.urlopen(
            cell19c_request_object,
            timeout=cell19c_timeout,
        ) as cell19c_response_object:
            return cell19c_response_object.read().decode("utf-8", errors="replace")

    def cell19c_json_list_once(cell19c_prefix, cell19c_page_token=None):
        cell19c_params = {
            "prefix": cell19c_prefix,
            "projection": "noAcl",
            "maxResults": "1000",
        }

        if cell19c_page_token is not None:
            cell19c_params["pageToken"] = cell19c_page_token

        cell19c_url = (
            cell19c_json_api
            + "?"
            + cell19c_urllib_parse.urlencode(cell19c_params)
        )

        cell19c_text = cell19c_read_url_text(cell19c_url, cell19c_timeout=120)
        return cell19c_json.loads(cell19c_text)

    def cell19c_json_list(cell19c_prefix, cell19c_max_pages=300):
        cell19c_objects = []
        cell19c_page_token = None
        cell19c_last_error = None

        for cell19c_page_index in range(int(cell19c_max_pages)):
            try:
                cell19c_payload = cell19c_json_list_once(
                    cell19c_prefix,
                    cell19c_page_token=cell19c_page_token,
                )
            except Exception as cell19c_error:
                cell19c_last_error = repr(cell19c_error)
                break

            for cell19c_item in cell19c_payload.get("items", []):
                cell19c_name = cell19c_item.get("name", None)
                cell19c_size = cell19c_item.get("size", None)

                if cell19c_name is not None:
                    cell19c_objects.append(
                        {
                            "name": str(cell19c_name),
                            "size": cell19c_size,
                        }
                    )

            cell19c_page_token = cell19c_payload.get("nextPageToken", None)

            if not cell19c_page_token:
                return cell19c_objects, True, None

        return cell19c_objects, False, cell19c_last_error or "json_max_pages_reached"

    def cell19c_xml_list_once(cell19c_prefix, cell19c_marker=None):
        cell19c_params = {
            "prefix": cell19c_prefix,
            "max-keys": "1000",
        }

        if cell19c_marker is not None:
            cell19c_params["marker"] = cell19c_marker

        cell19c_url = (
            cell19c_xml_api
            + "?"
            + cell19c_urllib_parse.urlencode(cell19c_params)
        )

        cell19c_text = cell19c_read_url_text(cell19c_url, cell19c_timeout=120)
        cell19c_root = cell19c_ElementTree.fromstring(cell19c_text)

        cell19c_objects = []
        cell19c_truncated = False
        cell19c_next_marker = None

        for cell19c_child in cell19c_root:
            cell19c_tag = cell19c_local_tag(cell19c_child)

            if cell19c_tag == "Contents":
                cell19c_key_text = None

                for cell19c_subchild in cell19c_child:
                    if cell19c_local_tag(cell19c_subchild) == "Key":
                        cell19c_key_text = cell19c_subchild.text

                if cell19c_key_text is not None:
                    cell19c_objects.append(
                        {
                            "name": str(cell19c_key_text),
                            "size": None,
                        }
                    )

            elif cell19c_tag == "IsTruncated":
                cell19c_truncated = (
                    str(cell19c_child.text or "").strip().lower() == "true"
                )

            elif cell19c_tag == "NextMarker":
                cell19c_next_marker = cell19c_child.text

        return cell19c_objects, cell19c_truncated, cell19c_next_marker

    def cell19c_xml_list(cell19c_prefix, cell19c_max_pages=300):
        cell19c_all_objects = []
        cell19c_marker = None
        cell19c_last_error = None

        for cell19c_page_index in range(int(cell19c_max_pages)):
            try:
                cell19c_objects, cell19c_truncated, cell19c_next_marker = (
                    cell19c_xml_list_once(
                        cell19c_prefix,
                        cell19c_marker=cell19c_marker,
                    )
                )
            except Exception as cell19c_error:
                cell19c_last_error = repr(cell19c_error)
                break

            cell19c_all_objects.extend(cell19c_objects)

            if not cell19c_truncated:
                return cell19c_all_objects, True, None

            if cell19c_next_marker:
                cell19c_marker = str(cell19c_next_marker)
            elif cell19c_all_objects:
                cell19c_marker = cell19c_all_objects[-1]["name"]
            else:
                return (
                    cell19c_all_objects,
                    False,
                    "xml_truncated_without_marker",
                )

        return (
            cell19c_all_objects,
            False,
            cell19c_last_error or "xml_max_pages_reached",
        )

    def cell19c_list_objects(cell19c_prefix):
        cell19c_objects_json, cell19c_ok_json, cell19c_error_json = cell19c_json_list(
            cell19c_prefix,
            cell19c_max_pages=300,
        )

        if cell19c_ok_json and len(cell19c_objects_json) > 0:
            return cell19c_objects_json, True, None

        cell19c_objects_xml, cell19c_ok_xml, cell19c_error_xml = cell19c_xml_list(
            cell19c_prefix,
            cell19c_max_pages=300,
        )

        if cell19c_ok_xml and len(cell19c_objects_xml) > 0:
            return cell19c_objects_xml, True, cell19c_error_json

        if cell19c_ok_json:
            return cell19c_objects_json, True, cell19c_error_json

        return (
            cell19c_objects_xml,
            False,
            str(cell19c_error_json or cell19c_error_xml or "unknown_listing_error"),
        )

    def cell19c_basename(cell19c_key):
        return str(cell19c_key).split("/")[-1]

    def cell19c_unique_preserve_order(cell19c_items):
        cell19c_seen = set()
        cell19c_out = []

        for cell19c_item in cell19c_items:
            cell19c_text = str(cell19c_item)

            if cell19c_text not in cell19c_seen:
                cell19c_seen.add(cell19c_text)
                cell19c_out.append(cell19c_text)

        return cell19c_out

    print("=" * 100)
    print("CELL 19c — CROSS-CONNECTOME GCS INVENTORY")
    print("=" * 100)
    print("[cfg] bucket=" + cell19c_bucket)
    print("[cfg] prefixes=" + str(cell19c_prefixes))
    print("[cfg] required_basenames=" + str(cell19c_required_basenames))
    print("[cfg] output_dir=" + str(cell19c_output_dir))
    print("=" * 100)

    cell19c_inventory = {}

    for cell19c_prefix in cell19c_prefixes:
        print("\n---- Prefix: " + str(cell19c_prefix) + " ----")

        cell19c_objects, cell19c_ok, cell19c_error = cell19c_list_objects(cell19c_prefix)

        cell19c_keys = []
        cell19c_basenames = []
        cell19c_size_map = {}

        for cell19c_object in cell19c_objects:
            cell19c_key = str(cell19c_object.get("name", ""))
            cell19c_size = cell19c_object.get("size", None)

            if cell19c_key == "":
                continue

            cell19c_keys.append(cell19c_key)
            cell19c_basenames.append(cell19c_basename(cell19c_key))

            if cell19c_size is not None:
                try:
                    cell19c_size_map[cell19c_key] = int(cell19c_size)
                except Exception:
                    cell19c_size_map[cell19c_key] = None

        cell19c_unique_basenames = cell19c_unique_preserve_order(cell19c_basenames)
        cell19c_unique_basenames_sorted = sorted(cell19c_unique_basenames)

        cell19c_present_required = {}

        for cell19c_required_name in cell19c_required_basenames:
            cell19c_matching_keys = []

            for cell19c_key in cell19c_keys:
                if cell19c_basename(cell19c_key) == cell19c_required_name:
                    cell19c_matching_keys.append(cell19c_key)

            cell19c_present_required[cell19c_required_name] = {
                "found": bool(len(cell19c_matching_keys) > 0),
                "keys": cell19c_matching_keys[:20],
            }

        cell19c_inventory[cell19c_prefix] = {
            "ok": bool(cell19c_ok),
            "error": cell19c_error,
            "n_objects_listed": int(len(cell19c_keys)),
            "unique_basenames": cell19c_unique_basenames_sorted,
            "present_required": cell19c_present_required,
            "sample_keys": cell19c_keys[:200],
            "size_map_sample": {
                cell19c_key: cell19c_size_map.get(cell19c_key, None)
                for cell19c_key in cell19c_keys[:200]
            },
        }

        print("[cell19c] ok=" + str(cell19c_ok))
        print("[cell19c] error=" + str(cell19c_error))
        print("[cell19c] n_objects_listed=" + str(len(cell19c_keys)))
        print("[cell19c] unique_basenames_count=" + str(len(cell19c_unique_basenames_sorted)))

        print("\n[cell19c] required basename check:")
        for cell19c_required_name in cell19c_required_basenames:
            cell19c_required_info = cell19c_present_required[cell19c_required_name]
            print(
                "  "
                + str(cell19c_required_name)
                + ": found="
                + str(cell19c_required_info["found"])
                + " n_keys="
                + str(len(cell19c_required_info["keys"]))
            )

            for cell19c_matching_key in cell19c_required_info["keys"][:5]:
                print("    " + str(cell19c_matching_key))

        print("\n[cell19c] unique basenames, first 100:")
        for cell19c_basename_value in cell19c_unique_basenames_sorted[:100]:
            print("  " + str(cell19c_basename_value))

        if len(cell19c_unique_basenames_sorted) > 100:
            print("  ... omitted " + str(len(cell19c_unique_basenames_sorted) - 100) + " more basenames")

        print("\n[cell19c] sample keys, first 50:")
        for cell19c_key in cell19c_keys[:50]:
            cell19c_key_size = cell19c_size_map.get(cell19c_key, None)
            print("  " + str(cell19c_key) + "  size=" + str(cell19c_key_size))

    cell19c_inventory_path = cell19c_output_dir / "cell19c_gcs_inventory.json"

    with open(cell19c_inventory_path, "w", encoding="utf-8") as cell19c_file_handle:
        cell19c_file_handle.write(
            cell19c_json.dumps(cell19c_inventory, indent=2, default=str)
        )

    print("\n" + "=" * 100)
    print("CELL 19c SUMMARY")
    print("=" * 100)

    for cell19c_prefix in cell19c_prefixes:
        cell19c_prefix_info = cell19c_inventory.get(cell19c_prefix, {})
        print(
            "  "
            + str(cell19c_prefix)
            + ": ok="
            + str(cell19c_prefix_info.get("ok", False))
            + " objects="
            + str(cell19c_prefix_info.get("n_objects_listed", 0))
            + " basenames="
            + str(len(cell19c_prefix_info.get("unique_basenames", [])))
        )

    print("Saved inventory: " + str(cell19c_inventory_path))
    print("=" * 100)

    return {
        "inventory": cell19c_inventory,
        "inventory_path": str(cell19c_inventory_path),
        "output_dir": str(cell19c_output_dir),
    }


cell19c_output = cell19c_main()

cell19c_inventory = cell19c_output["inventory"]
cell19c_inventory_path = cell19c_output["inventory_path"]
cell19c_output_dir = cell19c_output["output_dir"]

In [ ]:
# Cell 19d: Inspect actual cross-connectome schema for MANC / MAOL / MCNS.
#
# Purpose:
#   Cell 19c showed that non-FAFB connectomes do not contain:
#     classification.csv.gz
#     cell_stats.csv.gz
#     connections.csv.gz
#
#   Instead they contain:
#     neurons.csv.gz
#     connections_princeton.csv.gz
#     neuron_attributes.pickle.gz
#
#   This cell downloads those files and prints their actual columns / pickle structure.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell19d_.
#   - Imports are inside the cell function only.
#   - No experimental results are fabricated.


def cell19d_main():
    import urllib.request as cell19d_urllib_request
    import urllib.error as cell19d_urllib_error
    import urllib.parse as cell19d_urllib_parse
    import json as cell19d_json
    import gzip as cell19d_gzip
    import pickle as cell19d_pickle
    from pathlib import Path as cell19d_Path
    import pandas as cell19d_pd

    cell19d_bucket = "flywire-data"
    cell19d_root_prefix = "codex/data"
    cell19d_base_url = "https://storage.googleapis.com/" + cell19d_bucket + "/"

    cell19d_versions = {
        "manc": "1.2.1",
        "maol": "1.1",
        "mcns": "1.0",
    }

    cell19d_datasets = [
        "manc",
        "maol",
        "mcns",
    ]

    cell19d_files = [
        "neurons.csv.gz",
        "connections_princeton.csv.gz",
        "neuron_attributes.pickle.gz",
    ]

    cell19d_output_dir = cell19d_Path("./cross_connectome_probe_cell19d")
    cell19d_output_dir.mkdir(parents=True, exist_ok=True)

    cell19d_candidate_columns = {
        "id": [
            "root_id",
            "node_id",
            "id",
            "neuron_id",
            "soma_id",
        ],
        "pre": [
            "pre_root_id",
            "pre",
            "source",
            "src",
            "presynaptic_root_id",
            "pre_id",
        ],
        "post": [
            "post_root_id",
            "post",
            "target",
            "dst",
            "postsynaptic_root_id",
            "post_id",
        ],
        "syn_count": [
            "syn_count",
            "count",
            "n_syn",
            "weight",
            "edge_weight",
            "synapses",
        ],
        "nt_type": [
            "nt_type",
            "transmitter",
            "neurotransmitter",
            "nt",
        ],
        "label": [
            "class",
            "super_class",
            "sub_class",
            "cell_type",
            "primary_type",
            "additional_type",
            "annotation",
            "group",
            "hemilineage",
            "nerve",
            "flow",
            "type",
            "family",
            "subsystem",
            "category",
        ],
        "morphology": [
            "length_nm",
            "area_nm",
            "size_nm",
            "volume",
            "surface",
            "diameter",
            "soma_length",
            "soma_area",
            "soma_volume",
            "max_length",
            "bbox",
            "extent",
        ],
        "transmitter_score": [
            "da_avg",
            "ser_avg",
            "gaba_avg",
            "glut_avg",
            "ach_avg",
            "oct_avg",
            "nt_type_score",
        ],
    }

    def cell19d_truncate_text(cell19d_text_value, cell19d_limit_value=500):
        cell19d_text_string = str(cell19d_text_value)

        if len(cell19d_text_string) <= int(cell19d_limit_value):
            return cell19d_text_string

        return cell19d_text_string[: int(cell19d_limit_value)] + "...[truncated]"

    def cell19d_validate_gzip(cell19d_path_value):
        with cell19d_gzip.open(cell19d_path_value, "rb") as cell19d_gzip_handle:
            while cell19d_gzip_handle.read(1 << 20):
                pass

        return True

    def cell19d_download_one_file(cell19d_dataset_value, cell19d_filename_value):
        cell19d_version_value = cell19d_versions.get(str(cell19d_dataset_value), None)

        if cell19d_version_value is None:
            return None, "unknown_dataset"

        cell19d_object_key = (
            cell19d_root_prefix
            + "/"
            + str(cell19d_dataset_value)
            + "/"
            + str(cell19d_version_value)
            + "/"
            + str(cell19d_filename_value)
        )

        cell19d_url_value = cell19d_base_url + cell19d_urllib_parse.quote(
            cell19d_object_key,
            safe="/",
        )

        cell19d_dest_dir = cell19d_output_dir / str(cell19d_dataset_value)
        cell19d_dest_dir.mkdir(parents=True, exist_ok=True)

        cell19d_dest_path = cell19d_dest_dir / str(cell19d_filename_value)

        if cell19d_dest_path.exists() and cell19d_dest_path.stat().st_size > 1000:
            try:
                cell19d_validate_gzip(cell19d_dest_path)
                return cell19d_dest_path, "existing_valid"
            except Exception as cell19d_existing_error:
                try:
                    cell19d_dest_path.unlink()
                except Exception:
                    pass

                return None, "existing_invalid_removed:" + repr(cell19d_existing_error)

        cell19d_tmp_path = cell19d_Path(str(cell19d_dest_path) + ".part")

        try:
            cell19d_request_object = cell19d_urllib_request.Request(
                cell19d_url_value,
                headers={"User-Agent": "research"},
            )

            with cell19d_urllib_request.urlopen(
                cell19d_request_object,
                timeout=600,
            ) as cell19d_response_object:
                with open(cell19d_tmp_path, "wb") as cell19d_out_handle:
                    while True:
                        cell19d_chunk_value = cell19d_response_object.read(1 << 20)

                        if not cell19d_chunk_value:
                            break

                        cell19d_out_handle.write(cell19d_chunk_value)

            cell19d_validate_gzip(cell19d_tmp_path)
            cell19d_tmp_path.replace(cell19d_dest_path)

            return cell19d_dest_path, "downloaded_valid"

        except Exception as cell19d_download_error:
            try:
                if cell19d_tmp_path.exists():
                    cell19d_tmp_path.unlink()
            except Exception:
                pass

            return None, "download_failed:" + repr(cell19d_download_error)

    def cell19d_detect_candidate_columns(cell19d_columns_value):
        cell19d_lower_map = {}

        for cell19d_column_index, cell19d_column_value in enumerate(cell19d_columns_value):
            cell19d_lower_map[str(cell19d_column_value).lower()] = str(cell19d_column_value)

        cell19d_detected = {}

        for cell19d_category_value, cell19d_candidate_list in cell19d_candidate_columns.items():
            cell19d_matches = []

            for cell19d_candidate_value in cell19d_candidate_list:
                cell19d_actual_column = cell19d_lower_map.get(str(cell19d_candidate_value).lower(), None)

                if cell19d_actual_column is not None:
                    cell19d_matches.append(cell19d_actual_column)

            cell19d_detected[str(cell19d_category_value)] = cell19d_matches

        return cell19d_detected

    def cell19d_dataframe_summary(cell19d_frame_value, cell19d_name_value):
        cell19d_columns_list = [str(cell19d_column_value) for cell19d_column_value in cell19d_frame_value.columns]

        cell19d_dtypes_dict = {}

        for cell19d_column_value in cell19d_columns_list:
            cell19d_dtypes_dict[cell19d_column_value] = str(cell19d_frame_value[column := cell19d_column_value].dtype)

        cell19d_head_records = []

        try:
            cell19d_head_frame = cell19d_frame_value.head(5).astype(str)
            cell19d_head_records = cell19d_head_frame.to_dict(orient="records")
        except Exception as cell19d_head_error:
            cell19d_head_records = [{"error": repr(cell19d_head_error)}]

        return {
            "name": str(cell19d_name_value),
            "type": "DataFrame",
            "shape": [int(cell19d_frame_value.shape[0]), int(cell19d_frame_value.shape[1])],
            "columns": cell19d_columns_list,
            "dtypes": cell19d_dtypes_dict,
            "detected_columns": cell19d_detect_candidate_columns(cell19d_columns_list),
            "head_records": cell19d_head_records,
        }

    def cell19d_inspect_csv_file(cell19d_path_value, cell19d_label_value):
        try:
            cell19d_preview_frame = cell19d_pd.read_csv(cell19d_path_value, nrows=20)
        except Exception as cell19d_csv_error:
            return {
                "label": str(cell19d_label_value),
                "type": "csv_gz",
                "error": repr(cell19d_csv_error),
            }

        return cell19d_dataframe_summary(cell19d_preview_frame, str(cell19d_label_value))

    def cell19d_summarize_object(cell19d_object_value, cell19d_name_value, cell19d_depth_value=0):
        if cell19d_depth_value > 3:
            return {
                "name": str(cell19d_name_value),
                "type": str(type(cell19d_object_value)),
                "note": "depth_limit_reached",
            }

        if isinstance(cell19d_object_value, cell19d_pd.DataFrame):
            return cell19d_dataframe_summary(cell19d_object_value, str(cell19d_name_value))

        if isinstance(cell19d_object_value, dict):
            cell19d_child_items = {}

            for cell19d_child_key, cell19d_child_value in cell19d_object_value.items():
                cell19d_child_items[str(cell19d_child_key)] = cell19d_summarize_object(
                    cell19d_child_value,
                    str(cell19d_child_key),
                    cell19d_depth_value + 1,
                )

            return {
                "name": str(cell19d_name_value),
                "type": "dict",
                "keys": [str(cell19d_key_value) for cell19d_key_value in list(cell19d_object_value.keys())[:200]],
                "children": cell19d_child_items,
            }

        if isinstance(cell19d_object_value, (list, tuple)):
            cell19d_sample_items = []

            for cell19d_sample_index in range(min(5, len(cell19d_object_value))):
                cell19d_sample_items.append(
                    cell19d_summarize_object(
                        cell19d_object_value[cell19d_sample_index],
                        str(cell19d_sample_index),
                        cell19d_depth_value + 1,
                    )
                )

            return {
                "name": str(cell19d_name_value),
                "type": str(type(cell19d_object_value)),
                "length": int(len(cell19d_object_value)),
                "samples": cell19d_sample_items,
            }

        return {
            "name": str(cell19d_name_value),
            "type": str(type(cell19d_object_value)),
            "repr": cell19d_truncate_text(cell19d_object_value, 500),
        }

    def cell19d_inspect_pickle_file(cell19d_path_value, cell19d_label_value):
        try:
            with cell19d_gzip.open(cell19d_path_value, "rb") as cell19d_gzip_handle:
                cell19d_loaded_object = cell19d_pickle.load(cell19d_gzip_handle)
        except Exception as cell19d_pickle_error:
            return {
                "label": str(cell19d_label_value),
                "type": "pickle_gz",
                "error": repr(cell19d_pickle_error),
            }

        cell19d_summary_object = cell19d_summarize_object(cell19d_loaded_object, str(cell19d_label_value), 0)
        cell19d_summary_object["label"] = str(cell19d_label_value)
        cell19d_summary_object["type"] = "pickle_gz"

        return cell19d_summary_object

    print("=" * 100)
    print("CELL 19d — CROSS-CONNECTOME SCHEMA PROBE")
    print("=" * 100)
    print("[cfg] bucket=" + cell19d_bucket)
    print("[cfg] root_prefix=" + cell19d_root_prefix)
    print("[cfg] versions=" + str(cell19d_versions))
    print("[cfg] files=" + str(cell19d_files))
    print("[cfg] output_dir=" + str(cell19d_output_dir))
    print("=" * 100)

    cell19d_report = {}

    for cell19d_dataset_value in cell19d_datasets:
        print("\n---- Dataset: " + str(cell19d_dataset_value) + " ----")

        cell19d_dataset_report = {
            "version": str(cell19d_versions.get(cell19d_dataset_value, "unknown")),
            "files": {},
            "inspections": {},
        }

        for cell19d_filename_value in cell19d_files:
            cell19d_local_path, cell19d_download_status = cell19d_download_one_file(
                cell19d_dataset_value,
                cell19d_filename_value,
            )

            cell19d_dataset_report["files"][cell19d_filename_value] = {
                "status": str(cell19d_download_status),
                "path": str(cell19d_local_path) if cell19d_local_path is not None else None,
            }

            print(
                "[cell19d] "
                + str(cell19d_dataset_value)
                + "/"
                + str(cell19d_filename_value)
                + ": "
                + str(cell19d_download_status)
            )

            if cell19d_local_path is None:
                continue

            if str(cell19d_filename_value).endswith(".csv.gz"):
                cell19d_inspection_value = cell19d_inspect_csv_file(
                    cell19d_local_path,
                    str(cell19d_filename_value),
                )

                cell19d_dataset_report["inspections"][cell19d_filename_value] = cell19d_inspection_value

                print("[cell19d] columns=" + str(cell19d_inspection_value.get("columns", [])))

                if "detected_columns" in cell19d_inspection_value:
                    print(
                        "[cell19d] detected="
                        + str(cell19d_inspection_value["detected_columns"])
                    )

            elif str(cell19d_filename_value).endswith(".pickle.gz"):
                cell19d_inspection_value = cell19d_inspect_pickle_file(
                    cell19d_local_path,
                    str(cell19d_filename_value),
                )

                cell19d_dataset_report["inspections"][cell19d_filename_value] = cell19d_inspection_value

                print("[cell19d] pickle_type=" + str(cell19d_inspection_value.get("type", "unknown")))

                if "keys" in cell19d_inspection_value:
                    print("[cell19d] pickle_keys=" + str(cell19d_inspection_value.get("keys", [])))

                if "children" in cell19d_inspection_value:
                    for cell19d_child_key_value, cell19d_child_value in cell19d_inspection_value["children"].items():
                        if cell19d_child_value.get("type") == "DataFrame":
                            print(
                                "[cell19d] child_dataframe="
                                + str(cell19d_child_key_value)
                                + " shape="
                                + str(cell19d_child_value.get("shape", []))
                                + " columns="
                                + str(cell19d_child_value.get("columns", []))
                            )

        cell19d_report[cell19d_dataset_value] = cell19d_dataset_report

    cell19d_report_path = cell19d_output_dir / "cell19d_schema_probe.json"

    with open(cell19d_report_path, "w", encoding="utf-8") as cell19d_report_handle:
        cell19d_report_handle.write(
            cell19d_json.dumps(cell19d_report, indent=2, default=str)
        )

    print("\n" + "=" * 100)
    print("CELL 19d SUMMARY")
    print("=" * 100)

    for cell19d_dataset_value in cell19d_datasets:
        cell19d_dataset_info = cell19d_report.get(cell19d_dataset_value, {})
        print("\n" + str(cell19d_dataset_value) + ":")

        for cell19d_filename_value, cell19d_file_info in cell19d_dataset_info.get("files", {}).items():
            print(
                "  "
                + str(cell19d_filename_value)
                + ": "
                + str(cell19d_file_info.get("status", "missing"))
            )

        cell19d_neurons_info = cell19d_dataset_info.get("inspections", {}).get("neurons.csv.gz", {})
        cell19d_conn_info = cell19d_dataset_info.get("inspections", {}).get("connections_princeton.csv.gz", {})
        cell19d_attr_info = cell19d_dataset_info.get("inspections", {}).get("neuron_attributes.pickle.gz", {})

        if "columns" in cell19d_neurons_info:
            print("  neurons columns: " + str(cell19d_neurons_info.get("columns", [])))

        if "columns" in cell19d_conn_info:
            print("  connections columns: " + str(cell19d_conn_info.get("columns", [])))

        if "keys" in cell19d_attr_info:
            print("  neuron_attributes keys: " + str(cell19d_attr_info.get("keys", [])))

        if "children" in cell19d_attr_info:
            for cell19d_child_key_value, cell19d_child_value in cell19d_attr_info["children"].items():
                if cell19d_child_value.get("type") == "DataFrame":
                    print(
                        "  neuron_attributes DataFrame '"
                        + str(cell19d_child_key_value)
                        + "' columns: "
                        + str(cell19d_child_value.get("columns", []))
                    )

    print("\nSaved report: " + str(cell19d_report_path))
    print("=" * 100)

    print("\nNext step:")
    print("  Send the printed CELL 19d SUMMARY and the JSON file if needed.")
    print("  Then Cell 19 can be rewritten against the actual non-FAFB schema.")
    print("  Do not report cross-connectome results until this schema probe succeeds.")
    print("=" * 100)

    return {
        "output_dir": str(cell19d_output_dir),
        "report_path": str(cell19d_report_path),
        "report": cell19d_report,
    }


cell19d_output = cell19d_main()

cell19d_probe_report = cell19d_output["report"]
cell19d_report_path = cell19d_output["report_path"]
cell19d_output_dir = cell19d_output["output_dir"]

In [ ]:
# Cell 19e: Corrected cross-connectome structural coherence using actual MANC/MAOL/MCNS schema.
#
# Schema discovered by Cell 19d:
#   neurons.csv.gz:
#     Root ID, Predicted NT type, Predicted NT confidence, Verified NT type,
#     Super Class, Class, Sub Class, Hemilineage, Nerve, Flow, Function,
#     Cable length (nm), Surface area (nm^2), Volume (nm^3), ...
#
#   connections_princeton.csv.gz:
#     pre_root_id, post_root_id, neuropil, syn_count, nt_type
#
# This cell:
#   1. Loads the frozen FAFB SSPR model from Cell 17.
#   2. Downloads/reads MANC, MAOL, MCNS using the actual schema.
#   3. Builds a 61-dimensional feature matrix adapted to the target schema.
#   4. Applies the frozen encoder without retraining.
#   5. Clusters the frozen embedding with KMeans.
#   6. Computes ARI/NMI against the target connectome's native taxonomy.
#   7. Compares against a shuffled-label null.
#   8. Records failures honestly.
#
# Rules:
#   - No top-level imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell19e_.


def cell19e_main():
    cell19e_globals = globals()

    # ------------------------------------------------------------------
    # Acquire modules from globals when possible, otherwise import locally.
    # All aliases are prefixed.
    # ------------------------------------------------------------------
    cell19e_np = cell19e_globals.get("np", None)
    if cell19e_np is None:
        import numpy as cell19e_np

    cell19e_torch = cell19e_globals.get("torch", None)
    if cell19e_torch is None:
        import torch as cell19e_torch

    cell19e_pd = cell19e_globals.get("pd", None)
    if cell19e_pd is None:
        import pandas as cell19e_pd

    cell19e_Path = cell19e_globals.get("Path", None)
    if cell19e_Path is None:
        from pathlib import Path as cell19e_Path

    cell19e_gzip = cell19e_globals.get("gzip", None)
    if cell19e_gzip is None:
        import gzip as cell19e_gzip

    cell19e_json = cell19e_globals.get("json", None)
    if cell19e_json is None:
        import json as cell19e_json

    cell19e_pickle = cell19e_globals.get("pickle", None)
    if cell19e_pickle is None:
        import pickle as cell19e_pickle

    cell19e_csr_matrix = cell19e_globals.get("csr_matrix", None)
    if cell19e_csr_matrix is None:
        from scipy.sparse import csr_matrix as cell19e_csr_matrix

    cell19e_TruncatedSVD = cell19e_globals.get("TruncatedSVD", None)
    if cell19e_TruncatedSVD is None:
        from sklearn.decomposition import TruncatedSVD as cell19e_TruncatedSVD

    cell19e_KMeans = cell19e_globals.get("KMeans", None)
    if cell19e_KMeans is None:
        from sklearn.cluster import KMeans as cell19e_KMeans

    cell19e_ari_score = cell19e_globals.get("adjusted_rand_score", None)
    if cell19e_ari_score is None:
        from sklearn.metrics import adjusted_rand_score as cell19e_ari_score

    cell19e_nmi_score = cell19e_globals.get("normalized_mutual_info_score", None)
    if cell19e_nmi_score is None:
        from sklearn.metrics import normalized_mutual_info_score as cell19e_nmi_score

    cell19e_urllib_request = cell19e_globals.get("urllib_request", None)
    if cell19e_urllib_request is None:
        cell19e_urllib_module = cell19e_globals.get("urllib", None)
        if cell19e_urllib_module is not None and hasattr(cell19e_urllib_module, "request"):
            cell19e_urllib_request = cell19e_urllib_module.request
        else:
            import urllib.request as cell19e_urllib_request

    cell19e_urllib_error = cell19e_globals.get("urllib_error", None)
    if cell19e_urllib_error is None:
        cell19e_urllib_module_for_error = cell19e_globals.get("urllib", None)
        if (
            cell19e_urllib_module_for_error is not None
            and hasattr(cell19e_urllib_module_for_error, "error")
        ):
            cell19e_urllib_error = cell19e_urllib_module_for_error.error
        else:
            import urllib.error as cell19e_urllib_error

    # ------------------------------------------------------------------
    # Configuration.
    # ------------------------------------------------------------------
    cell19e_bucket = "flywire-data"
    cell19e_root_prefix = "codex/data"
    cell19e_base_url = "https://storage.googleapis.com/" + cell19e_bucket + "/"

    cell19e_versions = {
        "manc": "1.2.1",
        "maol": "1.1",
        "mcns": "1.0",
    }

    cell19e_datasets = [
        "manc",
        "maol",
        "mcns",
    ]

    cell19e_probe_dir = cell19e_Path("./cross_connectome_probe_cell19d")
    cell19e_output_dir = cell19e_Path("./cross_connectome_data_cell19e")
    cell19e_output_dir.mkdir(parents=True, exist_ok=True)

    cell19e_neuron_filename = "neurons.csv.gz"
    cell19e_connection_filename = "connections_princeton.csv.gz"

    cell19e_motif_exact_max_nodes = 30000
    cell19e_motif_exact_max_edges = 250000

    cell19e_null_permutations = 20
    cell19e_min_labeled_nodes = 50
    cell19e_min_positive_edges = 10
    cell19e_min_negative_edges = 10

    # ------------------------------------------------------------------
    # Small utilities.
    # ------------------------------------------------------------------
    def cell19e_first_global(cell19e_candidate_names):
        for cell19e_candidate_name in cell19e_candidate_names:
            if cell19e_candidate_name in cell19e_globals and cell19e_globals[cell19e_candidate_name] is not None:
                return cell19e_globals[cell19e_candidate_name]

        return None

    def cell19e_to_numpy(cell19e_object):
        if cell19e_object is None:
            return None

        cell19e_working_object = cell19e_object

        try:
            if hasattr(cell19e_working_object, "detach"):
                cell19e_working_object = cell19e_working_object.detach()

            if hasattr(cell19e_working_object, "cpu"):
                cell19e_working_object = cell19e_working_object.cpu()

            if hasattr(cell19e_working_object, "numpy"):
                cell19e_working_object = cell19e_working_object.numpy()

            return cell19e_np.asarray(cell19e_working_object)
        except Exception:
            return None

    def cell19e_is_valid_gzip(cell19e_path_value):
        try:
            with cell19e_gzip.open(cell19e_path_value, "rb") as cell19e_check_handle:
                while cell19e_check_handle.read(1 << 20):
                    pass

            return True
        except Exception:
            return False

    def cell19e_resolve_or_download(cell19e_dataset_name, cell19e_filename_value):
        cell19e_candidate_dirs = [
            cell19e_probe_dir / str(cell19e_dataset_name),
            cell19e_output_dir / str(cell19e_dataset_name),
        ]

        for cell19e_candidate_dir_value in cell19e_candidate_dirs:
            cell19e_candidate_path = cell19e_candidate_dir_value / str(cell19e_filename_value)

            if cell19e_candidate_path.exists() and cell19e_candidate_path.stat().st_size > 1000:
                if cell19e_is_valid_gzip(cell19e_candidate_path):
                    return cell19e_candidate_path, "existing_valid"

                try:
                    cell19e_candidate_path.unlink()
                except Exception:
                    pass

        cell19e_version_value = cell19e_versions.get(str(cell19e_dataset_name), None)

        if cell19e_version_value is None:
            return None, "unknown_dataset_version"

        cell19e_object_key = (
            cell19e_root_prefix
            + "/"
            + str(cell19e_dataset_name)
            + "/"
            + str(cell19e_version_value)
            + "/"
            + str(cell19e_filename_value)
        )

        cell19e_url_value = cell19e_base_url + str(cell19e_object_key)

        cell19e_dest_dir = cell19e_output_dir / str(cell19e_dataset_name)
        cell19e_dest_dir.mkdir(parents=True, exist_ok=True)

        cell19e_dest_path = cell19e_dest_dir / str(cell19e_filename_value)
        cell19e_tmp_path = cell19e_Path(str(cell19e_dest_path) + ".part")

        try:
            cell19e_request_object = cell19e_urllib_request.Request(
                cell19e_url_value,
                headers={"User-Agent": "research"},
            )

            with cell19e_urllib_request.urlopen(
                cell19e_request_object,
                timeout=900,
            ) as cell19e_response_object:
                with open(cell19e_tmp_path, "wb") as cell19e_out_handle:
                    while True:
                        cell19e_chunk_value = cell19e_response_object.read(1 << 20)

                        if not cell19e_chunk_value:
                            break

                        cell19e_out_handle.write(cell19e_chunk_value)

            if not cell19e_is_valid_gzip(cell19e_tmp_path):
                try:
                    cell19e_tmp_path.unlink()
                except Exception:
                    pass

                return None, "downloaded_but_invalid_gzip"

            cell19e_tmp_path.replace(cell19e_dest_path)
            return cell19e_dest_path, "downloaded_valid"

        except Exception as cell19e_download_error:
            try:
                if cell19e_tmp_path.exists():
                    cell19e_tmp_path.unlink()
            except Exception:
                pass

            return None, "download_failed:" + repr(cell19e_download_error)

    # ------------------------------------------------------------------
    # Load neuron table.
    # ------------------------------------------------------------------
    def cell19e_load_neuron_table(cell19e_neuron_path):
        cell19e_frame = cell19e_pd.read_csv(cell19e_neuron_path)

        if "Root ID" not in cell19e_frame.columns:
            return None, None, None, 0, "missing_Root_ID_column"

        cell19e_frame["Root ID"] = cell19e_pd.to_numeric(
            cell19e_frame["Root ID"],
            errors="coerce",
        )

        cell19e_frame = cell19e_frame.dropna(subset=["Root ID"]).copy()
        cell19e_frame = cell19e_frame.drop_duplicates(subset=["Root ID"], keep="first").copy()

        if cell19e_frame.shape[0] == 0:
            return None, None, None, 0, "empty_neuron_table"

        cell19e_ids = cell19e_frame["Root ID"].astype(cell19e_np.int64).to_numpy()
        cell19e_id_map = {
            int(cell19e_root_value): int(cell19e_index_value)
            for cell19e_index_value, cell19e_root_value in enumerate(cell19e_ids)
        }

        return cell19e_frame, cell19e_ids, cell19e_id_map, int(cell19e_ids.size), "ok"

    # ------------------------------------------------------------------
    # Transmitter feature construction.
    # ------------------------------------------------------------------
    cell19e_nt_aliases = {
        "DA": "DA",
        "DOPAMINE": "DA",
        "SER": "SER",
        "SEROTONIN": "SER",
        "GABA": "GABA",
        "GLUT": "GLUT",
        "GLUTAMATE": "GLUT",
        "ACH": "ACH",
        "ACETYLCHOLINE": "ACH",
        "OCT": "OCT",
        "OCTOPAMINE": "OCT",
    }

    cell19e_nt_to_index = {
        "DA": 0,
        "SER": 1,
        "GABA": 2,
        "GLUT": 3,
        "ACH": 4,
        "OCT": 5,
    }

    def cell19e_normalize_nt_value(cell19e_raw_value):
        cell19e_text = str(cell19e_raw_value).strip()

        if cell19e_text == "" or cell19e_text.lower() in {"nan", "none", "null", "unknown", "?"}:
            return None

        cell19e_upper = cell19e_text.upper()
        return cell19e_nt_aliases.get(cell19e_upper, None)

    def cell19e_build_transmitter_features(cell19e_neuron_frame, cell19e_node_count):
        cell19e_features = cell19e_np.zeros((int(cell19e_node_count), 6), dtype=cell19e_np.float64)

        cell19e_pred_type_series = None
        cell19e_pred_conf_series = None
        cell19e_ver_type_series = None

        if "Predicted NT type" in cell19e_neuron_frame.columns:
            cell19e_pred_type_series = cell19e_neuron_frame["Predicted NT type"].astype(str).to_numpy()

        if "Predicted NT confidence" in cell19e_neuron_frame.columns:
            cell19e_pred_conf_series = cell19e_pd.to_numeric(
                cell19e_neuron_frame["Predicted NT confidence"],
                errors="coerce",
            ).to_numpy()

        if "Verified NT type" in cell19e_neuron_frame.columns:
            cell19e_ver_type_series = cell19e_neuron_frame["Verified NT type"].astype(str).to_numpy()

        for cell19e_row_index in range(int(cell19e_node_count)):
            cell19e_pred_raw = None
            cell19e_conf_raw = None
            cell19e_ver_raw = None

            if cell19e_pred_type_series is not None and cell19e_row_index < len(cell19e_pred_type_series):
                cell19e_pred_raw = cell19e_pred_type_series[cell19e_row_index]

            if cell19e_pred_conf_series is not None and cell19e_row_index < len(cell19e_pred_conf_series):
                cell19e_conf_raw = cell19e_pred_conf_series[cell19e_row_index]

            if cell19e_ver_type_series is not None and cell19e_row_index < len(cell19e_ver_type_series):
                cell19e_ver_raw = cell19e_ver_type_series[cell19e_row_index]

            cell19e_pred_canonical = cell19e_normalize_nt_value(cell19e_pred_raw)

            cell19e_conf_value = float("nan")
            try:
                cell19e_conf_value = float(cell19e_conf_raw)
            except Exception:
                cell19e_conf_value = float("nan")

            if (
                cell19e_pred_canonical is not None
                and cell19e_np.isfinite(cell19e_conf_value)
                and cell19e_conf_value > 0.0
            ):
                cell19e_conf_value = max(0.0, min(1.0, cell19e_conf_value))
                cell19e_nt_index = cell19e_nt_to_index[cell19e_pred_canonical]
                cell19e_features[cell19e_row_index, cell19e_nt_index] = cell19e_conf_value
                continue

            cell19e_ver_canonical = cell19e_normalize_nt_value(cell19e_ver_raw)

            if cell19e_ver_canonical is not None:
                cell19e_nt_index = cell19e_nt_to_index[cell19e_ver_canonical]
                cell19e_features[cell19e_row_index, cell19e_nt_index] = 1.0

        return cell19e_features

    # ------------------------------------------------------------------
    # Morphology feature construction.
    # ------------------------------------------------------------------
    def cell19e_build_morphology_features(cell19e_neuron_frame, cell19e_node_count):
        cell19e_features = cell19e_np.zeros((int(cell19e_node_count), 3), dtype=cell19e_np.float64)

        cell19e_column_map = [
            ("Cable length (nm)", 0),
            ("Surface area (nm^2)", 1),
            ("Volume (nm^3)", 2),
        ]

        for cell19e_source_column, cell19e_target_index in cell19e_column_map:
            if cell19e_source_column not in cell19e_neuron_frame.columns:
                continue

            cell19e_raw_values = cell19e_pd.to_numeric(
                cell19e_neuron_frame[cell19e_source_column],
                errors="coerce",
            ).to_numpy()

            cell19e_clean_values = cell19e_np.nan_to_num(cell19e_raw_values, nan=0.0, posinf=0.0, neginf=0.0)
            cell19e_clean_values = cell19e_np.clip(cell19e_clean_values, 0.0, None)
            cell19e_features[:, cell19e_target_index] = cell19e_np.log1p(cell19e_clean_values)

        return cell19e_features

    # ------------------------------------------------------------------
    # Load and process connection table.
    # ------------------------------------------------------------------
    cell19e_sign_map = {
        "ACH": "exc",
        "ACETYLCHOLINE": "exc",
        "GLUT": "exc",
        "GLUTAMATE": "exc",
        "GABA": "inh",
    }

    def cell19e_load_connection_table(cell19e_connection_path, cell19e_id_map, cell19e_node_count):
        cell19e_frame = cell19e_pd.read_csv(cell19e_connection_path)

        cell19e_required_columns = [
            "pre_root_id",
            "post_root_id",
            "syn_count",
            "nt_type",
        ]

        for cell19e_required_column in cell19e_required_columns:
            if cell19e_required_column not in cell19e_frame.columns:
                return None, "missing_column:" + str(cell19e_required_column)

        cell19e_frame["pre_root_id"] = cell19e_pd.to_numeric(
            cell19e_frame["pre_root_id"],
            errors="coerce",
        )
        cell19e_frame["post_root_id"] = cell19e_pd.to_numeric(
            cell19e_frame["post_root_id"],
            errors="coerce",
        )
        cell19e_frame["syn_count"] = cell19e_pd.to_numeric(
            cell19e_frame["syn_count"],
            errors="coerce",
        )

        cell19e_frame = cell19e_frame.dropna(
            subset=["pre_root_id", "post_root_id"],
        ).copy()

        if cell19e_frame.shape[0] == 0:
            return None, "no_numeric_edges"

        cell19e_frame["pre_root_id"] = cell19e_frame["pre_root_id"].astype(cell19e_np.int64)
        cell19e_frame["post_root_id"] = cell19e_frame["post_root_id"].astype(cell19e_np.int64)

        cell19e_frame["src"] = cell19e_frame["pre_root_id"].map(cell19e_id_map)
        cell19e_frame["dst"] = cell19e_frame["post_root_id"].map(cell19e_id_map)

        cell19e_frame = cell19e_frame.dropna(subset=["src", "dst"]).copy()

        if cell19e_frame.shape[0] == 0:
            return None, "no_mapped_edges"

        cell19e_frame["src"] = cell19e_frame["src"].astype(cell19e_np.int64)
        cell19e_frame["dst"] = cell19e_frame["dst"].astype(cell19e_np.int64)

        cell19e_frame["nt_class"] = (
            cell19e_frame["nt_type"]
            .astype(str)
            .str.strip()
            .str.upper()
            .map(cell19e_sign_map)
        )

        cell19e_frame = cell19e_frame[cell19e_frame["nt_class"].isin(["exc", "inh"])].copy()

        if cell19e_frame.shape[0] == 0:
            return None, "no_signed_edges"

        cell19e_frame["syn_count"] = (
            cell19e_pd.to_numeric(cell19e_frame["syn_count"], errors="coerce")
            .fillna(1.0)
            .clip(lower=0.0)
        )

        cell19e_agg = (
            cell19e_frame
            .groupby(["src", "dst", "nt_class"], as_index=False)
            .agg(syn_count=("syn_count", "sum"))
        )

        cell19e_major = (
            cell19e_agg
            .sort_values("syn_count")
            .groupby(["src", "dst"], as_index=False)
            .last()
        )

        cell19e_src = cell19e_major["src"].to_numpy(dtype=cell19e_np.int64)
        cell19e_dst = cell19e_major["dst"].to_numpy(dtype=cell19e_np.int64)
        cell19e_sign_bin = (cell19e_major["nt_class"] == "exc").to_numpy(dtype=cell19e_np.float32)
        cell19e_syn = cell19e_major["syn_count"].to_numpy(dtype=cell19e_np.float32)

        if cell19e_src.size == 0:
            return None, "empty_major_edges"

        return {
            "src": cell19e_src,
            "dst": cell19e_dst,
            "sign_bin": cell19e_sign_bin,
            "syn": cell19e_syn,
        }, "ok"

    # ------------------------------------------------------------------
    # Build 61-dimensional target feature matrix.
    # ------------------------------------------------------------------
    def cell19e_build_target_features(
        cell19e_neuron_frame,
        cell19e_node_count,
        cell19e_connection_dict,
    ):
        cell19e_src = cell19e_connection_dict["src"]
        cell19e_dst = cell19e_connection_dict["dst"]
        cell19e_sign_bin = cell19e_connection_dict["sign_bin"]
        cell19e_syn = cell19e_connection_dict["syn"]

        cell19e_morphology_features = cell19e_build_morphology_features(
            cell19e_neuron_frame,
            cell19e_node_count,
        )

        cell19e_transmitter_features = cell19e_build_transmitter_features(
            cell19e_neuron_frame,
            cell19e_node_count,
        )

        cell19e_deg_in = cell19e_np.zeros(int(cell19e_node_count), dtype=cell19e_np.float64)
        cell19e_deg_out = cell19e_np.zeros(int(cell19e_node_count), dtype=cell19e_np.float64)

        cell19e_syn_float = cell19e_syn.astype(cell19e_np.float64)
        cell19e_np.add.at(cell19e_deg_in, cell19e_dst, cell19e_syn_float)
        cell19e_np.add.at(cell19e_deg_out, cell19e_src, cell19e_syn_float)

        cell19e_degree_features = cell19e_np.stack(
            [
                cell19e_np.log1p(cell19e_deg_in),
                cell19e_np.log1p(cell19e_deg_out),
            ],
            axis=1,
        )

        # Unsigned symmetric adjacency from signed major edges.
        cell19e_row_unsigned = cell19e_np.concatenate([cell19e_src, cell19e_dst])
        cell19e_col_unsigned = cell19e_np.concatenate([cell19e_dst, cell19e_src])
        cell19e_data_unsigned = cell19e_np.ones(cell19e_row_unsigned.size, dtype=cell19e_np.float32)

        cell19e_A_sym = cell19e_csr_matrix(
            (cell19e_data_unsigned, (cell19e_row_unsigned, cell19e_col_unsigned)),
            shape=(int(cell19e_node_count), int(cell19e_node_count)),
        )

        if int(cell19e_node_count) < 17:
            cell19e_spec_unsigned = cell19e_np.zeros((int(cell19e_node_count), 16), dtype=cell19e_np.float64)
        else:
            try:
                cell19e_svd_unsigned = cell19e_TruncatedSVD(n_components=16, random_state=42)
                cell19e_spec_unsigned = cell19e_svd_unsigned.fit_transform(cell19e_A_sym)
                cell19e_spec_unsigned_mean = cell19e_spec_unsigned.mean(axis=0)
                cell19e_spec_unsigned_std = cell19e_spec_unsigned.std(axis=0)
                cell19e_spec_unsigned = (
                    cell19e_spec_unsigned - cell19e_spec_unsigned_mean
                ) / (cell19e_spec_unsigned_std + 1e-8)
                cell19e_spec_unsigned = cell19e_np.nan_to_num(
                    cell19e_spec_unsigned,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )
            except Exception as cell19e_unsigned_svd_error:
                cell19e_spec_unsigned = cell19e_np.zeros(
                    (int(cell19e_node_count), 16),
                    dtype=cell19e_np.float64,
                )

        # Signed adjacency.
        cell19e_pos_mask = cell19e_sign_bin > 0.5
        cell19e_neg_mask = ~cell19e_pos_mask

        cell19e_row_positive = cell19e_src[cell19e_pos_mask]
        cell19e_col_positive = cell19e_dst[cell19e_pos_mask]
        cell19e_row_negative = cell19e_src[cell19e_neg_mask]
        cell19e_col_negative = cell19e_dst[cell19e_neg_mask]

        cell19e_row_signed = cell19e_np.concatenate([cell19e_row_positive, cell19e_row_negative])
        cell19e_col_signed = cell19e_np.concatenate([cell19e_col_positive, cell19e_col_negative])
        cell19e_data_signed = cell19e_np.concatenate(
            [
                cell19e_np.ones(cell19e_row_positive.size, dtype=cell19e_np.float32),
                -cell19e_np.ones(cell19e_row_negative.size, dtype=cell19e_np.float32),
            ]
        )

        cell19e_A_signed = cell19e_csr_matrix(
            (cell19e_data_signed, (cell19e_row_signed, cell19e_col_signed)),
            shape=(int(cell19e_node_count), int(cell19e_node_count)),
        )

        if int(cell19e_node_count) < 33:
            cell19e_spec_signed = cell19e_np.zeros((int(cell19e_node_count), 32), dtype=cell19e_np.float64)
        else:
            try:
                cell19e_svd_signed = cell19e_TruncatedSVD(n_components=32, random_state=42)
                cell19e_spec_signed = cell19e_svd_signed.fit_transform(cell19e_A_signed)
                cell19e_spec_signed_mean = cell19e_spec_signed.mean(axis=0)
                cell19e_spec_signed_std = cell19e_spec_signed.std(axis=0)
                cell19e_spec_signed = (
                    cell19e_spec_signed - cell19e_spec_signed_mean
                ) / (cell19e_spec_signed_std + 1e-8)
                cell19e_spec_signed = cell19e_np.nan_to_num(
                    cell19e_spec_signed,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )
            except Exception as cell19e_signed_svd_error:
                cell19e_spec_signed = cell19e_np.zeros(
                    (int(cell19e_node_count), 32),
                    dtype=cell19e_np.float64,
                )

        # Motif features.
        # Exact balanced/frustrated triangle counts require A^3 diagonals.
        # On million-edge connectomes this can blow up memory.
        # Therefore: exact only for small graphs; otherwise zero-omit and record.
        cell19e_motif_features = cell19e_np.zeros((int(cell19e_node_count), 2), dtype=cell19e_np.float64)
        cell19e_motif_mode = "zero_omitted_for_scale"

        if (
            int(cell19e_node_count) <= int(cell19e_motif_exact_max_nodes)
            and int(cell19e_src.size) <= int(cell19e_motif_exact_max_edges)
        ):
            try:
                cell19e_row_abs = cell19e_np.concatenate([cell19e_src, cell19e_dst])
                cell19e_col_abs = cell19e_np.concatenate([cell19e_dst, cell19e_src])
                cell19e_data_abs = cell19e_np.ones(cell19e_row_abs.size, dtype=cell19e_np.float32)

                cell19e_A_abs = cell19e_csr_matrix(
                    (cell19e_data_abs, (cell19e_row_abs, cell19e_col_abs)),
                    shape=(int(cell19e_node_count), int(cell19e_node_count)),
                )

                cell19e_A3_signed = cell19e_A_signed.dot(cell19e_A_signed).dot(cell19e_A_signed)
                cell19e_A3_abs = cell19e_A_abs.dot(cell19e_A_abs).dot(cell19e_A_abs)

                cell19e_diag_signed = cell19e_np.asarray(cell19e_A3_signed.diagonal()).flatten()
                cell19e_diag_abs = cell19e_np.asarray(cell19e_A3_abs.diagonal()).flatten()

                cell19e_tri_bal = (cell19e_diag_abs + cell19e_diag_signed) / 12.0
                cell19e_tri_fru = (cell19e_diag_abs - cell19e_diag_signed) / 12.0

                cell19e_motif_features = cell19e_np.stack(
                    [
                        cell19e_np.log1p(cell19e_np.clip(cell19e_tri_bal, 0.0, None)),
                        cell19e_np.log1p(cell19e_np.clip(cell19e_tri_fru, 0.0, None)),
                    ],
                    axis=1,
                )

                cell19e_motif_features = cell19e_np.nan_to_num(
                    cell19e_motif_features,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )

                cell19e_motif_mode = "exact_sparse"

            except Exception as cell19e_motif_error:
                cell19e_motif_features = cell19e_np.zeros(
                    (int(cell19e_node_count), 2),
                    dtype=cell19e_np.float64,
                )
                cell19e_motif_mode = "exact_failed_zero:" + repr(cell19e_motif_error)

        cell19e_X_raw = cell19e_np.concatenate(
            [
                cell19e_morphology_features,
                cell19e_transmitter_features,
                cell19e_degree_features,
                cell19e_spec_unsigned,
                cell19e_spec_signed,
                cell19e_motif_features,
            ],
            axis=1,
        )

        if int(cell19e_X_raw.shape[1]) != 61:
            return None, cell19e_motif_mode, "feature_shape_mismatch:" + str(cell19e_X_raw.shape)

        cell19e_X_raw = cell19e_np.nan_to_num(
            cell19e_X_raw,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return cell19e_X_raw, cell19e_motif_mode, "ok"

    # ------------------------------------------------------------------
    # Frozen model helpers.
    # ------------------------------------------------------------------
    def cell19e_collect_tensors(cell19e_output_object, cell19e_tensor_list=None):
        if cell19e_tensor_list is None:
            cell19e_tensor_list = []

        if cell19e_torch.is_tensor(cell19e_output_object):
            if int(cell19e_output_object.ndim) >= 2:
                cell19e_tensor_list.append(cell19e_output_object)
            return cell19e_tensor_list

        if isinstance(cell19e_output_object, dict):
            for cell19e_dict_value in cell19e_output_object.values():
                cell19e_collect_tensors(cell19e_dict_value, cell19e_tensor_list)
            return cell19e_tensor_list

        if isinstance(cell19e_output_object, (tuple, list)):
            for cell19e_sequence_value in cell19e_output_object:
                cell19e_collect_tensors(cell19e_sequence_value, cell19e_tensor_list)
            return cell19e_tensor_list

        return cell19e_tensor_list

    def cell19e_extract_embedding(cell19e_output_object, cell19e_model_object, cell19e_expected_rows):
        cell19e_candidate_tensors = cell19e_collect_tensors(cell19e_output_object)

        if len(cell19e_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 19e][FATAL] Frozen model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell19e_output_object))
            )

        cell19e_proto_parameter = getattr(cell19e_model_object, "proto_class", None)

        if cell19e_torch.is_tensor(cell19e_proto_parameter) and int(cell19e_proto_parameter.ndim) == 2:
            cell19e_proto_dim = int(cell19e_proto_parameter.shape[1])

            for cell19e_candidate_tensor in cell19e_candidate_tensors:
                if int(cell19e_candidate_tensor.shape[1]) == cell19e_proto_dim:
                    return cell19e_candidate_tensor

        for cell19e_candidate_tensor in cell19e_candidate_tensors:
            if int(cell19e_candidate_tensor.shape[0]) == int(cell19e_expected_rows):
                return cell19e_candidate_tensor

        return cell19e_candidate_tensors[0]

    def cell19e_make_model_from_checkpoint(cell19e_checkpoint):
        cell19e_d_features = int(cell19e_checkpoint.get("d_features", 61))
        cell19e_d_hidden = int(cell19e_checkpoint.get("d_cell5", 64))
        cell19e_d_proto = int(cell19e_checkpoint.get("d_proto_cell5", 32))
        cell19e_n_layers = int(cell19e_checkpoint.get("n_layers_cell5", 2))
        cell19e_n_classes = int(cell19e_checkpoint.get("n_classes_cell5", 29))
        cell19e_n_super = int(cell19e_checkpoint.get("n_super_cell5", 10))
        cell19e_temperature = float(cell19e_checkpoint.get("temperature_cell5", 0.1))
        cell19e_dropout = float(cell19e_checkpoint.get("dropout_cell5", 0.3))

        try:
            return cell19e_globals["SPGCN_cell5"](
                d_features_cell5=cell19e_d_features,
                d_cell5=cell19e_d_hidden,
                d_proto_cell5=cell19e_d_proto,
                n_layers_cell5=cell19e_n_layers,
                n_classes_cell5=cell19e_n_classes,
                n_super_cell5=cell19e_n_super,
                temperature_cell5=cell19e_temperature,
                dropout_cell5=cell19e_dropout,
            )
        except TypeError as cell19e_init_error:
            try:
                return cell19e_globals["SPGCN_cell5"](
                    d_features=cell19e_d_features,
                    d=cell19e_d_hidden,
                    d_proto=cell19e_d_proto,
                    n_layers=cell19e_n_layers,
                    n_classes=cell19e_n_classes,
                    n_super=cell19e_n_super,
                    temperature=cell19e_temperature,
                    dropout=cell19e_dropout,
                )
            except TypeError:
                raise TypeError(
                    "[Cell 19e][FATAL] Could not instantiate SPGCN_cell5 from checkpoint. "
                    "First TypeError: " + repr(cell19e_init_error)
                )

    def cell19e_forward_frozen_model(
        cell19e_model_object,
        cell19e_x_tensor,
        cell19e_ei_pos_tensor,
        cell19e_ew_pos_tensor,
        cell19e_dis_pos_tensor,
        cell19e_ei_neg_tensor,
        cell19e_ew_neg_tensor,
        cell19e_dis_neg_tensor,
    ):
        cell19e_last_type_error = None

        try:
            return cell19e_model_object(
                cell19e_x_tensor,
                cell19e_ei_pos_tensor,
                cell19e_ew_pos_tensor,
                cell19e_dis_pos_tensor,
                cell19e_ei_neg_tensor,
                cell19e_ew_neg_tensor,
                cell19e_dis_neg_tensor,
            )
        except TypeError as cell19e_type_error:
            cell19e_last_type_error = cell19e_type_error

        try:
            return cell19e_model_object(
                cell19e_x_tensor,
                cell19e_ei_pos_tensor,
                cell19e_dis_pos_tensor,
                cell19e_ei_neg_tensor,
                cell19e_dis_neg_tensor,
            )
        except TypeError as cell19e_type_error:
            cell19e_last_type_error = cell19e_type_error

        try:
            return cell19e_model_object(
                cell19e_x_tensor,
                cell19e_ei_pos_tensor,
                cell19e_ei_neg_tensor,
            )
        except TypeError as cell19e_type_error:
            cell19e_last_type_error = cell19e_type_error

        raise TypeError(
            "[Cell 19e][FATAL] Could not call frozen model with any known signature. "
            "Last TypeError: " + repr(cell19e_last_type_error)
        )

    def cell19e_normalize_edge_degree(cell19e_ei_tensor, cell19e_ew_tensor, cell19e_node_count, cell19e_target_device):
        if int(cell19e_ei_tensor.numel()) == 0:
            return cell19e_torch.ones(int(cell19e_node_count), device=cell19e_target_device)

        cell19e_degree_vector = cell19e_torch.zeros(int(cell19e_node_count), device=cell19e_target_device)
        cell19e_degree_vector.index_add_(0, cell19e_ei_tensor[0], cell19e_ew_tensor)
        cell19e_degree_vector.index_add_(0, cell19e_ei_tensor[1], cell19e_ew_tensor)

        return cell19e_degree_vector.clamp(min=1.0).rsqrt()

    # ------------------------------------------------------------------
    # Label selection.
    # ------------------------------------------------------------------
    cell19e_label_priority = [
        "Super Class",
        "Class",
        "Sub Class",
        "Primary Cell Type",
        "Function",
        "Flow",
        "Hemilineage",
        "Nerve",
        "Body Part",
    ]

    def cell19e_choose_label_column(cell19e_neuron_frame):
        for cell19e_candidate_column in cell19e_label_priority:
            if cell19e_candidate_column not in cell19e_neuron_frame.columns:
                continue

            cell19e_series = cell19e_neuron_frame[cell19e_candidate_column].astype(str).str.strip()

            cell19e_invalid_values = {
                "",
                "nan",
                "NaN",
                "None",
                "none",
                "null",
                "NULL",
                "unknown",
                "Unknown",
                "?",
            }

            cell19e_valid_mask = ~cell19e_series.isin(list(cell19e_invalid_values))
            cell19e_valid_count = int(cell19e_np.sum(cell19e_valid_mask.to_numpy()))

            if cell19e_valid_count < int(cell19e_min_labeled_nodes):
                continue

            cell19e_unique_values = sorted(
                set(
                    str(cell19e_label_value).strip()
                    for cell19e_label_value in cell19e_series[cell19e_valid_mask].to_numpy()
                )
            )

            if len(cell19e_unique_values) < 2:
                continue

            return cell19e_candidate_column, cell19e_unique_values, cell19e_valid_count

        return None, [], 0

    # ------------------------------------------------------------------
    # Resolve frozen model path.
    # ------------------------------------------------------------------
    cell19e_frozen_path_value = cell19e_first_global(
        [
            "cell17_frozen_path",
            "FROZEN_PATH_CELL17",
            "cell16_frozen_path",
            "FROZEN_PATH_CELL16",
        ]
    )

    if cell19e_frozen_path_value is None:
        cell19e_output_candidate = cell19e_first_global(["cell17_output"])

        if isinstance(cell19e_output_candidate, dict):
            cell19e_frozen_path_value = cell19e_output_candidate.get("frozen_path", None)

    if cell19e_frozen_path_value is None:
        cell19e_frozen_path_value = "./sspr_fafb_frozen_cell17.pt"

    cell19e_frozen_path = cell19e_Path(str(cell19e_frozen_path_value))

    if not cell19e_frozen_path.exists():
        cell19e_alternative_paths = [
            cell19e_Path("./sspr_fafb_frozen_cell17.pt"),
            cell19e_Path("./sspr_fafb_frozen_cell16.pt"),
            cell19e_Path("./sspr_fafb_frozen.pt"),
        ]

        cell19e_found_alternative = None

        for cell19e_alternative_path in cell19e_alternative_paths:
            if cell19e_alternative_path.exists():
                cell19e_found_alternative = cell19e_alternative_path
                break

        if cell19e_found_alternative is None:
            raise FileNotFoundError(
                "[Cell 19e][FATAL] Frozen FAFB SSPR file does not exist: "
                + str(cell19e_frozen_path)
                + ". Run Cell 17 first."
            )

        cell19e_frozen_path = cell19e_found_alternative

    if "SPGCN_cell5" not in cell19e_globals:
        raise NameError("[Cell 19e][FATAL] SPGCN_cell5 is not available. Run Cell 5 first.")

    if "DEVICE_cell1" not in cell19e_globals:
        raise NameError("[Cell 19e][FATAL] DEVICE_cell1 is not available. Run Cell 1 first.")

    cell19e_device = cell19e_globals["DEVICE_cell1"]

    print("=" * 100)
    print("CELL 19e — CORRECTED CROSS-CONNECTOME STRUCTURAL COHERENCE")
    print("=" * 100)
    print("[cfg] frozen_path=" + str(cell19e_frozen_path))
    print("[cfg] datasets=" + str(cell19e_datasets))
    print("[cfg] versions=" + str(cell19e_versions))
    print("[cfg] probe_dir=" + str(cell19e_probe_dir))
    print("[cfg] output_dir=" + str(cell19e_output_dir))
    print("[schema] using neurons.csv.gz + connections_princeton.csv.gz")
    print("[adaptation] transmitter channels are one-hot predicted/verified NT confidence")
    print("[adaptation] motif channels are zero-omitted for large graphs to avoid A^3 memory blow-up")
    print("=" * 100)

    # ------------------------------------------------------------------
    # Load frozen checkpoint and model.
    # ------------------------------------------------------------------
    try:
        cell19e_checkpoint = cell19e_torch.load(
            cell19e_frozen_path,
            map_location=cell19e_device,
            weights_only=False,
        )
    except TypeError:
        cell19e_checkpoint = cell19e_torch.load(
            cell19e_frozen_path,
            map_location=cell19e_device,
        )

    cell19e_feature_mean_raw = cell19e_checkpoint.get("feature_mean", None)
    cell19e_feature_std_raw = cell19e_checkpoint.get("feature_std", None)

    if cell19e_feature_mean_raw is None or cell19e_feature_std_raw is None:
        raise KeyError("[Cell 19e][FATAL] Frozen checkpoint missing feature_mean or feature_std.")

    cell19e_feature_mean = cell19e_torch.as_tensor(
        cell19e_feature_mean_raw,
        dtype=cell19e_torch.float32,
    ).to(cell19e_device)

    cell19e_feature_std = cell19e_torch.as_tensor(
        cell19e_feature_std_raw,
        dtype=cell19e_torch.float32,
    ).to(cell19e_device)

    cell19e_ckpt_mean_np = cell19e_to_numpy(cell19e_feature_mean)
    cell19e_ckpt_std_np = cell19e_to_numpy(cell19e_feature_std)

    if cell19e_ckpt_mean_np is None or cell19e_ckpt_std_np is None:
        raise TypeError("[Cell 19e][FATAL] Could not convert checkpoint feature mean/std to numpy.")

    cell19e_ckpt_mean_np = cell19e_np.asarray(cell19e_ckpt_mean_np, dtype=cell19e_np.float64).reshape(-1)
    cell19e_ckpt_std_np = cell19e_np.asarray(cell19e_ckpt_std_np, dtype=cell19e_np.float64).reshape(-1)

    cell19e_frozen_model = cell19e_make_model_from_checkpoint(cell19e_checkpoint)

    if "encoder_state" not in cell19e_checkpoint:
        raise KeyError("[Cell 19e][FATAL] Frozen checkpoint missing encoder_state.")

    try:
        cell19e_frozen_model.load_state_dict(cell19e_checkpoint["encoder_state"])
    except RuntimeError:
        cell19e_frozen_model.load_state_dict(cell19e_checkpoint["encoder_state"], strict=False)

    cell19e_frozen_model = cell19e_frozen_model.to(cell19e_device)
    cell19e_frozen_model.eval()

    cell19e_param_count = int(sum(int(parameter.numel()) for parameter in cell19e_frozen_model.parameters()))
    print("[Cell 19e] Loaded frozen SSPR params=" + f"{cell19e_param_count:,}")

    # ------------------------------------------------------------------
    # Main dataset loop.
    # ------------------------------------------------------------------
    cell19e_results_dict = {}
    cell19e_summary_rows = []

    for cell19e_dataset_name in cell19e_datasets:
        print("\n---- Dataset: " + str(cell19e_dataset_name) + " ----")

        cell19e_dataset_row = {
            "dataset": str(cell19e_dataset_name),
            "version": str(cell19e_versions.get(cell19e_dataset_name, "unknown")),
            "status": "pending",
            "N": "",
            "n_edges_signed": "",
            "n_positive_edges": "",
            "n_negative_edges": "",
            "label_column": "",
            "K": "",
            "n_labeled": "",
            "normalization_mode": "",
            "motif_mode": "",
            "ari": "",
            "ari_null": "",
            "ari_delta": "",
            "ari_p_empirical": "",
            "nmi": "",
            "nmi_null": "",
            "nmi_delta": "",
            "nmi_p_empirical": "",
            "error": "",
        }

        try:
            cell19e_neuron_path, cell19e_neuron_status = cell19e_resolve_or_download(
                cell19e_dataset_name,
                cell19e_neuron_filename,
            )

            cell19e_connection_path, cell19e_connection_status = cell19e_resolve_or_download(
                cell19e_dataset_name,
                cell19e_connection_filename,
            )

            print("[cell19e] neurons: " + str(cell19e_neuron_status))
            print("[cell19e] connections: " + str(cell19e_connection_status))

            if cell19e_neuron_path is None or cell19e_connection_path is None:
                cell19e_dataset_row["status"] = "download_failed"
                cell19e_dataset_row["error"] = (
                    "neurons=" + str(cell19e_neuron_status)
                    + "; connections=" + str(cell19e_connection_status)
                )
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            cell19e_neuron_frame, cell19e_ids, cell19e_id_map, cell19e_N, cell19e_neuron_load_status = (
                cell19e_load_neuron_table(cell19e_neuron_path)
            )

            if cell19e_neuron_frame is None:
                cell19e_dataset_row["status"] = "neuron_load_failed"
                cell19e_dataset_row["error"] = str(cell19e_neuron_load_status)
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            cell19e_connection_dict, cell19e_connection_load_status = cell19e_load_connection_table(
                cell19e_connection_path,
                cell19e_id_map,
                cell19e_N,
            )

            if cell19e_connection_dict is None:
                cell19e_dataset_row["status"] = "connection_load_failed"
                cell19e_dataset_row["error"] = str(cell19e_connection_load_status)
                cell19e_dataset_row["N"] = int(cell19e_N)
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            cell19e_src = cell19e_connection_dict["src"]
            cell19e_dst = cell19e_connection_dict["dst"]
            cell19e_sign_bin = cell19e_connection_dict["sign_bin"]
            cell19e_syn = cell19e_connection_dict["syn"]

            cell19e_pos_count = int(cell19e_np.sum(cell19e_sign_bin > 0.5))
            cell19e_neg_count = int(cell19e_np.sum(cell19e_sign_bin <= 0.5))

            cell19e_dataset_row["N"] = int(cell19e_N)
            cell19e_dataset_row["n_edges_signed"] = int(cell19e_src.size)
            cell19e_dataset_row["n_positive_edges"] = cell19e_pos_count
            cell19e_dataset_row["n_negative_edges"] = cell19e_neg_count

            print(
                "[cell19e] N=" + f"{cell19e_N:,}"
                + " signed_edges=" + f"{cell19e_src.size:,}"
                + " pos=" + f"{cell19e_pos_count:,}"
                + " neg=" + f"{cell19e_neg_count:,}"
            )

            if (
                cell19e_pos_count < int(cell19e_min_positive_edges)
                or cell19e_neg_count < int(cell19e_min_negative_edges)
            ):
                cell19e_dataset_row["status"] = "insufficient_sign_edges"
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            cell19e_X_raw, cell19e_motif_mode, cell19e_feature_status = cell19e_build_target_features(
                cell19e_neuron_frame,
                cell19e_N,
                cell19e_connection_dict,
            )

            cell19e_dataset_row["motif_mode"] = str(cell19e_motif_mode)

            if cell19e_X_raw is None:
                cell19e_dataset_row["status"] = "feature_build_failed"
                cell19e_dataset_row["error"] = str(cell19e_feature_status)
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            # ------------------------------------------------------------------
            # Normalization policy.
            #
            # If the checkpoint mean/std look like final standardized statistics,
            # standardize the target raw features using target statistics.
            # Otherwise use checkpoint statistics.
            # ------------------------------------------------------------------
            cell19e_ckpt_standardized_flag = False

            if (
                int(cell19e_ckpt_mean_np.size) == int(cell19e_X_raw.shape[1])
                and int(cell19e_ckpt_std_np.size) == int(cell19e_X_raw.shape[1])
            ):
                cell19e_ckpt_standardized_flag = bool(
                    cell19e_np.all(cell19e_np.abs(cell19e_ckpt_mean_np) < 0.10)
                    and cell19e_np.all(cell19e_np.abs(cell19e_ckpt_std_np - 1.0) < 0.25)
                )

            if cell19e_ckpt_standardized_flag:
                cell19e_target_mean = cell19e_X_raw.mean(axis=0)
                cell19e_target_std = cell19e_X_raw.std(axis=0)
                cell19e_X_model_np = (cell19e_X_raw - cell19e_target_mean) / (cell19e_target_std + 1e-8)
                cell19e_normalization_mode = "target_zscore_because_checkpoint_standardized"
            elif (
                int(cell19e_ckpt_mean_np.size) == int(cell19e_X_raw.shape[1])
                and int(cell19e_ckpt_std_np.size) == int(cell19e_X_raw.shape[1])
            ):
                cell19e_X_model_np = (
                    cell19e_X_raw - cell19e_ckpt_mean_np
                ) / (cell19e_ckpt_std_np + 1e-8)
                cell19e_normalization_mode = "checkpoint_zscore"
            else:
                cell19e_target_mean = cell19e_X_raw.mean(axis=0)
                cell19e_target_std = cell19e_X_raw.std(axis=0)
                cell19e_X_model_np = (cell19e_X_raw - cell19e_target_mean) / (cell19e_target_std + 1e-8)
                cell19e_normalization_mode = "target_zscore_because_checkpoint_dimension_mismatch"

            cell19e_X_model_np = cell19e_np.nan_to_num(
                cell19e_X_model_np,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            cell19e_dataset_row["normalization_mode"] = str(cell19e_normalization_mode)

            # ------------------------------------------------------------------
            # Build target edge tensors.
            # ------------------------------------------------------------------
            cell19e_X_model_tensor = cell19e_torch.tensor(
                cell19e_X_model_np,
                dtype=cell19e_torch.float32,
                device=cell19e_device,
            )

            cell19e_ei_full_new = cell19e_torch.tensor(
                cell19e_np.vstack([cell19e_src, cell19e_dst]),
                dtype=cell19e_torch.long,
                device=cell19e_device,
            )

            cell19e_syn_log_np = cell19e_np.log1p(cell19e_syn.astype(cell19e_np.float64))
            cell19e_syn_mean_value = float(cell19e_syn_log_np.mean()) if cell19e_syn_log_np.size > 0 else 1.0
            cell19e_edge_weight_np = (cell19e_syn_log_np / (cell19e_syn_mean_value + 1e-8)).astype(
                cell19e_np.float32
            )

            cell19e_edge_weight_tensor = cell19e_torch.tensor(
                cell19e_edge_weight_np,
                dtype=cell19e_torch.float32,
                device=cell19e_device,
            )

            cell19e_pos_mask_array = cell19e_sign_bin > 0.5
            cell19e_pos_mask_tensor = cell19e_torch.tensor(cell19e_pos_mask_array, device=cell19e_device)
            cell19e_neg_mask_tensor = ~cell19e_pos_mask_tensor

            cell19e_ei_pos_new = cell19e_ei_full_new[:, cell19e_pos_mask_tensor]
            cell19e_ew_pos_new = cell19e_edge_weight_tensor[cell19e_pos_mask_tensor]
            cell19e_ei_neg_new = cell19e_ei_full_new[:, cell19e_neg_mask_tensor]
            cell19e_ew_neg_new = cell19e_edge_weight_tensor[cell19e_neg_mask_tensor]

            cell19e_dis_pos_new = cell19e_normalize_edge_degree(
                cell19e_ei_pos_new,
                cell19e_ew_pos_new,
                cell19e_N,
                cell19e_device,
            )
            cell19e_dis_neg_new = cell19e_normalize_edge_degree(
                cell19e_ei_neg_new,
                cell19e_ew_neg_new,
                cell19e_N,
                cell19e_device,
            )

            # ------------------------------------------------------------------
            # Frozen forward pass.
            # ------------------------------------------------------------------
            try:
                with cell19e_torch.no_grad():
                    cell19e_model_output = cell19e_forward_frozen_model(
                        cell19e_frozen_model,
                        cell19e_X_model_tensor,
                        cell19e_ei_pos_new,
                        cell19e_ew_pos_new,
                        cell19e_dis_pos_new,
                        cell19e_ei_neg_new,
                        cell19e_ew_neg_new,
                        cell19e_dis_neg_new,
                    )

                    cell19e_embedding_tensor = cell19e_extract_embedding(
                        cell19e_model_output,
                        cell19e_frozen_model,
                        cell19e_N,
                    )

                    cell19e_embedding_np = cell19e_embedding_tensor.detach().cpu().numpy()
                    cell19e_embedding_np = cell19e_np.nan_to_num(
                        cell19e_embedding_np,
                        nan=0.0,
                        posinf=0.0,
                        neginf=0.0,
                    )

            except Exception as cell19e_forward_error:
                cell19e_dataset_row["status"] = "forward_failed"
                cell19e_dataset_row["error"] = repr(cell19e_forward_error)
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                print("[cell19e][fail] forward_failed: " + repr(cell19e_forward_error))
                continue

            # ------------------------------------------------------------------
            # Label selection and clustering.
            # ------------------------------------------------------------------
            cell19e_label_column, cell19e_unique_labels, cell19e_valid_count_preview = (
                cell19e_choose_label_column(cell19e_neuron_frame)
            )

            cell19e_dataset_row["label_column"] = str(cell19e_label_column) if cell19e_label_column is not None else ""

            if cell19e_label_column is None:
                cell19e_dataset_row["status"] = "too_few_labels"
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            cell19e_label_series = cell19e_neuron_frame[cell19e_label_column].astype(str).str.strip()
            cell19e_label_strings = cell19e_label_series.to_numpy()

            cell19e_label_to_index = {
                cell19e_label_string: cell19e_label_index
                for cell19e_label_index, cell19e_label_string in enumerate(cell19e_unique_labels)
            }

            cell19e_y_all = cell19e_np.array(
                [
                    cell19e_label_to_index.get(str(cell19e_label_value).strip(), -1)
                    for cell19e_label_value in cell19e_label_strings
                ],
                dtype=cell19e_np.int64,
            )

            cell19e_valid_mask = cell19e_y_all >= 0
            cell19e_n_labeled = int(cell19e_np.sum(cell19e_valid_mask))

            cell19e_dataset_row["n_labeled"] = cell19e_n_labeled
            cell19e_dataset_row["K"] = len(cell19e_unique_labels)

            if cell19e_n_labeled < int(cell19e_min_labeled_nodes):
                cell19e_dataset_row["status"] = "too_few_labeled"
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            cell19e_K = len(cell19e_unique_labels)

            if cell19e_K < 2:
                cell19e_dataset_row["status"] = "too_few_classes"
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            if cell19e_K > int(cell19e_N):
                cell19e_dataset_row["status"] = "too_many_classes_for_nodes"
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                continue

            try:
                cell19e_kmeans_object = cell19e_KMeans(
                    n_clusters=cell19e_K,
                    random_state=0,
                    n_init=10,
                )
                cell19e_kmeans_object.fit(cell19e_embedding_np)
                cell19e_pred_all = cell19e_kmeans_object.labels_
            except Exception as cell19e_cluster_error:
                cell19e_dataset_row["status"] = "cluster_failed"
                cell19e_dataset_row["error"] = repr(cell19e_cluster_error)
                cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
                cell19e_summary_rows.append(cell19e_dataset_row)
                print("[cell19e][fail] cluster_failed: " + repr(cell19e_cluster_error))
                continue

            cell19e_y_true = cell19e_y_all[cell19e_valid_mask]
            cell19e_y_pred = cell19e_pred_all[cell19e_valid_mask]

            cell19e_ari = float(cell19e_ari_score(cell19e_y_true, cell19e_y_pred))
            cell19e_nmi = float(cell19e_nmi_score(cell19e_y_true, cell19e_y_pred))

            cell19e_rng = cell19e_np.random.default_rng(0)
            cell19e_null_aris = []
            cell19e_null_nmis = []

            for cell19e_null_index in range(int(cell19e_null_permutations)):
                cell19e_y_perm = cell19e_y_true.copy()
                cell19e_rng.shuffle(cell19e_y_perm)
                cell19e_null_aris.append(float(cell19e_ari_score(cell19e_y_perm, cell19e_y_pred)))
                cell19e_null_nmis.append(float(cell19e_nmi_score(cell19e_y_perm, cell19e_y_pred)))

            cell19e_ari_null = float(cell19e_np.mean(cell19e_null_aris))
            cell19e_nmi_null = float(cell19e_np.mean(cell19e_null_nmis))
            cell19e_ari_delta = float(cell19e_ari - cell19e_ari_null)
            cell19e_nmi_delta = float(cell19e_nmi - cell19e_nmi_null)

            cell19e_ari_p_empirical = float(
                (
                    sum(
                        1
                        for cell19e_null_ari_value in cell19e_null_aris
                        if cell19e_null_ari_value >= cell19e_ari
                    )
                    + 1
                )
                / float(len(cell19e_null_aris) + 1)
            )

            cell19e_nmi_p_empirical = float(
                (
                    sum(
                        1
                        for cell19e_null_nmi_value in cell19e_null_nmis
                        if cell19e_null_nmi_value >= cell19e_nmi
                    )
                    + 1
                )
                / float(len(cell19e_null_nmis) + 1)
            )

            cell19e_dataset_row["status"] = "ok"
            cell19e_dataset_row["ari"] = float(cell19e_ari)
            cell19e_dataset_row["ari_null"] = float(cell19e_ari_null)
            cell19e_dataset_row["ari_delta"] = float(cell19e_ari_delta)
            cell19e_dataset_row["ari_p_empirical"] = float(cell19e_ari_p_empirical)
            cell19e_dataset_row["nmi"] = float(cell19e_nmi)
            cell19e_dataset_row["nmi_null"] = float(cell19e_nmi_null)
            cell19e_dataset_row["nmi_delta"] = float(cell19e_nmi_delta)
            cell19e_dataset_row["nmi_p_empirical"] = float(cell19e_nmi_p_empirical)

            cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
            cell19e_summary_rows.append(cell19e_dataset_row)

            print(
                "[cell19e][ok] label_column=" + str(cell19e_label_column)
                + " K=" + str(cell19e_K)
                + " labeled=" + f"{cell19e_n_labeled:,}"
            )
            print(
                "[cell19e][ok] ARI=" + f"{cell19e_ari:.4f}"
                + " null=" + f"{cell19e_ari_null:.4f}"
                + " delta=" + f"{cell19e_ari_delta:+.4f}"
                + " p_emp=" + f"{cell19e_ari_p_empirical:.3f}"
            )
            print(
                "[cell19e][ok] NMI=" + f"{cell19e_nmi:.4f}"
                + " null=" + f"{cell19e_nmi_null:.4f}"
                + " delta=" + f"{cell19e_nmi_delta:+.4f}"
                + " p_emp=" + f"{cell19e_nmi_p_empirical:.3f}"
            )

        except Exception as cell19e_dataset_error:
            cell19e_dataset_row["status"] = "exception"
            cell19e_dataset_row["error"] = repr(cell19e_dataset_error)
            cell19e_results_dict[cell19e_dataset_name] = cell19e_dataset_row.copy()
            cell19e_summary_rows.append(cell19e_dataset_row)
            print("[cell19e][fail] exception: " + repr(cell19e_dataset_error))

    # ------------------------------------------------------------------
    # Print final summary.
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("CELL 19e — CORRECTED CROSS-CONNECTOME SUMMARY")
    print("=" * 100)

    print(
        f"{'dataset':>8s} {'status':>22s} {'N':>10s} {'edges':>10s} "
        f"{'label':>16s} {'K':>4s} {'labeled':>10s} "
        f"{'ARI':>8s} {'dARI':>8s} {'p':>6s} "
        f"{'NMI':>8s} {'dNMI':>8s} {'p':>6s}"
    )

    for cell19e_dataset_name in cell19e_datasets:
        cell19e_row_dict = cell19e_results_dict.get(cell19e_dataset_name, {"status": "missing"})

        if cell19e_row_dict.get("status") == "ok":
            print(
                f"{str(cell19e_row_dict.get('dataset', '')):>8s} "
                f"{str(cell19e_row_dict.get('status', '')):>22s} "
                f"{int(cell19e_row_dict.get('N', 0)):>10,d} "
                f"{int(cell19e_row_dict.get('n_edges_signed', 0)):>10,d} "
                f"{str(cell19e_row_dict.get('label_column', ''))[:16]:>16s} "
                f"{int(cell19e_row_dict.get('K', 0)):>4d} "
                f"{int(cell19e_row_dict.get('n_labeled', 0)):>10,d} "
                f"{float(cell19e_row_dict.get('ari', float('nan'))):>8.4f} "
                f"{float(cell19e_row_dict.get('ari_delta', float('nan'))):>+8.4f} "
                f"{float(cell19e_row_dict.get('ari_p_empirical', float('nan'))):>6.3f} "
                f"{float(cell19e_row_dict.get('nmi', float('nan'))):>8.4f} "
                f"{float(cell19e_row_dict.get('nmi_delta', float('nan'))):>+8.4f} "
                f"{float(cell19e_row_dict.get('nmi_p_empirical', float('nan'))):>6.3f}"
            )
        else:
            print(
                f"{str(cell19e_row_dict.get('dataset', cell19e_dataset_name)):>8s} "
                f"{str(cell19e_row_dict.get('status', 'missing')):>22s} "
                f"{str(cell19e_row_dict.get('N', '')):>10s} "
                f"{str(cell19e_row_dict.get('n_edges_signed', '')):>10s} "
                f"{str(cell19e_row_dict.get('label_column', ''))[:16]:>16s} "
                f"{str(cell19e_row_dict.get('K', '')):>4s} "
                f"{str(cell19e_row_dict.get('n_labeled', '')):>10s} "
                f"{'':>8s} {'':>8s} {'':>6s} "
                f"{'':>8s} {'':>8s} {'':>6s}"
            )

    print("=" * 100)

    # ------------------------------------------------------------------
    # Save artifacts.
    # ------------------------------------------------------------------
    cell19e_summary_columns = [
        "dataset",
        "version",
        "status",
        "N",
        "n_edges_signed",
        "n_positive_edges",
        "n_negative_edges",
        "label_column",
        "K",
        "n_labeled",
        "normalization_mode",
        "motif_mode",
        "ari",
        "ari_null",
        "ari_delta",
        "ari_p_empirical",
        "nmi",
        "nmi_null",
        "nmi_delta",
        "nmi_p_empirical",
        "error",
    ]

    cell19e_summary_df = cell19e_pd.DataFrame(cell19e_summary_rows, columns=cell19e_summary_columns)

    cell19e_summary_csv_path = cell19e_output_dir / "cell19e_cross_connectome_summary.csv"
    cell19e_results_json_path = cell19e_output_dir / "cell19e_cross_connectome_results.json"
    cell19e_results_pickle_path = cell19e_output_dir / "cell19e_cross_connectome_results.pkl"

    cell19e_summary_df.to_csv(cell19e_summary_csv_path, index=False)

    cell19e_results_json_path.write_text(
        cell19e_json.dumps(cell19e_results_dict, indent=2, default=str),
        encoding="utf-8",
    )

    with open(cell19e_results_pickle_path, "wb") as cell19e_pickle_handle:
        cell19e_pickle.dump(cell19e_results_dict, cell19e_pickle_handle)

    print("Saved summary CSV: " + str(cell19e_summary_csv_path))
    print("Saved results JSON: " + str(cell19e_results_json_path))
    print("Saved results PKL:  " + str(cell19e_results_pickle_path))

    # ------------------------------------------------------------------
    # Optional Hugging Face checkpoint/upload.
    # ------------------------------------------------------------------
    cell19e_hf_save_checkpoint = cell19e_globals.get("hf_save_checkpoint", None)

    if callable(cell19e_hf_save_checkpoint):
        try:
            cell19e_hf_save_checkpoint(
                "cell19e_cross_connectome_corrected",
                {
                    "results": cell19e_results_dict,
                    "summary_rows": cell19e_summary_rows,
                    "frozen_path": str(cell19e_frozen_path),
                    "schema_note": (
                        "Uses neurons.csv.gz + connections_princeton.csv.gz. "
                        "Transmitter channels are one-hot predicted/verified NT confidence. "
                        "Motif channels are zero-omitted for large graphs."
                    ),
                    "paths": {
                        "summary_csv": str(cell19e_summary_csv_path),
                        "results_json": str(cell19e_results_json_path),
                        "results_pickle": str(cell19e_results_pickle_path),
                    },
                },
                description=(
                    "Cell 19e: corrected cross-connectome structural coherence using actual "
                    "MANC/MAOL/MCNS schema. Failures are recorded, not fabricated."
                ),
                upload=True,
            )
        except Exception as cell19e_checkpoint_error:
            print("[ckpt][warn] Cell 19e checkpoint failed: " + repr(cell19e_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    cell19e_hf_upload_file_to_bucket = cell19e_globals.get("hf_upload_file_to_bucket", None)

    if callable(cell19e_hf_upload_file_to_bucket):
        cell19e_remote_paths = {
            str(cell19e_summary_csv_path): "checkpoints/cell19e_cross_connectome/cell19e_cross_connectome_summary.csv",
            str(cell19e_results_json_path): "checkpoints/cell19e_cross_connectome/cell19e_cross_connectome_results.json",
            str(cell19e_results_pickle_path): "checkpoints/cell19e_cross_connectome/cell19e_cross_connectome_results.pkl",
        }

        for cell19e_local_path_string, cell19e_remote_path_string in cell19e_remote_paths.items():
            try:
                cell19e_upload_ok, cell19e_upload_info = cell19e_hf_upload_file_to_bucket(
                    cell19e_Path(cell19e_local_path_string),
                    cell19e_remote_path_string,
                )
                print(
                    "[ckpt:cell19e] upload "
                    + cell19e_Path(cell19e_local_path_string).name
                    + ": "
                    + ("OK" if cell19e_upload_ok else "FAILED")
                    + " | "
                    + str(cell19e_upload_info)
                )
            except Exception as cell19e_upload_error:
                print(
                    "[ckpt][warn] Cell 19e upload failed for "
                    + cell19e_Path(cell19e_local_path_string).name
                    + ": "
                    + repr(cell19e_upload_error)
                )
    else:
        print(
            "[ckpt][skip] hf_upload_file_to_bucket not defined. "
            "Cross-connectome files remain local only."
        )

    return {
        "results": cell19e_results_dict,
        "summary_rows": cell19e_summary_rows,
        "output_dir": str(cell19e_output_dir),
        "summary_csv_path": str(cell19e_summary_csv_path),
        "results_json_path": str(cell19e_results_json_path),
        "results_pickle_path": str(cell19e_results_pickle_path),
        "frozen_path": str(cell19e_frozen_path),
    }


cell19e_output = cell19e_main()

cell19e_cross_results = cell19e_output["results"]
cell19e_cross_summary_rows = cell19e_output["summary_rows"]
cell19e_cross_output_dir = cell19e_output["output_dir"]
cell19e_cross_summary_csv_path = cell19e_output["summary_csv_path"]
cell19e_cross_results_json_path = cell19e_output["results_json_path"]
cell19e_cross_results_pickle_path = cell19e_output["results_pickle_path"]
cell19e_cross_frozen_path = cell19e_output["frozen_path"]

In [ ]:
# Cell 19f: Diagnose connection_load_failed and run corrected cross-connectome coherence.
#
# This replaces Cell 19e for the current failure mode.
#
# Known actual schema from Cell 19d:
#   neurons.csv.gz:
#     Root ID, Predicted NT type, Predicted NT confidence, Verified NT type,
#     Super Class, Class, Sub Class, Hemilineage, Nerve, Flow, Function,
#     Cable length (nm), Surface area (nm^2), Volume (nm^3), ...
#
#   connections_princeton.csv.gz:
#     pre_root_id, post_root_id, neuropil, syn_count, nt_type
#
# This cell:
#   1. Loads the frozen FAFB SSPR model from Cell 17.
#   2. Reads MANC / MAOL / MCNS using the actual schema.
#   3. Prints the exact reason connection loading failed in Cell 19e.
#   4. Expands transmitter alias handling.
#   5. Falls back to presynaptic neuron transmitter type when edge nt_type is unusable.
#   6. Builds adapted 61-dimensional features.
#   7. Applies frozen SSPR encoder without retraining.
#   8. Clusters embedding and computes ARI/NMI against native taxonomy.
#   9. Compares against shuffled-label null.
#  10. Saves diagnostics and results.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell19f_.


def cell19f_main():
    cell19f_globals = globals()

    # ------------------------------------------------------------------
    # Modules from globals when available, otherwise local imports.
    # ------------------------------------------------------------------
    cell19f_np = cell19f_globals.get("np", None)
    if cell19f_np is None:
        import numpy as cell19f_np

    cell19f_torch = cell19f_globals.get("torch", None)
    if cell19f_torch is None:
        import torch as cell19f_torch

    cell19f_pd = cell19f_globals.get("pd", None)
    if cell19f_pd is None:
        import pandas as cell19f_pd

    cell19f_Path = cell19f_globals.get("Path", None)
    if cell19f_Path is None:
        from pathlib import Path as cell19f_Path

    cell19f_gzip = cell19f_globals.get("gzip", None)
    if cell19f_gzip is None:
        import gzip as cell19f_gzip

    cell19f_json = cell19f_globals.get("json", None)
    if cell19f_json is None:
        import json as cell19f_json

    cell19f_pickle = cell19f_globals.get("pickle", None)
    if cell19f_pickle is None:
        import pickle as cell19f_pickle

    cell19f_csr_matrix = cell19f_globals.get("csr_matrix", None)
    if cell19f_csr_matrix is None:
        from scipy.sparse import csr_matrix as cell19f_csr_matrix

    cell19f_TruncatedSVD = cell19f_globals.get("TruncatedSVD", None)
    if cell19f_TruncatedSVD is None:
        from sklearn.decomposition import TruncatedSVD as cell19f_TruncatedSVD

    cell19f_KMeans = cell19f_globals.get("KMeans", None)
    if cell19f_KMeans is None:
        from sklearn.cluster import KMeans as cell19f_KMeans

    cell19f_ari_score = cell19f_globals.get("adjusted_rand_score", None)
    if cell19f_ari_score is None:
        from sklearn.metrics import adjusted_rand_score as cell19f_ari_score

    cell19f_nmi_score = cell19f_globals.get("normalized_mutual_info_score", None)
    if cell19f_nmi_score is None:
        from sklearn.metrics import normalized_mutual_info_score as cell19f_nmi_score

    cell19f_urllib_request = cell19f_globals.get("urllib_request", None)
    if cell19f_urllib_request is None:
        cell19f_urllib_module = cell19f_globals.get("urllib", None)
        if cell19f_urllib_module is not None and hasattr(cell19f_urllib_module, "request"):
            cell19f_urllib_request = cell19f_urllib_module.request
        else:
            import urllib.request as cell19f_urllib_request

    # ------------------------------------------------------------------
    # Configuration.
    # ------------------------------------------------------------------
    cell19f_bucket = "flywire-data"
    cell19f_root_prefix = "codex/data"
    cell19f_base_url = "https://storage.googleapis.com/" + cell19f_bucket + "/"

    cell19f_versions = {
        "manc": "1.2.1",
        "maol": "1.1",
        "mcns": "1.0",
    }

    cell19f_datasets = [
        "manc",
        "maol",
        "mcns",
    ]

    cell19f_probe_dir = cell19f_Path("./cross_connectome_probe_cell19d")
    cell19f_output_dir = cell19f_Path("./cross_connectome_data_cell19f")
    cell19f_output_dir.mkdir(parents=True, exist_ok=True)

    cell19f_neuron_filename = "neurons.csv.gz"
    cell19f_connection_filename = "connections_princeton.csv.gz"

    cell19f_motif_exact_max_nodes = 30000
    cell19f_motif_exact_max_edges = 250000

    cell19f_null_permutations = 200
    cell19f_min_labeled_nodes = 50
    cell19f_min_positive_edges = 10
    cell19f_min_negative_edges = 1
    cell19f_warn_negative_edges = 10
    cell19f_max_classes = 200

    # ------------------------------------------------------------------
    # Generic helpers.
    # ------------------------------------------------------------------
    def cell19f_first_global(cell19f_candidate_names):
        for cell19f_candidate_name in cell19f_candidate_names:
            if cell19f_candidate_name in cell19f_globals and cell19f_globals[cell19f_candidate_name] is not None:
                return cell19f_globals[cell19f_candidate_name]

        return None

    def cell19f_to_numpy(cell19f_object):
        if cell19f_object is None:
            return None

        cell19f_working_object = cell19f_object

        try:
            if hasattr(cell19f_working_object, "detach"):
                cell19f_working_object = cell19f_working_object.detach()

            if hasattr(cell19f_working_object, "cpu"):
                cell19f_working_object = cell19f_working_object.cpu()

            if hasattr(cell19f_working_object, "numpy"):
                cell19f_working_object = cell19f_working_object.numpy()

            return cell19f_np.asarray(cell19f_working_object)
        except Exception:
            return None

    def cell19f_is_valid_gzip(cell19f_path_value):
        try:
            with cell19f_gzip.open(cell19f_path_value, "rb") as cell19f_check_handle:
                while cell19f_check_handle.read(1 << 20):
                    pass

            return True
        except Exception:
            return False

    def cell19f_resolve_or_download(cell19f_dataset_name, cell19f_filename_value):
        cell19f_candidate_dirs = [
            cell19f_probe_dir / str(cell19f_dataset_name),
            cell19f_output_dir / str(cell19f_dataset_name),
        ]

        for cell19f_candidate_dir_value in cell19f_candidate_dirs:
            cell19f_candidate_path = cell19f_candidate_dir_value / str(cell19f_filename_value)

            if cell19f_candidate_path.exists() and cell19f_candidate_path.stat().st_size > 1000:
                if cell19f_is_valid_gzip(cell19f_candidate_path):
                    return cell19f_candidate_path, "existing_valid"

                try:
                    cell19f_candidate_path.unlink()
                except Exception:
                    pass

        cell19f_version_value = cell19f_versions.get(str(cell19f_dataset_name), None)

        if cell19f_version_value is None:
            return None, "unknown_dataset_version"

        cell19f_object_key = (
            cell19f_root_prefix
            + "/"
            + str(cell19f_dataset_name)
            + "/"
            + str(cell19f_version_value)
            + "/"
            + str(cell19f_filename_value)
        )

        cell19f_url_value = cell19f_base_url + str(cell19f_object_key)

        cell19f_dest_dir = cell19f_output_dir / str(cell19f_dataset_name)
        cell19f_dest_dir.mkdir(parents=True, exist_ok=True)

        cell19f_dest_path = cell19f_dest_dir / str(cell19f_filename_value)
        cell19f_tmp_path = cell19f_Path(str(cell19f_dest_path) + ".part")

        try:
            cell19f_request_object = cell19f_urllib_request.Request(
                cell19f_url_value,
                headers={"User-Agent": "research"},
            )

            with cell19f_urllib_request.urlopen(
                cell19f_request_object,
                timeout=900,
            ) as cell19f_response_object:
                with open(cell19f_tmp_path, "wb") as cell19f_out_handle:
                    while True:
                        cell19f_chunk_value = cell19f_response_object.read(1 << 20)

                        if not cell19f_chunk_value:
                            break

                        cell19f_out_handle.write(cell19f_chunk_value)

            if not cell19f_is_valid_gzip(cell19f_tmp_path):
                try:
                    cell19f_tmp_path.unlink()
                except Exception:
                    pass

                return None, "downloaded_but_invalid_gzip"

            cell19f_tmp_path.replace(cell19f_dest_path)
            return cell19f_dest_path, "downloaded_valid"

        except Exception as cell19f_download_error:
            try:
                if cell19f_tmp_path.exists():
                    cell19f_tmp_path.unlink()
            except Exception:
                pass

            return None, "download_failed:" + repr(cell19f_download_error)

    def cell19f_strip_frame_columns(cell19f_frame_value):
        cell19f_stripped_columns = []

        for cell19f_column_value in cell19f_frame_value.columns:
            cell19f_stripped_columns.append(str(cell19f_column_value).strip())

        cell19f_frame_value.columns = cell19f_stripped_columns
        return cell19f_frame_value

    def cell19f_choose_column(cell19f_frame_value, cell19f_candidate_names, cell19f_default_value=None):
        cell19f_lower_to_actual = {}

        for cell19f_column_value in cell19f_frame_value.columns:
            cell19f_lower_to_actual[str(cell19f_column_value).lower()] = str(cell19f_column_value)

        for cell19f_candidate_value in cell19f_candidate_names:
            cell19f_actual_column = cell19f_lower_to_actual.get(str(cell19f_candidate_value).lower(), None)

            if cell19f_actual_column is not None:
                return cell19f_actual_column

        return cell19f_default_value

    # ------------------------------------------------------------------
    # Transmitter text handling.
    # ------------------------------------------------------------------
    cell19f_invalid_text_values = {
        "",
        "nan",
        "NaN",
        "None",
        "none",
        "null",
        "NULL",
        "NA",
        "na",
        "unknown",
        "Unknown",
        "UNKNOWN",
        "?",
        "-",
        "--",
    }

    cell19f_nt_index_aliases = {
        "DA": 0,
        "DOPAMINE": 0,
        "DOPAMINERGIC": 0,
        "SER": 1,
        "SEROTONIN": 1,
        "SEROTONERGIC": 1,
        "GABA": 2,
        "GABAERGIC": 2,
        "AMINOBUTYRIC": 2,
        "GLUT": 3,
        "GLUTAMATE": 3,
        "GLUTAMATERGIC": 3,
        "GLU": 3,
        "ACH": 4,
        "ACETYLCHOLINE": 4,
        "CHOLINERGIC": 4,
        "CHOL": 4,
        "OCT": 5,
        "OCTOPAMINE": 5,
        "OCTOPAMINERGIC": 5,
    }

    cell19f_nt_sorted_alias_items = sorted(
        cell19f_nt_index_aliases.items(),
        key=lambda cell19f_alias_pair: len(str(cell19f_alias_pair[0])),
        reverse=True,
    )

    def cell19f_normalize_nt_text(cell19f_raw_text):
        cell19f_text_value = str(cell19f_raw_text).strip()

        if cell19f_text_value in cell19f_invalid_text_values:
            return ""

        cell19f_upper_text = cell19f_text_value.upper()
        cell19f_normalized_characters = []

        for cell19f_character_value in cell19f_upper_text:
            if cell19f_character_value.isalnum():
                cell19f_normalized_characters.append(cell19f_character_value)

        return "".join(cell19f_normalized_characters)

    def cell19f_nt_index_from_text(cell19f_raw_text):
        cell19f_normalized_text = cell19f_normalize_nt_text(cell19f_raw_text)

        if cell19f_normalized_text == "":
            return None

        if cell19f_normalized_text in cell19f_nt_index_aliases:
            return int(cell19f_nt_index_aliases[cell19f_normalized_text])

        for cell19f_alias_key, cell19f_alias_index in cell19f_nt_sorted_alias_items:
            cell19f_alias_key_text = str(cell19f_alias_key)

            if len(cell19f_alias_key_text) >= 4 and cell19f_alias_key_text in cell19f_normalized_text:
                return int(cell19f_alias_index)

        return None

    def cell19f_classify_nt_text(cell19f_raw_text):
        cell19f_normalized_text = cell19f_normalize_nt_text(cell19f_raw_text)

        if cell19f_normalized_text == "":
            return None

        cell19f_exact_exc_values = {
            "ACH",
            "GLUT",
            "GLU",
            "EXC",
            "EXCITATORY",
            "E",
        }

        cell19f_exact_inh_values = {
            "GABA",
            "GLY",
            "INH",
            "INHIBITORY",
            "I",
        }

        if cell19f_normalized_text in cell19f_exact_exc_values:
            return "exc"

        if cell19f_normalized_text in cell19f_exact_inh_values:
            return "inh"

        cell19f_exc_tokens = [
            "ACETYLCHOLINE",
            "CHOLINERGIC",
            "CHOL",
            "GLUTAMATE",
            "GLUTAMATERGIC",
            "GLUT",
            "GLU",
            "EXCITATORY",
            "EXC",
        ]

        cell19f_inh_tokens = [
            "GABAERGIC",
            "GABA",
            "AMINOBUTYRIC",
            "GLYCINE",
            "GLYCINERGIC",
            "GLY",
            "INHIBITORY",
            "INH",
        ]

        for cell19f_exc_token_value in cell19f_exc_tokens:
            if cell19f_exc_token_value in cell19f_normalized_text:
                return "exc"

        for cell19f_inh_token_value in cell19f_inh_tokens:
            if cell19f_inh_token_value in cell19f_normalized_text:
                return "inh"

        return None

    # ------------------------------------------------------------------
    # Neuron table loading.
    # ------------------------------------------------------------------
    def cell19f_build_neuron_nt_class(cell19f_neuron_frame, cell19f_node_count):
        cell19f_nt_class_array = cell19f_np.full(int(cell19f_node_count), None, dtype=object)

        cell19f_verified_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Verified NT type",
                "verified_nt_type",
                "Verified neurotransmitter",
                "nt_type",
            ],
            None,
        )

        cell19f_predicted_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Predicted NT type",
                "predicted_nt_type",
                "Predicted neurotransmitter",
            ],
            None,
        )

        cell19f_verified_array = None
        cell19f_predicted_array = None

        if cell19f_verified_col is not None:
            cell19f_verified_series = cell19f_neuron_frame[cell19f_verified_col].astype(str)
            cell19f_verified_unique_values = cell19f_pd.unique(cell19f_verified_series)
            cell19f_verified_lookup = {}

            for cell19f_verified_unique_value in cell19f_verified_unique_values:
                cell19f_verified_lookup[str(cell19f_verified_unique_value)] = cell19f_classify_nt_text(
                    cell19f_verified_unique_value
                )

            cell19f_verified_array = cell19f_verified_series.map(cell19f_verified_lookup).to_numpy(dtype=object)

        if cell19f_predicted_col is not None:
            cell19f_predicted_series = cell19f_neuron_frame[cell19f_predicted_col].astype(str)
            cell19f_predicted_unique_values = cell19f_pd.unique(cell19f_predicted_series)
            cell19f_predicted_lookup = {}

            for cell19f_predicted_unique_value in cell19f_predicted_unique_values:
                cell19f_predicted_lookup[str(cell19f_predicted_unique_value)] = cell19f_classify_nt_text(
                    cell19f_predicted_unique_value
                )

            cell19f_predicted_array = cell19f_predicted_series.map(cell19f_predicted_lookup).to_numpy(dtype=object)

        for cell19f_row_index in range(int(cell19f_node_count)):
            cell19f_verified_value = None
            cell19f_predicted_value = None

            if cell19f_verified_array is not None and cell19f_row_index < len(cell19f_verified_array):
                cell19f_verified_value = cell19f_verified_array[cell19f_row_index]

            if cell19f_predicted_array is not None and cell19f_row_index < len(cell19f_predicted_array):
                cell19f_predicted_value = cell19f_predicted_array[cell19f_row_index]

            if cell19f_verified_value in {"exc", "inh"}:
                cell19f_nt_class_array[cell19f_row_index] = str(cell19f_verified_value)
            elif cell19f_predicted_value in {"exc", "inh"}:
                cell19f_nt_class_array[cell19f_row_index] = str(cell19f_predicted_value)

        return cell19f_nt_class_array

    def cell19f_load_neuron_table(cell19f_neuron_path):
        try:
            cell19f_frame = cell19f_pd.read_csv(cell19f_neuron_path, low_memory=False)
        except Exception as cell19f_read_error:
            return None, None, None, 0, None, "read_failed:" + repr(cell19f_read_error)

        cell19f_frame = cell19f_strip_frame_columns(cell19f_frame)

        cell19f_id_col = cell19f_choose_column(
            cell19f_frame,
            [
                "Root ID",
                "root_id",
                "ID",
                "Id",
                "id",
                "neuron_id",
                "soma_id",
            ],
            None,
        )

        if cell19f_id_col is None:
            return None, None, None, 0, None, "missing_id_column"

        cell19f_frame[cell19f_id_col] = cell19f_pd.to_numeric(
            cell19f_frame[cell19f_id_col],
            errors="coerce",
        )

        cell19f_frame = cell19f_frame.dropna(subset=[cell19f_id_col]).copy()
        cell19f_frame = cell19f_frame.drop_duplicates(subset=[cell19f_id_col], keep="first").copy()
        cell19f_frame = cell19f_frame.reset_index(drop=True)

        if cell19f_frame.shape[0] == 0:
            return None, None, None, 0, None, "empty_neuron_table"

        try:
            cell19f_ids = cell19f_frame[cell19f_id_col].astype(cell19f_np.int64).to_numpy()
        except Exception as cell19f_id_cast_error:
            return None, None, None, 0, None, "id_cast_failed:" + repr(cell19f_id_cast_error)

        cell19f_node_count = int(cell19f_ids.size)

        cell19f_id_map = {
            int(cell19f_root_value): int(cell19f_root_index)
            for cell19f_root_index, cell19f_root_value in enumerate(cell19f_ids)
        }

        cell19f_nt_class_array = cell19f_build_neuron_nt_class(cell19f_frame, cell19f_node_count)

        return cell19f_frame, cell19f_ids, cell19f_id_map, cell19f_node_count, cell19f_nt_class_array, "ok"

    # ------------------------------------------------------------------
    # Connection table loading with diagnostics and fallback.
    # ------------------------------------------------------------------
    def cell19f_load_connection_table(
        cell19f_connection_path,
        cell19f_id_map,
        cell19f_node_count,
        cell19f_neuron_nt_class,
        cell19f_diagnostic_dict,
    ):
        try:
            cell19f_frame = cell19f_pd.read_csv(cell19f_connection_path, low_memory=False)
        except Exception as cell19f_read_error:
            cell19f_diagnostic_dict["connection_read_error"] = repr(cell19f_read_error)
            return None, "connection_read_failed:" + repr(cell19f_read_error)

        cell19f_frame = cell19f_strip_frame_columns(cell19f_frame)

        cell19f_diagnostic_dict["connection_columns"] = [
            str(cell19f_column_value)
            for cell19f_column_value in cell19f_frame.columns
        ]

        cell19f_diagnostic_dict["connection_rows_total"] = int(cell19f_frame.shape[0])

        cell19f_src_col = cell19f_choose_column(
            cell19f_frame,
            [
                "pre_root_id",
                "pre",
                "source",
                "src",
                "presynaptic_root_id",
                "pre_id",
                "from",
            ],
            None,
        )

        cell19f_dst_col = cell19f_choose_column(
            cell19f_frame,
            [
                "post_root_id",
                "post",
                "target",
                "dst",
                "postsynaptic_root_id",
                "post_id",
                "to",
            ],
            None,
        )

        cell19f_syn_col = cell19f_choose_column(
            cell19f_frame,
            [
                "syn_count",
                "count",
                "n_syn",
                "weight",
                "edge_weight",
                "synapses",
                "syn",
            ],
            None,
        )

        cell19f_nt_col = cell19f_choose_column(
            cell19f_frame,
            [
                "nt_type",
                "transmitter",
                "neurotransmitter",
                "nt",
                "edge_nt_type",
            ],
            None,
        )

        cell19f_diagnostic_dict["detected_src_col"] = str(cell19f_src_col) if cell19f_src_col is not None else ""
        cell19f_diagnostic_dict["detected_dst_col"] = str(cell19f_dst_col) if cell19f_dst_col is not None else ""
        cell19f_diagnostic_dict["detected_syn_col"] = str(cell19f_syn_col) if cell19f_syn_col is not None else ""
        cell19f_diagnostic_dict["detected_nt_col"] = str(cell19f_nt_col) if cell19f_nt_col is not None else ""

        if cell19f_src_col is None:
            return None, "missing_pre_root_id_column"

        if cell19f_dst_col is None:
            return None, "missing_post_root_id_column"

        cell19f_frame[cell19f_src_col] = cell19f_pd.to_numeric(
            cell19f_frame[cell19f_src_col],
            errors="coerce",
        )

        cell19f_frame[cell19f_dst_col] = cell19f_pd.to_numeric(
            cell19f_frame[cell19f_dst_col],
            errors="coerce",
        )

        cell19f_frame = cell19f_frame.dropna(subset=[cell19f_src_col, cell19f_dst_col]).copy()

        if cell19f_frame.shape[0] == 0:
            cell19f_diagnostic_dict["connection_rows_numeric_endpoint"] = 0
            return None, "no_numeric_endpoint_rows"

        cell19f_frame = cell19f_frame.reset_index(drop=True)

        try:
            cell19f_frame[cell19f_src_col] = cell19f_frame[cell19f_src_col].astype(cell19f_np.int64)
            cell19f_frame[cell19f_dst_col] = cell19f_frame[cell19f_dst_col].astype(cell19f_np.int64)
        except Exception as cell19f_endpoint_cast_error:
            return None, "endpoint_cast_failed:" + repr(cell19f_endpoint_cast_error)

        cell19f_frame["src"] = cell19f_frame[cell19f_src_col].map(cell19f_id_map)
        cell19f_frame["dst"] = cell19f_frame[cell19f_dst_col].map(cell19f_id_map)

        cell19f_mapped_mask = cell19f_frame["src"].notna() & cell19f_frame["dst"].notna()
        cell19f_diagnostic_dict["connection_rows_mapped"] = int(cell19f_np.sum(cell19f_mapped_mask.to_numpy()))

        cell19f_frame = cell19f_frame.loc[cell19f_mapped_mask].copy()
        cell19f_frame = cell19f_frame.reset_index(drop=True)

        if cell19f_frame.shape[0] == 0:
            return None, "no_mapped_edges"

        try:
            cell19f_frame["src"] = cell19f_frame["src"].astype(cell19f_np.int64)
            cell19f_frame["dst"] = cell19f_frame["dst"].astype(cell19f_np.int64)
        except Exception as cell19f_mapped_cast_error:
            return None, "mapped_endpoint_cast_failed:" + repr(cell19f_mapped_cast_error)

        cell19f_edge_nt_array = cell19f_np.full(cell19f_frame.shape[0], None, dtype=object)

        if cell19f_nt_col is not None:
            cell19f_nt_series = cell19f_frame[cell19f_nt_col].astype(str)
            cell19f_nt_unique_values = cell19f_pd.unique(cell19f_nt_series)

            cell19f_nt_lookup = {}

            for cell19f_nt_unique_value in cell19f_nt_unique_values:
                cell19f_nt_lookup[str(cell19f_nt_unique_value)] = cell19f_classify_nt_text(
                    cell19f_nt_unique_value
                )

            cell19f_edge_nt_array = cell19f_nt_series.map(cell19f_nt_lookup).to_numpy(dtype=object)

            cell19f_nt_value_counts = cell19f_nt_series.value_counts().head(50)

            cell19f_diagnostic_dict["nt_type_top_values"] = {
                str(cell19f_nt_count_key_value): int(cell19f_nt_count_value_value)
                for cell19f_nt_count_key_value, cell19f_nt_count_value_value in cell19f_nt_value_counts.items()
            }
        else:
            cell19f_diagnostic_dict["nt_type_top_values"] = {}

        cell19f_signed_before_fallback_count = int(
            cell19f_np.sum(
                cell19f_np.isin(
                    cell19f_edge_nt_array,
                    cell19f_np.array(["exc", "inh"], dtype=object),
                )
            )
        )

        cell19f_diagnostic_dict["signed_rows_before_neuron_fallback"] = cell19f_signed_before_fallback_count

        cell19f_src_array = cell19f_frame["src"].to_numpy(dtype=cell19f_np.int64)
        cell19f_fallback_array = cell19f_neuron_nt_class[cell19f_src_array]

        cell19f_edge_nt_series = cell19f_pd.Series(cell19f_edge_nt_array, index=cell19f_frame.index)
        cell19f_fallback_series = cell19f_pd.Series(cell19f_fallback_array, index=cell19f_frame.index)

        cell19f_final_nt_series = cell19f_edge_nt_series.where(
            cell19f_edge_nt_series.notna(),
            cell19f_fallback_series,
        )

        cell19f_signed_mask = cell19f_final_nt_series.isin(["exc", "inh"]).to_numpy()
        cell19f_signed_after_fallback_count = int(cell19f_np.sum(cell19f_signed_mask))

        cell19f_diagnostic_dict["signed_rows_after_neuron_fallback"] = cell19f_signed_after_fallback_count

        if cell19f_signed_after_fallback_count == 0:
            return None, "no_signed_edges_after_nt_fallback"

        cell19f_frame = cell19f_frame.loc[cell19f_signed_mask].copy()
        cell19f_frame = cell19f_frame.reset_index(drop=True)

        cell19f_frame["nt_class"] = cell19f_final_nt_series.loc[cell19f_signed_mask].to_numpy(dtype=object)

        if cell19f_syn_col is not None:
            cell19f_frame["syn_count"] = cell19f_pd.to_numeric(
                cell19f_frame[cell19f_syn_col],
                errors="coerce",
            ).fillna(1.0).clip(lower=0.0)
        else:
            cell19f_frame["syn_count"] = 1.0

        cell19f_pos_count_before_agg = int(cell19f_np.sum(cell19f_frame["nt_class"].to_numpy() == "exc"))
        cell19f_neg_count_before_agg = int(cell19f_np.sum(cell19f_frame["nt_class"].to_numpy() == "inh"))

        cell19f_diagnostic_dict["signed_rows_exc_before_agg"] = cell19f_pos_count_before_agg
        cell19f_diagnostic_dict["signed_rows_inh_before_agg"] = cell19f_neg_count_before_agg

        cell19f_agg = (
            cell19f_frame
            .groupby(["src", "dst", "nt_class"], as_index=False)
            .agg(syn_count=("syn_count", "sum"))
        )

        cell19f_major = (
            cell19f_agg
            .sort_values("syn_count")
            .groupby(["src", "dst"], as_index=False)
            .last()
        )

        cell19f_src_out = cell19f_major["src"].to_numpy(dtype=cell19f_np.int64)
        cell19f_dst_out = cell19f_major["dst"].to_numpy(dtype=cell19f_np.int64)
        cell19f_sign_bin_out = (cell19f_major["nt_class"] == "exc").to_numpy(dtype=cell19f_np.float32)
        cell19f_syn_out = cell19f_major["syn_count"].to_numpy(dtype=cell19f_np.float32)

        cell19f_diagnostic_dict["major_edges_total"] = int(cell19f_src_out.size)
        cell19f_diagnostic_dict["major_edges_positive"] = int(cell19f_np.sum(cell19f_sign_bin_out > 0.5))
        cell19f_diagnostic_dict["major_edges_negative"] = int(cell19f_np.sum(cell19f_sign_bin_out <= 0.5))

        if cell19f_src_out.size == 0:
            return None, "empty_major_edges"

        return {
            "src": cell19f_src_out,
            "dst": cell19f_dst_out,
            "sign_bin": cell19f_sign_bin_out,
            "syn": cell19f_syn_out,
        }, "ok"

    # ------------------------------------------------------------------
    # Feature construction.
    # ------------------------------------------------------------------
    def cell19f_build_morphology_features(cell19f_neuron_frame, cell19f_node_count):
        cell19f_features = cell19f_np.zeros((int(cell19f_node_count), 3), dtype=cell19f_np.float64)

        cell19f_length_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Cable length (nm)",
                "length_nm",
                "Cable length",
                "soma_length",
                "length",
            ],
            None,
        )

        cell19f_area_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Surface area (nm^2)",
                "area_nm",
                "Surface area",
                "soma_area",
                "area",
            ],
            None,
        )

        cell19f_volume_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Volume (nm^3)",
                "size_nm",
                "Volume",
                "soma_volume",
                "volume",
                "size",
            ],
            None,
        )

        cell19f_column_targets = [
            (cell19f_length_col, 0),
            (cell19f_area_col, 1),
            (cell19f_volume_col, 2),
        ]

        for cell19f_source_column, cell19f_target_index in cell19f_column_targets:
            if cell19f_source_column is None:
                continue

            cell19f_raw_values = cell19f_pd.to_numeric(
                cell19f_neuron_frame[cell19f_source_column],
                errors="coerce",
            ).to_numpy()

            cell19f_clean_values = cell19f_np.nan_to_num(
                cell19f_raw_values,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            cell19f_clean_values = cell19f_np.clip(cell19f_clean_values, 0.0, None)
            cell19f_features[:, cell19f_target_index] = cell19f_np.log1p(cell19f_clean_values)

        return cell19f_features

    def cell19f_build_transmitter_features(cell19f_neuron_frame, cell19f_node_count):
        cell19f_features = cell19f_np.zeros((int(cell19f_node_count), 6), dtype=cell19f_np.float64)

        cell19f_verified_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Verified NT type",
                "verified_nt_type",
                "Verified neurotransmitter",
                "nt_type",
            ],
            None,
        )

        cell19f_predicted_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Predicted NT type",
                "predicted_nt_type",
                "Predicted neurotransmitter",
            ],
            None,
        )

        cell19f_confidence_col = cell19f_choose_column(
            cell19f_neuron_frame,
            [
                "Predicted NT confidence",
                "predicted_nt_confidence",
                "nt_type_score",
                "confidence",
            ],
            None,
        )

        cell19f_verified_array = None
        cell19f_predicted_array = None
        cell19f_confidence_array = None

        if cell19f_verified_col is not None:
            cell19f_verified_series = cell19f_neuron_frame[cell19f_verified_col].astype(str)
            cell19f_verified_unique_values = cell19f_pd.unique(cell19f_verified_series)
            cell19f_verified_lookup = {}

            for cell19f_verified_unique_value in cell19f_verified_unique_values:
                cell19f_verified_lookup[str(cell19f_verified_unique_value)] = cell19f_nt_index_from_text(
                    cell19f_verified_unique_value
                )

            cell19f_verified_array = cell19f_verified_series.map(cell19f_verified_lookup).to_numpy(dtype=object)

        if cell19f_predicted_col is not None:
            cell19f_predicted_series = cell19f_neuron_frame[cell19f_predicted_col].astype(str)
            cell19f_predicted_unique_values = cell19f_pd.unique(cell19f_predicted_series)
            cell19f_predicted_lookup = {}

            for cell19f_predicted_unique_value in cell19f_predicted_unique_values:
                cell19f_predicted_lookup[str(cell19f_predicted_unique_value)] = cell19f_nt_index_from_text(
                    cell19f_predicted_unique_value
                )

            cell19f_predicted_array = cell19f_predicted_series.map(cell19f_predicted_lookup).to_numpy(dtype=object)

        if cell19f_confidence_col is not None:
            cell19f_confidence_array = cell19f_pd.to_numeric(
                cell19f_neuron_frame[cell19f_confidence_col],
                errors="coerce",
            ).to_numpy()

        for cell19f_row_index in range(int(cell19f_node_count)):
            cell19f_verified_index = None
            cell19f_predicted_index = None
            cell19f_confidence_value = float("nan")

            if cell19f_verified_array is not None and cell19f_row_index < len(cell19f_verified_array):
                cell19f_verified_index = cell19f_verified_array[cell19f_row_index]

            if cell19f_predicted_array is not None and cell19f_row_index < len(cell19f_predicted_array):
                cell19f_predicted_index = cell19f_predicted_array[cell19f_row_index]

            if cell19f_confidence_array is not None and cell19f_row_index < len(cell19f_confidence_array):
                try:
                    cell19f_confidence_value = float(cell19f_confidence_array[cell19f_row_index])
                except Exception:
                    cell19f_confidence_value = float("nan")

            if cell19f_verified_index is not None:
                try:
                    cell19f_features[cell19f_row_index, int(cell19f_verified_index)] = 1.0
                except Exception:
                    pass

                continue

            if cell19f_predicted_index is not None:
                if cell19f_np.isfinite(cell19f_confidence_value) and cell19f_confidence_value > 0.0:
                    cell19f_use_confidence = max(0.0, min(1.0, cell19f_confidence_value))
                else:
                    cell19f_use_confidence = 0.5

                try:
                    cell19f_features[cell19f_row_index, int(cell19f_predicted_index)] = float(cell19f_use_confidence)
                except Exception:
                    pass

        return cell19f_features

    def cell19f_build_target_features(
        cell19f_neuron_frame,
        cell19f_node_count,
        cell19f_connection_dict,
    ):
        cell19f_src = cell19f_connection_dict["src"]
        cell19f_dst = cell19f_connection_dict["dst"]
        cell19f_sign_bin = cell19f_connection_dict["sign_bin"]
        cell19f_syn = cell19f_connection_dict["syn"]

        cell19f_morphology_features = cell19f_build_morphology_features(
            cell19f_neuron_frame,
            cell19f_node_count,
        )

        cell19f_transmitter_features = cell19f_build_transmitter_features(
            cell19f_neuron_frame,
            cell19f_node_count,
        )

        cell19f_deg_in = cell19f_np.zeros(int(cell19f_node_count), dtype=cell19f_np.float64)
        cell19f_deg_out = cell19f_np.zeros(int(cell19f_node_count), dtype=cell19f_np.float64)

        cell19f_syn_float = cell19f_syn.astype(cell19f_np.float64)
        cell19f_np.add.at(cell19f_deg_in, cell19f_dst, cell19f_syn_float)
        cell19f_np.add.at(cell19f_deg_out, cell19f_src, cell19f_syn_float)

        cell19f_degree_features = cell19f_np.stack(
            [
                cell19f_np.log1p(cell19f_deg_in),
                cell19f_np.log1p(cell19f_deg_out),
            ],
            axis=1,
        )

        cell19f_row_unsigned = cell19f_np.concatenate([cell19f_src, cell19f_dst])
        cell19f_col_unsigned = cell19f_np.concatenate([cell19f_dst, cell19f_src])
        cell19f_data_unsigned = cell19f_np.ones(cell19f_row_unsigned.size, dtype=cell19f_np.float32)

        cell19f_A_sym = cell19f_csr_matrix(
            (cell19f_data_unsigned, (cell19f_row_unsigned, cell19f_col_unsigned)),
            shape=(int(cell19f_node_count), int(cell19f_node_count)),
        )

        if int(cell19f_node_count) < 17:
            cell19f_spec_unsigned = cell19f_np.zeros((int(cell19f_node_count), 16), dtype=cell19f_np.float64)
        else:
            try:
                cell19f_svd_unsigned = cell19f_TruncatedSVD(n_components=16, random_state=42)
                cell19f_spec_unsigned = cell19f_svd_unsigned.fit_transform(cell19f_A_sym)
                cell19f_spec_unsigned_mean = cell19f_spec_unsigned.mean(axis=0)
                cell19f_spec_unsigned_std = cell19f_spec_unsigned.std(axis=0)
                cell19f_spec_unsigned = (
                    cell19f_spec_unsigned - cell19f_spec_unsigned_mean
                ) / (cell19f_spec_unsigned_std + 1e-8)
                cell19f_spec_unsigned = cell19f_np.nan_to_num(
                    cell19f_spec_unsigned,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )
            except Exception as cell19f_unsigned_svd_error:
                cell19f_spec_unsigned = cell19f_np.zeros(
                    (int(cell19f_node_count), 16),
                    dtype=cell19f_np.float64,
                )

        cell19f_pos_mask = cell19f_sign_bin > 0.5
        cell19f_neg_mask = ~cell19f_pos_mask

        cell19f_row_positive = cell19f_src[cell19f_pos_mask]
        cell19f_col_positive = cell19f_dst[cell19f_pos_mask]
        cell19f_row_negative = cell19f_src[cell19f_neg_mask]
        cell19f_col_negative = cell19f_dst[cell19f_neg_mask]

        cell19f_row_signed = cell19f_np.concatenate([cell19f_row_positive, cell19f_row_negative])
        cell19f_col_signed = cell19f_np.concatenate([cell19f_col_positive, cell19f_col_negative])
        cell19f_data_signed = cell19f_np.concatenate(
            [
                cell19f_np.ones(cell19f_row_positive.size, dtype=cell19f_np.float32),
                -cell19f_np.ones(cell19f_row_negative.size, dtype=cell19f_np.float32),
            ]
        )

        cell19f_A_signed = cell19f_csr_matrix(
            (cell19f_data_signed, (cell19f_row_signed, cell19f_col_signed)),
            shape=(int(cell19f_node_count), int(cell19f_node_count)),
        )

        if int(cell19f_node_count) < 33:
            cell19f_spec_signed = cell19f_np.zeros((int(cell19f_node_count), 32), dtype=cell19f_np.float64)
        else:
            try:
                cell19f_svd_signed = cell19f_TruncatedSVD(n_components=32, random_state=42)
                cell19f_spec_signed = cell19f_svd_signed.fit_transform(cell19f_A_signed)
                cell19f_spec_signed_mean = cell19f_spec_signed.mean(axis=0)
                cell19f_spec_signed_std = cell19f_spec_signed.std(axis=0)
                cell19f_spec_signed = (
                    cell19f_spec_signed - cell19f_spec_signed_mean
                ) / (cell19f_spec_signed_std + 1e-8)
                cell19f_spec_signed = cell19f_np.nan_to_num(
                    cell19f_spec_signed,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )
            except Exception as cell19f_signed_svd_error:
                cell19f_spec_signed = cell19f_np.zeros(
                    (int(cell19f_node_count), 32),
                    dtype=cell19f_np.float64,
                )

        cell19f_motif_features = cell19f_np.zeros((int(cell19f_node_count), 2), dtype=cell19f_np.float64)
        cell19f_motif_mode = "zero_omitted_for_scale"

        if (
            int(cell19f_node_count) <= int(cell19f_motif_exact_max_nodes)
            and int(cell19f_src.size) <= int(cell19f_motif_exact_max_edges)
        ):
            try:
                cell19f_row_abs = cell19f_np.concatenate([cell19f_src, cell19f_dst])
                cell19f_col_abs = cell19f_np.concatenate([cell19f_dst, cell19f_src])
                cell19f_data_abs = cell19f_np.ones(cell19f_row_abs.size, dtype=cell19f_np.float32)

                cell19f_A_abs = cell19f_csr_matrix(
                    (cell19f_data_abs, (cell19f_row_abs, cell19f_col_abs)),
                    shape=(int(cell19f_node_count), int(cell19f_node_count)),
                )

                cell19f_A3_signed = cell19f_A_signed.dot(cell19f_A_signed).dot(cell19f_A_signed)
                cell19f_A3_abs = cell19f_A_abs.dot(cell19f_A_abs).dot(cell19f_A_abs)

                cell19f_diag_signed = cell19f_np.asarray(cell19f_A3_signed.diagonal()).flatten()
                cell19f_diag_abs = cell19f_np.asarray(cell19f_A3_abs.diagonal()).flatten()

                cell19f_tri_bal = (cell19f_diag_abs + cell19f_diag_signed) / 12.0
                cell19f_tri_fru = (cell19f_diag_abs - cell19f_diag_signed) / 12.0

                cell19f_motif_features = cell19f_np.stack(
                    [
                        cell19f_np.log1p(cell19f_np.clip(cell19f_tri_bal, 0.0, None)),
                        cell19f_np.log1p(cell19f_np.clip(cell19f_tri_fru, 0.0, None)),
                    ],
                    axis=1,
                )

                cell19f_motif_features = cell19f_np.nan_to_num(
                    cell19f_motif_features,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )

                cell19f_motif_mode = "exact_sparse"

            except Exception as cell19f_motif_error:
                cell19f_motif_features = cell19f_np.zeros(
                    (int(cell19f_node_count), 2),
                    dtype=cell19f_np.float64,
                )
                cell19f_motif_mode = "exact_failed_zero:" + repr(cell19f_motif_error)

        cell19f_X_raw = cell19f_np.concatenate(
            [
                cell19f_morphology_features,
                cell19f_transmitter_features,
                cell19f_degree_features,
                cell19f_spec_unsigned,
                cell19f_spec_signed,
                cell19f_motif_features,
            ],
            axis=1,
        )

        if int(cell19f_X_raw.shape[1]) != 61:
            return None, cell19f_motif_mode, "feature_shape_mismatch:" + str(cell19f_X_raw.shape)

        cell19f_X_raw = cell19f_np.nan_to_num(
            cell19f_X_raw,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return cell19f_X_raw, cell19f_motif_mode, "ok"

    # ------------------------------------------------------------------
    # Frozen model helpers.
    # ------------------------------------------------------------------
    def cell19f_collect_tensors(cell19f_output_object, cell19f_tensor_list=None):
        if cell19f_tensor_list is None:
            cell19f_tensor_list = []

        if cell19f_torch.is_tensor(cell19f_output_object):
            if int(cell19f_output_object.ndim) >= 2:
                cell19f_tensor_list.append(cell19f_output_object)
            return cell19f_tensor_list

        if isinstance(cell19f_output_object, dict):
            for cell19f_dict_value in cell19f_output_object.values():
                cell19f_collect_tensors(cell19f_dict_value, cell19f_tensor_list)
            return cell19f_tensor_list

        if isinstance(cell19f_output_object, (tuple, list)):
            for cell19f_sequence_value in cell19f_output_object:
                cell19f_collect_tensors(cell19f_sequence_value, cell19f_tensor_list)
            return cell19f_tensor_list

        return cell19f_tensor_list

    def cell19f_extract_embedding(cell19f_output_object, cell19f_model_object, cell19f_expected_rows):
        cell19f_candidate_tensors = cell19f_collect_tensors(cell19f_output_object)

        if len(cell19f_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 19f][FATAL] Frozen model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell19f_output_object))
            )

        cell19f_proto_parameter = getattr(cell19f_model_object, "proto_class", None)

        if cell19f_torch.is_tensor(cell19f_proto_parameter) and int(cell19f_proto_parameter.ndim) == 2:
            cell19f_proto_dim = int(cell19f_proto_parameter.shape[1])

            for cell19f_candidate_tensor in cell19f_candidate_tensors:
                if int(cell19f_candidate_tensor.shape[1]) == cell19f_proto_dim:
                    return cell19f_candidate_tensor

        for cell19f_candidate_tensor in cell19f_candidate_tensors:
            if int(cell19f_candidate_tensor.shape[0]) == int(cell19f_expected_rows):
                return cell19f_candidate_tensor

        return cell19f_candidate_tensors[0]

    def cell19f_make_model_from_checkpoint(cell19f_checkpoint):
        cell19f_d_features = int(cell19f_checkpoint.get("d_features", 61))
        cell19f_d_hidden = int(cell19f_checkpoint.get("d_cell5", 64))
        cell19f_d_proto = int(cell19f_checkpoint.get("d_proto_cell5", 32))
        cell19f_n_layers = int(cell19f_checkpoint.get("n_layers_cell5", 2))
        cell19f_n_classes = int(cell19f_checkpoint.get("n_classes_cell5", 29))
        cell19f_n_super = int(cell19f_checkpoint.get("n_super_cell5", 10))
        cell19f_temperature = float(cell19f_checkpoint.get("temperature_cell5", 0.1))
        cell19f_dropout = float(cell19f_checkpoint.get("dropout_cell5", 0.3))

        try:
            return cell19f_globals["SPGCN_cell5"](
                d_features_cell5=cell19f_d_features,
                d_cell5=cell19f_d_hidden,
                d_proto_cell5=cell19f_d_proto,
                n_layers_cell5=cell19f_n_layers,
                n_classes_cell5=cell19f_n_classes,
                n_super_cell5=cell19f_n_super,
                temperature_cell5=cell19f_temperature,
                dropout_cell5=cell19f_dropout,
            )
        except TypeError as cell19f_init_error:
            try:
                return cell19f_globals["SPGCN_cell5"](
                    d_features=cell19f_d_features,
                    d=cell19f_d_hidden,
                    d_proto=cell19f_d_proto,
                    n_layers=cell19f_n_layers,
                    n_classes=cell19f_n_classes,
                    n_super=cell19f_n_super,
                    temperature=cell19f_temperature,
                    dropout=cell19f_dropout,
                )
            except TypeError:
                raise TypeError(
                    "[Cell 19f][FATAL] Could not instantiate SPGCN_cell5 from checkpoint. "
                    "First TypeError: " + repr(cell19f_init_error)
                )

    def cell19f_forward_frozen_model(
        cell19f_model_object,
        cell19f_x_tensor,
        cell19f_ei_pos_tensor,
        cell19f_ew_pos_tensor,
        cell19f_dis_pos_tensor,
        cell19f_ei_neg_tensor,
        cell19f_ew_neg_tensor,
        cell19f_dis_neg_tensor,
    ):
        cell19f_last_type_error = None

        try:
            return cell19f_model_object(
                cell19f_x_tensor,
                cell19f_ei_pos_tensor,
                cell19f_ew_pos_tensor,
                cell19f_dis_pos_tensor,
                cell19f_ei_neg_tensor,
                cell19f_ew_neg_tensor,
                cell19f_dis_neg_tensor,
            )
        except TypeError as cell19f_type_error:
            cell19f_last_type_error = cell19f_type_error

        try:
            return cell19f_model_object(
                cell19f_x_tensor,
                cell19f_ei_pos_tensor,
                cell19f_dis_pos_tensor,
                cell19f_ei_neg_tensor,
                cell19f_dis_neg_tensor,
            )
        except TypeError as cell19f_type_error:
            cell19f_last_type_error = cell19f_type_error

        try:
            return cell19f_model_object(
                cell19f_x_tensor,
                cell19f_ei_pos_tensor,
                cell19f_ei_neg_tensor,
            )
        except TypeError as cell19f_type_error:
            cell19f_last_type_error = cell19f_type_error

        raise TypeError(
            "[Cell 19f][FATAL] Could not call frozen model with any known signature. "
            "Last TypeError: " + repr(cell19f_last_type_error)
        )

    def cell19f_normalize_edge_degree(cell19f_ei_tensor, cell19f_ew_tensor, cell19f_node_count, cell19f_target_device):
        if int(cell19f_ei_tensor.numel()) == 0:
            return cell19f_torch.ones(int(cell19f_node_count), device=cell19f_target_device)

        cell19f_degree_vector = cell19f_torch.zeros(int(cell19f_node_count), device=cell19f_target_device)
        cell19f_degree_vector.index_add_(0, cell19f_ei_tensor[0], cell19f_ew_tensor)
        cell19f_degree_vector.index_add_(0, cell19f_ei_tensor[1], cell19f_ew_tensor)

        return cell19f_degree_vector.clamp(min=1.0).rsqrt()

    # ------------------------------------------------------------------
    # Label selection.
    # ------------------------------------------------------------------
    cell19f_label_priority = [
        "Super Class",
        "Class",
        "Sub Class",
        "Primary Cell Type",
        "Function",
        "Flow",
        "Hemilineage",
        "Nerve",
        "Body Part",
    ]

    def cell19f_choose_label_column(cell19f_neuron_frame):
        for cell19f_candidate_column in cell19f_label_priority:
            if cell19f_candidate_column not in cell19f_neuron_frame.columns:
                continue

            cell19f_series = cell19f_neuron_frame[cell19f_candidate_column].astype(str).str.strip()

            cell19f_invalid_values = [
                str(cell19f_invalid_value)
                for cell19f_invalid_value in cell19f_invalid_text_values
            ]

            cell19f_valid_mask = ~cell19f_series.isin(cell19f_invalid_values)
            cell19f_valid_count = int(cell19f_np.sum(cell19f_valid_mask.to_numpy()))

            if cell19f_valid_count < int(cell19f_min_labeled_nodes):
                continue

            cell19f_unique_values = sorted(
                set(
                    str(cell19f_label_value).strip()
                    for cell19f_label_value in cell19f_series[cell19f_valid_mask].to_numpy()
                )
            )

            if len(cell19f_unique_values) < 2:
                continue

            if len(cell19f_unique_values) > int(cell19f_max_classes):
                continue

            return cell19f_candidate_column, cell19f_unique_values, cell19f_valid_count

        return None, [], 0

    # ------------------------------------------------------------------
    # Resolve frozen model.
    # ------------------------------------------------------------------
    cell19f_frozen_path_value = cell19f_first_global(
        [
            "cell17_frozen_path",
            "FROZEN_PATH_CELL17",
            "cell16_frozen_path",
            "FROZEN_PATH_CELL16",
        ]
    )

    if cell19f_frozen_path_value is None:
        cell19f_output_candidate = cell19f_first_global(["cell17_output"])

        if isinstance(cell19f_output_candidate, dict):
            cell19f_frozen_path_value = cell19f_output_candidate.get("frozen_path", None)

    if cell19f_frozen_path_value is None:
        cell19f_frozen_path_value = "./sspr_fafb_frozen_cell17.pt"

    cell19f_frozen_path = cell19f_Path(str(cell19f_frozen_path_value))

    if not cell19f_frozen_path.exists():
        cell19f_alternative_paths = [
            cell19f_Path("./sspr_fafb_frozen_cell17.pt"),
            cell19f_Path("./sspr_fafb_frozen_cell16.pt"),
            cell19f_Path("./sspr_fafb_frozen.pt"),
        ]

        cell19f_found_alternative = None

        for cell19f_alternative_path in cell19f_alternative_paths:
            if cell19f_alternative_path.exists():
                cell19f_found_alternative = cell19f_alternative_path
                break

        if cell19f_found_alternative is None:
            raise FileNotFoundError(
                "[Cell 19f][FATAL] Frozen FAFB SSPR file does not exist: "
                + str(cell19f_frozen_path)
                + ". Run Cell 17 first."
            )

        cell19f_frozen_path = cell19f_found_alternative

    if "SPGCN_cell5" not in cell19f_globals:
        raise NameError("[Cell 19f][FATAL] SPGCN_cell5 is not available. Run Cell 5 first.")

    if "DEVICE_cell1" not in cell19f_globals:
        raise NameError("[Cell 19f][FATAL] DEVICE_cell1 is not available. Run Cell 1 first.")

    cell19f_device = cell19f_globals["DEVICE_cell1"]

    print("=" * 100)
    print("CELL 19f — CONNECTION DIAGNOSTIC + CORRECTED CROSS-CONNECTOME COHERENCE")
    print("=" * 100)
    print("[cfg] frozen_path=" + str(cell19f_frozen_path))
    print("[cfg] datasets=" + str(cell19f_datasets))
    print("[cfg] versions=" + str(cell19f_versions))
    print("[cfg] probe_dir=" + str(cell19f_probe_dir))
    print("[cfg] output_dir=" + str(cell19f_output_dir))
    print("[schema] using neurons.csv.gz + connections_princeton.csv.gz")
    print("[fix] expanded nt_type aliases")
    print("[fix] fallback edge sign from presynaptic neuron Verified/Predicted NT type")
    print("[fix] column names stripped and alternative names accepted")
    print("=" * 100)

    # ------------------------------------------------------------------
    # Load frozen checkpoint.
    # ------------------------------------------------------------------
    try:
        cell19f_checkpoint = cell19f_torch.load(
            cell19f_frozen_path,
            map_location=cell19f_device,
            weights_only=False,
        )
    except TypeError:
        cell19f_checkpoint = cell19f_torch.load(
            cell19f_frozen_path,
            map_location=cell19f_device,
        )

    cell19f_feature_mean_raw = cell19f_checkpoint.get("feature_mean", None)
    cell19f_feature_std_raw = cell19f_checkpoint.get("feature_std", None)

    if cell19f_feature_mean_raw is None or cell19f_feature_std_raw is None:
        raise KeyError("[Cell 19f][FATAL] Frozen checkpoint missing feature_mean or feature_std.")

    cell19f_feature_mean = cell19f_torch.as_tensor(
        cell19f_feature_mean_raw,
        dtype=cell19f_torch.float32,
    ).to(cell19f_device)

    cell19f_feature_std = cell19f_torch.as_tensor(
        cell19f_feature_std_raw,
        dtype=cell19f_torch.float32,
    ).to(cell19f_device)

    cell19f_ckpt_mean_np = cell19f_to_numpy(cell19f_feature_mean)
    cell19f_ckpt_std_np = cell19f_to_numpy(cell19f_feature_std)

    if cell19f_ckpt_mean_np is None or cell19f_ckpt_std_np is None:
        raise TypeError("[Cell 19f][FATAL] Could not convert checkpoint feature mean/std to numpy.")

    cell19f_ckpt_mean_np = cell19f_np.asarray(cell19f_ckpt_mean_np, dtype=cell19f_np.float64).reshape(-1)
    cell19f_ckpt_std_np = cell19f_np.asarray(cell19f_ckpt_std_np, dtype=cell19f_np.float64).reshape(-1)

    cell19f_frozen_model = cell19f_make_model_from_checkpoint(cell19f_checkpoint)

    if "encoder_state" not in cell19f_checkpoint:
        raise KeyError("[Cell 19f][FATAL] Frozen checkpoint missing encoder_state.")

    try:
        cell19f_frozen_model.load_state_dict(cell19f_checkpoint["encoder_state"])
    except RuntimeError:
        cell19f_frozen_model.load_state_dict(cell19f_checkpoint["encoder_state"], strict=False)

    cell19f_frozen_model = cell19f_frozen_model.to(cell19f_device)
    cell19f_frozen_model.eval()

    cell19f_param_count = int(sum(int(parameter.numel()) for parameter in cell19f_frozen_model.parameters()))
    print("[Cell 19f] Loaded frozen SSPR params=" + f"{cell19f_param_count:,}")

    # ------------------------------------------------------------------
    # Main dataset loop.
    # ------------------------------------------------------------------
    cell19f_results_dict = {}
    cell19f_summary_rows = []

    for cell19f_dataset_name in cell19f_datasets:
        print("\n---- Dataset: " + str(cell19f_dataset_name) + " ----")

        cell19f_dataset_row = {
            "dataset": str(cell19f_dataset_name),
            "version": str(cell19f_versions.get(cell19f_dataset_name, "unknown")),
            "status": "pending",
            "N": "",
            "n_edges_signed": "",
            "n_positive_edges": "",
            "n_negative_edges": "",
            "label_column": "",
            "K": "",
            "n_labeled": "",
            "normalization_mode": "",
            "motif_mode": "",
            "signed_graph_warning": "",
            "ari": "",
            "ari_null": "",
            "ari_delta": "",
            "ari_p_empirical": "",
            "nmi": "",
            "nmi_null": "",
            "nmi_delta": "",
            "nmi_p_empirical": "",
            "error": "",
            "connection_rows_total": "",
            "connection_rows_mapped": "",
            "signed_rows_before_neuron_fallback": "",
            "signed_rows_after_neuron_fallback": "",
            "major_edges_positive": "",
            "major_edges_negative": "",
            "neuron_nt_known_count": "",
            "nt_type_top_values": "",
        }

        cell19f_diagnostic_dict = {}

        try:
            cell19f_neuron_path, cell19f_neuron_status = cell19f_resolve_or_download(
                cell19f_dataset_name,
                cell19f_neuron_filename,
            )

            cell19f_connection_path, cell19f_connection_status = cell19f_resolve_or_download(
                cell19f_dataset_name,
                cell19f_connection_filename,
            )

            print("[cell19f] neurons: " + str(cell19f_neuron_status))
            print("[cell19f] connections: " + str(cell19f_connection_status))

            if cell19f_neuron_path is None or cell19f_connection_path is None:
                cell19f_dataset_row["status"] = "download_failed"
                cell19f_dataset_row["error"] = (
                    "neurons=" + str(cell19f_neuron_status)
                    + "; connections=" + str(cell19f_connection_status)
                )
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            cell19f_neuron_frame, cell19f_ids, cell19f_id_map, cell19f_N, cell19f_neuron_nt_class, cell19f_neuron_load_status = (
                cell19f_load_neuron_table(cell19f_neuron_path)
            )

            if cell19f_neuron_frame is None:
                cell19f_dataset_row["status"] = "neuron_load_failed"
                cell19f_dataset_row["error"] = str(cell19f_neuron_load_status)
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            cell19f_neuron_nt_known_count = int(
                sum(
                    1
                    for cell19f_nt_value in cell19f_neuron_nt_class
                    if cell19f_nt_value in {"exc", "inh"}
                )
            )

            cell19f_dataset_row["N"] = int(cell19f_N)
            cell19f_dataset_row["neuron_nt_known_count"] = cell19f_neuron_nt_known_count

            print("[cell19f] N=" + f"{cell19f_N:,}" + " neuron_nt_known=" + f"{cell19f_neuron_nt_known_count:,}")

            cell19f_connection_dict, cell19f_connection_load_status = cell19f_load_connection_table(
                cell19f_connection_path,
                cell19f_id_map,
                cell19f_N,
                cell19f_neuron_nt_class,
                cell19f_diagnostic_dict,
            )

            cell19f_dataset_row["connection_rows_total"] = cell19f_diagnostic_dict.get("connection_rows_total", "")
            cell19f_dataset_row["connection_rows_mapped"] = cell19f_diagnostic_dict.get("connection_rows_mapped", "")
            cell19f_dataset_row["signed_rows_before_neuron_fallback"] = cell19f_diagnostic_dict.get(
                "signed_rows_before_neuron_fallback",
                "",
            )
            cell19f_dataset_row["signed_rows_after_neuron_fallback"] = cell19f_diagnostic_dict.get(
                "signed_rows_after_neuron_fallback",
                "",
            )
            cell19f_dataset_row["major_edges_positive"] = cell19f_diagnostic_dict.get("major_edges_positive", "")
            cell19f_dataset_row["major_edges_negative"] = cell19f_diagnostic_dict.get("major_edges_negative", "")

            if "nt_type_top_values" in cell19f_diagnostic_dict:
                cell19f_dataset_row["nt_type_top_values"] = str(cell19f_diagnostic_dict["nt_type_top_values"])[:1000]

            print("[cell19f] connection diagnostic keys=" + str(list(cell19f_diagnostic_dict.keys())))
            print("[cell19f] connection_rows_total=" + str(cell19f_diagnostic_dict.get("connection_rows_total", "")))
            print("[cell19f] connection_rows_mapped=" + str(cell19f_diagnostic_dict.get("connection_rows_mapped", "")))
            print("[cell19f] signed_before_fallback=" + str(cell19f_diagnostic_dict.get("signed_rows_before_neuron_fallback", "")))
            print("[cell19f] signed_after_fallback=" + str(cell19f_diagnostic_dict.get("signed_rows_after_neuron_fallback", "")))

            if "nt_type_top_values" in cell19f_diagnostic_dict:
                print("[cell19f] nt_type_top_values=" + str(cell19f_diagnostic_dict["nt_type_top_values"])[:1000])

            if cell19f_connection_dict is None:
                cell19f_dataset_row["status"] = "connection_load_failed"
                cell19f_dataset_row["error"] = str(cell19f_connection_load_status)
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                print("[cell19f][fail] connection_load_failed: " + str(cell19f_connection_load_status))
                continue

            cell19f_src = cell19f_connection_dict["src"]
            cell19f_dst = cell19f_connection_dict["dst"]
            cell19f_sign_bin = cell19f_connection_dict["sign_bin"]
            cell19f_syn = cell19f_connection_dict["syn"]

            cell19f_pos_count = int(cell19f_np.sum(cell19f_sign_bin > 0.5))
            cell19f_neg_count = int(cell19f_np.sum(cell19f_sign_bin <= 0.5))

            cell19f_dataset_row["n_edges_signed"] = int(cell19f_src.size)
            cell19f_dataset_row["n_positive_edges"] = cell19f_pos_count
            cell19f_dataset_row["n_negative_edges"] = cell19f_neg_count

            print(
                "[cell19e-equivalent] signed_edges=" + f"{cell19f_src.size:,}"
                + " pos=" + f"{cell19f_pos_count:,}"
                + " neg=" + f"{cell19f_neg_count:,}"
            )

            if cell19f_pos_count < int(cell19f_min_positive_edges):
                cell19f_dataset_row["status"] = "insufficient_positive_edges"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            if cell19f_neg_count < int(cell19f_min_negative_edges):
                cell19f_dataset_row["status"] = "insufficient_negative_edges"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            if cell19f_neg_count < int(cell19f_warn_negative_edges):
                cell19f_dataset_row["signed_graph_warning"] = "few_inhibitory_edges"

            cell19f_X_raw, cell19f_motif_mode, cell19f_feature_status = cell19f_build_target_features(
                cell19f_neuron_frame,
                cell19f_N,
                cell19f_connection_dict,
            )

            cell19f_dataset_row["motif_mode"] = str(cell19f_motif_mode)

            if cell19f_X_raw is None:
                cell19f_dataset_row["status"] = "feature_build_failed"
                cell19f_dataset_row["error"] = str(cell19f_feature_status)
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            cell19f_ckpt_standardized_flag = False

            if (
                int(cell19f_ckpt_mean_np.size) == int(cell19f_X_raw.shape[1])
                and int(cell19f_ckpt_std_np.size) == int(cell19f_X_raw.shape[1])
            ):
                cell19f_ckpt_standardized_flag = bool(
                    cell19f_np.all(cell19f_np.abs(cell19f_ckpt_mean_np) < 0.10)
                    and cell19f_np.all(cell19f_np.abs(cell19f_ckpt_std_np - 1.0) < 0.25)
                )

            if cell19f_ckpt_standardized_flag:
                cell19f_target_mean = cell19f_X_raw.mean(axis=0)
                cell19f_target_std = cell19f_X_raw.std(axis=0)
                cell19f_X_model_np = (cell19f_X_raw - cell19f_target_mean) / (cell19f_target_std + 1e-8)
                cell19f_normalization_mode = "target_zscore_because_checkpoint_standardized"
            elif (
                int(cell19f_ckpt_mean_np.size) == int(cell19f_X_raw.shape[1])
                and int(cell19f_ckpt_std_np.size) == int(cell19f_X_raw.shape[1])
            ):
                cell19f_X_model_np = (
                    cell19f_X_raw - cell19f_ckpt_mean_np
                ) / (cell19f_ckpt_std_np + 1e-8)
                cell19f_normalization_mode = "checkpoint_zscore"
            else:
                cell19f_target_mean = cell19f_X_raw.mean(axis=0)
                cell19f_target_std = cell19f_X_raw.std(axis=0)
                cell19f_X_model_np = (cell19f_X_raw - cell19f_target_mean) / (cell19f_target_std + 1e-8)
                cell19f_normalization_mode = "target_zscore_because_checkpoint_dimension_mismatch"

            cell19f_X_model_np = cell19f_np.nan_to_num(
                cell19f_X_model_np,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            cell19f_dataset_row["normalization_mode"] = str(cell19f_normalization_mode)

            cell19f_X_model_tensor = cell19f_torch.tensor(
                cell19f_X_model_np,
                dtype=cell19f_torch.float32,
                device=cell19f_device,
            )

            cell19f_ei_full_new = cell19f_torch.tensor(
                cell19f_np.vstack([cell19f_src, cell19f_dst]),
                dtype=cell19f_torch.long,
                device=cell19f_device,
            )

            cell19f_syn_log_np = cell19f_np.log1p(cell19f_syn.astype(cell19f_np.float64))
            cell19f_syn_mean_value = float(cell19f_syn_log_np.mean()) if cell19f_syn_log_np.size > 0 else 1.0
            cell19f_edge_weight_np = (cell19f_syn_log_np / (cell19f_syn_mean_value + 1e-8)).astype(
                cell19f_np.float32
            )

            cell19f_edge_weight_tensor = cell19f_torch.tensor(
                cell19f_edge_weight_np,
                dtype=cell19f_torch.float32,
                device=cell19f_device,
            )

            cell19f_pos_mask_array = cell19f_sign_bin > 0.5
            cell19f_pos_mask_tensor = cell19f_torch.tensor(cell19f_pos_mask_array, device=cell19f_device)
            cell19f_neg_mask_tensor = ~cell19f_pos_mask_tensor

            cell19f_ei_pos_new = cell19f_ei_full_new[:, cell19f_pos_mask_tensor]
            cell19f_ew_pos_new = cell19f_edge_weight_tensor[cell19f_pos_mask_tensor]
            cell19f_ei_neg_new = cell19f_ei_full_new[:, cell19f_neg_mask_tensor]
            cell19f_ew_neg_new = cell19f_edge_weight_tensor[cell19f_neg_mask_tensor]

            cell19f_dis_pos_new = cell19f_normalize_edge_degree(
                cell19f_ei_pos_new,
                cell19f_ew_pos_new,
                cell19f_N,
                cell19f_device,
            )
            cell19f_dis_neg_new = cell19f_normalize_edge_degree(
                cell19f_ei_neg_new,
                cell19f_ew_neg_new,
                cell19f_N,
                cell19f_device,
            )

            try:
                with cell19f_torch.no_grad():
                    cell19f_model_output = cell19f_forward_frozen_model(
                        cell19f_frozen_model,
                        cell19f_X_model_tensor,
                        cell19f_ei_pos_new,
                        cell19f_ew_pos_new,
                        cell19f_dis_pos_new,
                        cell19f_ei_neg_new,
                        cell19f_ew_neg_new,
                        cell19f_dis_neg_new,
                    )

                    cell19f_embedding_tensor = cell19f_extract_embedding(
                        cell19f_model_output,
                        cell19f_frozen_model,
                        cell19f_N,
                    )

                    cell19f_embedding_np = cell19f_embedding_tensor.detach().cpu().numpy()
                    cell19f_embedding_np = cell19f_np.nan_to_num(
                        cell19f_embedding_np,
                        nan=0.0,
                        posinf=0.0,
                        neginf=0.0,
                    )

            except Exception as cell19f_forward_error:
                cell19f_dataset_row["status"] = "forward_failed"
                cell19f_dataset_row["error"] = repr(cell19f_forward_error)
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                print("[cell19f][fail] forward_failed: " + repr(cell19f_forward_error))
                continue

            if cell19f_embedding_np.size == 0:
                cell19f_dataset_row["status"] = "empty_embedding"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            cell19f_embedding_std = float(cell19f_np.std(cell19f_embedding_np))

            if not cell19f_np.isfinite(cell19f_embedding_std) or cell19f_embedding_std < 1e-12:
                cell19f_dataset_row["status"] = "degenerate_embedding"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            cell19f_label_column, cell19f_unique_labels, cell19e_valid_count_preview = (
                cell19f_choose_label_column(cell19f_neuron_frame)
            )

            cell19f_dataset_row["label_column"] = str(cell19f_label_column) if cell19f_label_column is not None else ""

            if cell19f_label_column is None:
                cell19f_dataset_row["status"] = "too_few_usable_labels"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            cell19f_label_series = cell19f_neuron_frame[cell19f_label_column].astype(str).str.strip()
            cell19f_label_strings = cell19f_label_series.to_numpy()

            cell19f_label_to_index = {
                cell19f_label_string: cell19f_label_index
                for cell19f_label_index, cell19f_label_string in enumerate(cell19f_unique_labels)
            }

            cell19f_y_all = cell19f_np.array(
                [
                    cell19f_label_to_index.get(str(cell19f_label_value).strip(), -1)
                    for cell19f_label_value in cell19f_label_strings
                ],
                dtype=cell19f_np.int64,
            )

            cell19f_valid_mask = cell19f_y_all >= 0
            cell19f_n_labeled = int(cell19f_np.sum(cell19f_valid_mask))

            cell19f_dataset_row["n_labeled"] = cell19f_n_labeled
            cell19f_dataset_row["K"] = len(cell19f_unique_labels)

            if cell19f_n_labeled < int(cell19f_min_labeled_nodes):
                cell19f_dataset_row["status"] = "too_few_labeled"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            cell19f_K = len(cell19f_unique_labels)

            if cell19f_K < 2:
                cell19f_dataset_row["status"] = "too_few_classes"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            if cell19f_K > int(cell19f_n_labeled):
                cell19f_dataset_row["status"] = "too_many_classes_for_labeled_nodes"
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                continue

            try:
                cell19f_kmeans_object = cell19f_KMeans(
                    n_clusters=cell19f_K,
                    random_state=0,
                    n_init=10,
                )
                cell19f_kmeans_object.fit(cell19f_embedding_np)
                cell19f_pred_all = cell19f_kmeans_object.labels_
            except Exception as cell19f_cluster_error:
                cell19f_dataset_row["status"] = "cluster_failed"
                cell19f_dataset_row["error"] = repr(cell19f_cluster_error)
                cell19f_results_dict[cell19f_dataset_name] = {
                    "row": cell19f_dataset_row.copy(),
                    "diagnostics": cell19f_diagnostic_dict,
                }
                cell19f_summary_rows.append(cell19f_dataset_row)
                print("[cell19f][fail] cluster_failed: " + repr(cell19f_cluster_error))
                continue

            cell19f_y_true = cell19f_y_all[cell19f_valid_mask]
            cell19f_y_pred = cell19f_pred_all[cell19f_valid_mask]

            cell19f_ari = float(cell19f_ari_score(cell19f_y_true, cell19f_y_pred))
            cell19f_nmi = float(cell19f_nmi_score(cell19f_y_true, cell19f_y_pred))

            cell19f_rng = cell19f_np.random.default_rng(0)
            cell19f_null_aris = []
            cell19f_null_nmis = []

            for cell19f_null_index in range(int(cell19f_null_permutations)):
                cell19f_y_perm = cell19f_y_true.copy()
                cell19f_rng.shuffle(cell19f_y_perm)
                cell19f_null_aris.append(float(cell19f_ari_score(cell19f_y_perm, cell19f_y_pred)))
                cell19f_null_nmis.append(float(cell19f_nmi_score(cell19f_y_perm, cell19f_y_pred)))

            cell19f_ari_null = float(cell19f_np.mean(cell19f_null_aris))
            cell19f_nmi_null = float(cell19f_np.mean(cell19f_null_nmis))
            cell19f_ari_delta = float(cell19f_ari - cell19f_ari_null)
            cell19f_nmi_delta = float(cell19f_nmi - cell19f_nmi_null)

            cell19f_ari_p_empirical = float(
                (
                    sum(
                        1
                        for cell19f_null_ari_value in cell19f_null_aris
                        if cell19f_null_ari_value >= cell19f_ari
                    )
                    + 1
                )
                / float(len(cell19f_null_aris) + 1)
            )

            cell19f_nmi_p_empirical = float(
                (
                    sum(
                        1
                        for cell19f_null_nmi_value in cell19f_null_nmis
                        if cell19f_null_nmi_value >= cell19f_nmi
                    )
                    + 1
                )
                / float(len(cell19f_null_nmis) + 1)
            )

            cell19f_dataset_row["status"] = "ok"
            cell19f_dataset_row["ari"] = float(cell19f_ari)
            cell19f_dataset_row["ari_null"] = float(cell19f_ari_null)
            cell19f_dataset_row["ari_delta"] = float(cell19f_ari_delta)
            cell19f_dataset_row["ari_p_empirical"] = float(cell19f_ari_p_empirical)
            cell19f_dataset_row["nmi"] = float(cell19f_nmi)
            cell19f_dataset_row["nmi_null"] = float(cell19f_nmi_null)
            cell19f_dataset_row["nmi_delta"] = float(cell19f_nmi_delta)
            cell19f_dataset_row["nmi_p_empirical"] = float(cell19f_nmi_p_empirical)

            cell19f_results_dict[cell19f_dataset_name] = {
                "row": cell19f_dataset_row.copy(),
                "diagnostics": cell19f_diagnostic_dict,
            }
            cell19f_summary_rows.append(cell19f_dataset_row)

            print(
                "[cell19f][ok] label_column=" + str(cell19f_label_column)
                + " K=" + str(cell19f_K)
                + " labeled=" + f"{cell19f_n_labeled:,}"
            )
            print(
                "[cell19f][ok] ARI=" + f"{cell19f_ari:.4f}"
                + " null=" + f"{cell19f_ari_null:.4f}"
                + " delta=" + f"{cell19f_ari_delta:+.4f}"
                + " p_emp=" + f"{cell19f_ari_p_empirical:.3f}"
            )
            print(
                "[cell19f][ok] NMI=" + f"{cell19f_nmi:.4f}"
                + " null=" + f"{cell19f_nmi_null:.4f}"
                + " delta=" + f"{cell19f_nmi_delta:+.4f}"
                + " p_emp=" + f"{cell19f_nmi_p_empirical:.3f}"
            )

        except Exception as cell19f_dataset_error:
            cell19f_dataset_row["status"] = "exception"
            cell19f_dataset_row["error"] = repr(cell19f_dataset_error)
            cell19f_results_dict[cell19f_dataset_name] = {
                "row": cell19f_dataset_row.copy(),
                "diagnostics": cell19f_diagnostic_dict,
            }
            cell19f_summary_rows.append(cell19f_dataset_row)
            print("[cell19f][fail] exception: " + repr(cell19f_dataset_error))

    # ------------------------------------------------------------------
    # Final summary.
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("CELL 19f — CORRECTED CROSS-CONNECTOME SUMMARY")
    print("=" * 100)

    print(
        f"{'dataset':>8s} {'status':>26s} {'N':>10s} {'edges':>10s} "
        f"{'label':>16s} {'K':>4s} {'labeled':>10s} "
        f"{'ARI':>8s} {'dARI':>8s} {'p':>6s} "
        f"{'NMI':>8s} {'dNMI':>8s} {'p':>6s}"
    )

    for cell19f_dataset_name in cell19f_datasets:
        cell19f_result_container = cell19f_results_dict.get(cell19f_dataset_name, {})
        cell19f_row_dict = cell19f_result_container.get("row", {"status": "missing"})

        if cell19f_row_dict.get("status") == "ok":
            print(
                f"{str(cell19f_row_dict.get('dataset', '')):>8s} "
                f"{str(cell19f_row_dict.get('status', '')):>26s} "
                f"{int(cell19f_row_dict.get('N', 0)):>10,d} "
                f"{int(cell19f_row_dict.get('n_edges_signed', 0)):>10,d} "
                f"{str(cell19f_row_dict.get('label_column', ''))[:16]:>16s} "
                f"{int(cell19f_row_dict.get('K', 0)):>4d} "
                f"{int(cell19f_row_dict.get('n_labeled', 0)):>10,d} "
                f"{float(cell19f_row_dict.get('ari', float('nan'))):>8.4f} "
                f"{float(cell19f_row_dict.get('ari_delta', float('nan'))):>+8.4f} "
                f"{float(cell19f_row_dict.get('ari_p_empirical', float('nan'))):>6.3f} "
                f"{float(cell19f_row_dict.get('nmi', float('nan'))):>8.4f} "
                f"{float(cell19f_row_dict.get('nmi_delta', float('nan'))):>+8.4f} "
                f"{float(cell19f_row_dict.get('nmi_p_empirical', float('nan'))):>6.3f}"
            )
        else:
            print(
                f"{str(cell19f_row_dict.get('dataset', cell19f_dataset_name)):>8s} "
                f"{str(cell19f_row_dict.get('status', 'missing')):>26s} "
                f"{str(cell19f_row_dict.get('N', '')):>10s} "
                f"{str(cell19f_row_dict.get('n_edges_signed', '')):>10s} "
                f"{str(cell19f_row_dict.get('label_column', ''))[:16]:>16s} "
                f"{str(cell19f_row_dict.get('K', '')):>4s} "
                f"{str(cell19f_row_dict.get('n_labeled', '')):>10s} "
                f"{'':>8s} {'':>8s} {'':>6s} "
                f"{'':>8s} {'':>8s} {'':>6s}"
            )

            if str(cell19f_row_dict.get("error", "")) != "":
                print("    error=" + str(cell19f_row_dict.get("error", ""))[:500])

            if "connection_rows_total" in cell19f_row_dict:
                print("    connection_rows_total=" + str(cell19f_row_dict.get("connection_rows_total", "")))
                print("    connection_rows_mapped=" + str(cell19f_row_dict.get("connection_rows_mapped", "")))
                print("    signed_before_fallback=" + str(cell19f_row_dict.get("signed_rows_before_neuron_fallback", "")))
                print("    signed_after_fallback=" + str(cell19f_row_dict.get("signed_rows_after_neuron_fallback", "")))

            if "nt_type_top_values" in cell19f_row_dict:
                print("    nt_type_top_values=" + str(cell19f_row_dict.get("nt_type_top_values", ""))[:1000])

    print("=" * 100)

    # ------------------------------------------------------------------
    # Save artifacts.
    # ------------------------------------------------------------------
    cell19f_summary_columns = [
        "dataset",
        "version",
        "status",
        "N",
        "n_edges_signed",
        "n_positive_edges",
        "n_negative_edges",
        "label_column",
        "K",
        "n_labeled",
        "normalization_mode",
        "motif_mode",
        "signed_graph_warning",
        "ari",
        "ari_null",
        "ari_delta",
        "ari_p_empirical",
        "nmi",
        "nmi_null",
        "nmi_delta",
        "nmi_p_empirical",
        "error",
        "connection_rows_total",
        "connection_rows_mapped",
        "signed_rows_before_neuron_fallback",
        "signed_rows_after_neuron_fallback",
        "major_edges_positive",
        "major_edges_negative",
        "neuron_nt_known_count",
        "nt_type_top_values",
    ]

    cell19f_summary_df = cell19f_pd.DataFrame(cell19f_summary_rows, columns=cell19f_summary_columns)

    cell19f_summary_csv_path = cell19f_output_dir / "cell19f_cross_connectome_summary.csv"
    cell19f_results_json_path = cell19f_output_dir / "cell19f_cross_connectome_results.json"
    cell19f_results_pickle_path = cell19f_output_dir / "cell19f_cross_connectome_results.pkl"

    cell19f_summary_df.to_csv(cell19f_summary_csv_path, index=False)

    cell19f_results_json_path.write_text(
        cell19f_json.dumps(cell19f_results_dict, indent=2, default=str),
        encoding="utf-8",
    )

    with open(cell19f_results_pickle_path, "wb") as cell19f_pickle_handle:
        cell19f_pickle.dump(cell19f_results_dict, cell19f_pickle_handle)

    print("Saved summary CSV: " + str(cell19f_summary_csv_path))
    print("Saved results JSON: " + str(cell19f_results_json_path))
    print("Saved results PKL:  " + str(cell19f_results_pickle_path))

    # ------------------------------------------------------------------
    # Optional Hugging Face checkpoint/upload.
    # ------------------------------------------------------------------
    cell19f_hf_save_checkpoint = cell19f_globals.get("hf_save_checkpoint", None)

    if callable(cell19f_hf_save_checkpoint):
        try:
            cell19f_hf_save_checkpoint(
                "cell19f_cross_connectome_corrected",
                {
                    "results": cell19f_results_dict,
                    "summary_rows": cell19f_summary_rows,
                    "frozen_path": str(cell19f_frozen_path),
                    "schema_note": (
                        "Uses neurons.csv.gz + connections_princeton.csv.gz. "
                        "Edge signs are inferred from nt_type with expanded aliases, "
                        "falling back to presynaptic neuron Verified/Predicted NT type. "
                        "Motif channels are zero-omitted for large graphs."
                    ),
                    "paths": {
                        "summary_csv": str(cell19f_summary_csv_path),
                        "results_json": str(cell19f_results_json_path),
                        "results_pickle": str(cell19f_results_pickle_path),
                    },
                },
                description=(
                    "Cell 19f: corrected cross-connectome structural coherence with connection-schema diagnostics."
                ),
                upload=True,
            )
        except Exception as cell19f_checkpoint_error:
            print("[ckpt][warn] Cell 19f checkpoint failed: " + repr(cell19f_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    cell19f_hf_upload_file_to_bucket = cell19f_globals.get("hf_upload_file_to_bucket", None)

    if callable(cell19f_hf_upload_file_to_bucket):
        cell19f_remote_paths = {
            str(cell19f_summary_csv_path): "checkpoints/cell19f_cross_connectome/cell19f_cross_connectome_summary.csv",
            str(cell19f_results_json_path): "checkpoints/cell19f_cross_connectome/cell19f_cross_connectome_results.json",
            str(cell19f_results_pickle_path): "checkpoints/cell19f_cross_connectome/cell19f_cross_connectome_results.pkl",
        }

        for cell19f_local_path_string, cell19f_remote_path_string in cell19f_remote_paths.items():
            try:
                cell19f_upload_ok, cell19f_upload_info = cell19f_hf_upload_file_to_bucket(
                    cell19f_Path(cell19f_local_path_string),
                    cell19f_remote_path_string,
                )
                print(
                    "[ckpt:cell19f] upload "
                    + cell19f_Path(cell19f_local_path_string).name
                    + ": "
                    + ("OK" if cell19f_upload_ok else "FAILED")
                    + " | "
                    + str(cell19f_upload_info)
                )
            except Exception as cell19f_upload_error:
                print(
                    "[ckpt][warn] Cell 19f upload failed for "
                    + cell19f_Path(cell19f_local_path_string).name
                    + ": "
                    + repr(cell19f_upload_error)
                )
    else:
        print(
            "[ckpt][skip] hf_upload_file_to_bucket not defined. "
            "Cross-connectome files remain local only."
        )

    return {
        "results": cell19f_results_dict,
        "summary_rows": cell19f_summary_rows,
        "output_dir": str(cell19f_output_dir),
        "summary_csv_path": str(cell19f_summary_csv_path),
        "results_json_path": str(cell19f_results_json_path),
        "results_pickle_path": str(cell19f_results_pickle_path),
        "frozen_path": str(cell19f_frozen_path),
    }


cell19f_output = cell19f_main()

cell19f_cross_results = cell19f_output["results"]
cell19f_cross_summary_rows = cell19f_output["summary_rows"]
cell19f_cross_output_dir = cell19f_output["output_dir"]
cell19f_cross_summary_csv_path = cell19f_output["summary_csv_path"]
cell19f_cross_results_json_path = cell19f_output["results_json_path"]
cell19f_cross_results_pickle_path = cell19f_output["results_pickle_path"]
cell19f_cross_frozen_path = cell19f_output["frozen_path"]

In [ ]:
# Cell 20: Define Hierarchical Empirical Bayes (HEB) prototype head.
#
# HEB is a defensive ablation for SSPR:
#   - class prototypes shrink toward superclass prototypes
#   - shrinkage weight is n_c / (kappa0 + n_c)
#   - distance uses Student-t posterior predictive likelihood
#   - same signed spectral encoder as SPGCN/SSPR
#
# This cell only installs local helper functions/classes into cell20_heb_api.
# It does not contact any external service.
# It does not install a package.
# It does not run the 10-seed experiment.
#
# Rules:
#   - No top-level imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every notebook-local/helper/loop variable is prefixed with cell20_.


def cell20_install_heb_api():
    cell20_g = globals()

    cell20_torch = cell20_g.get("torch", None)

    if cell20_torch is None:
        raise NameError(
            "[Cell 20][FATAL] torch is not available. "
            "Run earlier cells first."
        )

    cell20_nn = cell20_torch.nn
    cell20_F = cell20_torch.nn.functional

    def cell20_get_kwarg(cell20_kwargs, cell20_names, cell20_default=None):
        for cell20_name in cell20_names:
            if cell20_name in cell20_kwargs:
                cell20_value = cell20_kwargs[cell20_name]

                if cell20_value is not None:
                    return cell20_value

        return cell20_default

    def cell20_collect_tensors(cell20_obj, cell20_out=None):
        if cell20_out is None:
            cell20_out = []

        if cell20_torch.is_tensor(cell20_obj):
            if int(cell20_obj.ndim) >= 2:
                cell20_out.append(cell20_obj)
            return cell20_out

        if isinstance(cell20_obj, dict):
            for cell20_value in cell20_obj.values():
                cell20_collect_tensors(cell20_value, cell20_out)
            return cell20_out

        if isinstance(cell20_obj, (tuple, list)):
            for cell20_value in cell20_obj:
                cell20_collect_tensors(cell20_value, cell20_out)
            return cell20_out

        return cell20_out

    def cell20_extract_embedding(cell20_obj, cell20_expected_rows):
        cell20_tensors = cell20_collect_tensors(cell20_obj)

        if len(cell20_tensors) == 0:
            raise TypeError(
                "[Cell 20][FATAL] Encoder output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell20_obj))
            )

        for cell20_tensor in cell20_tensors:
            if int(cell20_tensor.shape[0]) == int(cell20_expected_rows):
                return cell20_tensor

        return cell20_tensors[0]

    class cell20_HEBPrototypeWrapper(cell20_nn.Module):
        """
        Wraps an existing SPGCN-style encoder and replaces the prototype
        classification rule with a hierarchical empirical Bayes head.

        The encoder is expected to expose:
            proto_class: [n_classes, d_proto]
            proto_super: [n_super, d_proto]

        The HEB head adds:
            log_scale: per-class Student-t scale parameter
            sup_of_class: class -> superclass map
        """

        def __init__(
            self,
            cell20_encoder_model,
            cell20_n_classes,
            cell20_n_super,
            cell20_d_proto,
            cell20_kappa0=4.0,
        ):
            super().__init__()

            self.encoder = cell20_encoder_model
            self.n_classes = int(cell20_n_classes)
            self.n_super = int(cell20_n_super)
            self.kappa0 = float(cell20_kappa0)

            if not hasattr(cell20_encoder_model, "proto_class"):
                raise AttributeError(
                    "[Cell 20][FATAL] Encoder model has no proto_class parameter."
                )

            if not hasattr(cell20_encoder_model, "proto_super"):
                raise AttributeError(
                    "[Cell 20][FATAL] Encoder model has no proto_super parameter."
                )

            cell20_proto_class_parameter = getattr(cell20_encoder_model, "proto_class")
            cell20_proto_super_parameter = getattr(cell20_encoder_model, "proto_super")

            if not cell20_torch.is_tensor(cell20_proto_class_parameter):
                raise TypeError(
                    "[Cell 20][FATAL] encoder.proto_class is not a torch tensor/parameter."
                )

            if not cell20_torch.is_tensor(cell20_proto_super_parameter):
                raise TypeError(
                    "[Cell 20][FATAL] encoder.proto_super is not a torch tensor/parameter."
                )

            if int(cell20_proto_class_parameter.ndim) != 2:
                raise RuntimeError(
                    "[Cell 20][FATAL] encoder.proto_class must be 2D, got shape "
                    + str(tuple(cell20_proto_class_parameter.shape))
                )

            if int(cell20_proto_super_parameter.ndim) != 2:
                raise RuntimeError(
                    "[Cell 20][FATAL] encoder.proto_super must be 2D, got shape "
                    + str(tuple(cell20_proto_super_parameter.shape))
                )

            if int(cell20_proto_class_parameter.shape[0]) != self.n_classes:
                raise RuntimeError(
                    "[Cell 20][FATAL] proto_class row count mismatch: "
                    + str(int(cell20_proto_class_parameter.shape[0]))
                    + " vs n_classes="
                    + str(self.n_classes)
                )

            if int(cell20_proto_super_parameter.shape[0]) != self.n_super:
                raise RuntimeError(
                    "[Cell 20][FATAL] proto_super row count mismatch: "
                    + str(int(cell20_proto_super_parameter.shape[0]))
                    + " vs n_super="
                    + str(self.n_super)
                )

            self.d_proto = int(cell20_proto_class_parameter.shape[1])

            if int(cell20_d_proto) != self.d_proto:
                raise RuntimeError(
                    "[Cell 20][FATAL] Requested d_proto="
                    + str(int(cell20_d_proto))
                    + " but encoder proto_class has d="
                    + str(self.d_proto)
                )

            self.log_scale = cell20_nn.Parameter(
                cell20_torch.zeros(
                    self.n_classes,
                    dtype=cell20_proto_class_parameter.dtype,
                    device=cell20_proto_class_parameter.device,
                )
            )

            self.register_buffer(
                "sup_of_class",
                cell20_torch.zeros(
                    self.n_classes,
                    dtype=cell20_torch.long,
                    device=cell20_proto_class_parameter.device,
                ),
            )

            self.register_buffer(
                "nu",
                cell20_torch.tensor(
                    float(self.d_proto),
                    dtype=cell20_proto_class_parameter.dtype,
                    device=cell20_proto_class_parameter.device,
                ),
            )

        @property
        def proto_class(self):
            return self.encoder.proto_class

        @property
        def proto_super(self):
            return self.encoder.proto_super

        def set_superclass_map(self, cell20_sup_of_class):
            cell20_map_tensor = cell20_torch.as_tensor(
                cell20_sup_of_class,
                dtype=cell20_torch.long,
                device=self.sup_of_class.device,
            )

            if int(cell20_map_tensor.numel()) != self.n_classes:
                raise RuntimeError(
                    "[Cell 20][FATAL] superclass map length mismatch: "
                    + str(int(cell20_map_tensor.numel()))
                    + " vs n_classes="
                    + str(self.n_classes)
                )

            self.sup_of_class.copy_(cell20_map_tensor)

        def shrunk_centers(self, cell20_n_c):
            cell20_n_c_f = cell20_n_c.float().to(
                device=self.proto_class.device,
                dtype=self.proto_class.dtype,
            )

            cell20_w = (
                cell20_n_c_f / (self.kappa0 + cell20_n_c_f)
            ).unsqueeze(-1)

            cell20_super_centers = self.proto_super[self.sup_of_class]

            cell20_centers = (
                cell20_w * self.proto_class
                + (1.0 - cell20_w) * cell20_super_centers
            )

            return cell20_centers

        def student_t_nll(self, cell20_h, cell20_centers):
            cell20_dist2 = cell20_torch.cdist(cell20_h, cell20_centers).pow(2)

            cell20_scales = self.log_scale.exp().clamp(min=1e-3, max=1e3)

            cell20_nu = self.nu.to(dtype=cell20_h.dtype)
            cell20_d = float(self.d_proto)

            cell20_pi = cell20_torch.tensor(
                3.141592653589793,
                dtype=cell20_h.dtype,
                device=cell20_h.device,
            )

            cell20_log_norm = (
                cell20_torch.lgamma((cell20_nu + cell20_d) / 2.0)
                - cell20_torch.lgamma(cell20_nu / 2.0)
                - 0.5 * cell20_d * cell20_torch.log(cell20_nu * cell20_pi)
                - cell20_d * cell20_torch.log(cell20_scales)
            )

            cell20_log_kernel = (
                (cell20_nu + cell20_d) / 2.0
            ) * cell20_torch.log1p(
                cell20_dist2 / (cell20_nu * cell20_scales.pow(2))
            )

            cell20_nll = cell20_log_kernel - cell20_log_norm.unsqueeze(0)

            return cell20_torch.nan_to_num(
                cell20_nll,
                nan=1e4,
                posinf=1e4,
                neginf=-1e4,
            )

        def compute_nll(self, cell20_h, cell20_labels, cell20_train_idx):
            cell20_train_labels = cell20_labels[cell20_train_idx].clamp(min=0).long()

            cell20_n_c = cell20_torch.bincount(
                cell20_train_labels,
                minlength=self.n_classes,
            ).to(dtype=cell20_h.dtype)

            cell20_centers = self.shrunk_centers(cell20_n_c)

            return self.student_t_nll(cell20_h, cell20_centers)

        def _encode_raw(
            self,
            cell20_X,
            cell20_ei_pos,
            cell20_ew_pos,
            cell20_dis_pos,
            cell20_ei_neg,
            cell20_ew_neg,
            cell20_dis_neg,
        ):
            cell20_last_type_error = None

            try:
                return self.encoder(
                    cell20_X,
                    cell20_ei_pos,
                    cell20_ew_pos,
                    cell20_dis_pos,
                    cell20_ei_neg,
                    cell20_ew_neg,
                    cell20_dis_neg,
                )
            except TypeError as cell20_type_error:
                cell20_last_type_error = cell20_type_error

            try:
                return self.encoder(
                    cell20_X,
                    cell20_ei_pos,
                    cell20_dis_pos,
                    cell20_ei_neg,
                    cell20_dis_neg,
                )
            except TypeError as cell20_type_error:
                cell20_last_type_error = cell20_type_error

            try:
                return self.encoder(
                    cell20_X,
                    cell20_ei_pos,
                    cell20_ei_neg,
                )
            except TypeError as cell20_type_error:
                cell20_last_type_error = cell20_type_error

            raise TypeError(
                "[Cell 20][FATAL] Could not call wrapped encoder with any known signature. "
                "Last TypeError: " + repr(cell20_last_type_error)
            )

        def forward(
            self,
            cell20_X,
            cell20_ei_pos,
            cell20_ew_pos,
            cell20_dis_pos,
            cell20_ei_neg,
            cell20_ew_neg,
            cell20_dis_neg,
        ):
            cell20_raw_output = self._encode_raw(
                cell20_X,
                cell20_ei_pos,
                cell20_ew_pos,
                cell20_dis_pos,
                cell20_ei_neg,
                cell20_ew_neg,
                cell20_dis_neg,
            )

            return cell20_extract_embedding(
                cell20_raw_output,
                int(cell20_X.shape[0]),
            )

    def cell20_make_encoder(*cell20_args, **cell20_kwargs):
        cell20_d_features = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_d_features",
                "cell21_d_features",
                "d_features",
            ],
            None,
        )

        if cell20_d_features is None and len(cell20_args) > 0:
            cell20_d_features = cell20_args[0]

        if cell20_d_features is None:
            raise TypeError(
                "[Cell 20][FATAL] cell20_make_encoder requires d_features."
            )

        cell20_d = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_d",
                "cell21_d",
                "d",
            ],
            64,
        )

        cell20_d_proto = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_d_proto",
                "cell21_d_proto",
                "d_proto",
            ],
            32,
        )

        cell20_n_layers = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_n_layers",
                "cell21_n_layers",
                "n_layers",
            ],
            2,
        )

        cell20_n_classes = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_n_classes",
                "cell21_n_classes",
                "n_classes",
            ],
            29,
        )

        cell20_n_super = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_n_super",
                "cell21_n_super",
                "n_super",
            ],
            10,
        )

        cell20_dropout = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_dropout",
                "cell21_dropout",
                "dropout",
            ],
            0.3,
        )

        cell20_spgcn_class = cell20_g.get("SPGCN_cell5", None)

        if cell20_spgcn_class is None:
            raise NameError(
                "[Cell 20][FATAL] SPGCN_cell5 is not available. Run Cell 5 first."
            )

        cell20_last_type_error = None

        try:
            return cell20_spgcn_class(
                d_features_cell5=int(cell20_d_features),
                d_cell5=int(cell20_d),
                d_proto_cell5=int(cell20_d_proto),
                n_layers_cell5=int(cell20_n_layers),
                n_classes_cell5=int(cell20_n_classes),
                n_super_cell5=int(cell20_n_super),
                temperature_cell5=0.1,
                dropout_cell5=float(cell20_dropout),
            )
        except TypeError as cell20_type_error:
            cell20_last_type_error = cell20_type_error

        try:
            return cell20_spgcn_class(
                d_features=int(cell20_d_features),
                d=int(cell20_d),
                d_proto=int(cell20_d_proto),
                n_layers=int(cell20_n_layers),
                n_classes=int(cell20_n_classes),
                n_super=int(cell20_n_super),
                temperature=0.1,
                dropout=float(cell20_dropout),
            )
        except TypeError as cell20_type_error:
            cell20_last_type_error = cell20_type_error

        try:
            return cell20_spgcn_class(
                int(cell20_d_features),
                int(cell20_d),
                int(cell20_d_proto),
                int(cell20_n_layers),
                int(cell20_n_classes),
                int(cell20_n_super),
                0.1,
                float(cell20_dropout),
            )
        except TypeError as cell20_type_error:
            cell20_last_type_error = cell20_type_error

        raise TypeError(
            "[Cell 20][FATAL] Could not instantiate SPGCN_cell5 encoder. "
            "Last TypeError: " + repr(cell20_last_type_error)
        )

    def cell20_make_model(*cell20_args, **cell20_kwargs):
        cell20_d_features = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_d_features",
                "cell21_d_features",
                "d_features",
            ],
            None,
        )

        if cell20_d_features is None and len(cell20_args) > 0:
            cell20_d_features = cell20_args[0]

        if cell20_d_features is None:
            raise TypeError(
                "[Cell 20][FATAL] cell20_make_model requires d_features."
            )

        cell20_d = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_d",
                "cell21_d",
                "d",
            ],
            64,
        )

        cell20_d_proto = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_d_proto",
                "cell21_d_proto",
                "d_proto",
            ],
            32,
        )

        cell20_n_layers = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_n_layers",
                "cell21_n_layers",
                "n_layers",
            ],
            2,
        )

        cell20_n_classes = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_n_classes",
                "cell21_n_classes",
                "n_classes",
            ],
            29,
        )

        cell20_n_super = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_n_super",
                "cell21_n_super",
                "n_super",
            ],
            10,
        )

        cell20_dropout = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_dropout",
                "cell21_dropout",
                "dropout",
            ],
            0.3,
        )

        cell20_kappa0 = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell20_kappa0",
                "cell21_kappa0",
                "kappa0",
            ],
            4.0,
        )

        cell20_encoder_model = cell20_make_encoder(
            cell20_d_features=cell20_d_features,
            cell20_d=cell20_d,
            cell20_d_proto=cell20_d_proto,
            cell20_n_layers=cell20_n_layers,
            cell20_n_classes=cell20_n_classes,
            cell20_n_super=cell20_n_super,
            cell20_dropout=cell20_dropout,
        )

        cell20_actual_d_proto = int(cell20_encoder_model.proto_class.shape[1])

        return cell20_HEBPrototypeWrapper(
            cell20_encoder_model=cell20_encoder_model,
            cell20_n_classes=cell20_n_classes,
            cell20_n_super=cell20_n_super,
            cell20_d_proto=cell20_actual_d_proto,
            cell20_kappa0=cell20_kappa0,
        )

    def cell20_infer_superclass_map(
        cell20_labels,
        cell20_super_labels,
        cell20_n_classes,
        cell20_n_super,
    ):
        cell20_labels_t = cell20_torch.as_tensor(
            cell20_labels,
            dtype=cell20_torch.long,
        )

        cell20_super_t = cell20_torch.as_tensor(
            cell20_super_labels,
            dtype=cell20_torch.long,
        )

        if int(cell20_labels_t.numel()) != int(cell20_super_t.numel()):
            raise RuntimeError(
                "[Cell 20][FATAL] labels and super_labels length mismatch in superclass map inference."
            )

        cell20_sup_of_class = cell20_torch.zeros(
            int(cell20_n_classes),
            dtype=cell20_torch.long,
            device=cell20_labels_t.device,
        )

        for cell20_c in range(int(cell20_n_classes)):
            cell20_mask = (cell20_labels_t == cell20_c) & (cell20_super_t >= 0)

            if bool(cell20_torch.any(cell20_mask)):
                cell20_values = cell20_super_t[cell20_mask].long()
                cell20_counts = cell20_torch.bincount(
                    cell20_values,
                    minlength=int(cell20_n_super),
                )
                cell20_sup_of_class[cell20_c] = int(cell20_counts.argmax().item())

        return cell20_sup_of_class

    def cell20_move_tensor(cell20_obj, cell20_device):
        if cell20_torch.is_tensor(cell20_obj):
            return cell20_obj.to(cell20_device)

        return cell20_obj

    def cell20_index_tensor(cell20_obj, cell20_device):
        if cell20_torch.is_tensor(cell20_obj):
            return cell20_obj.to(device=cell20_device, dtype=cell20_torch.long)

        return cell20_torch.as_tensor(
            cell20_obj,
            dtype=cell20_torch.long,
            device=cell20_device,
        )

    def cell20_label_tensor(cell20_obj, cell20_device):
        if cell20_torch.is_tensor(cell20_obj):
            return cell20_obj.to(device=cell20_device, dtype=cell20_torch.long)

        return cell20_torch.as_tensor(
            cell20_obj,
            dtype=cell20_torch.long,
            device=cell20_device,
        )

    def cell20_train_heb(
        cell20_model=None,
        cell20_X=None,
        cell20_ei_pos=None,
        cell20_ew_pos=None,
        cell20_dis_pos=None,
        cell20_ei_neg=None,
        cell20_ew_neg=None,
        cell20_dis_neg=None,
        cell20_train_idx=None,
        cell20_val_idx=None,
        cell20_test_idx=None,
        cell20_labels=None,
        cell20_super_labels=None,
        cell20_rare_classes=None,
        cell20_seed=None,
        cell20_epochs=200,
        cell20_device=None,
        cell20_C=29,
        cell20_C_super=10,
        cell20_use_reinit=True,
        cell20_reinit_epoch=40,
        cell20_eval_every=25,
        **cell20_kwargs,
    ):
        cell20_epochs = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell21_epochs",
                "epochs",
            ],
            cell20_epochs,
        )

        cell20_device = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell21_device",
                "device",
            ],
            cell20_device,
        )

        cell20_C = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell21_C",
                "C",
            ],
            cell20_C,
        )

        cell20_C_super = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell21_C_super",
                "C_super",
            ],
            cell20_C_super,
        )

        cell20_use_reinit = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell21_use_reinit",
                "use_reinit",
            ],
            cell20_use_reinit,
        )

        cell20_reinit_epoch = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell21_reinit_epoch",
                "reinit_epoch",
            ],
            cell20_reinit_epoch,
        )

        cell20_eval_every = cell20_get_kwarg(
            cell20_kwargs,
            [
                "cell21_eval_every",
                "eval_every",
            ],
            cell20_eval_every,
        )

        cell20_required_args = [
            ("cell20_model", cell20_model),
            ("cell20_X", cell20_X),
            ("cell20_ei_pos", cell20_ei_pos),
            ("cell20_ew_pos", cell20_ew_pos),
            ("cell20_dis_pos", cell20_dis_pos),
            ("cell20_ei_neg", cell20_ei_neg),
            ("cell20_ew_neg", cell20_ew_neg),
            ("cell20_dis_neg", cell20_dis_neg),
            ("cell20_train_idx", cell20_train_idx),
            ("cell20_val_idx", cell20_val_idx),
            ("cell20_test_idx", cell20_test_idx),
            ("cell20_labels", cell20_labels),
            ("cell20_super_labels", cell20_super_labels),
            ("cell20_rare_classes", cell20_rare_classes),
            ("cell20_seed", cell20_seed),
        ]

        cell20_missing_args = []

        for cell20_arg_name, cell20_arg_value in cell20_required_args:
            if cell20_arg_value is None:
                cell20_missing_args.append(cell20_arg_name)

        if len(cell20_missing_args) > 0:
            raise TypeError(
                "[Cell 20][FATAL] cell20_train_heb missing required arguments: "
                + str(cell20_missing_args)
            )

        cell20_rare_acc_fn = cell20_g.get("rare_balanced_acc_cell6", None)

        if not callable(cell20_rare_acc_fn):
            raise NameError(
                "[Cell 20][FATAL] rare_balanced_acc_cell6 is not available. "
                "Run Cell 6 first."
            )

        if cell20_device is None:
            cell20_device = cell20_g.get("DEVICE_cell1", None)

        if cell20_device is None:
            raise NameError(
                "[Cell 20][FATAL] DEVICE_cell1 is not available and no device was passed."
            )

        cell20_seed_fn = cell20_g.get("seed_all_cell1", None)

        if callable(cell20_seed_fn):
            cell20_seed_fn(int(cell20_seed))
        else:
            cell20_torch.manual_seed(int(cell20_seed))

            cell20_np_module = cell20_g.get("np", None)

            if cell20_np_module is not None:
                cell20_np_module.random.seed(int(cell20_seed))

        if not cell20_torch.is_tensor(cell20_X):
            cell20_X = cell20_torch.as_tensor(
                cell20_X,
                dtype=cell20_torch.float32,
                device=cell20_device,
            )
        else:
            cell20_X = cell20_X.to(cell20_device)

        cell20_ei_pos = cell20_move_tensor(cell20_ei_pos, cell20_device)
        cell20_ew_pos = cell20_move_tensor(cell20_ew_pos, cell20_device)
        cell20_dis_pos = cell20_move_tensor(cell20_dis_pos, cell20_device)
        cell20_ei_neg = cell20_move_tensor(cell20_ei_neg, cell20_device)
        cell20_ew_neg = cell20_move_tensor(cell20_ew_neg, cell20_device)
        cell20_dis_neg = cell20_move_tensor(cell20_dis_neg, cell20_device)

        cell20_tr = cell20_index_tensor(cell20_train_idx, cell20_device)
        cell20_va = cell20_index_tensor(cell20_val_idx, cell20_device)
        cell20_te = cell20_index_tensor(cell20_test_idx, cell20_device)

        cell20_y = cell20_label_tensor(cell20_labels, cell20_device)
        cell20_y_super = cell20_label_tensor(cell20_super_labels, cell20_device)

        cell20_model = cell20_model.to(cell20_device)

        cell20_sup_map = cell20_infer_superclass_map(
            cell20_y,
            cell20_y_super,
            int(cell20_C),
            int(cell20_C_super),
        )

        cell20_model.set_superclass_map(cell20_sup_map)

        cell20_opt = cell20_torch.optim.AdamW(
            cell20_model.parameters(),
            lr=1e-3,
            weight_decay=1e-4,
        )

        cell20_sched = cell20_torch.optim.lr_scheduler.CosineAnnealingLR(
            cell20_opt,
            T_max=int(cell20_epochs),
        )

        cell20_cnt = cell20_torch.bincount(
            cell20_y[cell20_tr].clamp(min=0).long(),
            minlength=int(cell20_C),
        ).float()

        cell20_class_weight = (
            cell20_cnt.sum() / cell20_cnt.clamp(min=1.0)
        ) ** 0.5

        cell20_class_weight = cell20_class_weight / cell20_class_weight.mean()

        cell20_best_val = -1.0
        cell20_best_state = None
        cell20_reinit_done = False

        cell20_rare_classes_list = [int(cell20_x) for cell20_x in cell20_rare_classes]

        for cell20_epoch in range(int(cell20_epochs)):
            cell20_model.train()
            cell20_opt.zero_grad()

            cell20_h = cell20_model(
                cell20_X,
                cell20_ei_pos,
                cell20_ew_pos,
                cell20_dis_pos,
                cell20_ei_neg,
                cell20_ew_neg,
                cell20_dis_neg,
            )

            cell20_nll = cell20_model.compute_nll(
                cell20_h,
                cell20_y,
                cell20_tr,
            )

            cell20_logits = -cell20_nll[cell20_tr]
            cell20_logits = cell20_torch.nan_to_num(
                cell20_logits,
                nan=-1e4,
                posinf=1e4,
                neginf=-1e4,
            )

            cell20_loss = cell20_F.cross_entropy(
                cell20_logits,
                cell20_y[cell20_tr],
                weight=cell20_class_weight,
            )

            if bool(cell20_torch.isfinite(cell20_loss)):
                cell20_loss.backward()
                cell20_torch.nn.utils.clip_grad_norm_(
                    cell20_model.parameters(),
                    1.0,
                )
                cell20_opt.step()

            cell20_sched.step()

            if (
                bool(cell20_use_reinit)
                and int(cell20_epoch) == int(cell20_reinit_epoch)
                and not cell20_reinit_done
            ):
                with cell20_torch.no_grad():
                    cell20_h_train = cell20_h[cell20_tr]
                    cell20_y_train = cell20_y[cell20_tr]
                    cell20_y_super_train = cell20_y_super[cell20_tr]

                    for cell20_c in range(int(cell20_C)):
                        cell20_mask_c = cell20_y_train == cell20_c

                        if int(cell20_torch.sum(cell20_mask_c).item()) > 0:
                            cell20_model.encoder.proto_class.data[cell20_c] = (
                                cell20_h_train[cell20_mask_c].mean(dim=0)
                            )

                    for cell20_s in range(int(cell20_C_super)):
                        cell20_mask_s = cell20_y_super_train == cell20_s

                        if int(cell20_torch.sum(cell20_mask_s).item()) > 0:
                            cell20_model.encoder.proto_super.data[cell20_s] = (
                                cell20_h_train[cell20_mask_s].mean(dim=0)
                            )

                cell20_reinit_done = True

            cell20_should_eval = bool(
                ((int(cell20_epoch) + 1) % int(cell20_eval_every) == 0)
                or (int(cell20_epoch) == int(cell20_epochs) - 1)
            )

            if cell20_should_eval:
                cell20_model.eval()

                with cell20_torch.no_grad():
                    cell20_h_eval = cell20_model(
                        cell20_X,
                        cell20_ei_pos,
                        cell20_ew_pos,
                        cell20_dis_pos,
                        cell20_ei_neg,
                        cell20_ew_neg,
                        cell20_dis_neg,
                    )

                    cell20_nll_eval = cell20_model.compute_nll(
                        cell20_h_eval,
                        cell20_y,
                        cell20_tr,
                    )

                    cell20_pred_va = (
                        (-cell20_nll_eval[cell20_va])
                        .argmax(dim=-1)
                        .detach()
                        .cpu()
                        .numpy()
                    )

                    cell20_true_va = cell20_y[cell20_va].detach().cpu().numpy()

                    cell20_val_rare = float(
                        cell20_rare_acc_fn(
                            cell20_true_va,
                            cell20_pred_va,
                            cell20_rare_classes_list,
                        )
                    )

                    if (
                        cell20_val_rare == cell20_val_rare
                        and cell20_val_rare > cell20_best_val
                    ):
                        cell20_best_val = cell20_val_rare
                        cell20_best_state = {
                            cell20_state_key: cell20_state_value.detach().clone()
                            for cell20_state_key, cell20_state_value
                            in cell20_model.state_dict().items()
                        }

        if cell20_best_state is not None:
            cell20_model.load_state_dict(cell20_best_state)

        cell20_model.eval()

        with cell20_torch.no_grad():
            cell20_h_final = cell20_model(
                cell20_X,
                cell20_ei_pos,
                cell20_ew_pos,
                cell20_dis_pos,
                cell20_ei_neg,
                cell20_ew_neg,
                cell20_dis_neg,
            )

            cell20_nll_final = cell20_model.compute_nll(
                cell20_h_final,
                cell20_y,
                cell20_tr,
            )

            cell20_pred_all = (
                (-cell20_nll_final)
                .argmax(dim=-1)
                .detach()
                .cpu()
                .numpy()
            )

        cell20_true_all = cell20_y.detach().cpu().numpy()
        cell20_tr_np = cell20_tr.detach().cpu().numpy()
        cell20_te_np = cell20_te.detach().cpu().numpy()

        cell20_train_rare = float(
            cell20_rare_acc_fn(
                cell20_true_all[cell20_tr_np],
                cell20_pred_all[cell20_tr_np],
                cell20_rare_classes_list,
            )
        )

        cell20_test_rare = float(
            cell20_rare_acc_fn(
                cell20_true_all[cell20_te_np],
                cell20_pred_all[cell20_te_np],
                cell20_rare_classes_list,
            )
        )

        return {
            "train_rare": cell20_train_rare,
            "val_rare": float(cell20_best_val),
            "test_rare": cell20_test_rare,
            "reinit_done": bool(cell20_reinit_done),
            "kappa0": float(cell20_model.kappa0),
            "d_proto": int(cell20_model.d_proto),
        }

    return {
        "HEBPrototypeWrapper": cell20_HEBPrototypeWrapper,
        "make_encoder": cell20_make_encoder,
        "make_model": cell20_make_model,
        "infer_superclass_map": cell20_infer_superclass_map,
        "train_heb": cell20_train_heb,
    }


cell20_heb_api = cell20_install_heb_api()

print(
    "[Cell 20] HEB local helper bundle ready. "
    "This is not an external API/app/package. "
    "Run Cell 21 to execute the 10-seed ablation."
)

In [ ]:
# Cell 21: Run HEB prototype head on 10 FlyWire seeds.
#
# This is a defensive ablation against SSPR.
#
# Rules:
#   - No top-level imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every local/helper/loop variable is prefixed with cell21_.
#   - No fabricated result. If SSPR array is missing, paired test is skipped.


def cell21_extract_numeric_array(cell21_obj):
    cell21_g = globals()

    cell21_np = cell21_g.get("np", None)

    if cell21_np is None:
        import numpy as cell21_np

    cell21_torch = cell21_g.get("torch", None)

    cell21_numeric_keys = [
        "test_rare",
        "rare_ba",
        "rare_balanced_acc",
        "test_rare_array",
        "values",
        "array",
        "results",
        "sspr_after",
        "sspr_before",
        "heb_test_rare",
    ]

    if cell21_obj is None:
        return None

    if cell21_torch is not None and cell21_torch.is_tensor(cell21_obj):
        try:
            return cell21_np.asarray(
                cell21_obj.detach().cpu().numpy(),
                dtype=float,
            ).reshape(-1)
        except Exception:
            return None

    if isinstance(cell21_obj, dict):
        for cell21_key in cell21_numeric_keys:
            if cell21_key in cell21_obj:
                cell21_child_array = cell21_extract_numeric_array(cell21_obj[cell21_key])

                if cell21_child_array is not None:
                    return cell21_child_array

        return None

    if isinstance(cell21_obj, (list, tuple)):
        if len(cell21_obj) == 0:
            return None

        if all(isinstance(cell21_item, dict) for cell21_item in cell21_obj):
            cell21_values = []

            for cell21_item in cell21_obj:
                cell21_item_array = None

                for cell21_key in cell21_numeric_keys:
                    if cell21_key in cell21_item:
                        cell21_item_array = cell21_extract_numeric_array(cell21_item[cell21_key])

                        if cell21_item_array is not None and cell21_item_array.size > 0:
                            cell21_values.append(float(cell21_item_array.reshape(-1)[0]))
                            break

                if cell21_item_array is None:
                    cell21_values.append(float("nan"))

            return cell21_np.asarray(cell21_values, dtype=float).reshape(-1)

        try:
            return cell21_np.asarray(cell21_obj, dtype=float).reshape(-1)
        except Exception:
            return None

    try:
        return cell21_np.asarray(cell21_obj, dtype=float).reshape(-1)
    except Exception:
        return None


def cell21_main():
    cell21_g = globals()

    cell21_np = cell21_g.get("np", None)

    if cell21_np is None:
        import numpy as cell21_np

    cell21_torch = cell21_g.get("torch", None)

    if cell21_torch is None:
        raise NameError("[Cell 21][FATAL] torch is not available. Run earlier cells first.")

    cell21_api = cell21_g.get("cell20_heb_api", None)

    if cell21_api is None:
        raise NameError(
            "[Cell 21][FATAL] cell20_heb_api is not available. Run Cell 20 first."
        )

    cell21_required_names = [
        "DEVICE_cell1",
        "X_sspr_tensor_cell11",
        "labels_tensor_cell3",
        "labeled_mask_cell3",
        "super_labels_tensor_cell3",
        "C_cell3",
        "C_super_cell3",
        "rare_classes_list_cell3",
        "ei_pos_cell7",
        "ew_pos_cell7",
        "dis_pos_cell7",
        "ei_neg_cell7",
        "ew_neg_cell7",
        "dis_neg_cell7",
        "make_split_cell4",
        "rare_balanced_acc_cell6",
    ]

    cell21_missing_names = []

    for cell21_required_name in cell21_required_names:
        if cell21_required_name not in cell21_g:
            cell21_missing_names.append(cell21_required_name)

    if len(cell21_missing_names) > 0:
        raise NameError(
            "[Cell 21][FATAL] Missing upstream variables: "
            + str(cell21_missing_names)
            + ". Run Cells 1-11 first."
        )

    cell21_device = cell21_g["DEVICE_cell1"]
    cell21_C = int(cell21_g["C_cell3"])
    cell21_C_super = int(cell21_g["C_super_cell3"])

    cell21_X_source = cell21_g["X_sspr_tensor_cell11"]

    if cell21_torch.is_tensor(cell21_X_source):
        cell21_X = cell21_X_source.to(cell21_device).float()
    else:
        cell21_X_np = cell21_np.asarray(cell21_X_source, dtype=cell21_np.float32)
        cell21_X = cell21_torch.tensor(
            cell21_X_np,
            dtype=cell21_torch.float32,
            device=cell21_device,
        )

    if int(cell21_X.ndim) != 2:
        raise RuntimeError(
            "[Cell 21][FATAL] X_sspr_tensor_cell11 must be 2D, got shape "
            + str(tuple(cell21_X.shape))
        )

    cell21_d_features = int(cell21_X.shape[1])

    cell21_rare_obj = cell21_g["rare_classes_list_cell3"]

    if cell21_torch.is_tensor(cell21_rare_obj):
        cell21_rare_np = cell21_rare_obj.detach().cpu().numpy()
    else:
        cell21_rare_np = cell21_np.asarray(cell21_rare_obj)

    cell21_rare_classes = sorted(
        set(
            int(cell21_rare_value)
            for cell21_rare_value in cell21_rare_np.reshape(-1).tolist()
        )
    )

    cell21_seeds = list(range(10))

    cell21_kappa0 = 4.0
    cell21_d_hidden = 64
    cell21_d_proto = 32
    cell21_n_layers = 2
    cell21_dropout = 0.3
    cell21_epochs = 200
    cell21_reinit_epoch = 40
    cell21_eval_every = 25

    print("=" * 100)
    print("CELL 21 — HEB PROTOTYPE HEAD, 10 SEEDS")
    print("=" * 100)
    print("[cfg] kappa0=" + str(cell21_kappa0))
    print("[cfg] d_hidden=" + str(cell21_d_hidden))
    print("[cfg] d_proto=" + str(cell21_d_proto))
    print("[cfg] n_layers=" + str(cell21_n_layers))
    print("[cfg] dropout=" + str(cell21_dropout))
    print("[cfg] epochs=" + str(cell21_epochs))
    print("[cfg] reinit_epoch=" + str(cell21_reinit_epoch))
    print("[cfg] eval_every=" + str(cell21_eval_every))
    print("[data] X shape=" + str(tuple(cell21_X.shape)))
    print("[data] C=" + str(cell21_C) + ", C_super=" + str(cell21_C_super))
    print("[data] rare_classes=" + str(cell21_rare_classes))
    print("=" * 100)

    cell21_rows = []

    for cell21_seed_value in cell21_seeds:
        print("\n[Cell 21] seed " + str(cell21_seed_value), flush=True)

        cell21_train_idx, cell21_val_idx, cell21_test_idx = cell21_g["make_split_cell4"](
            cell21_seed_value,
            cell21_g["labeled_mask_cell3"],
            cell21_g["labels_tensor_cell3"],
            cell21_g["C_cell3"],
        )

        cell21_model = cell21_api["make_model"](
            cell21_d_features=cell21_d_features,
            cell21_d=cell21_d_hidden,
            cell21_d_proto=cell21_d_proto,
            cell21_n_layers=cell21_n_layers,
            cell21_n_classes=cell21_C,
            cell21_n_super=cell21_C_super,
            cell21_dropout=cell21_dropout,
            cell21_kappa0=cell21_kappa0,
        )

        cell21_result = cell21_api["train_heb"](
            cell21_model,
            cell21_X,
            cell21_g["ei_pos_cell7"],
            cell21_g["ew_pos_cell7"],
            cell21_g["dis_pos_cell7"],
            cell21_g["ei_neg_cell7"],
            cell21_g["ew_neg_cell7"],
            cell21_g["dis_neg_cell7"],
            cell21_train_idx,
            cell21_val_idx,
            cell21_test_idx,
            cell21_g["labels_tensor_cell3"],
            cell21_g["super_labels_tensor_cell3"],
            cell21_rare_classes,
            cell21_seed_value,
            cell20_epochs=cell21_epochs,
            cell20_device=cell21_device,
            cell20_C=cell21_C,
            cell20_C_super=cell21_C_super,
            cell20_use_reinit=True,
            cell20_reinit_epoch=cell21_reinit_epoch,
            cell20_eval_every=cell21_eval_every,
        )

        cell21_row = {
            "seed": int(cell21_seed_value),
            "train_rare": float(cell21_result.get("train_rare", float("nan"))),
            "val_rare": float(cell21_result.get("val_rare", float("nan"))),
            "test_rare": float(cell21_result.get("test_rare", float("nan"))),
            "reinit_done": bool(cell21_result.get("reinit_done", False)),
            "kappa0": float(cell21_result.get("kappa0", cell21_kappa0)),
            "d_proto": int(cell21_result.get("d_proto", cell21_d_proto)),
        }

        cell21_rows.append(cell21_row)

        print(
            "  train="
            + f"{cell21_row['train_rare']:.4f}"
            + " val="
            + f"{cell21_row['val_rare']:.4f}"
            + " test="
            + f"{cell21_row['test_rare']:.4f}",
            flush=True,
        )

    cell21_test_array = cell21_np.asarray(
        [cell21_row["test_rare"] for cell21_row in cell21_rows],
        dtype=float,
    )

    cell21_finite_array = cell21_test_array[cell21_np.isfinite(cell21_test_array)]

    if cell21_finite_array.size > 0:
        cell21_mean = float(cell21_np.mean(cell21_finite_array))
        cell21_std = float(
            cell21_np.std(cell21_finite_array, ddof=1)
            if cell21_finite_array.size > 1
            else 0.0
        )
    else:
        cell21_mean = float("nan")
        cell21_std = float("nan")

    cell21_summary = {
        "method": "HEB prototype head",
        "kappa0": float(cell21_kappa0),
        "n_seeds": int(cell21_finite_array.size),
        "mean_test_rare": float(cell21_mean),
        "std_test_rare": float(cell21_std),
        "rows": cell21_rows,
    }

    print("\n" + "-" * 100)
    print(
        "HEB rare_ba: "
        + f"{cell21_mean:.4f}"
        + " +/- "
        + f"{cell21_std:.4f}"
        + " (n="
        + str(cell21_finite_array.size)
        + ")"
    )

    cell21_comparison = {
        "status": "skipped",
        "reason": "SSPR array not found.",
    }

    cell21_sspr_candidates = [
        "sspr_after_cell11c",
        "sspr_rare_cell11c",
        "results_sspr_cell11c",
        "sspr_full_cell11c",
        "cell11c_sspr_after",
        "cell11c_sspr_rare",
    ]

    cell21_sspr_array = None
    cell21_sspr_source = None

    for cell21_candidate_name in cell21_sspr_candidates:
        cell21_candidate_array = cell21_extract_numeric_array(
            cell21_g.get(cell21_candidate_name, None)
        )

        if (
            cell21_candidate_array is not None
            and cell21_candidate_array.size >= 2
            and bool(cell21_np.any(cell21_np.isfinite(cell21_candidate_array)))
        ):
            cell21_sspr_array = cell21_candidate_array
            cell21_sspr_source = cell21_candidate_name
            break

    if cell21_sspr_array is None:
        print("[Cell 21][warn] SSPR 10-seed array not found. Paired comparison skipped.")
    else:
        print("SSPR source variable: " + str(cell21_sspr_source))
        print("SSPR n=" + str(int(cell21_sspr_array.size)))

        if int(cell21_sspr_array.size) == int(cell21_test_array.size):
            try:
                import scipy.stats as cell21_stats

                cell21_t_stat, cell21_p_ttest = cell21_stats.ttest_rel(
                    cell21_sspr_array,
                    cell21_test_array,
                )

                cell21_diff = cell21_sspr_array - cell21_test_array
                cell21_finite_diff = cell21_diff[cell21_np.isfinite(cell21_diff)]

                if cell21_finite_diff.size > 0 and bool(cell21_np.any(cell21_finite_diff != 0.0)):
                    cell21_w_stat, cell21_p_wilcoxon = cell21_stats.wilcoxon(
                        cell21_finite_diff,
                        alternative="greater",
                        zero_method="zsplit",
                    )
                else:
                    cell21_w_stat = float("nan")
                    cell21_p_wilcoxon = float("nan")

                cell21_rng = cell21_np.random.default_rng(0)
                cell21_boot_means = []

                for cell21_boot_index in range(10000):
                    cell21_boot_sample = cell21_rng.choice(
                        cell21_finite_diff,
                        size=int(cell21_finite_diff.size),
                        replace=True,
                    )
                    cell21_boot_means.append(float(cell21_np.mean(cell21_boot_sample)))

                cell21_boot_array = cell21_np.asarray(cell21_boot_means, dtype=float)
                cell21_ci_low, cell21_ci_high = cell21_np.percentile(
                    cell21_boot_array,
                    [2.5, 97.5],
                )

                if cell21_ci_low > 0.0:
                    cell21_verdict = "SSPR > HEB"
                elif cell21_ci_high < 0.0:
                    cell21_verdict = "HEB > SSPR"
                else:
                    cell21_verdict = "SSPR ~= HEB"

                cell21_comparison = {
                    "status": "paired",
                    "sspr_source": str(cell21_sspr_source),
                    "n_pairs": int(cell21_finite_diff.size),
                    "mean_delta_sspr_minus_heb": float(cell21_np.mean(cell21_finite_diff)),
                    "std_delta_sspr_minus_heb": float(
                        cell21_np.std(cell21_finite_diff, ddof=1)
                        if cell21_finite_diff.size > 1
                        else 0.0
                    ),
                    "ttest_t": float(cell21_t_stat),
                    "ttest_p": float(cell21_p_ttest),
                    "wilcoxon_stat": float(cell21_w_stat),
                    "wilcoxon_p": float(cell21_p_wilcoxon),
                    "bootstrap_ci_low": float(cell21_ci_low),
                    "bootstrap_ci_high": float(cell21_ci_high),
                    "verdict": str(cell21_verdict),
                }

                print(
                    "Paired delta SSPR - HEB: "
                    + f"{cell21_comparison['mean_delta_sspr_minus_heb']:+.4f}"
                    + " +/- "
                    + f"{cell21_comparison['std_delta_sspr_minus_heb']:.4f}"
                )
                print(
                    "Paired t-test: t="
                    + f"{cell21_comparison['ttest_t']:+.3f}"
                    + " p="
                    + f"{cell21_comparison['ttest_p']:.4f}"
                )
                print(
                    "Wilcoxon signed-rank, alternative=greater: p="
                    + f"{cell21_comparison['wilcoxon_p']:.4f}"
                )
                print(
                    "95% bootstrap CI on delta: ["
                    + f"{cell21_comparison['bootstrap_ci_low']:+.4f}"
                    + ", "
                    + f"{cell21_comparison['bootstrap_ci_high']:+.4f}"
                    + "]"
                )
                print("Verdict: " + str(cell21_comparison["verdict"]))

            except Exception as cell21_compare_error:
                cell21_comparison = {
                    "status": "paired_failed",
                    "reason": repr(cell21_compare_error),
                    "sspr_source": str(cell21_sspr_source),
                }
                print("[Cell 21][warn] Paired comparison failed: " + repr(cell21_compare_error))
        else:
            cell21_sspr_mean = float(cell21_np.nanmean(cell21_sspr_array))
            cell21_independent_delta = float(cell21_sspr_mean - cell21_mean)

            cell21_comparison = {
                "status": "independent",
                "reason": "SSPR and HEB arrays have different lengths.",
                "sspr_source": str(cell21_sspr_source),
                "sspr_mean": cell21_sspr_mean,
                "heb_mean": float(cell21_mean),
                "independent_delta_sspr_minus_heb": cell21_independent_delta,
            }

            print(
                "Independent mean delta SSPR - HEB: "
                + f"{cell21_independent_delta:+.4f}"
            )
            print("Paired test skipped because array lengths differ.")

    print("=" * 100)

    from pathlib import Path as cell21_Path
    import json as cell21_json

    cell21_artifact_dir = cell21_Path("artifacts")
    cell21_artifact_dir.mkdir(parents=True, exist_ok=True)

    cell21_json_path = cell21_artifact_dir / "cell21_heb_10seeds.json"
    cell21_csv_path = cell21_artifact_dir / "cell21_heb_10seeds.csv"

    cell21_output_dict = {
        "summary": cell21_summary,
        "comparison": cell21_comparison,
        "config": {
            "kappa0": float(cell21_kappa0),
            "d_hidden": int(cell21_d_hidden),
            "d_proto": int(cell21_d_proto),
            "n_layers": int(cell21_n_layers),
            "dropout": float(cell21_dropout),
            "epochs": int(cell21_epochs),
            "reinit_epoch": int(cell21_reinit_epoch),
            "eval_every": int(cell21_eval_every),
            "seeds": cell21_seeds,
        },
    }

    cell21_json_path.write_text(
        cell21_json.dumps(cell21_output_dict, indent=2, default=str),
        encoding="utf-8",
    )

    cell21_csv_header = [
        "seed",
        "train_rare",
        "val_rare",
        "test_rare",
        "reinit_done",
        "kappa0",
        "d_proto",
    ]

    cell21_csv_lines = [",".join(cell21_csv_header)]

    for cell21_row in cell21_rows:
        cell21_csv_lines.append(
            ",".join(
                str(cell21_row.get(cell21_field_name, ""))
                for cell21_field_name in cell21_csv_header
            )
        )

    cell21_csv_path.write_text(
        "\n".join(cell21_csv_lines) + "\n",
        encoding="utf-8",
    )

    print("Saved JSON: " + str(cell21_json_path))
    print("Saved CSV:  " + str(cell21_csv_path))

    cell21_hf_save_checkpoint = cell21_g.get("hf_save_checkpoint", None)

    if callable(cell21_hf_save_checkpoint):
        try:
            cell21_hf_save_checkpoint(
                "cell21_heb_10seed_ablation",
                {
                    "summary": cell21_summary,
                    "comparison": cell21_comparison,
                    "config": cell21_output_dict["config"],
                    "paths": {
                        "json": str(cell21_json_path),
                        "csv": str(cell21_csv_path),
                    },
                },
                description=(
                    "Cell 21: HEB prototype head 10-seed defensive ablation. "
                    "No fabricated result; comparison is skipped if SSPR array is missing."
                ),
                upload=True,
            )
        except Exception as cell21_checkpoint_error:
            print("[ckpt][warn] Cell 21 checkpoint failed: " + repr(cell21_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    return {
        "rows": cell21_rows,
        "test_rare_array": cell21_test_array,
        "summary": cell21_summary,
        "comparison": cell21_comparison,
        "json_path": str(cell21_json_path),
        "csv_path": str(cell21_csv_path),
    }


cell21_output = cell21_main()

cell21_heb_rows = cell21_output["rows"]
cell21_heb_test_rare_array = cell21_output["test_rare_array"]
cell21_heb_summary = cell21_output["summary"]
cell21_heb_comparison = cell21_output["comparison"]
cell21_heb_json_path = cell21_output["json_path"]
cell21_heb_csv_path = cell21_output["csv_path"]

In [ ]:
# Cell 22: Add HEB row to the final comparison table.
#
# This cell prefers runtime arrays from your notebook.
# If a runtime array is missing, it can fall back to prior exported constants
# only when cell22_use_prior_constants = True.
#
# The prior constants below are taken from the table you supplied earlier.
# If your actual notebook differs, set cell22_use_prior_constants = False
# or replace the constants with your exported values.
#
# Rules:
#   - No top-level imports.
#   - No @app.cell.
#   - No top-level underscore names.
#   - Every local/helper/loop variable is prefixed with cell22_.


def cell22_extract_numeric_array(cell22_obj):
    cell22_g = globals()

    cell22_np = cell22_g.get("np", None)

    if cell22_np is None:
        import numpy as cell22_np

    cell22_torch = cell22_g.get("torch", None)

    cell22_numeric_keys = [
        "test_rare",
        "rare_ba",
        "rare_balanced_acc",
        "test_rare_array",
        "values",
        "array",
        "results",
        "sspr_after",
        "sspr_before",
        "heb_test_rare",
    ]

    if cell22_obj is None:
        return None

    if cell22_torch is not None and cell22_torch.is_tensor(cell22_obj):
        try:
            return cell22_np.asarray(
                cell22_obj.detach().cpu().numpy(),
                dtype=float,
            ).reshape(-1)
        except Exception:
            return None

    if isinstance(cell22_obj, dict):
        for cell22_key in cell22_numeric_keys:
            if cell22_key in cell22_obj:
                cell22_child_array = cell22_extract_numeric_array(cell22_obj[cell22_key])

                if cell22_child_array is not None:
                    return cell22_child_array

        return None

    if isinstance(cell22_obj, (list, tuple)):
        if len(cell22_obj) == 0:
            return None

        if all(isinstance(cell22_item, dict) for cell22_item in cell22_obj):
            cell22_values = []

            for cell22_item in cell22_obj:
                cell22_item_array = None

                for cell22_key in cell22_numeric_keys:
                    if cell22_key in cell22_item:
                        cell22_item_array = cell22_extract_numeric_array(cell22_item[cell22_key])

                        if cell22_item_array is not None and cell22_item_array.size > 0:
                            cell22_values.append(float(cell22_item_array.reshape(-1)[0]))
                            break

                if cell22_item_array is None:
                    cell22_values.append(float("nan"))

            return cell22_np.asarray(cell22_values, dtype=float).reshape(-1)

        try:
            return cell22_np.asarray(cell22_obj, dtype=float).reshape(-1)
        except Exception:
            return None

    try:
        return cell22_np.asarray(cell22_obj, dtype=float).reshape(-1)
    except Exception:
        return None


def cell22_main():
    cell22_g = globals()

    cell22_np = cell22_g.get("np", None)

    if cell22_np is None:
        import numpy as cell22_np

    cell22_use_prior_constants = True

    cell22_prior_constants = {
        "lp": (0.0661, 0.0200, 10),
        "gcn": (0.2064, 0.0328, 10),
        "alt_faithful": (0.0285, 0.0093, 10),
        "alt_balanced": (0.1596, 0.0251, 10),
        "spgcn": (0.4317, 0.0446, 10),
        "ntac_25k": (0.4283, 0.0430, 10),
        "sspr_no_refine": (0.4590, 0.0651, 10),
        "sspr_full": (0.5954, 0.0391, 10),
    }

    cell22_methods = [
        (
            "Label propagation",
            "lp",
            [
                "lp_rare_cell10",
                "results_lp_cell10",
                "lp_rare_cell10_list",
                "label_prop_rare_cell10",
            ],
        ),
        (
            "GCN + all features baseline",
            "gcn",
            [
                "results_linear_cell8s",
                "linear_rare_cell8s",
                "baseline_rare_cell8",
                "gcn_baseline_rare_cell8",
            ],
        ),
        (
            "Altiparmak faithful reimpl",
            "alt_faithful",
            [
                "alt_faithful_rare_cell11b",
                "results_alt_faithful_cell11b",
                "altiparmak_faithful_rare_cell11b",
                "faithful_portion6_rare_cell11b",
            ],
        ),
        (
            "Altiparmak balanced_target500",
            "alt_balanced",
            [
                "alt_balanced_rare_cell11b",
                "results_alt_balanced_cell11b",
                "altiparmak_balanced_rare_cell11b",
                "balanced_target500_rare_cell11b",
            ],
        ),
        (
            "SPGCN unsigned spectral",
            "spgcn",
            [
                "results_spgcn_cell8",
                "spgcn_rare_cell8",
                "spgcn_test_rare_cell8",
            ],
        ),
        (
            "NTAC 25K subgraph diagnostic",
            "ntac_25k",
            [
                "cell18_ntac_test_rare_array",
                "cell18_ntac_results",
                "cell12b_ntac_test_rare_array",
                "cell12b_ntac_results",
            ],
        ),
        (
            "SSPR without refinement",
            "sspr_no_refine",
            [
                "sspr_before_cell11c",
                "results_sspr_before_cell11c",
                "sspr_no_refine_rare_cell11c",
            ],
        ),
        (
            "SSPR full",
            "sspr_full",
            [
                "sspr_after_cell11c",
                "results_sspr_cell11c",
                "sspr_full_rare_cell11c",
                "cell11c_sspr_after",
            ],
        ),
        (
            "HEB prototype head (ablation)",
            "heb",
            [
                "cell21_heb_test_rare_array",
            ],
        ),
    ]

    def cell22_stats_from_array(cell22_array):
        cell22_finite = cell22_array[cell22_np.isfinite(cell22_array)]

        if cell22_finite.size == 0:
            return None

        cell22_mean = float(cell22_np.mean(cell22_finite))
        cell22_std = float(
            cell22_np.std(cell22_finite, ddof=1)
            if cell22_finite.size > 1
            else 0.0
        )

        return {
            "mean": cell22_mean,
            "std": cell22_std,
            "n": int(cell22_finite.size),
        }

    cell22_rows = []
    cell22_used_prior = False

    for cell22_display_name, cell22_key, cell22_candidates in cell22_methods:
        cell22_runtime_array = None
        cell22_runtime_source = None

        for cell22_candidate_name in cell22_candidates:
            cell22_candidate_array = cell22_extract_numeric_array(
                cell22_g.get(cell22_candidate_name, None)
            )

            if (
                cell22_candidate_array is not None
                and cell22_candidate_array.size >= 1
                and bool(cell22_np.any(cell22_np.isfinite(cell22_candidate_array)))
            ):
                cell22_runtime_array = cell22_candidate_array
                cell22_runtime_source = cell22_candidate_name
                break

        if cell22_runtime_array is not None:
            cell22_stat_dict = cell22_stats_from_array(cell22_runtime_array)

            if cell22_stat_dict is not None:
                cell22_rows.append(
                    {
                        "method": cell22_display_name,
                        "key": cell22_key,
                        "mean": cell22_stat_dict["mean"],
                        "std": cell22_stat_dict["std"],
                        "n": cell22_stat_dict["n"],
                        "status": "ok",
                        "source": "runtime:" + str(cell22_runtime_source),
                    }
                )
                continue

        if cell22_use_prior_constants and cell22_key in cell22_prior_constants:
            cell22_prior_mean, cell22_prior_std, cell22_prior_n = cell22_prior_constants[cell22_key]
            cell22_used_prior = True

            cell22_rows.append(
                {
                    "method": cell22_display_name,
                    "key": cell22_key,
                    "mean": float(cell22_prior_mean),
                    "std": float(cell22_prior_std),
                    "n": int(cell22_prior_n),
                    "status": "ok",
                    "source": "prior_constant",
                }
            )
            continue

        cell22_rows.append(
            {
                "method": cell22_display_name,
                "key": cell22_key,
                "mean": float("nan"),
                "std": float("nan"),
                "n": 0,
                "status": "missing",
                "source": "",
            }
        )

    print("\n" + "=" * 100)
    print("CELL 22 — MAIN COMPARISON TABLE WITH HEB ABLATION")
    print("=" * 100)

    if cell22_used_prior:
        print(
            "[warning] Some rows use prior exported constants because runtime arrays were missing. "
            "If these do not match your notebook, set cell22_use_prior_constants = False."
        )
        print("=" * 100)

    print(
        f"{'method':42s} {'rare_ba':>20s} {'n':>4s} {'source':>28s}"
    )
    print("-" * 100)

    for cell22_row in cell22_rows:
        if cell22_row["status"] == "ok":
            cell22_value_string = (
                f"{cell22_row['mean']:.4f} +/- {cell22_row['std']:.4f}"
            )
        else:
            cell22_value_string = "missing"

        print(
            f"{cell22_row['method']:42s} "
            f"{cell22_value_string:>20s} "
            f"{str(cell22_row['n']):>4s} "
            f"{str(cell22_row['source'])[:28]:>28s}"
        )

    print("=" * 100)

    cell22_heb_row = None
    cell22_sspr_row = None

    for cell22_row in cell22_rows:
        if cell22_row["key"] == "heb":
            cell22_heb_row = cell22_row
        if cell22_row["key"] == "sspr_full":
            cell22_sspr_row = cell22_row

    if cell22_heb_row is not None and cell22_heb_row["status"] == "ok":
        print("HEB ablation result:")
        print(
            "  HEB rare_ba = "
            + f"{cell22_heb_row['mean']:.4f}"
            + " +/- "
            + f"{cell22_heb_row['std']:.4f}"
            + " (n="
            + str(cell22_heb_row["n"])
            + ")"
        )

        if cell22_sspr_row is not None and cell22_sspr_row["status"] == "ok":
            cell22_delta = float(cell22_sspr_row["mean"] - cell22_heb_row["mean"])
            print(
                "  Independent delta SSPR - HEB = "
                + f"{cell22_delta:+.4f}"
            )

            if cell22_delta > 0.03:
                print("  Interpretation: HEB is below SSPR; use as defensive ablation.")
            elif abs(cell22_delta) <= 0.03:
                print("  Interpretation: HEB is roughly indistinguishable from SSPR at this n.")
            else:
                print(
                    "  Interpretation: HEB is above SSPR. Do not pivot unless "
                    "confirmed with more seeds and a pre-specified test."
                )
    else:
        print("HEB row is missing. Run Cell 21 before using this table.")

    print("=" * 100)

    from pathlib import Path as cell22_Path
    import json as cell22_json

    cell22_artifact_dir = cell22_Path("artifacts")
    cell22_artifact_dir.mkdir(parents=True, exist_ok=True)

    cell22_json_path = cell22_artifact_dir / "cell22_main_table_with_heb.json"
    cell22_markdown_path = cell22_artifact_dir / "cell22_main_table_with_heb.md"

    cell22_json_path.write_text(
        cell22_json.dumps(
            {
                "rows": cell22_rows,
                "used_prior_constants": bool(cell22_used_prior),
                "prior_constants": cell22_prior_constants,
            },
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    cell22_markdown_lines = [
        "| Method | Rare-class balanced accuracy | n | Source |",
        "|---|---:|---:|---|",
    ]

    for cell22_row in cell22_rows:
        if cell22_row["status"] == "ok":
            cell22_md_value = (
                f"{cell22_row['mean']:.4f} ± {cell22_row['std']:.4f}"
            )
        else:
            cell22_md_value = "missing"

        cell22_markdown_lines.append(
            "| "
            + str(cell22_row["method"])
            + " | "
            + cell22_md_value
            + " | "
            + str(cell22_row["n"])
            + " | "
            + str(cell22_row["source"])
            + " |"
        )

    cell22_markdown_path.write_text(
        "\n".join(cell22_markdown_lines) + "\n",
        encoding="utf-8",
    )

    print("Saved JSON:     " + str(cell22_json_path))
    print("Saved Markdown: " + str(cell22_markdown_path))

    return {
        "rows": cell22_rows,
        "used_prior_constants": bool(cell22_used_prior),
        "json_path": str(cell22_json_path),
        "markdown_path": str(cell22_markdown_path),
    }


cell22_output = cell22_main()

cell22_table_rows = cell22_output["rows"]
cell22_used_prior_constants = cell22_output["used_prior_constants"]
cell22_table_json_path = cell22_output["json_path"]
cell22_table_markdown_path = cell22_output["markdown_path"]

In [ ]:
# Cell 23: Run SSPR full and HEB prototype head on the same 100 seeds.
#
# Purpose:
#   Produce a paired SSPR-vs-HEB comparison under identical splits.
#
# Requirements:
#   - Cells 1-11 must already define the FlyWire data, splits, SPGCN, training loop,
#     rare-class metric, and X_sspr_tensor_cell11.
#   - Cell 20 must already define cell20_heb_api.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No unnecessary top-level imports.
#   - Every notebook-local/helper/loop variable is prefixed with cell23_.
#   - No fabricated experimental numbers.
#   - Failed seeds are recorded, not invented.


def cell23_main():
    cell23_globals = globals()

    cell23_np = cell23_globals.get("np", None)
    if cell23_np is None:
        import numpy as cell23_np

    cell23_torch = cell23_globals.get("torch", None)
    if cell23_torch is None:
        import torch as cell23_torch

    cell23_Path = cell23_globals.get("Path", None)
    if cell23_Path is None:
        from pathlib import Path as cell23_Path

    cell23_json = cell23_globals.get("json", None)
    if cell23_json is None:
        import json as cell23_json

    cell23_gc = cell23_globals.get("gc", None)
    if cell23_gc is None:
        import gc as cell23_gc

    cell23_time = cell23_globals.get("time", None)
    if cell23_time is None:
        import time as cell23_time

    cell23_required_names = [
        "DEVICE_cell1",
        "X_sspr_tensor_cell11",
        "labels_tensor_cell3",
        "labeled_mask_cell3",
        "super_labels_tensor_cell3",
        "C_cell3",
        "C_super_cell3",
        "rare_classes_list_cell3",
        "ei_pos_cell7",
        "ew_pos_cell7",
        "dis_pos_cell7",
        "ei_neg_cell7",
        "ew_neg_cell7",
        "dis_neg_cell7",
        "make_split_cell4",
        "SPGCN_cell5",
        "train_spgcn_cell6",
        "rare_balanced_acc_cell6",
        "cell20_heb_api",
    ]

    cell23_missing_names = []

    for cell23_required_name in cell23_required_names:
        if cell23_required_name not in cell23_globals:
            cell23_missing_names.append(cell23_required_name)

    if len(cell23_missing_names) > 0:
        raise NameError(
            "[Cell 23][FATAL] Missing upstream variables: "
            + str(cell23_missing_names)
            + ". Run Cells 1-11 and Cell 20 first."
        )

    cell23_device = cell23_globals["DEVICE_cell1"]
    cell23_C = int(cell23_globals["C_cell3"])
    cell23_C_super = int(cell23_globals["C_super_cell3"])

    # ------------------------------------------------------------------
    # Experiment configuration.
    #
    # For a quick smoke test, temporarily change seeds to range(3)
    # and epochs to 20.
    # ------------------------------------------------------------------
    cell23_seeds = list(range(100))
    cell23_epochs = 200

    cell23_lambda_refine = 0.3
    cell23_tau_refine = 1.0

    cell23_kappa0 = 4.0
    cell23_d_hidden = 64
    cell23_d_proto = 32
    cell23_n_layers = 2
    cell23_dropout = 0.3
    cell23_heb_reinit_epoch = 40
    cell23_heb_eval_every = 25

    cell23_max_consecutive_failures = 3
    cell23_bootstrap_iterations = 10000

    # ------------------------------------------------------------------
    # Conversion helpers.
    # ------------------------------------------------------------------
    def cell23_to_numpy(cell23_object):
        if cell23_object is None:
            return None

        cell23_working_object = cell23_object

        try:
            if hasattr(cell23_working_object, "detach"):
                cell23_working_object = cell23_working_object.detach()

            if hasattr(cell23_working_object, "cpu"):
                cell23_working_object = cell23_working_object.cpu()

            if hasattr(cell23_working_object, "numpy"):
                cell23_working_object = cell23_working_object.numpy()

            return cell23_np.asarray(cell23_working_object)
        except Exception:
            return None

    def cell23_to_int_vector(cell23_object, cell23_description):
        cell23_array_object = cell23_to_numpy(cell23_object)

        if cell23_array_object is None:
            raise TypeError(
                "[Cell 23][FATAL] Could not convert "
                + str(cell23_description)
                + " to numpy."
            )

        try:
            return cell23_array_object.astype(cell23_np.int64).reshape(-1)
        except Exception as cell23_cast_error:
            raise TypeError(
                "[Cell 23][FATAL] Could not cast "
                + str(cell23_description)
                + " to int64: "
                + repr(cell23_cast_error)
            )

    def cell23_to_bool_vector(cell23_object, cell23_description):
        cell23_array_object = cell23_to_numpy(cell23_object)

        if cell23_array_object is None:
            raise TypeError(
                "[Cell 23][FATAL] Could not convert "
                + str(cell23_description)
                + " to numpy."
            )

        try:
            return cell23_array_object.astype(bool).reshape(-1)
        except Exception as cell23_cast_error:
            raise TypeError(
                "[Cell 23][FATAL] Could not cast "
                + str(cell23_description)
                + " to bool: "
                + repr(cell23_cast_error)
            )

    def cell23_as_long_tensor(cell23_object, cell23_description):
        cell23_array_object = cell23_to_int_vector(cell23_object, cell23_description)

        return cell23_torch.as_tensor(
            cell23_array_object,
            dtype=cell23_torch.long,
            device=cell23_device,
        )

    def cell23_move_tensor(cell23_tensor_object, cell23_target_device):
        if cell23_torch.is_tensor(cell23_tensor_object):
            return cell23_tensor_object.to(cell23_target_device)

        return cell23_tensor_object

    def cell23_fmt(cell23_numeric_value):
        try:
            cell23_float_value = float(cell23_numeric_value)
        except Exception:
            return "n/a"

        if not cell23_np.isfinite(cell23_float_value):
            return "n/a"

        return f"{cell23_float_value:.4f}"

    # ------------------------------------------------------------------
    # Load core arrays.
    # ------------------------------------------------------------------
    cell23_X_source = cell23_globals["X_sspr_tensor_cell11"]

    if cell23_torch.is_tensor(cell23_X_source):
        cell23_X = cell23_X_source.to(cell23_device).float()
    else:
        cell23_X_np = cell23_to_numpy(cell23_X_source)

        if cell23_X_np is None:
            raise TypeError("[Cell 23][FATAL] X_sspr_tensor_cell11 could not be converted.")

        cell23_X = cell23_torch.tensor(
            cell23_X_np.astype(cell23_np.float32),
            dtype=cell23_torch.float32,
            device=cell23_device,
        )

    if int(cell23_X.ndim) != 2:
        raise RuntimeError(
            "[Cell 23][FATAL] X_sspr_tensor_cell11 must be 2D, got shape "
            + str(tuple(cell23_X.shape))
        )

    cell23_feature_dim = int(cell23_X.shape[1])
    cell23_N = int(cell23_X.shape[0])

    cell23_labels_np = cell23_to_int_vector(
        cell23_globals["labels_tensor_cell3"],
        "Cell 23 labels",
    )

    cell23_labeled_mask_np = cell23_to_bool_vector(
        cell23_globals["labeled_mask_cell3"],
        "Cell 23 labeled mask",
    )

    if int(cell23_labels_np.size) != cell23_N:
        raise RuntimeError(
            "[Cell 23][FATAL] Label length mismatch: "
            + str(int(cell23_labels_np.size))
            + " vs X rows="
            + str(cell23_N)
        )

    if int(cell23_labeled_mask_np.size) != cell23_N:
        raise RuntimeError(
            "[Cell 23][FATAL] Labeled mask length mismatch: "
            + str(int(cell23_labeled_mask_np.size))
            + " vs X rows="
            + str(cell23_N)
        )

    cell23_labeled_mask_t = cell23_torch.as_tensor(
        cell23_labeled_mask_np,
        dtype=cell23_torch.bool,
        device=cell23_device,
    )

    cell23_rare_source_np = cell23_to_numpy(cell23_globals["rare_classes_list_cell3"])

    if cell23_rare_source_np is None:
        raise TypeError("[Cell 23][FATAL] rare_classes_list_cell3 could not be converted.")

    cell23_rare_classes = sorted(
        set(
            int(cell23_rare_value)
            for cell23_rare_value in cell23_rare_source_np.reshape(-1).tolist()
        )
    )

    cell23_ei_pos = cell23_move_tensor(cell23_globals["ei_pos_cell7"], cell23_device)
    cell23_ew_pos = cell23_move_tensor(cell23_globals["ew_pos_cell7"], cell23_device)
    cell23_dis_pos = cell23_move_tensor(cell23_globals["dis_pos_cell7"], cell23_device)
    cell23_ei_neg = cell23_move_tensor(cell23_globals["ei_neg_cell7"], cell23_device)
    cell23_ew_neg = cell23_move_tensor(cell23_globals["ew_neg_cell7"], cell23_device)
    cell23_dis_neg = cell23_move_tensor(cell23_globals["dis_neg_cell7"], cell23_device)

    cell23_labels_t = cell23_torch.as_tensor(
        cell23_labels_np,
        dtype=cell23_torch.long,
        device=cell23_device,
    )

    cell23_super_labels_t = cell23_as_long_tensor(
        cell23_globals["super_labels_tensor_cell3"],
        "Cell 23 super labels",
    )

    cell23_eye = cell23_torch.eye(
        cell23_C,
        dtype=cell23_torch.float32,
        device=cell23_device,
    )

    # ------------------------------------------------------------------
    # Model/output helpers.
    # ------------------------------------------------------------------
    def cell23_collect_tensors(cell23_output_object, cell23_tensor_list=None):
        if cell23_tensor_list is None:
            cell23_tensor_list = []

        if cell23_torch.is_tensor(cell23_output_object):
            if int(cell23_output_object.ndim) >= 2:
                cell23_tensor_list.append(cell23_output_object)
            return cell23_tensor_list

        if isinstance(cell23_output_object, dict):
            for cell23_dict_value in cell23_output_object.values():
                cell23_collect_tensors(cell23_dict_value, cell23_tensor_list)
            return cell23_tensor_list

        if isinstance(cell23_output_object, (tuple, list)):
            for cell23_sequence_value in cell23_output_object:
                cell23_collect_tensors(cell23_sequence_value, cell23_tensor_list)
            return cell23_tensor_list

        return cell23_tensor_list

    def cell23_get_proto_dim(cell23_model_object):
        cell23_proto_parameter = getattr(cell23_model_object, "proto_class", None)

        if cell23_torch.is_tensor(cell23_proto_parameter) and int(cell23_proto_parameter.ndim) == 2:
            return int(cell23_proto_parameter.shape[1])

        return None

    def cell23_extract_embedding(cell23_output_object, cell23_model_object, cell23_expected_rows):
        cell23_candidate_tensors = cell23_collect_tensors(cell23_output_object)

        if len(cell23_candidate_tensors) == 0:
            raise TypeError(
                "[Cell 23][FATAL] Model output contained no 2D+ torch tensor. "
                "Output type: " + str(type(cell23_output_object))
            )

        cell23_proto_dim = cell23_get_proto_dim(cell23_model_object)

        if cell23_proto_dim is not None:
            for cell23_candidate_tensor in cell23_candidate_tensors:
                if int(cell23_candidate_tensor.shape[1]) == cell23_proto_dim:
                    return cell23_candidate_tensor

        for cell23_candidate_tensor in cell23_candidate_tensors:
            if int(cell23_candidate_tensor.shape[0]) == int(cell23_expected_rows):
                return cell23_candidate_tensor

        return cell23_candidate_tensors[0]

    def cell23_forward_spgcn(
        cell23_model_object,
        cell23_x_tensor,
        cell23_ei_pos_tensor,
        cell23_ew_pos_tensor,
        cell23_dis_pos_tensor,
        cell23_ei_neg_tensor,
        cell23_ew_neg_tensor,
        cell23_dis_neg_tensor,
    ):
        cell23_last_type_error = None

        try:
            return cell23_model_object(
                cell23_x_tensor,
                cell23_ei_pos_tensor,
                cell23_ew_pos_tensor,
                cell23_dis_pos_tensor,
                cell23_ei_neg_tensor,
                cell23_ew_neg_tensor,
                cell23_dis_neg_tensor,
            )
        except TypeError as cell23_type_error:
            cell23_last_type_error = cell23_type_error

        try:
            return cell23_model_object(
                cell23_x_tensor,
                cell23_ei_pos_tensor,
                cell23_dis_pos_tensor,
                cell23_ei_neg_tensor,
                cell23_dis_neg_tensor,
            )
        except TypeError as cell23_type_error:
            cell23_last_type_error = cell23_type_error

        try:
            return cell23_model_object(
                cell23_x_tensor,
                cell23_ei_pos_tensor,
                cell23_ei_neg_tensor,
            )
        except TypeError as cell23_type_error:
            cell23_last_type_error = cell23_type_error

        raise TypeError(
            "[Cell 23][FATAL] Could not call SPGCN model with any known signature. "
            "Last TypeError: " + repr(cell23_last_type_error)
        )

    def cell23_predict_spgcn(
        cell23_model_object,
        cell23_x_tensor,
        cell23_ei_pos_tensor,
        cell23_ew_pos_tensor,
        cell23_dis_pos_tensor,
        cell23_ei_neg_tensor,
        cell23_ew_neg_tensor,
        cell23_dis_neg_tensor,
    ):
        cell23_model_object.eval()
        cell23_model_device = next(cell23_model_object.parameters()).device

        cell23_x_device = cell23_x_tensor.to(cell23_model_device)
        cell23_ei_pos_device = cell23_move_tensor(cell23_ei_pos_tensor, cell23_model_device)
        cell23_ew_pos_device = cell23_move_tensor(cell23_ew_pos_tensor, cell23_model_device)
        cell23_dis_pos_device = cell23_move_tensor(cell23_dis_pos_tensor, cell23_model_device)
        cell23_ei_neg_device = cell23_move_tensor(cell23_ei_neg_tensor, cell23_model_device)
        cell23_ew_neg_device = cell23_move_tensor(cell23_ew_neg_tensor, cell23_model_device)
        cell23_dis_neg_device = cell23_move_tensor(cell23_dis_neg_tensor, cell23_model_device)

        with cell23_torch.no_grad():
            cell23_output_object = cell23_forward_spgcn(
                cell23_model_object,
                cell23_x_device,
                cell23_ei_pos_device,
                cell23_ew_pos_device,
                cell23_dis_pos_device,
                cell23_ei_neg_device,
                cell23_ew_neg_device,
                cell23_dis_neg_device,
            )

            cell23_candidate_tensors = cell23_collect_tensors(cell23_output_object)

            if len(cell23_candidate_tensors) == 0:
                raise TypeError("[Cell 23][FATAL] No 2D+ tensor in model output.")

            cell23_proto_parameter = getattr(cell23_model_object, "proto_class", None)

            if cell23_torch.is_tensor(cell23_proto_parameter) and int(cell23_proto_parameter.ndim) == 2:
                cell23_proto_float = cell23_proto_parameter.detach().float()
                cell23_proto_dim = int(cell23_proto_float.shape[1])

                for cell23_candidate_tensor in cell23_candidate_tensors:
                    if int(cell23_candidate_tensor.shape[1]) == cell23_proto_dim:
                        cell23_distance_matrix = cell23_torch.cdist(
                            cell23_candidate_tensor.detach().float(),
                            cell23_proto_float,
                        )
                        return (
                            cell23_distance_matrix
                            .argmin(dim=1)
                            .detach()
                            .cpu()
                            .numpy()
                            .astype(cell23_np.int64)
                        )

            for cell23_candidate_tensor in cell23_candidate_tensors:
                if int(cell23_candidate_tensor.shape[1]) == cell23_C:
                    return (
                        cell23_candidate_tensor
                        .argmax(dim=1)
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(cell23_np.int64)
                    )

            return (
                cell23_candidate_tensors[0]
                .argmax(dim=-1)
                .detach()
                .cpu()
                .numpy()
                .astype(cell23_np.int64)
            )

    def cell23_refine_prototypes(
        cell23_model_object,
        cell23_x_tensor,
        cell23_ei_pos_tensor,
        cell23_ew_pos_tensor,
        cell23_dis_pos_tensor,
        cell23_ei_neg_tensor,
        cell23_ew_neg_tensor,
        cell23_dis_neg_tensor,
        cell23_train_index,
        cell23_unlabeled_index,
        cell23_labels_np_input,
        cell23_class_count,
    ):
        cell23_model_object.eval()
        cell23_model_device = next(cell23_model_object.parameters()).device

        cell23_x_device = cell23_x_tensor.to(cell23_model_device)
        cell23_ei_pos_device = cell23_move_tensor(cell23_ei_pos_tensor, cell23_model_device)
        cell23_ew_pos_device = cell23_move_tensor(cell23_ew_pos_tensor, cell23_model_device)
        cell23_dis_pos_device = cell23_move_tensor(cell23_dis_pos_tensor, cell23_model_device)
        cell23_ei_neg_device = cell23_move_tensor(cell23_ei_neg_tensor, cell23_model_device)
        cell23_ew_neg_device = cell23_move_tensor(cell23_ew_neg_tensor, cell23_model_device)
        cell23_dis_neg_device = cell23_move_tensor(cell23_dis_neg_tensor, cell23_model_device)

        cell23_train_index_tensor = cell23_as_long_tensor(
            cell23_train_index,
            "Cell 23 train index for refinement",
        )

        cell23_unlabeled_index_tensor = cell23_as_long_tensor(
            cell23_unlabeled_index,
            "Cell 23 unlabeled index for refinement",
        )

        cell23_labels_tensor = cell23_torch.as_tensor(
            cell23_labels_np_input,
            dtype=cell23_torch.long,
            device=cell23_model_device,
        )

        if int(cell23_unlabeled_index_tensor.numel()) == 0:
            return cell23_model_object, 0

        with cell23_torch.no_grad():
            cell23_output_object = cell23_forward_spgcn(
                cell23_model_object,
                cell23_x_device,
                cell23_ei_pos_device,
                cell23_ew_pos_device,
                cell23_dis_pos_device,
                cell23_ei_neg_device,
                cell23_ew_neg_device,
                cell23_dis_neg_device,
            )

            cell23_embedding_tensor = cell23_extract_embedding(
                cell23_output_object,
                cell23_model_object,
                int(cell23_x_device.shape[0]),
            )

            cell23_train_embedding = cell23_embedding_tensor[cell23_train_index_tensor]
            cell23_unlabeled_embedding = cell23_embedding_tensor[cell23_unlabeled_index_tensor]
            cell23_train_labels = cell23_labels_tensor[cell23_train_index_tensor]

            cell23_refined_count = 0

            for cell23_class_index in range(int(cell23_class_count)):
                cell23_class_mask = cell23_train_labels == cell23_class_index
                cell23_support_count = int(cell23_torch.sum(cell23_class_mask).item())

                if cell23_support_count == 0 or cell23_support_count >= 100:
                    continue

                cell23_proto_parameter = cell23_model_object.proto_class.data[cell23_class_index]

                cell23_unlabeled_distance = cell23_torch.cdist(
                    cell23_unlabeled_embedding,
                    cell23_proto_parameter.unsqueeze(0),
                ).squeeze(-1)

                cell23_similarity = cell23_torch.exp(
                    -cell23_unlabeled_distance / float(cell23_tau_refine)
                )

                cell23_similarity_sum = cell23_similarity.sum() + 1e-8
                cell23_similarity_normalized = cell23_similarity / cell23_similarity_sum

                cell23_unlabeled_centroid = (
                    cell23_similarity_normalized.unsqueeze(-1)
                    * cell23_unlabeled_embedding
                ).sum(0)

                cell23_labeled_centroid = cell23_train_embedding[cell23_class_mask].mean(0)

                cell23_refined_proto = (
                    cell23_labeled_centroid
                    + float(cell23_lambda_refine) * cell23_unlabeled_centroid
                ) / (1.0 + float(cell23_lambda_refine))

                cell23_model_object.proto_class.data[cell23_class_index] = cell23_refined_proto
                cell23_refined_count = cell23_refined_count + 1

        return cell23_model_object, cell23_refined_count

    def cell23_call_train_spgcn(
        cell23_model_object,
        cell23_x_tensor,
        cell23_train_idx,
        cell23_val_idx,
        cell23_test_idx,
        cell23_seed_value,
    ):
        cell23_train_function = cell23_globals["train_spgcn_cell6"]

        try:
            return cell23_train_function(
                cell23_model_object,
                cell23_x_tensor,
                cell23_ei_pos,
                cell23_ew_pos,
                cell23_dis_pos,
                cell23_ei_neg,
                cell23_ew_neg,
                cell23_dis_neg,
                cell23_train_idx,
                cell23_val_idx,
                cell23_test_idx,
                cell23_globals["labels_tensor_cell3"],
                cell23_globals["super_labels_tensor_cell3"],
                cell23_globals["rare_classes_list_cell3"],
                cell23_seed_value,
                epochs_cell6=cell23_epochs,
                C_cell6=cell23_C,
                C_super_cell6=cell23_C_super,
                use_hierarchical_cell6=False,
                use_reinit_cell6=True,
            )
        except TypeError:
            try:
                return cell23_train_function(
                    cell23_model_object,
                    cell23_x_tensor,
                    cell23_ei_pos,
                    cell23_ew_pos,
                    cell23_dis_pos,
                    cell23_ei_neg,
                    cell23_ew_neg,
                    cell23_dis_neg,
                    cell23_train_idx,
                    cell23_val_idx,
                    cell23_test_idx,
                    cell23_globals["labels_tensor_cell3"],
                    cell23_globals["super_labels_tensor_cell3"],
                    cell23_globals["rare_classes_list_cell3"],
                    cell23_seed_value,
                    cell23_epochs,
                    cell23_C,
                    cell23_C_super,
                    False,
                    True,
                )
            except TypeError:
                return cell23_train_function(
                    cell23_model_object,
                    cell23_x_tensor,
                    cell23_ei_pos,
                    cell23_ew_pos,
                    cell23_dis_pos,
                    cell23_ei_neg,
                    cell23_ew_neg,
                    cell23_dis_neg,
                    cell23_train_idx,
                    cell23_val_idx,
                    cell23_test_idx,
                    cell23_globals["labels_tensor_cell3"],
                    cell23_globals["super_labels_tensor_cell3"],
                    cell23_globals["rare_classes_list_cell3"],
                    cell23_seed_value,
                )

    def cell23_make_spgcn(cell23_input_feature_dim):
        try:
            return cell23_globals["SPGCN_cell5"](
                d_features_cell5=int(cell23_input_feature_dim),
                d_cell5=cell23_d_hidden,
                d_proto_cell5=cell23_d_proto,
                n_layers_cell5=cell23_n_layers,
                n_classes_cell5=cell23_C,
                n_super_cell5=cell23_C_super,
                temperature_cell5=0.1,
                dropout_cell5=cell23_dropout,
            )
        except TypeError:
            return cell23_globals["SPGCN_cell5"](
                d_features=int(cell23_input_feature_dim),
                d=cell23_d_hidden,
                d_proto=cell23_d_proto,
                n_layers=cell23_n_layers,
                n_classes=cell23_C,
                n_super=cell23_C_super,
                temperature=0.1,
                dropout=cell23_dropout,
            )

    # ------------------------------------------------------------------
    # Metrics.
    # ------------------------------------------------------------------
    def cell23_f1_one_class(cell23_y_true_np, cell23_y_pred_np, cell23_class_value):
        cell23_true_mask = cell23_y_true_np == cell23_class_value
        cell23_pred_mask = cell23_y_pred_np == cell23_class_value

        cell23_support_count = int(cell23_np.sum(cell23_true_mask))

        if cell23_support_count == 0:
            return None

        cell23_tp_count = int(cell23_np.sum(cell23_true_mask & cell23_pred_mask))
        cell23_pred_count = int(cell23_np.sum(cell23_pred_mask))

        if cell23_pred_count == 0:
            cell23_precision_value = 0.0
        else:
            cell23_precision_value = float(cell23_tp_count) / float(cell23_pred_count)

        cell23_recall_value = float(cell23_tp_count) / float(cell23_support_count)

        if cell23_precision_value + cell23_recall_value == 0.0:
            return 0.0

        return (
            2.0
            * cell23_precision_value
            * cell23_recall_value
            / (cell23_precision_value + cell23_recall_value)
        )

    def cell23_macro_f1(cell23_y_true_np, cell23_y_pred_np, cell23_class_list=None):
        if cell23_class_list is None:
            cell23_class_iter = range(cell23_C)
        else:
            cell23_class_iter = cell23_class_list

        cell23_f1_values = []

        for cell23_class_value in cell23_class_iter:
            cell23_f1_value = cell23_f1_one_class(
                cell23_y_true_np,
                cell23_y_pred_np,
                int(cell23_class_value),
            )

            if cell23_f1_value is not None:
                cell23_f1_values.append(float(cell23_f1_value))

        if len(cell23_f1_values) == 0:
            return float("nan")

        return float(cell23_np.mean(cell23_f1_values))

    # ------------------------------------------------------------------
    # Paired statistics.
    # ------------------------------------------------------------------
    def cell23_paired_compare(cell23_a_array, cell23_b_array, cell23_a_name, cell23_b_name):
        cell23_a_array = cell23_np.asarray(cell23_a_array, dtype=float).reshape(-1)
        cell23_b_array = cell23_np.asarray(cell23_b_array, dtype=float).reshape(-1)

        cell23_output_dict = {
            "a_name": str(cell23_a_name),
            "b_name": str(cell23_b_name),
            "n_pairs": 0,
            "mean_a": float("nan"),
            "std_a": float("nan"),
            "mean_b": float("nan"),
            "std_b": float("nan"),
            "mean_delta_a_minus_b": float("nan"),
            "std_delta_a_minus_b": float("nan"),
            "win_a_count": 0,
            "win_b_count": 0,
            "tie_count": 0,
            "ttest_t": float("nan"),
            "ttest_p_two_sided": float("nan"),
            "wilcoxon_p_two_sided": float("nan"),
            "wilcoxon_p_a_greater": float("nan"),
            "bootstrap_ci_low": float("nan"),
            "bootstrap_ci_high": float("nan"),
            "verdict": "insufficient_pairs",
            "error": "",
        }

        if cell23_a_array.size != cell23_b_array.size:
            cell23_output_dict["error"] = "Array length mismatch."
            return cell23_output_dict

        cell23_finite_mask = cell23_np.isfinite(cell23_a_array) & cell23_np.isfinite(cell23_b_array)
        cell23_a_finite = cell23_a_array[cell23_finite_mask]
        cell23_b_finite = cell23_b_array[cell23_finite_mask]

        if cell23_a_finite.size < 2:
            cell23_output_dict["error"] = "Fewer than 2 finite paired observations."
            return cell23_output_dict

        cell23_delta = cell23_a_finite - cell23_b_finite

        cell23_output_dict["n_pairs"] = int(cell23_a_finite.size)
        cell23_output_dict["mean_a"] = float(cell23_np.mean(cell23_a_finite))
        cell23_output_dict["std_a"] = float(cell23_np.std(cell23_a_finite, ddof=1))
        cell23_output_dict["mean_b"] = float(cell23_np.mean(cell23_b_finite))
        cell23_output_dict["std_b"] = float(cell23_np.std(cell23_b_finite, ddof=1))
        cell23_output_dict["mean_delta_a_minus_b"] = float(cell23_np.mean(cell23_delta))
        cell23_output_dict["std_delta_a_minus_b"] = float(cell23_np.std(cell23_delta, ddof=1))
        cell23_output_dict["win_a_count"] = int(cell23_np.sum(cell23_a_finite > cell23_b_finite))
        cell23_output_dict["win_b_count"] = int(cell23_np.sum(cell23_b_finite > cell23_a_finite))
        cell23_output_dict["tie_count"] = int(cell23_np.sum(cell23_a_finite == cell23_b_finite))

        try:
            import scipy.stats as cell23_stats

            cell23_t_stat, cell23_p_ttest = cell23_stats.ttest_rel(cell23_a_finite, cell23_b_finite)
            cell23_output_dict["ttest_t"] = float(cell23_t_stat)
            cell23_output_dict["ttest_p_two_sided"] = float(cell23_p_ttest)

            if bool(cell23_np.any(cell23_delta != 0.0)):
                cell23_w_two_stat, cell23_p_w_two = cell23_stats.wilcoxon(
                    cell23_delta,
                    alternative="two-sided",
                    zero_method="zsplit",
                )
                cell23_w_greater_stat, cell23_p_w_greater = cell23_stats.wilcoxon(
                    cell23_delta,
                    alternative="greater",
                    zero_method="zsplit",
                )
                cell23_output_dict["wilcoxon_p_two_sided"] = float(cell23_p_w_two)
                cell23_output_dict["wilcoxon_p_a_greater"] = float(cell23_p_w_greater)
            else:
                cell23_output_dict["wilcoxon_p_two_sided"] = 1.0
                cell23_output_dict["wilcoxon_p_a_greater"] = 1.0

        except Exception as cell23_stats_error:
            cell23_output_dict["error"] = repr(cell23_stats_error)

        try:
            cell23_rng = cell23_np.random.default_rng(0)
            cell23_boot_means = []

            for cell23_boot_index in range(int(cell23_bootstrap_iterations)):
                cell23_boot_sample = cell23_rng.choice(
                    cell23_delta,
                    size=int(cell23_delta.size),
                    replace=True,
                )
                cell23_boot_means.append(float(cell23_np.mean(cell23_boot_sample)))

            cell23_boot_array = cell23_np.asarray(cell23_boot_means, dtype=float)
            cell23_ci_low, cell23_ci_high = cell23_np.percentile(cell23_boot_array, [2.5, 97.5])

            cell23_output_dict["bootstrap_ci_low"] = float(cell23_ci_low)
            cell23_output_dict["bootstrap_ci_high"] = float(cell23_ci_high)

            if cell23_ci_low > 0.0:
                cell23_output_dict["verdict"] = str(cell23_a_name) + " > " + str(cell23_b_name)
            elif cell23_ci_high < 0.0:
                cell23_output_dict["verdict"] = str(cell23_b_name) + " > " + str(cell23_a_name)
            else:
                cell23_output_dict["verdict"] = str(cell23_a_name) + " ~= " + str(cell23_b_name)

        except Exception as cell23_boot_error:
            if cell23_output_dict["error"] == "":
                cell23_output_dict["error"] = repr(cell23_boot_error)
            else:
                cell23_output_dict["error"] = cell23_output_dict["error"] + " | bootstrap: " + repr(cell23_boot_error)

        return cell23_output_dict

    # ------------------------------------------------------------------
    # Header print.
    # ------------------------------------------------------------------
    print("=" * 100)
    print("CELL 23 — SSPR + HEB PAIRED 100-SEED RUN")
    print("=" * 100)
    print("[cfg] seeds=" + str(cell23_seeds[:10]) + " ... total=" + str(len(cell23_seeds)))
    print("[cfg] epochs=" + str(cell23_epochs))
    print("[cfg] SSPR refinement lambda=" + str(cell23_lambda_refine))
    print("[cfg] SSPR refinement tau=" + str(cell23_tau_refine))
    print("[cfg] HEB kappa0=" + str(cell23_kappa0))
    print("[cfg] HEB d_hidden=" + str(cell23_d_hidden))
    print("[cfg] HEB d_proto=" + str(cell23_d_proto))
    print("[cfg] HEB n_layers=" + str(cell23_n_layers))
    print("[cfg] HEB dropout=" + str(cell23_dropout))
    print("[cfg] HEB reinit_epoch=" + str(cell23_heb_reinit_epoch))
    print("[cfg] HEB eval_every=" + str(cell23_heb_eval_every))
    print("[data] N=" + f"{cell23_N:,}")
    print("[data] X shape=" + str(tuple(cell23_X.shape)))
    print("[data] C=" + str(cell23_C) + ", C_super=" + str(cell23_C_super))
    print("[data] rare_classes=" + str(cell23_rare_classes))
    print("=" * 100)

    cell23_rows = []
    cell23_consecutive_failures = 0

    cell23_heb_api = cell23_globals["cell20_heb_api"]

    for cell23_seed_value in cell23_seeds:
        cell23_seed_start_time = cell23_time.perf_counter()

        print(
            "\n>>> Cell 23 seed "
            + str(cell23_seed_value)
            + " / "
            + str(cell23_seeds[-1]),
            flush=True,
        )

        cell23_row = {
            "seed": int(cell23_seed_value),
            "status": "pending",
            "sspr_before_rare": float("nan"),
            "sspr_after_rare": float("nan"),
            "sspr_macro_f1": float("nan"),
            "sspr_refined_count": 0,
            "heb_train_rare": float("nan"),
            "heb_val_rare": float("nan"),
            "heb_test_rare": float("nan"),
            "error": "",
            "elapsed_sec": float("nan"),
        }

        try:
            cell23_train_idx, cell23_val_idx, cell23_test_idx = cell23_globals["make_split_cell4"](
                cell23_seed_value,
                cell23_globals["labeled_mask_cell3"],
                cell23_globals["labels_tensor_cell3"],
                cell23_globals["C_cell3"],
            )

            cell23_train_np = cell23_to_int_vector(cell23_train_idx, "Cell 23 train idx")
            cell23_val_np = cell23_to_int_vector(cell23_val_idx, "Cell 23 val idx")
            cell23_test_np = cell23_to_int_vector(cell23_test_idx, "Cell 23 test idx")

            cell23_y_test_np = cell23_labels_np[cell23_test_np].copy()

            # ----------------------------------------------------------
            # SSPR full.
            # ----------------------------------------------------------
            cell23_sspr_model = cell23_make_spgcn(cell23_feature_dim).to(cell23_device)

            cell23_sspr_train_result = cell23_call_train_spgcn(
                cell23_sspr_model,
                cell23_X,
                cell23_train_idx,
                cell23_val_idx,
                cell23_test_idx,
                cell23_seed_value,
            )

            cell23_sspr_pred_before_full = cell23_predict_spgcn(
                cell23_sspr_model,
                cell23_X,
                cell23_ei_pos,
                cell23_ew_pos,
                cell23_dis_pos,
                cell23_ei_neg,
                cell23_ew_neg,
                cell23_dis_neg,
            )

            cell23_sspr_pred_before_test = cell23_sspr_pred_before_full[cell23_test_np].copy()

            cell23_sspp_before_rare = float(
                cell23_globals["rare_balanced_acc_cell6"](
                    cell23_y_test_np,
                    cell23_sspr_pred_before_test,
                    cell23_rare_classes,
                )
            )

            # Build unlabeled refinement set, excluding any labeled split nodes.
            cell23_unlabeled_all_t = cell23_torch.where(~cell23_labeled_mask_t)[0]
            cell23_unlabeled_all_np = cell23_unlabeled_all_t.detach().cpu().numpy().astype(cell23_np.int64)

            cell23_excluded_np = cell23_np.unique(
                cell23_np.concatenate([cell23_train_np, cell23_val_np, cell23_test_np])
            )

            if cell23_excluded_np.size > 0:
                cell23_unlabeled_keep_mask = ~cell23_np.isin(cell23_unlabeled_all_np, cell23_excluded_np)
                cell23_unlabeled_np = cell23_unlabeled_all_np[cell23_unlabeled_keep_mask]
            else:
                cell23_unlabeled_np = cell23_unlabeled_all_np

            cell23_unlabeled_t = cell23_torch.as_tensor(
                cell23_unlabeled_np,
                dtype=cell23_torch.long,
                device=cell23_device,
            )

            cell23_sspr_model, cell23_refined_count = cell23_refine_prototypes(
                cell23_sspr_model,
                cell23_X,
                cell23_ei_pos,
                cell23_ew_pos,
                cell23_dis_pos,
                cell23_ei_neg,
                cell23_ew_neg,
                cell23_dis_neg,
                cell23_train_idx,
                cell23_unlabeled_t,
                cell23_labels_np,
                cell23_C,
            )

            cell23_sspr_pred_after_full = cell23_predict_spgcn(
                cell23_sspr_model,
                cell23_X,
                cell23_ei_pos,
                cell23_ew_pos,
                cell23_dis_pos,
                cell23_ei_neg,
                cell23_ew_neg,
                cell23_dis_neg,
            )

            cell23_sspr_pred_after_test = cell23_sspr_pred_after_full[cell23_test_np].copy()

            cell23_sspp_after_rare = float(
                cell23_globals["rare_balanced_acc_cell6"](
                    cell23_y_test_np,
                    cell23_sspr_pred_after_test,
                    cell23_rare_classes,
                )
            )

            cell23_sspp_macro = float(
                cell23_macro_f1(
                    cell23_y_test_np,
                    cell23_sspr_pred_after_test,
                    list(range(cell23_C)),
                )
            )

            cell23_row["sspr_before_rare"] = cell23_sspp_before_rare
            cell23_row["sspr_after_rare"] = cell23_sspp_after_rare
            cell23_row["sspr_macro_f1"] = cell23_sspp_macro
            cell23_row["sspr_refined_count"] = int(cell23_refined_count)

            # Free SSPR model before HEB.
            del cell23_sspr_model
            del cell23_sspr_pred_before_full
            del cell23_sspr_pred_after_full

            if cell23_torch.cuda.is_available():
                cell23_torch.cuda.empty_cache()

            cell23_gc.collect()

            # ----------------------------------------------------------
            # HEB.
            # ----------------------------------------------------------
            cell23_heb_model = cell23_heb_api["make_model"](
                cell20_d_features=cell23_feature_dim,
                cell20_d=cell23_d_hidden,
                cell20_d_proto=cell23_d_proto,
                cell20_n_layers=cell23_n_layers,
                cell20_n_classes=cell23_C,
                cell20_n_super=cell23_C_super,
                cell20_dropout=cell23_dropout,
                cell20_kappa0=cell23_kappa0,
            )

            cell23_heb_result = cell23_heb_api["train_heb"](
                cell20_model=cell23_heb_model,
                cell20_X=cell23_X,
                cell20_ei_pos=cell23_ei_pos,
                cell20_ew_pos=cell23_ew_pos,
                cell20_dis_pos=cell23_dis_pos,
                cell20_ei_neg=cell23_ei_neg,
                cell20_ew_neg=cell23_ew_neg,
                cell20_dis_neg=cell23_dis_neg,
                cell20_train_idx=cell23_train_idx,
                cell20_val_idx=cell23_val_idx,
                cell20_test_idx=cell23_test_idx,
                cell20_labels=cell23_globals["labels_tensor_cell3"],
                cell20_super_labels=cell23_globals["super_labels_tensor_cell3"],
                cell20_rare_classes=cell23_rare_classes,
                cell20_seed=cell23_seed_value,
                cell20_epochs=cell23_epochs,
                cell20_device=cell23_device,
                cell20_C=cell23_C,
                cell20_C_super=cell23_C_super,
                cell20_use_reinit=True,
                cell20_reinit_epoch=cell23_heb_reinit_epoch,
                cell20_eval_every=cell23_heb_eval_every,
            )

            cell23_row["heb_train_rare"] = float(cell23_heb_result.get("train_rare", float("nan")))
            cell23_row["heb_val_rare"] = float(cell23_heb_result.get("val_rare", float("nan")))
            cell23_row["heb_test_rare"] = float(cell23_heb_result.get("test_rare", float("nan")))

            del cell23_heb_model

            if cell23_torch.cuda.is_available():
                cell23_torch.cuda.empty_cache()

            cell23_gc.collect()

            cell23_row["status"] = "ok"
            cell23_consecutive_failures = 0

        except Exception as cell23_seed_error:
            cell23_row["status"] = "failed"
            cell23_row["error"] = repr(cell23_seed_error)
            cell23_consecutive_failures = cell23_consecutive_failures + 1
            print("[Cell 23][fail] seed " + str(cell23_seed_value) + ": " + repr(cell23_seed_error), flush=True)

        cell23_row["elapsed_sec"] = float(cell23_time.perf_counter() - cell23_seed_start_time)
        cell23_rows.append(cell23_row)

        print(
            "  SSPR before="
            + cell23_fmt(cell23_row["sspr_before_rare"])
            + " | SSPR after="
            + cell23_fmt(cell23_row["sspr_after_rare"])
            + " | HEB="
            + cell23_fmt(cell23_row["heb_test_rare"])
            + " | elapsed="
            + f"{cell23_row['elapsed_sec']:.1f}s",
            flush=True,
        )

        if cell23_consecutive_failures >= int(cell23_max_consecutive_failures):
            print(
                "[Cell 23][abort] "
                + str(cell23_consecutive_failures)
                + " consecutive seed failures. Stopping to avoid wasting compute."
            )
            break

    # ------------------------------------------------------------------
    # Build arrays.
    # ------------------------------------------------------------------
    cell23_sspp_before_array = cell23_np.asarray(
        [cell23_row.get("sspr_before_rare", float("nan")) for cell23_row in cell23_rows],
        dtype=float,
    )

    cell23_sspp_after_array = cell23_np.asarray(
        [cell23_row.get("sspr_after_rare", float("nan")) for cell23_row in cell23_rows],
        dtype=float,
    )

    cell23_sspp_macro_array = cell23_np.asarray(
        [cell23_row.get("sspr_macro_f1", float("nan")) for cell23_row in cell23_rows],
        dtype=float,
    )

    cell23_heb_array = cell23_np.asarray(
        [cell23_row.get("heb_test_rare", float("nan")) for cell23_row in cell23_rows],
        dtype=float,
    )

    cell23_paired_mask = (
        cell23_np.isfinite(cell23_sspp_after_array)
        & cell23_np.isfinite(cell23_heb_array)
    )

    cell23_paired_sspp_array = cell23_sspp_after_array[cell23_paired_mask]
    cell23_paired_heb_array = cell23_heb_array[cell23_paired_mask]

    cell23_comparison_sspr_vs_heb = cell23_paired_compare(
        cell23_paired_sspp_array,
        cell23_paired_heb_array,
        "SSPR_full",
        "HEB",
    )

    cell23_comparison_sspr_before_vs_heb = cell23_paired_compare(
        cell23_np.asarray(
            [
                cell23_row.get("sspr_before_rare", float("nan"))
                for cell23_index, cell23_row in enumerate(cell23_rows)
                if bool(cell23_paired_mask[cell23_index])
            ],
            dtype=float,
        ),
        cell23_paired_heb_array,
        "SSPR_before_refine",
        "HEB",
    )

    # ------------------------------------------------------------------
    # Summary.
    # ------------------------------------------------------------------
    cell23_sspp_after_finite = cell23_sspp_after_array[cell23_np.isfinite(cell23_sspp_after_array)]
    cell23_heb_finite = cell23_heb_array[cell23_np.isfinite(cell23_heb_array)]

    cell23_summary = {
        "n_requested_seeds": int(len(cell23_seeds)),
        "n_completed_rows": int(len(cell23_rows)),
        "n_ok_rows": int(sum(1 for cell23_row in cell23_rows if cell23_row.get("status") == "ok")),
        "n_failed_rows": int(sum(1 for cell23_row in cell23_rows if cell23_row.get("status") == "failed")),
        "n_finite_sspr_after": int(cell23_sspp_after_finite.size),
        "n_finite_heb": int(cell23_heb_finite.size),
        "n_paired": int(cell23_paired_sspp_array.size),
        "sspr_after_mean": float(cell23_np.mean(cell23_sspp_after_finite)) if cell23_sspp_after_finite.size > 0 else float("nan"),
        "sspr_after_std": float(cell23_np.std(cell23_sspp_after_finite, ddof=1)) if cell23_sspp_after_finite.size > 1 else 0.0,
        "heb_mean": float(cell23_np.mean(cell23_heb_finite)) if cell23_heb_finite.size > 0 else float("nan"),
        "heb_std": float(cell23_np.std(cell23_heb_finite, ddof=1)) if cell23_heb_finite.size > 1 else 0.0,
        "sspr_macro_mean": float(cell23_np.mean(cell23_sspp_macro_array[cell23_np.isfinite(cell23_sspp_macro_array)]))
        if bool(cell23_np.any(cell23_np.isfinite(cell23_sspp_macro_array)))
        else float("nan"),
        "comparison_sspr_vs_heb": cell23_comparison_sspr_vs_heb,
        "comparison_sspr_before_vs_heb": cell23_comparison_sspr_before_vs_heb,
        "config": {
            "seeds": cell23_seeds,
            "epochs": int(cell23_epochs),
            "lambda_refine": float(cell23_lambda_refine),
            "tau_refine": float(cell23_tau_refine),
            "kappa0": float(cell23_kappa0),
            "d_hidden": int(cell23_d_hidden),
            "d_proto": int(cell23_d_proto),
            "n_layers": int(cell23_n_layers),
            "dropout": float(cell23_dropout),
            "heb_reinit_epoch": int(cell23_heb_reinit_epoch),
            "heb_eval_every": int(cell23_heb_eval_every),
            "bootstrap_iterations": int(cell23_bootstrap_iterations),
        },
    }

    print("\n" + "=" * 100)
    print("CELL 23 — SSPR + HEB 100-SEED SUMMARY")
    print("=" * 100)
    print("Requested seeds:       " + str(cell23_summary["n_requested_seeds"]))
    print("Completed rows:        " + str(cell23_summary["n_completed_rows"]))
    print("OK rows:               " + str(cell23_summary["n_ok_rows"]))
    print("Failed rows:           " + str(cell23_summary["n_failed_rows"]))
    print("Finite SSPR-after:     " + str(cell23_summary["n_finite_sspr_after"]))
    print("Finite HEB:            " + str(cell23_summary["n_finite_heb"]))
    print("Paired comparisons:    " + str(cell23_summary["n_paired"]))
    print("")
    print(
        "SSPR full rare_ba: "
        + cell23_fmt(cell23_summary["sspr_after_mean"])
        + " +/- "
        + cell23_fmt(cell23_summary["sspr_after_std"])
    )
    print(
        "HEB rare_ba:       "
        + cell23_fmt(cell23_summary["heb_mean"])
        + " +/- "
        + cell23_fmt(cell23_summary["heb_std"])
    )
    print(
        "SSPR macro-F1:     "
        + cell23_fmt(cell23_summary["sspr_macro_mean"])
    )
    print("")
    print("SSPR full vs HEB:")
    print("  mean delta SSPR-HEB: " + cell23_fmt(cell23_comparison_sspr_vs_heb["mean_delta_a_minus_b"]))
    print("  std delta:           " + cell23_fmt(cell23_comparison_sspr_vs_heb["std_delta_a_minus_b"]))
    print("  wins SSPR/HEB/tie:   "
          + str(cell23_comparison_sspr_vs_heb["win_a_count"])
          + "/"
          + str(cell23_comparison_sspr_vs_heb["win_b_count"])
          + "/"
          + str(cell23_comparison_sspr_vs_heb["tie_count"]))
    print("  paired t-test p:     " + cell23_fmt(cell23_comparison_sspr_vs_heb["ttest_p_two_sided"]))
    print("  Wilcoxon two-sided:  " + cell23_fmt(cell23_comparison_sspr_vs_heb["wilcoxon_p_two_sided"]))
    print("  Wilcoxon SSPR>HEB:   " + cell23_fmt(cell23_comparison_sspr_vs_heb["wilcoxon_p_a_greater"]))
    print("  bootstrap CI:        ["
          + cell23_fmt(cell23_comparison_sspr_vs_heb["bootstrap_ci_low"])
          + ", "
          + cell23_fmt(cell23_comparison_sspr_vs_heb["bootstrap_ci_high"])
          + "]")
    print("  verdict:             " + str(cell23_comparison_sspr_vs_heb["verdict"]))
    print("=" * 100)

    # ------------------------------------------------------------------
    # Save artifacts.
    # ------------------------------------------------------------------
    cell23_artifact_dir = cell23_Path("artifacts")
    cell23_artifact_dir.mkdir(parents=True, exist_ok=True)

    cell23_csv_path = cell23_artifact_dir / "cell23_sspr_heb_100seeds.csv"
    cell23_json_path = cell23_artifact_dir / "cell23_sspr_heb_100seeds.json"

    cell23_csv_header = [
        "seed",
        "status",
        "sspr_before_rare",
        "sspr_after_rare",
        "sspr_macro_f1",
        "sspr_refined_count",
        "heb_train_rare",
        "heb_val_rare",
        "heb_test_rare",
        "error",
        "elapsed_sec",
    ]

    def cell23_csv_escape(cell23_text_value):
        cell23_text_string = str(cell23_text_value)
        cell23_needs_quote_flag = False

        for cell23_character_value in cell23_text_string:
            if cell23_character_value in ',\"\n\r':
                cell23_needs_quote_flag = True
                break

        if not cell23_needs_quote_flag:
            return cell23_text_string

        cell23_escaped_string = cell23_text_string.replace('"', '""')
        return '"' + cell23_escaped_string + '"'

    cell23_csv_lines = [",".join(cell23_csv_escape(cell23_field_value) for cell23_field_value in cell23_csv_header)]

    for cell23_row in cell23_rows:
        cell23_row_parts = []

        for cell23_field_value in cell23_csv_header:
            cell23_row_parts.append(cell23_csv_escape(cell23_row.get(cell23_field_value, "")))

        cell23_csv_lines.append(",".join(cell23_row_parts))

    cell23_csv_path.write_text(
        "\n".join(cell23_csv_lines) + "\n",
        encoding="utf-8",
    )

    cell23_output_dict = {
        "summary": cell23_summary,
        "rows": cell23_rows,
        "arrays": {
            "sspr_before_rare": cell23_sspp_before_array.tolist(),
            "sspr_after_rare": cell23_sspp_after_array.tolist(),
            "sspr_macro_f1": cell23_sspp_macro_array.tolist(),
            "heb_test_rare": cell23_heb_array.tolist(),
        },
    }

    cell23_json_path.write_text(
        cell23_json.dumps(cell23_output_dict, indent=2, default=str),
        encoding="utf-8",
    )

    print("Saved CSV:  " + str(cell23_csv_path))
    print("Saved JSON: " + str(cell23_json_path))

    # ------------------------------------------------------------------
    # Optional Hugging Face checkpoint/upload.
    # ------------------------------------------------------------------
    cell23_hf_save_checkpoint = cell23_globals.get("hf_save_checkpoint", None)

    if callable(cell23_hf_save_checkpoint):
        try:
            cell23_hf_save_checkpoint(
                "cell23_sspr_heb_100seed_paired",
                {
                    "summary": cell23_summary,
                    "rows": cell23_rows,
                    "paths": {
                        "csv": str(cell23_csv_path),
                        "json": str(cell23_json_path),
                    },
                },
                description=(
                    "Cell 23: paired SSPR and HEB 100-seed run. "
                    "Failures are recorded, not fabricated."
                ),
                upload=True,
            )
        except Exception as cell23_checkpoint_error:
            print("[ckpt][warn] Cell 23 checkpoint failed: " + repr(cell23_checkpoint_error))
    else:
        print(
            "[ckpt][skip] hf_save_checkpoint not defined. "
            "Run Cell 00 first to enable Hugging Face checkpoints."
        )

    cell23_hf_upload_file_to_bucket = cell23_globals.get("hf_upload_file_to_bucket", None)

    if callable(cell23_hf_upload_file_to_bucket):
        cell23_remote_paths = {
            str(cell23_csv_path): "checkpoints/cell23_sspr_heb_100seed_paired/cell23_sspr_heb_100seeds.csv",
            str(cell23_json_path): "checkpoints/cell23_sspr_heb_100seed_paired/cell23_sspr_heb_100seeds.json",
        }

        for cell23_local_path_string, cell23_remote_path_string in cell23_remote_paths.items():
            try:
                cell23_upload_ok, cell23_upload_info = cell23_hf_upload_file_to_bucket(
                    cell23_Path(cell23_local_path_string),
                    cell23_remote_path_string,
                )
                print(
                    "[ckpt:cell23] upload "
                    + cell23_Path(cell23_local_path_string).name
                    + ": "
                    + ("OK" if cell23_upload_ok else "FAILED")
                    + " | "
                    + str(cell23_upload_info)
                )
            except Exception as cell23_upload_error:
                print(
                    "[ckpt][warn] Cell 23 upload failed for "
                    + cell23_Path(cell23_local_path_string).name
                    + ": "
                    + repr(cell23_upload_error)
                )
    else:
        print(
            "[ckpt][skip] hf_upload_file_to_bucket not defined. "
            "Cell 23 files remain local only."
        )

    return {
        "rows": cell23_rows,
        "summary": cell23_summary,
        "sspr_before_array": cell23_sspp_before_array,
        "sspr_after_array": cell23_sspp_after_array,
        "sspr_macro_array": cell23_sspp_macro_array,
        "heb_array": cell23_heb_array,
        "paired_sspr_array": cell23_paired_sspp_array,
        "paired_heb_array": cell23_paired_heb_array,
        "comparison_sspr_vs_heb": cell23_comparison_sspr_vs_heb,
        "comparison_sspr_before_vs_heb": cell23_comparison_sspr_before_vs_heb,
        "csv_path": str(cell23_csv_path),
        "json_path": str(cell23_json_path),
    }


cell23_output = cell23_main()

cell23_rows = cell23_output["rows"]
cell23_summary = cell23_output["summary"]
cell23_sspr_before_array = cell23_output["sspr_before_array"]
cell23_sspr_after_array = cell23_output["sspr_after_array"]
cell23_sspr_macro_array = cell23_output["sspr_macro_array"]
cell23_heb_array = cell23_output["heb_array"]
cell23_paired_sspr_array = cell23_output["paired_sspr_array"]
cell23_paired_heb_array = cell23_output["paired_heb_array"]
cell23_comparison_sspr_vs_heb = cell23_output["comparison_sspr_vs_heb"]
cell23_comparison_sspr_before_vs_heb = cell23_output["comparison_sspr_before_vs_heb"]
cell23_csv_path = cell23_output["csv_path"]
cell23_json_path = cell23_output["json_path"]

In [ ]:
# Cell 24a: Rare-class balanced accuracy audit.
#
# Purpose:
#   Verify that rare_balanced_acc_cell6 equals a from-scratch per-class recall
#   computation over rare classes.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell24a_.


def cell24a_rare_ba_from_scratch(
    cell24a_y_true,
    cell24a_y_pred,
    cell24a_rare_classes,
):
    cell24a_np = globals().get("np", None)
    if cell24a_np is None:
        import numpy as cell24a_np

    cell24a_y_true_array = cell24a_np.asarray(cell24a_y_true).reshape(-1).astype(cell24a_np.int64)
    cell24a_y_pred_array = cell24a_np.asarray(cell24a_y_pred).reshape(-1).astype(cell24a_np.int64)

    if cell24a_y_true_array.size != cell24a_y_pred_array.size:
        raise ValueError(
            "[Cell 24a][FATAL] y_true and y_pred length mismatch: "
            + str(cell24a_y_true_array.size)
            + " vs "
            + str(cell24a_y_pred_array.size)
        )

    cell24a_rare_list = sorted(set(int(cell24a_x) for cell24a_x in cell24a_rare_classes))

    if len(cell24a_rare_list) == 0:
        raise ValueError("[Cell 24a][FATAL] rare_classes is empty.")

    cell24a_recalls = []
    cell24a_supports = []
    cell24a_class_details = []

    for cell24a_class_value in cell24a_rare_list:
        cell24a_true_mask = cell24a_y_true_array == cell24a_class_value
        cell24a_support_count = int(cell24a_np.sum(cell24a_true_mask))

        if cell24a_support_count == 0:
            continue

        cell24a_tp_count = int(
            cell24a_np.sum(cell24a_true_mask & (cell24a_y_pred_array == cell24a_class_value))
        )

        cell24a_recall_value = float(cell24a_tp_count) / float(cell24a_support_count)

        cell24a_recalls.append(cell24a_recall_value)
        cell24a_supports.append(cell24a_support_count)

        cell24a_class_details.append(
            {
                "class": int(cell24a_class_value),
                "support": cell24a_support_count,
                "tp": cell24a_tp_count,
                "recall": cell24a_recall_value,
            }
        )

    if len(cell24a_recalls) == 0:
        return {
            "rare_ba": float("nan"),
            "n_supported_rare_classes": 0,
            "class_details": [],
        }

    return {
        "rare_ba": float(cell24a_np.mean(cell24a_recalls)),
        "n_supported_rare_classes": int(len(cell24a_recalls)),
        "class_details": cell24a_class_details,
    }


def cell24a_audit_metric_for_predictions(
    cell24a_y_true,
    cell24a_y_pred,
    cell24a_rare_classes,
    cell24a_method_name="method",
):
    cell24a_g = globals()

    cell24a_np = cell24a_g.get("np", None)
    if cell24a_np is None:
        import numpy as cell24a_np

    cell24a_existing_metric = cell24a_g.get("rare_balanced_acc_cell6", None)

    if not callable(cell24a_existing_metric):
        raise NameError(
            "[Cell 24a][FATAL] rare_balanced_acc_cell6 is not available. "
            "Run earlier cells first."
        )

    cell24a_value_existing = float(
        cell24a_existing_metric(
            cell24a_y_true,
            cell24a_y_pred,
            cell24a_rare_classes,
        )
    )

    cell24a_scratch_dict = cell24a_rare_ba_from_scratch(
        cell24a_y_true,
        cell24a_y_pred,
        cell24a_rare_classes,
    )

    cell24a_value_scratch = float(cell24a_scratch_dict["rare_ba"])

    cell24a_abs_diff = float(abs(cell24a_value_existing - cell24a_value_scratch))

    cell24a_audit_dict = {
        "method": str(cell24a_method_name),
        "rare_ba_existing_metric": cell24a_value_existing,
        "rare_ba_from_scratch": cell24a_value_scratch,
        "abs_difference": cell24a_abs_diff,
        "n_supported_rare_classes": int(cell24a_scratch_dict["n_supported_rare_classes"]),
        "class_details": cell24a_scratch_dict["class_details"],
        "status": "ok" if cell24a_abs_diff < 1e-6 else "mismatch",
    }

    return cell24a_audit_dict

In [ ]:
# Cell 24b: Train-only superclass map for HEB.
#
# Purpose:
#   Ensure HEB's hierarchical shrinkage uses only training-split labels.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell24b_.


def cell24b_infer_superclass_map_train_only(
    cell24b_labels,
    cell24b_super_labels,
    cell24b_train_idx,
    cell24b_C,
    cell24b_C_super,
    cell24b_device=None,
):
    cell24b_torch = globals().get("torch", None)
    if cell24b_torch is None:
        import torch as cell24b_torch

    cell24b_labels_t = cell24b_torch.as_tensor(
        cell24b_labels,
        dtype=cell24b_torch.long,
    )

    cell24b_super_t = cell24b_torch.as_tensor(
        cell24b_super_labels,
        dtype=cell24b_torch.long,
    )

    if cell24b_device is None:
        cell24b_device = cell24b_labels_t.device

    cell24b_labels_t = cell24b_labels_t.to(cell24b_device)
    cell24b_super_t = cell24b_super_t.to(cell24b_device)

    cell24b_train_idx_t = cell24b_torch.as_tensor(
        cell24b_train_idx,
        dtype=cell24b_torch.long,
        device=cell24b_device,
    )

    cell24b_train_mask = cell24b_torch.zeros(
        int(cell24b_labels_t.numel()),
        dtype=cell24b_torch.bool,
        device=cell24b_device,
    )

    cell24b_train_mask[cell24b_train_idx_t] = True

    cell24b_sup_of_class = cell24b_torch.zeros(
        int(cell24b_C),
        dtype=cell24b_torch.long,
        device=cell24b_device,
    )

    for cell24b_c in range(int(cell24b_C)):
        cell24b_class_mask = (
            (cell24b_labels_t == cell24b_c)
            & cell24b_train_mask
            & (cell24b_super_t >= 0)
        )

        if bool(cell24b_torch.any(cell24b_class_mask)):
            cell24b_values = cell24b_super_t[cell24b_class_mask].long()
            cell24b_counts = cell24b_torch.bincount(
                cell24b_values,
                minlength=int(cell24b_C_super),
            )
            cell24b_sup_of_class[cell24b_c] = int(cell24b_counts.argmax().item())

    return cell24b_sup_of_class

In [ ]:
# Cell 24c: Export raw predictions.
#
# Purpose:
#   Save per-seed predictions for reproducibility and metric audit.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell24c_.


def cell24c_export_predictions(
    cell24c_method_name,
    cell24c_seed,
    cell24c_y_true,
    cell24c_y_pred,
    cell24c_test_idx=None,
    cell24c_y_prob=None,
    cell24c_extra_dict=None,
    cell24c_output_dir="artifacts/predictions",
):
    cell24c_np = globals().get("np", None)
    if cell24c_np is None:
        import numpy as cell24c_np

    cell24c_Path = globals().get("Path", None)
    if cell24c_Path is None:
        from pathlib import Path as cell24c_Path

    cell24c_datetime = globals().get("datetime", None)
    if cell24c_datetime is None:
        import datetime as cell24c_datetime

    cell24c_output_path = cell24c_Path(cell24c_output_dir)
    cell24c_output_path.mkdir(parents=True, exist_ok=True)

    cell24c_safe_method = str(cell24c_method_name).replace("/", "_").replace(" ", "_")

    cell24c_file_path = (
        cell24c_output_path
        / ("predictions_" + cell24c_safe_method + "_seed" + str(int(cell24c_seed)) + ".npz")
    )

    cell24c_save_dict = {
        "method": str(cell24c_method_name),
        "seed": int(cell24c_seed),
        "y_true": cell24c_np.asarray(cell24c_y_true).astype(cell24c_np.int64),
        "y_pred": cell24c_np.asarray(cell24c_y_pred).astype(cell24c_np.int64),
        "created_utc": cell24c_datetime.datetime.now(
            cell24c_datetime.timezone.utc
        ).isoformat(),
    }

    if cell24c_test_idx is not None:
        cell24c_save_dict["test_idx"] = cell24c_np.asarray(cell24c_test_idx).astype(cell24c_np.int64)

    if cell24c_y_prob is not None:
        cell24c_save_dict["y_prob"] = cell24c_np.asarray(cell24c_y_prob).astype(cell24c_np.float32)

    if cell24c_extra_dict is not None:
        for cell24c_key_value, cell24c_value_value in cell24c_extra_dict.items():
            cell24c_save_dict[str(cell24c_key_value)] = cell24c_value_value

    cell24c_np.savez_compressed(cell24c_file_path, **cell24c_save_dict)

    return str(cell24c_file_path)

In [ ]:
# Cell 24d: Artifact manifest.
#
# Purpose:
#   Hash all final artifacts for reproducibility.
#
# Rules:
#   - No @app.cell.
#   - No top-level underscore names.
#   - No top-level generic variables.
#   - Every local/helper/loop variable is prefixed with cell24d_.


def cell24d_build_manifest(
    cell24d_roots=None,
    cell24d_output_dir="artifacts/manifests",
):
    cell24d_Path = globals().get("Path", None)
    if cell24d_Path is None:
        from pathlib import Path as cell24d_Path

    cell24d_json = globals().get("json", None)
    if cell24d_json is None:
        import json as cell24d_json

    cell24d_hashlib = globals().get("hashlib", None)
    if cell24d_hashlib is None:
        import hashlib as cell24d_hashlib

    cell24d_datetime = globals().get("datetime", None)
    if cell24d_datetime is None:
        import datetime as cell24d_datetime

    if cell24d_roots is None:
        cell24d_roots = [
            "artifacts",
            "cross_connectome_data_cell19f",
            "cross_connectome_probe_cell19d",
        ]

    cell24d_include_suffixes = {
        ".csv",
        ".json",
        ".npz",
        ".pt",
        ".pth",
        ".pkl",
        ".txt",
        ".md",
        ".yaml",
        ".yml",
    }

    cell24d_exclude_parts = {
        "__pycache__",
        ".git",
        ".ipynb_checkpoints",
    }

    def cell24d_should_include(cell24d_path_value):
        if not cell24d_path_value.is_file():
            return False

        if cell24d_path_value.suffix.lower() not in cell24d_include_suffixes:
            return False

        for cell24d_part_value in cell24d_path_value.parts:
            if str(cell24d_part_value) in cell24d_exclude_parts:
                return False

        return True

    def cell24d_sha256_file(cell24d_path_value, cell24d_chunk_size=1024 * 1024):
        cell24d_hash_object = cell24d_hashlib.sha256()

        with open(cell24d_path_value, "rb") as cell24d_file_handle:
            while True:
                cell24d_chunk_value = cell24d_file_handle.read(int(cell24d_chunk_size))

                if not cell24d_chunk_value:
                    break

                cell24d_hash_object.update(cell24d_chunk_value)

        return cell24d_hash_object.hexdigest()

    cell24d_entries = []

    for cell24d_root_string in cell24d_roots:
        cell24d_root_path = cell24d_Path(cell24d_root_string)

        if not cell24d_root_path.exists():
            continue

        for cell24d_path_value in sorted(cell24d_root_path.rglob("*")):
            if not cell24d_should_include(cell24d_path_value):
                continue

            try:
                cell24d_stat_value = cell24d_path_value.stat()
                cell24d_sha_value = cell24d_sha256_file(cell24d_path_value)

                cell24d_entries.append(
                    {
                        "path": str(cell24d_path_value),
                        "size_bytes": int(cell24d_stat_value.st_size),
                        "sha256": str(cell24d_sha_value),
                        "modified_unix": float(cell24d_stat_value.st_mtime),
                    }
                )
            except Exception as cell24d_entry_error:
                print("[Cell 24d][warn] failed to hash " + str(cell24d_path_value) + ": " + repr(cell24d_entry_error))

    cell24d_output_path = cell24d_Path(cell24d_output_dir)
    cell24d_output_path.mkdir(parents=True, exist_ok=True)

    cell24d_manifest_json_path = cell24d_output_path / "artifact_manifest.json"
    cell24d_sha_txt_path = cell24d_output_path / "sha256sums.txt"

    cell24d_created_utc = cell24d_datetime.datetime.now(
        cell24d_datetime.timezone.utc
    ).isoformat()

    cell24d_manifest_dict = {
        "created_utc": cell24d_created_utc,
        "roots": [str(cell24d_x) for cell24d_x in cell24d_roots],
        "n_files": int(len(cell24d_entries)),
        "entries": cell24d_entries,
    }

    cell24d_manifest_json_path.write_text(
        cell24d_json.dumps(cell24d_manifest_dict, indent=2, default=str),
        encoding="utf-8",
    )

    cell24d_txt_lines = []

    for cell24d_entry_value in cell24d_entries:
        cell24d_txt_lines.append(
            str(cell24d_entry_value["sha256"])
            + "  "
            + str(cell24d_entry_value["path"])
        )

    cell24d_sha_txt_path.write_text(
        "\n".join(cell24d_txt_lines) + "\n",
        encoding="utf-8",
    )

    print("Files hashed: " + str(len(cell24d_entries)))
    print("Saved manifest JSON: " + str(cell24d_manifest_json_path))
    print("Saved SHA256 text:   " + str(cell24d_sha_txt_path))

    return {
        "n_files": int(len(cell24d_entries)),
        "manifest_json_path": str(cell24d_manifest_json_path),
        "sha256_txt_path": str(cell24d_sha_txt_path),
        "entries": cell24d_entries,
    }

In [ ]:
# Cell 27: IMMEDIATE EXECUTABLE AUDIT
# This cell runs top-to-bottom and prints the verdicts immediately.
# No function definitions that just sit there. It executes now.

import numpy as cell27_np
import torch as cell27_torch

print("=" * 80)
print("CELL 27: IMMEDIATE EXECUTABLE AUDIT")
print("=" * 80)

cell27_g = globals()

# =========================================================================
# PART 1: THE METRIC FUNCTION AUDIT (Synthetic Test)
# =========================================================================
print("\n[PART 1] AUDITING rare_balanced_acc_cell6 FUNCTION LOGIC")
print("-" * 80)

cell27_metric_fn = cell27_g.get("rare_balanced_acc_cell6", None)

if not callable(cell27_metric_fn):
    print("[BLOCKER] rare_balanced_acc_cell6 is not in globals. Cannot audit metric.")
else:
    # Synthetic test: 
    # Classes 0, 1 are rare. Class 2 is majority (non-rare).
    # True labels: three 0s, three 1s, three 2s.
    # Pred labels: Class 0 makes 1 error into Class 2 (non-rare).
    #              Class 1 makes 1 error into Class 2 (non-rare).
    # Correct Recall for 0: 2/3. Correct Recall for 1: 2/3.
    # Correct Rare BA = (2/3 + 2/3) / 2 = 0.666666...

    cell27_y_true_synth = cell27_np.array([0, 0, 0, 1, 1, 1, 2, 2, 2], dtype=cell27_np.int64)
    cell27_y_pred_synth = cell27_np.array([0, 0, 2, 1, 1, 2, 0, 1, 2], dtype=cell27_np.int64)
    cell27_rare_synth = [0, 1]

    cell27_expected_ba = 2.0 / 3.0

    try:
        cell27_computed_ba = float(cell27_metric_fn(
            cell27_y_true_synth, 
            cell27_y_pred_synth, 
            cell27_rare_synth
        ))
    
        print(f"Expected Rare BA:  {cell27_expected_ba:.6f}")
        print(f"Computed Rare BA:  {cell27_computed_ba:.6f}")
    
        if abs(cell27_computed_ba - cell27_expected_ba) < 1e-5:
            print("[PASS] The metric function correctly counts errors into non-rare classes.")
            print("       Your headline numbers are mathematically safe.")
        else:
            print("[FAIL] THE METRIC FUNCTION IS BROKEN.")
            print("       It is likely dropping errors that fall into non-rare classes.")
            print("       ALL HEADLINE NUMBERS MUST BE REGENERATED.")
    except Exception as cell27_met_err:
        print(f"[FAIL] Metric function threw an exception: {repr(cell27_met_err)}")


# =========================================================================
# PART 2: THE HEB SUPERCLASS LEAKAGE AUDIT (Seed 0)
# =========================================================================
print("\n[PART 2] AUDITING HEB SUPERCLASS MAP LEAKAGE (Seed 0)")
print("-" * 80)

cell27_labels = cell27_g.get("labels_tensor_cell3", None)
cell27_super = cell27_g.get("super_labels_tensor_cell3", None)
cell27_mask = cell27_g.get("labeled_mask_cell3", None)
cell27_C = cell27_g.get("C_cell3", None)
cell27_C_super = cell27_g.get("C_super_cell3", None)
cell27_split_fn = cell27_g.get("make_split_cell4", None)

if None in [cell27_labels, cell27_super, cell27_C, cell27_C_super, cell27_split_fn]:
    print("[BLOCKER] Missing core FlyWire variables in globals. Cannot audit leakage.")
else:
    cell27_labels_t = cell27_torch.as_tensor(cell27_labels, dtype=cell27_torch.long)
    cell27_super_t = cell27_torch.as_tensor(cell27_super, dtype=cell27_torch.long)

    # 1. Compute map using ALL labeled neurons (The Leak)
    cell27_map_all = cell27_torch.zeros(int(cell27_C), dtype=cell27_torch.long)
    for cell27_c in range(int(cell27_C)):
        cell27_m = (cell27_labels_t == cell27_c) & cell27_torch.as_tensor(cell27_mask, dtype=cell27_torch.bool) & (cell27_super_t >= 0)
        if cell27_m.any():
            cell27_map_all[cell27_c] = cell27_torch.bincount(cell27_super_t[cell27_m], minlength=int(cell27_C_super)).argmax()

    # 2. Compute map using ONLY Seed 0 Train neurons (The Fix)
    cell27_tr, _, _ = cell27_split_fn(0, cell27_mask, cell27_labels, cell27_C)
    cell27_tr_t = cell27_torch.as_tensor(cell27_tr, dtype=cell27_torch.long)
    cell27_tr_mask = cell27_torch.zeros(cell27_labels_t.numel(), dtype=cell27_torch.bool)
    cell27_tr_mask[cell27_tr_t] = True

    cell27_map_train = cell27_torch.zeros(int(cell27_C), dtype=cell27_torch.long)
    for cell27_c in range(int(cell27_C)):
        cell27_m = (cell27_labels_t == cell27_c) & cell27_tr_mask & (cell27_super_t >= 0)
        if cell27_m.any():
            cell27_map_train[cell27_c] = cell27_torch.bincount(cell27_super_t[cell27_m], minlength=int(cell27_C_super)).argmax()

    # 3. Compare
    cell27_diffs = (cell27_map_all != cell27_map_train).nonzero(as_tuple=True)[0]

    if len(cell27_diffs) == 0:
        print("[PASS] The full-labeled superclass map perfectly matches the train-only map.")
        print("       HEB leakage is not an issue on this specific split.")
    else:
        print(f"[FAIL] LEAKAGE DETECTED. {len(cell27_diffs)} classes have different superclasses")
        print(f"       when using all labeled neurons vs only train neurons.")
        print(f"       Classes affected: {cell27_diffs.tolist()}")
        print("       You MUST patch HEB to use train-only maps before submitting.")


# =========================================================================
# PART 3: THE 100-SEED ARRAY VERDICT
# =========================================================================
print("\n[PART 3] CHECKING 100-SEED HEB vs SSPR ARRAYS")
print("-" * 80)

cell27_heb_arr = cell27_g.get("cell23_heb_array", cell27_g.get("heb_test_rare_array", None))
cell27_sspr_arr = cell27_g.get("cell23_sspr_after_array", cell27_g.get("sspr_after_cell11c", None))

if cell27_heb_arr is None or cell27_sspr_arr is None:
    print("[BLOCKER] 100-seed arrays not found in globals. Run Cell 23 first.")
else:
    cell27_heb_np = cell27_np.asarray(cell27_heb_arr, dtype=float).flatten()
    cell27_sspr_np = cell27_np.asarray(cell27_sspr_arr, dtype=float).flatten()

    cell27_finite_mask = cell27_np.isfinite(cell27_heb_np) & cell27_np.isfinite(cell27_sspr_np)
    cell27_heb_fin = cell27_heb_np[cell27_finite_mask]
    cell27_sspr_fin = cell27_sspr_np[cell27_finite_mask]

    if len(cell27_heb_fin) < 2:
        print("[BLOCKER] Not enough finite paired seeds to compute statistics.")
    else:
        cell27_delta = cell27_heb_fin - cell27_sspr_fin
        cell27_mean_d = cell27_np.mean(cell27_delta)
        cell27_std_d = cell27_np.std(cell27_delta, ddof=1)
        cell27_wins = cell27_np.sum(cell27_delta > 0)
    
        # Bootstrap CI
        cell27_rng = cell27_np.random.default_rng(42)
        cell27_boots = [cell27_np.mean(cell27_rng.choice(cell27_delta, size=len(cell27_delta), replace=True)) for _ in range(2000)]
        cell27_lo, cell27_hi = cell27_np.percentile(cell27_boots, [2.5, 97.5])
    
        print(f"Paired Seeds:      {len(cell27_heb_fin)}")
        print(f"HEB Mean:          {cell27_np.mean(cell27_heb_fin):.4f} ± {cell27_np.std(cell27_heb_fin, ddof=1):.4f}")
        print(f"SSPR Mean:         {cell27_np.mean(cell27_sspr_fin):.4f} ± {cell27_np.std(cell27_sspr_fin, ddof=1):.4f}")
        print(f"Delta (HEB-SSPR):  {cell27_mean_d:+.4f} ± {cell27_std_d:.4f}")
        print(f"HEB Wins:          {cell27_wins}/{len(cell27_heb_fin)}")
        print(f"95% Bootstrap CI:  [{cell27_lo:+.4f}, {cell27_hi:+.4f}]")
    
        if cell27_lo > 0:
            print("\n[VERDICT] HEB STRICTLY BEATS SSPR. Pivot the paper to HEB.")
        elif cell27_hi < 0:
            print("\n[VERDICT] SSPR STRICTLY BEATS HEB. Keep SSPR as headline.")
        else:
            print("\n[VERDICT] STATISTICAL TIE. Do not claim HEB superiority. Frame as prototype equivalence.")

print("\n" + "=" * 80)
print("AUDIT COMPLETE.")
print("=" * 80)

In [ ]:
# Cell 28: Factorial Ablation (30 seeds)
# Tests: SSPR-Full, HEB-Full, HEB-No-Shrinkage, HEB-Gaussian
# Rules: No @app.cell, no top-level imports, cell28_ prefix.

def cell28_main():
    cell28_g = globals()
    import numpy as cell28_np
    import torch as cell28_torch
    from pathlib import Path as cell28_Path
    import json as cell28_json
    import time as cell28_time

    cell28_required = [
        "DEVICE_cell1", "X_sspr_tensor_cell11", "labels_tensor_cell3", 
        "labeled_mask_cell3", "super_labels_tensor_cell3", "C_cell3", 
        "C_super_cell3", "rare_classes_list_cell3", "ei_pos_cell7", 
        "ew_pos_cell7", "dis_pos_cell7", "ei_neg_cell7", "ew_neg_cell7", 
        "dis_neg_cell7", "make_split_cell4", "SPGCN_cell5", 
        "train_spgcn_cell6", "rare_balanced_acc_cell6", "cell20_heb_api"
    ]
    for cell28_req in cell28_required:
        if cell28_req not in cell28_g:
            raise NameError(f"[Cell 28][FATAL] Missing {cell28_req}")

    cell28_device = cell28_g["DEVICE_cell1"]
    cell28_C = int(cell28_g["C_cell3"])
    cell28_C_super = int(cell28_g["C_super_cell3"])
    cell28_seeds = list(range(30))
    cell28_epochs = 200

    cell28_X = cell28_g["X_sspr_tensor_cell11"].to(cell28_device).float()
    cell28_d_features = int(cell28_X.shape[1])

    # Train-only superclass map (Bulletproof against leakage)
    def cell28_get_sup_map(cell28_y, cell28_y_super, cell28_tr_idx):
        cell28_tr_mask = cell28_torch.zeros(cell28_y.numel(), dtype=cell28_torch.bool, device=cell28_y.device)
        cell28_tr_mask[cell28_tr_idx] = True
        cell28_sup_map = cell28_torch.zeros(cell28_C, dtype=cell28_torch.long, device=cell28_y.device)
        for cell28_c in range(cell28_C):
            cell28_mask = (cell28_y == cell28_c) & cell28_tr_mask & (cell28_y_super >= 0)
            if cell28_mask.any():
                cell28_sup_map[cell28_c] = cell28_torch.bincount(cell28_y_super[cell28_mask], minlength=cell28_C_super).argmax()
        return cell28_sup_map

    # Gaussian Wrapper
    cell28_BaseHEB = cell28_g["cell20_heb_api"]["HEBPrototypeWrapper"]
    class cell28_GaussianHEB(cell28_BaseHEB):
        def student_t_nll(self, h, centers):
            dist2 = cell28_torch.cdist(h, centers).pow(2)
            scales = self.log_scale.exp().clamp(min=1e-3, max=1e3)
            nll = 0.5 * dist2 / scales.pow(2) + cell28_torch.log(scales).unsqueeze(0)
            return cell28_torch.nan_to_num(nll, nan=1e4, posinf=1e4, neginf=-1e4)

    def cell28_make_heb(cell28_kappa0, cell28_use_gaussian):
        cell28_enc = cell28_g["cell20_heb_api"]["make_encoder"](
            cell20_d_features=cell28_d_features, cell20_d=64, cell20_d_proto=32,
            cell20_n_layers=2, cell20_n_classes=cell28_C, cell20_n_super=cell28_C_super, cell20_dropout=0.3
        )
        if cell28_use_gaussian:
            return cell28_GaussianHEB(cell28_enc, cell28_C, cell28_C_super, 32, cell28_kappa0)
        return cell28_BaseHEB(cell28_enc, cell28_C, cell28_C_super, 32, cell28_kappa0)

    def cell28_make_sspr():
        try:
            return cell28_g["SPGCN_cell5"](d_features_cell5=cell28_d_features, d_cell5=64, d_proto_cell5=32, n_layers_cell5=2, n_classes_cell5=cell28_C, n_super_cell5=cell28_C_super, temperature_cell5=0.1, dropout_cell5=0.3)
        except TypeError:
            return cell28_g["SPGCN_cell5"](cell28_d_features, 64, 32, 2, cell28_C, cell28_C_super, 0.1, 0.3)

    variants = [
        ("SSPR-Full", "sspr", 4.0, False),
        ("HEB-Full", "heb", 4.0, False),
        ("HEB-No-Shrinkage", "heb", 0.0, False),
        ("HEB-Gaussian", "heb", 4.0, True)
    ]

    print("=" * 80)
    print("CELL 28: FACTORIAL ABLATION (30 SEEDS)")
    print("=" * 80)

    cell28_results = {v[0]: [] for v in variants}

    for cell28_seed in cell28_seeds:
        print(f"\n>>> Seed {cell28_seed}", flush=True)
        cell28_tr, cell28_va, cell28_te = cell28_g["make_split_cell4"](cell28_seed, cell28_g["labeled_mask_cell3"], cell28_g["labels_tensor_cell3"], cell28_C)
        cell28_tr_t = cell28_tr.to(cell28_device).long()
    
        for cell28_name, cell28_type, cell28_kappa, cell28_gauss in variants:
            if cell28_type == "sspr":
                cell28_model = cell28_make_sspr().to(cell28_device)
                cell28_g["train_spgcn_cell6"](
                    cell28_model, cell28_X, cell28_g["ei_pos_cell7"], cell28_g["ew_pos_cell7"], cell28_g["dis_pos_cell7"],
                    cell28_g["ei_neg_cell7"], cell28_g["ew_neg_cell7"], cell28_g["dis_neg_cell7"],
                    cell28_tr, cell28_va, cell28_te, cell28_g["labels_tensor_cell3"], 
                    cell28_g["super_labels_tensor_cell3"], cell28_g["rare_classes_list_cell3"], 
                    cell28_seed, epochs_cell6=cell28_epochs, C_cell6=cell28_C, C_super_cell6=cell28_C_super, use_reinit_cell6=True
                )
                # SSPR Refinement
                cell28_model.eval()
                with cell28_torch.no_grad():
                    cell28_h = cell28_model(cell28_X, cell28_g["ei_pos_cell7"], cell28_g["ew_pos_cell7"], cell28_g["dis_pos_cell7"], cell28_g["ei_neg_cell7"], cell28_g["ew_neg_cell7"], cell28_g["dis_neg_cell7"])
                    if isinstance(cell28_h, tuple): cell28_h = cell28_h[0]
                    cell28_y_tr = cell28_g["labels_tensor_cell3"].to(cell28_device)[cell28_tr_t]
                    for cell28_c in range(cell28_C):
                        cell28_mask = cell28_y_tr == cell28_c
                        if cell28_mask.sum() > 0:
                            cell28_model.proto_class.data[cell28_c] = cell28_h[cell28_tr_t][cell28_mask].mean(0)
            else:
                cell28_model = cell28_make_heb(cell28_kappa, cell28_gauss).to(cell28_device)
                cell28_model.set_superclass_map(cell28_get_sup_map(
                    cell28_g["labels_tensor_cell3"].to(cell28_device), 
                    cell28_g["super_labels_tensor_cell3"].to(cell28_device), 
                    cell28_tr_t
                ))
                cell28_g["cell20_heb_api"]["train_heb"](
                    cell20_model=cell28_model, cell20_X=cell28_X, 
                    cell20_ei_pos=cell28_g["ei_pos_cell7"], cell20_ew_pos=cell28_g["ew_pos_cell7"], cell20_dis_pos=cell28_g["dis_pos_cell7"],
                    cell20_ei_neg=cell28_g["ei_neg_cell7"], cell20_ew_neg=cell28_g["ew_neg_cell7"], cell20_dis_neg=cell28_g["dis_neg_cell7"],
                    cell20_train_idx=cell28_tr, cell20_val_idx=cell28_va, cell20_test_idx=cell28_te,
                    cell20_labels=cell28_g["labels_tensor_cell3"], cell20_super_labels=cell28_g["super_labels_tensor_cell3"],
                    cell20_rare_classes=cell28_g["rare_classes_list_cell3"], cell20_seed=cell28_seed,
                    cell20_epochs=cell28_epochs, cell20_device=cell28_device, cell20_C=cell28_C, cell20_C_super=cell28_C_super,
                    cell20_use_reinit=True, cell20_reinit_epoch=40, cell20_eval_every=25
                )

            # Evaluate
            cell28_model.eval()
            with cell28_torch.no_grad():
                if cell28_type == "sspr":
                    cell28_h = cell28_model(cell28_X, cell28_g["ei_pos_cell7"], cell28_g["ew_pos_cell7"], cell28_g["dis_pos_cell7"], cell28_g["ei_neg_cell7"], cell28_g["ew_neg_cell7"], cell28_g["dis_neg_cell7"])
                    if isinstance(cell28_h, tuple): cell28_h = cell28_h[0]
                    cell28_pred = cell28_torch.cdist(cell28_h, cell28_model.proto_class).argmin(1).cpu().numpy()
                else:
                    cell28_h = cell28_model(cell28_X, cell28_g["ei_pos_cell7"], cell28_g["ew_pos_cell7"], cell28_g["dis_pos_cell7"], cell28_g["ei_neg_cell7"], cell28_g["ew_neg_cell7"], cell28_g["dis_neg_cell7"])
                    cell28_nll = cell28_model.compute_nll(cell28_h, cell28_g["labels_tensor_cell3"].to(cell28_device), cell28_tr_t)
                    cell28_pred = (-cell28_nll).argmax(1).cpu().numpy()

            cell28_te_np = cell28_te.numpy()
            cell28_true = cell28_g["labels_tensor_cell3"].numpy()[cell28_te_np]
            cell28_pred_te = cell28_pred[cell28_te_np]
            cell28_rare_ba = cell28_g["rare_balanced_acc_cell6"](cell28_true, cell28_pred_te, cell28_g["rare_classes_list_cell3"])
            cell28_results[cell28_name].append(cell28_rare_ba)
            print(f"  {cell28_name}: {cell28_rare_ba:.4f}", flush=True)
        
            del cell28_model
            if cell28_torch.cuda.is_available(): cell28_torch.cuda.empty_cache()

    print("\n" + "=" * 80)
    print("CELL 28 ABLATION SUMMARY (30 Seeds)")
    print("=" * 80)
    for cell28_name, cell28_arr in cell28_results.items():
        cell28_arr_np = cell28_np.array(cell28_arr)
        print(f"{cell28_name:20s}: {cell28_arr_np.mean():.4f} +/- {cell28_arr_np.std(ddof=1):.4f}")

    cell28_out_dir = cell28_Path("artifacts")
    cell28_out_dir.mkdir(exist_ok=True)
    with open(cell28_out_dir / "cell28_ablation.json", "w") as f:
        cell28_json.dump({k: v for k, v in cell28_results.items()}, f, indent=2)
    return cell28_results

cell28_ablation_results = cell28_main()

In [ ]:
# Cell 29: Reproducibility Artifact Export
# Dumps raw predictions for HEB and SSPR to artifacts/predictions/
# Rules: No @app.cell, no top-level imports, cell29_ prefix.

def cell29_main():
    cell29_g = globals()
    import numpy as cell29_np
    import torch as cell29_torch
    from pathlib import Path as cell29_Path

    cell29_device = cell29_g["DEVICE_cell1"]
    cell29_C = int(cell29_g["C_cell3"])
    cell29_C_super = int(cell29_g["C_super_cell3"])

    # Change to list(range(100)) if you want to export all 100 seeds. 
    # 10 is usually enough for a reviewer spot-check and saves hours of compute.
    cell29_seeds = list(range(10)) 

    cell29_X = cell29_g["X_sspr_tensor_cell11"].to(cell29_device).float()
    cell29_d_features = int(cell29_X.shape[1])

    cell29_out_dir = cell29_Path("artifacts/predictions")
    cell29_out_dir.mkdir(parents=True, exist_ok=True)

    def cell29_get_sup_map(cell29_y, cell29_y_super, cell29_tr_idx):
        cell29_tr_mask = cell29_torch.zeros(cell29_y.numel(), dtype=cell29_torch.bool, device=cell29_y.device)
        cell29_tr_mask[cell29_tr_idx] = True
        cell29_sup_map = cell29_torch.zeros(cell29_C, dtype=cell29_torch.long, device=cell29_y.device)
        for cell29_c in range(cell29_C):
            cell29_mask = (cell29_y == cell29_c) & cell29_tr_mask & (cell29_y_super >= 0)
            if cell29_mask.any():
                cell29_sup_map[cell29_c] = cell29_torch.bincount(cell29_y_super[cell29_mask], minlength=cell29_C_super).argmax()
        return cell29_sup_map

    print("=" * 80)
    print(f"CELL 29: EXPORTING PREDICTIONS FOR {len(cell29_seeds)} SEEDS")
    print("=" * 80)

    for cell29_seed in cell29_seeds:
        cell29_tr, cell29_va, cell29_te = cell29_g["make_split_cell4"](cell29_seed, cell29_g["labeled_mask_cell3"], cell29_g["labels_tensor_cell3"], cell29_C)
        cell29_tr_t = cell29_tr.to(cell29_device).long()
        cell29_te_np = cell29_te.numpy()
        cell29_true_np = cell29_g["labels_tensor_cell3"].numpy()[cell29_te_np]

        # 1. SSPR
        try:
            cell29_m_sspr = cell29_g["SPGCN_cell5"](d_features_cell5=cell29_d_features, d_cell5=64, d_proto_cell5=32, n_layers_cell5=2, n_classes_cell5=cell29_C, n_super_cell5=cell29_C_super, temperature_cell5=0.1, dropout_cell5=0.3).to(cell29_device)
        except TypeError:
            cell29_m_sspr = cell29_g["SPGCN_cell5"](cell29_d_features, 64, 32, 2, cell29_C, cell29_C_super, 0.1, 0.3).to(cell29_device)
        
        cell29_g["train_spgcn_cell6"](
            cell29_m_sspr, cell29_X, cell29_g["ei_pos_cell7"], cell29_g["ew_pos_cell7"], cell29_g["dis_pos_cell7"],
            cell29_g["ei_neg_cell7"], cell29_g["ew_neg_cell7"], cell29_g["dis_neg_cell7"],
            cell29_tr, cell29_va, cell29_te, cell29_g["labels_tensor_cell3"], 
            cell29_g["super_labels_tensor_cell3"], cell29_g["rare_classes_list_cell3"], 
            cell29_seed, epochs_cell6=200, C_cell6=cell29_C, C_super_cell6=cell29_C_super, use_reinit_cell6=True
        )
        cell29_m_sspr.eval()
        with cell29_torch.no_grad():
            cell29_h = cell29_m_sspr(cell29_X, cell29_g["ei_pos_cell7"], cell29_g["ew_pos_cell7"], cell29_g["dis_pos_cell7"], cell29_g["ei_neg_cell7"], cell29_g["ew_neg_cell7"], cell29_g["dis_neg_cell7"])
            if isinstance(cell29_h, tuple): cell29_h = cell29_h[0]
            cell29_pred_sspr = cell29_torch.cdist(cell29_h, cell29_m_sspr.proto_class).argmin(1).cpu().numpy()[cell29_te_np]
        
        cell29_np.savez_compressed(cell29_out_dir / f"sspr_seed_{cell29_seed:03d}.npz", y_true=cell29_true_np, y_pred=cell29_pred_sspr, seed=cell29_seed)

        # 2. HEB
        cell29_m_heb = cell29_g["cell20_heb_api"]["make_model"](
            cell20_d_features=cell29_d_features, cell20_d=64, cell20_d_proto=32, cell20_n_layers=2,
            cell20_n_classes=cell29_C, cell20_n_super=cell29_C_super, cell20_dropout=0.3, cell20_kappa0=4.0
        ).to(cell29_device)
        cell29_m_heb.set_superclass_map(cell29_get_sup_map(
            cell29_g["labels_tensor_cell3"].to(cell29_device), 
            cell29_g["super_labels_tensor_cell3"].to(cell29_device), 
            cell29_tr_t
        ))
        cell29_g["cell20_heb_api"]["train_heb"](
            cell20_model=cell29_m_heb, cell20_X=cell29_X, 
            cell20_ei_pos=cell29_g["ei_pos_cell7"], cell20_ew_pos=cell29_g["ew_pos_cell7"], cell20_dis_pos=cell29_g["dis_pos_cell7"],
            cell20_ei_neg=cell29_g["ei_neg_cell7"], cell20_ew_neg=cell29_g["ew_neg_cell7"], cell20_dis_neg=cell29_g["dis_neg_cell7"],
            cell20_train_idx=cell29_tr, cell20_val_idx=cell29_va, cell20_test_idx=cell29_te,
            cell20_labels=cell29_g["labels_tensor_cell3"], cell20_super_labels=cell29_g["super_labels_tensor_cell3"],
            cell20_rare_classes=cell29_g["rare_classes_list_cell3"], cell20_seed=cell29_seed,
            cell20_epochs=200, cell20_device=cell29_device, cell20_C=cell29_C, cell20_C_super=cell29_C_super,
            cell20_use_reinit=True, cell20_reinit_epoch=40, cell20_eval_every=25
        )
        cell29_m_heb.eval()
        with cell29_torch.no_grad():
            cell29_h = cell29_m_heb(cell29_X, cell29_g["ei_pos_cell7"], cell29_g["ew_pos_cell7"], cell29_g["dis_pos_cell7"], cell29_g["ei_neg_cell7"], cell29_g["ew_neg_cell7"], cell29_g["dis_neg_cell7"])
            cell29_nll = cell29_m_heb.compute_nll(cell29_h, cell29_g["labels_tensor_cell3"].to(cell29_device), cell29_tr_t)
            cell29_pred_heb = (-cell29_nll).argmax(1).cpu().numpy()[cell29_te_np]

        cell29_np.savez_compressed(cell29_out_dir / f"heb_seed_{cell29_seed:03d}.npz", y_true=cell29_true_np, y_pred=cell29_pred_heb, seed=cell29_seed)
        print(f"Exported Seed {cell29_seed} SSPR & HEB")
    
        del cell29_m_sspr, cell29_m_heb
        if cell29_torch.cuda.is_available(): cell29_torch.cuda.empty_cache()

    print("=" * 80)
    print(f"EXPORT COMPLETE. Files saved to {cell29_out_dir}")
    print("Zip this folder and include it in your supplementary material.")
    print("=" * 80)

cell29_main()

In [ ]:
# Cell 30: Freeze HEB Encoder (Seed 0) for Cross-Connectome
# Rules: No @app.cell, no top-level imports, cell30_ prefix.

def cell30_main():
    cell30_g = globals()
    import torch as cell30_torch
    import numpy as cell30_np
    from pathlib import Path as cell30_Path

    cell30_device = cell30_g["DEVICE_cell1"]
    cell30_C = int(cell30_g["C_cell3"])
    cell30_C_super = int(cell30_g["C_super_cell3"])

    cell30_X = cell30_g["X_sspr_tensor_cell11"].to(cell30_device).float()
    cell30_d_features = int(cell30_X.shape[1])

    cell30_tr, cell30_va, cell30_te = cell30_g["make_split_cell4"](0, cell30_g["labeled_mask_cell3"], cell30_g["labels_tensor_cell3"], cell30_C)
    cell30_tr_t = cell30_tr.to(cell30_device).long()

    def cell30_get_sup_map(cell30_y, cell30_y_super, cell30_tr_idx):
        cell30_tr_mask = cell30_torch.zeros(cell30_y.numel(), dtype=cell30_torch.bool, device=cell30_y.device)
        cell30_tr_mask[cell30_tr_idx] = True
        cell30_sup_map = cell30_torch.zeros(cell30_C, dtype=cell30_torch.long, device=cell30_y.device)
        for cell30_c in range(cell30_C):
            cell30_mask = (cell30_y == cell30_c) & cell30_tr_mask & (cell30_y_super >= 0)
            if cell30_mask.any():
                cell30_sup_map[cell30_c] = cell30_torch.bincount(cell30_y_super[cell30_mask], minlength=cell30_C_super).argmax()
        return cell30_sup_map

    print("=" * 80)
    print("CELL 30: TRAINING & FREEZING HEB ENCODER (SEED 0)")
    print("=" * 80)

    cell30_model = cell30_g["cell20_heb_api"]["make_model"](
        cell20_d_features=cell30_d_features, cell20_d=64, cell20_d_proto=32, cell20_n_layers=2,
        cell20_n_classes=cell30_C, cell20_n_super=cell30_C_super, cell20_dropout=0.3, cell20_kappa0=4.0
    ).to(cell30_device)

    cell30_model.set_superclass_map(cell30_get_sup_map(
        cell30_g["labels_tensor_cell3"].to(cell30_device), 
        cell30_g["super_labels_tensor_cell3"].to(cell30_device), 
        cell30_tr_t
    ))

    cell30_g["cell20_heb_api"]["train_heb"](
        cell20_model=cell30_model, cell20_X=cell30_X, 
        cell20_ei_pos=cell30_g["ei_pos_cell7"], cell20_ew_pos=cell30_g["ew_pos_cell7"], cell20_dis_pos=cell30_g["dis_pos_cell7"],
        cell20_ei_neg=cell30_g["ei_neg_cell7"], cell20_ew_neg=cell30_g["ew_neg_cell7"], cell20_dis_neg=cell30_g["dis_neg_cell7"],
        cell20_train_idx=cell30_tr, cell20_val_idx=cell30_va, cell20_test_idx=cell30_te,
        cell20_labels=cell30_g["labels_tensor_cell3"], cell20_super_labels=cell30_g["super_labels_tensor_cell3"],
        cell20_rare_classes=cell30_g["rare_classes_list_cell3"], cell20_seed=0,
        cell20_epochs=200, cell20_device=cell30_device, cell20_C=cell30_C, cell20_C_super=cell30_C_super,
        cell20_use_reinit=True, cell20_reinit_epoch=40, cell20_eval_every=25
    )

    # Compute feature stats for normalization during transfer
    cell30_feat_mean = cell30_X.mean(dim=0).cpu()
    cell30_feat_std = cell30_X.std(dim=0).cpu()

    cell30_save_dict = {
        "method": "HEB-Proto",
        "seed": 0,
        "encoder_state": {k: v.cpu() for k, v in cell30_model.encoder.state_dict().items()},
        "proto_class": cell30_model.encoder.proto_class.data.cpu(),
        "proto_super": cell30_model.encoder.proto_super.data.cpu(),
        "log_scale": cell30_model.log_scale.data.cpu(),
        "sup_of_class": cell30_model.sup_of_class.cpu(),
        "feature_mean": cell30_feat_mean,
        "feature_std": cell30_feat_std,
        "d_features": cell30_d_features,
        "d_hidden": 64,
        "d_proto": 32,
        "n_layers": 2,
        "n_classes": cell30_C,
        "n_super": cell30_C_super,
        "kappa0": 4.0,
        "dropout": 0.3,
    }

    cell30_out_path = cell30_Path("./heb_fafb_frozen.pt")
    cell30_torch.save(cell30_save_dict, cell30_out_path)

    print("=" * 80)
    print(f"FROZEN HEB SAVED TO: {cell30_out_path}")
    print("You can now pass this path to Cell 19f to evaluate HEB cross-connectome coherence.")
    print("=" * 80)

    return str(cell30_out_path)

cell30_heb_frozen_path = cell30_main()

In [ ]:
# Cell 31: Cross-Connectome Coherence with Frozen HEB Encoder (PATCHED)
# Fixes numpy string sorting crash and handles missing labels safely.

def cell31_heb_cross_connectome():
    import torch
    import numpy as np
    import pandas as pd
    import gzip
    import urllib.request
    from pathlib import Path
    from scipy.sparse import csr_matrix
    from sklearn.decomposition import TruncatedSVD
    from sklearn.cluster import KMeans
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

    print("=" * 80)
    print("CELL 31: CROSS-CONNECTOME COHERENCE (FROZEN HEB ENCODER)")
    print("=" * 80)

    # 1. Load Frozen HEB Checkpoint
    ckpt_path = Path("./heb_fafb_frozen.pt")
    if not ckpt_path.exists():
        print("[FATAL] heb_fafb_frozen.pt not found. Run Cell 30 first.")
        return None
    
    ckpt = torch.load(ckpt_path, map_location="cpu")
    feat_mean = ckpt["feature_mean"].numpy()
    feat_std = ckpt["feature_std"].numpy()

    # Instantiate Encoder
    SPGCN = globals().get("SPGCN_cell5")
    if SPGCN is None:
        print("[FATAL] SPGCN_cell5 not in globals.")
        return None
    
    try:
        encoder = SPGCN(d_features_cell5=ckpt["d_features"], d_cell5=ckpt["d_hidden"], 
                        d_proto_cell5=ckpt["d_proto"], n_layers_cell5=ckpt["n_layers"], 
                        n_classes_cell5=ckpt["n_classes"], n_super_cell5=ckpt["n_super"], 
                        temperature_cell5=0.1, dropout_cell5=ckpt["dropout"])
    except TypeError:
        encoder = SPGCN(ckpt["d_features"], ckpt["d_hidden"], ckpt["d_proto"], 
                        ckpt["n_layers"], ckpt["n_classes"], ckpt["n_super"], 0.1, ckpt["dropout"])
                    
    encoder.load_state_dict(ckpt["encoder_state"])
    encoder.eval()
    print(f"[ok] Loaded HEB Encoder (d_features={ckpt['d_features']})")

    # 2. Target Connectomes
    targets = {
        "manc": "1.2.1", "maol": "1.1", "mcns": "1.0"
    }
    probe_dir = Path("./cross_connectome_probe_cell19d")
    out_dir = Path("./cross_connectome_data_cell19f")

    results = []

    for name, ver in targets.items():
        print(f"\n--- Target: {name.upper()} v{ver} ---")
    
        # Locate files (downloaded by Cell 19f/19d)
        n_path = None
        c_path = None
        for d in [probe_dir / name, out_dir / name]:
            if (d / "neurons.csv.gz").exists(): n_path = d / "neurons.csv.gz"
            if (d / "connections_princeton.csv.gz").exists(): c_path = d / "connections_princeton.csv.gz"
        
        if not n_path or not c_path:
            print(f"[skip] Missing files for {name}")
            continue
        
        # Load Neurons
        df_n = pd.read_csv(n_path, low_memory=False)
        df_n.columns = [str(c).strip() for c in df_n.columns]
        df_n["Root ID"] = pd.to_numeric(df_n["Root ID"], errors="coerce")
        df_n = df_n.dropna(subset=["Root ID"]).drop_duplicates("Root ID")
        ids = df_n["Root ID"].astype(np.int64).values
        N = len(ids)
        id_map = {r: i for i, r in enumerate(ids)}
    
        # Load Connections
        df_c = pd.read_csv(c_path, low_memory=False)
        df_c.columns = [str(c).strip() for c in df_c.columns]
        df_c["pre_root_id"] = pd.to_numeric(df_c["pre_root_id"], errors="coerce")
        df_c["post_root_id"] = pd.to_numeric(df_c["post_root_id"], errors="coerce")
        df_c = df_c.dropna(subset=["pre_root_id", "post_root_id"])
    
        # Map edges
        src = df_c["pre_root_id"].map(id_map).dropna().astype(np.int64).values
        dst = df_c["post_root_id"].map(id_map).dropna().astype(np.int64).values
    
        if len(src) == 0:
            print(f"[skip] No mapped edges for {name}")
            continue

        # Build Adjacencies (Unsigned)
        row_u = np.concatenate([src, dst])
        col_u = np.concatenate([dst, src])
        A_sym = csr_matrix((np.ones(len(row_u)), (row_u, col_u)), shape=(N, N))
    
        # Features: 16 Unsigned SVD + Bio/Degree
        X = np.zeros((N, 61), dtype=np.float32)
    
        # Bio/Degree (Indices 0-10)
        if "Cable length (nm)" in df_n.columns:
            X[:, 0] = np.log1p(pd.to_numeric(df_n["Cable length (nm)"], errors="coerce").fillna(0).values)
        if "Surface area (nm^2)" in df_n.columns:
            X[:, 1] = np.log1p(pd.to_numeric(df_n["Surface area (nm^2)"], errors="coerce").fillna(0).values)
        if "Volume (nm^3)" in df_n.columns:
            X[:, 2] = np.log1p(pd.to_numeric(df_n["Volume (nm^3)"], errors="coerce").fillna(0).values)
        
        # Degree
        deg_in = np.zeros(N); deg_out = np.zeros(N)
        np.add.at(deg_in, dst, 1); np.add.at(deg_out, src, 1)
        X[:, 9] = np.log1p(deg_in); X[:, 10] = np.log1p(deg_out)
    
        # Spectral
        if N > 50:
            try:
                X[:, 11:27] = TruncatedSVD(n_components=16, random_state=42).fit_transform(A_sym)
            except Exception as e:
                print(f"[warn] SVD failed: {e}")
            
        # Standardize with FAFB stats
        X = (X - feat_mean) / (feat_std + 1e-8)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
        # Forward Pass
        X_t = torch.tensor(X, dtype=torch.float32)
        ei = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
        ew = torch.ones(len(src), dtype=torch.float32)
        dis = torch.ones(N, dtype=torch.float32)
    
        with torch.no_grad():
            try:
                h = encoder.encode(X_t, ei, ew, dis, ei, ew, dis)
            except TypeError:
                h = encoder(X_t, ei, ew, dis, ei, ew, dis)
                if isinstance(h, tuple): h = h[0]
            
        emb = h.numpy()
    
        # --- PATCHED CLUSTERING & LABEL EXTRACTION ---
        label_col = None
        for cand in ["Super Class", "Class", "Sub Class", "Primary Cell Type"]:
            if cand in df_n.columns:
                label_col = cand
                break
            
        if label_col is None:
            print(f"[skip] No suitable label column found for {name}")
            continue

        # Robust string extraction (fixes the numpy sort crash)
        labels_series = df_n[label_col].fillna("__MISSING__").astype(str)
        labels_str = labels_series.to_numpy(dtype=str) # force strict string dtype
    
        invalid_strings = {"__MISSING__", "nan", "None", "", "NaN", "NULL", "null", "unknown", "?"}
        valid_mask = np.array([s not in invalid_strings for s in labels_str], dtype=bool)
    
        if valid_mask.sum() < 50:
            print(f"[skip] Too few valid labels ({valid_mask.sum()}) for {name}")
            continue
        
        unique_labels = np.unique(labels_str[valid_mask])
        K = len(unique_labels)
    
        if K < 2 or K > 200:
            print(f"[skip] K={K} is out of reasonable bounds for {name}")
            continue
    
        kmeans = KMeans(n_clusters=K, random_state=0, n_init=5).fit(emb)
        pred = kmeans.labels_
    
        # Map strings to ints
        l_map = {l: i for i, l in enumerate(unique_labels)}
        y_true = np.array([l_map[l] for l in labels_str[valid_mask]])
        y_pred = pred[valid_mask]
    
        ari = adjusted_rand_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
    
        # Null
        rng = np.random.default_rng(0)
        null_aris = [adjusted_rand_score(rng.permutation(y_true), y_pred) for _ in range(50)]
        p_ari = (sum(np.array(null_aris) >= ari) + 1) / 51
    
        print(f"[ok] N={N:,} | K={K} | ARI={ari:.4f} (null={np.mean(null_aris):.4f}, p={p_ari:.3f}) | NMI={nmi:.4f}")
        results.append({"dataset": name, "N": N, "K": K, "ARI": ari, "NMI": nmi, "p_ari": p_ari})

    print("\n" + "=" * 80)
    print("CROSS-CONNECTOME HEB SUMMARY")
    print("=" * 80)
    for r in results:
        print(f"{r['dataset'].upper():>6s} | ARI: {r['ARI']:.4f} (p={r['p_ari']:.3f}) | NMI: {r['NMI']:.4f}")
    
    return results

cell31_cross_connectome_heb_results = cell31_heb_cross_connectome()